In [88]:
### Making preliminary functions and dictionaries and loading packages
cell1_start_time = time(); start_load_pkgs = time()
    using Pkg;  
#    Pkg.add("Plots"); using Plots;  
    Pkg.add("JSON3"); using JSON3;
    Pkg.add("Dates"); using Dates;
    Pkg.add("JLD2"); using JLD2
    Pkg.add("HypothesisTests"); using HypothesisTests
    Pkg.add("DataFrames"); using DataFrames
    Pkg.add("CSV"); using CSV
    Pkg.add("FASTX"); using FASTX
    Pkg.add("Combinatorics"); using Combinatorics
    Pkg.add("StatsBase"); using StatsBase
    using Statistics
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
println(pwd()); cd("/Users/ryhisner"); println(pwd())
#    Pkg.add("LaTeXStrings");  using LaTeXStrings
#    Pkg.add("PyPlot"); using PyPlot
#    Pkg.add("PlotlyJS"); using PlotlyJS
#    Pkg.add("PGFPlotsX"); using PGFPlotsX
#    Pkg.add("UnicodePlots"); using UnicodePlots
#    Pkg.add("InspectDR"); using InspectDR
#    Pkg.add("GLMakie"); using GLMakie
#    Pkg.add("CairoMakie"); using CairoMakie
#    Pkg.add("WGLMakie"); using WGLMakie
#    Pkg.add("GMT"); using GMT
#####################################################################################################################################
### Turns time in seconds to hours, minutes, & seconds
function seconds_to_hrs_min_sec(t)
    hours = 0
    minutes = 0
    seconds = 0
    hours = t÷3600
    minutes = (t%3600)÷60
    if t > 0.0001
        seconds = (t%3600)%60
    end
    hours_int = Int(hours)
    minutes_int = Int(minutes)
    minutes_str = split(string(minutes_int), ".")[1]
    hours_fin = split(string(hours_int), ".")[1]
    minutes_fin = ""
    minutes_fin = lpad(minutes_str, 2, "0")
    seconds_rd = round(digits=2, seconds)
    seconds_1 = string(split(string(seconds_rd), ".")[1])
    seconds_2 = string(split(string(seconds_rd), ".")[2])
    seconds_left = lpad(seconds_1, 2, "0")
    seconds_right = rpad(seconds_2, 2, "0")
    seconds_fin = "$(seconds_left).$(seconds_right)"
    final_time = "$hours_int:$minutes_fin:$seconds_fin"
    final_time2 = "$hours_int hr, $minutes_int min, $seconds_fin sec"
    return final_time, final_time2
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
######################################################################################################################################
### Adds zero to truncated digit so all numbers have same # of digits & line up nicely E.g. if number = 9.1 & digits_rd = 3, it returns 9.100
function add_zeros_to_rounded(number::Float64, digits_rd::Int)
    if number ≥ 1
        num_final = 0.0
        num_whole = number*10^digits_rd
        num_whole_str = split(string(num_whole), ".")[1]
        len = length(num_whole_str)
        if len ≤ 2
            num_final = "."*"0"^(digits_rd-len)*num_whole_str
        else
            num_final = num_whole_str[1:end-digits_rd]*"."*num_whole_str[end-digits_rd+1:end]
        end
    return num_final
    end
end
#####################################################################################################################################
## Makes all digits go to 2 decimal places so they line up nicely
function number_to_string_to_two_decimal_places(number::Float64, decimals::Int)  
    num_str = string(number)
    before_dec = split(num_str, ".")[1]
    before_dec_str = string(before_dec)
    after_dec = split(num_str, ".")[2]
    after_dec_str = ""
    if length(after_dec) ≥ decimals
        after_dec_str = after_dec[1:decimals]
    else
        after_dec_len = length(after_dec)
        zero_array = Vector{String}()
        missing_zeros_to_fill = decimals - after_dec_len
        for i in 1:missing_zeros_to_fill
            push!(zero_array, "0")
        end
        zeros_str = join(zero_array)
        after_dec_str = after_dec[1:after_dec_len]*zeros_str
    end
    final_num_string = before_dec_str*"."*after_dec_str
    return final_num_string
end
#####################################################################################################################################
#####################################################################################################################################
load_pkgs_runtime = time() - start_load_pkgs
load_pkgs_hms1, load_pkgs_hms2 = seconds_to_hrs_min_sec(load_pkgs_runtime)
println("Time to Load Packages = ", load_pkgs_hms1); println("Time to Load Packages = ", load_pkgs_hms2)
#####################################################################################################################################
hydrophobic_index_dict = Dict{String, Int}("L"=>97, "I"=>99, "F"=>100, "W"=>97, "V"=>76, "M"=>74, "C"=>49, "Y"=>63, "A"=>41, "T"=>13, "E"=>-31, "H"=>8, "G"=>0, "S"=>-5, "Q"=>-10, "D"=>-55, "R"=>-14, "K"=>-23, "N"=>-28, "P"=>5, "*"=>0)
#####################################################################################################################################
wuhan_ref_seq = "..."
ref_seq = wuhan_ref_seq
#####################################################################################################################################
######################################################################################################################################
clade_set_complete = Set(["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"])
clade_arr_complete = ["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"]
clade_to_pango = Dict("recombinant"=>"recombinant", "19A"=>"B", "19B"=>"A", "20A"=>"B.1", "20B"=>"B.1.1", "20C"=>"B.1", "20D"=>"B.1.1.1", "20E"=>"B.1.177", "20F"=>"D.2", "20G"=>"B.1.2", "20H"=>"B.1.351", "20I"=>"B.1.1.7", "20J"=>"P.1", "21A"=>"B.1.617.2", "21B"=>"B.1.617.1", "21C"=>"B.1.427", "21D"=>"B.1.525", "21E"=>"P.3", "21F"=>"B.1.526", "21G"=>"C.37", "21H"=>"B.1.621", "21I"=>"B.1.617.2", "21J"=>"B.1.617.2", "21K"=>"BA.1", "21L"=>"BA.2", "21M"=>"BA.3", "22A"=>"BA.4", "22B"=>"BA.5", "22C"=>"BA.2.12.1", "22D"=>"BA.2.75", "22E"=>"BQ.1", "22F"=>"XBB", "23A"=>"XBB.1.5", "23B"=>"XBB.1.16", "23C"=>"CH.1.1", "23D"=>"XBB.1.9", "23E"=>"XBB.2.3", "23F"=>"EG.5.1", "23G"=>"XBB.1.5.70", "23H"=>"HK.3", "23I"=>"BA.2.86", "24A"=>"JN.1", "24B"=>"JN.1.11.1", "24C"=>"KP.3", "24D"=>"XDV.1", "24E"=>"KP.3.1.1", "24F"=>"XEC", "24G"=>"KP.2.3", "24H"=>"LF.7", "24I"=>"MV.1", "25A"=>"LP.8.1", "25B"=>"NB.1.8.1", "25C"=>"XFG", "25D"=>"MC.10.2.1", "25E"=>"PY.1", "25F"=>"NW.1.2", "25G"=>"XFC", "25H"=>"XFJ", "25I"=>"BA.3.2")
######################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
mut_gene(n) = split(string(n), ":")[1]
######################################################################################################################################
######################################################################################################################################
AAsub_gene(n) = aa_gene_comprehensive_dict[n]
AAsub_gene_num(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
AAsub_gene_num_pos_only(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################
AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
AA_gene_pos_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_gene_pos_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
######################################################################################################################################
function AA_order_key(mut)
    gene = aa_gene_comprehensive_dict[mut]
    AApos = aa_pos_comprehensive_dict[mut]
    gene_pos = gene_print_order[gene]
    return (gene_pos, AApos)
end
#####################################################################################################################################
function pango_variant_sort_key(pango::String)
    dotparts = split(pango, ".")
    k1 = string(dotparts[1])
    k2 = 0
    k3 = 0
    k4 = 0
    if length(dotparts) ≥ 2
        k2 = parse(Int, string(dotparts[2]))
    end
    if length(dotparts) ≥ 3
        k3 = parse(Int, string(dotparts[3]))
    end
    if length(dotparts) ≥ 4
        k4 = parse(Int, string(dotparts[4]))
    end
    return (k1, k2, k3, k4)
end
######################################################################################################################################
gene_print_order = Dict{String, Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
function pango_minus_X_fx(pango::String, minus::Int)
    unaliased = pango_to_pango_unaliased_v2[pango]
    dot_ct = count(".", unaliased)
    println(dot_ct)
    if dot_ct ≥ minus
        dotsplits = split(unaliased, ".")
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(pango), $(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X_pango
    else
        return pango
    end
end
####################################################################################
function pango_unaliased_minus_X_fx(unaliased::String, minus::Int)
    dot_ct = count(".", unaliased)
    if dot_ct ≥ minus 
        dotsplits = split(unaliased, ".")
        println(dotsplits)
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X
    else
        return minus_X_pango
    end
end
######################################################################################################################################
######################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
##########################################################################################################################################################################
##########################################################################################################################################################################
wuhan_ref_seq = "..."
refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_AA_dict = Dict{String, String}()
gene_AA_dict["ORF1a"] = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV*"
gene_AA_dict["ORF1b"] = "RVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
gene_AA_dict["S"] = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
gene_AA_dict["E"] = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
gene_AA_dict["M"] = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
gene_AA_dict["N"] = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
gene_AA_dict["ORF3a"] = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
gene_AA_dict["ORF6"] = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
gene_AA_dict["ORF7a"] = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
gene_AA_dict["ORF7b"] = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
gene_AA_dict["ORF8"] = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
gene_AA_dict["ORF9b"] = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
NSP_AA_size = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
NSP_ranges = Dict{String, String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98))
NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401))
NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696))
NSP1a_add = Dict{Int,Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
NSP1b_add = Dict{Int,Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
NSP1ab_add = Dict{Int,Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
######################################################################################################################################
ORF_size_dict = Dict{String, Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
aa_pos_comprehensive_dict = Dict{String, Int}()
aa_gene_comprehensive_dict = Dict{String, String}()
aa_gene_and_pos_comprehensive_dict = Dict{String, String}()
refAA_comprehensive_dict = Dict{String, String}()
qryAA_comprehensive_dict = Dict{String, String}()
global nonspike_muts = Set{String}()
for (orf, orf_len) in ORF_size_dict
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:orf_len
                orf_mut = "$(orf):$(aa1)$(i)$(aa2)"
                orf_pos_mut = "$(orf):$(i)"
                aa_pos_comprehensive_dict[orf_mut] = i
                aa_pos_comprehensive_dict[orf_pos_mut] = i
                aa_gene_comprehensive_dict[orf_mut] = orf
                aa_gene_comprehensive_dict[orf_pos_mut] = orf
                aa_gene_and_pos_comprehensive_dict[orf_mut] = orf_pos_mut
                aa_gene_and_pos_comprehensive_dict[orf_pos_mut] = orf_pos_mut
                if orf ≠ "S"
                    push!(nonspike_muts, orf_mut)
                    push!(nonspike_muts, orf_pos_mut)
                end
                refAA_comprehensive_dict[orf_mut] = aa1
                qryAA_comprehensive_dict[orf_mut] = aa2
            end
        end
    end
end
NTD_disulfide_variations = Set(["NTD:disulfide", "NTD_disulfide", "S:NTD_disulfide", "NTDdisulfide", "NTD:Disulfide"])
aa_gene_comprehensive_dict["NTD_disulfide"] = "S"
aa_pos_comprehensive_dict["NTD_disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD_disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD_disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD_disulfide"] = "NTD_disulfide"
aa_gene_comprehensive_dict["NTD:disulfide"] = "S"
aa_pos_comprehensive_dict["NTD:disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD:disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD:disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD:disulfide"] = "NTD_disulfide"
######################################################## Below: 100% comprehensive ref_nuc & qry_nuc dicts #################################################################
ref_nuc_comprehensive_dict = Dict{String, String}()
qry_nuc_comprehensive_dict = Dict{String, String}()
nuc_mut_int_comprehensive_dict = Dict{String, Int}()
nuc_mut_int_string_comprehensive_dict = Dict{String, String}()
nuc_residues1 = ["T", "C", "A", "G"]
nuc_residues2 = ["T", "C", "A", "G", "Y", "R", "K", "W", "M", "S", "-", "N"]
for nres1 in nuc_residues1
    for nres2 in nuc_residues2
        for i in 1:30000
            mut = "$(nres1)$(i)$(nres2)"
            nucmpos = i
            ref_nuc_comprehensive_dict[mut] = nres1
            qry_nuc_comprehensive_dict[mut] = nres2
            nuc_mut_int_comprehensive_dict[mut] = i
            nuc_mut_int_string_comprehensive_dict[mut] = string(i)
        end
    end
end
##########################################################################################################################################################################
##########################################################################################################################################################################
NSP_muts_pos_dict = Dict{String, Int}()
NSP_muts_gene_dict = Dict{String, String}()
NSP_ref_AA_dict = Dict{String, String}()
NSP_qry_AA_dict = Dict{String, String}()
NSP_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"])
for nsp in NSP_set
    nsp_len = NSP_AA_size[nsp]
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:nsp_len
                nspmut = "$(nsp):$(aa1)$(i)$(aa2)"
                nsppos = "$(nsp):$(i)"
                NSP_muts_gene_dict[nspmut] = nsp
                NSP_muts_pos_dict[nspmut] = i
                NSP_muts_gene_dict[nsppos] = nsp
                NSP_muts_pos_dict[nsppos] = i
                NSP_ref_AA_dict[nspmut] = aa1
                NSP_qry_AA_dict[nspmut] = aa2  
            end
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function AAmut_to_AApos(m)
    gn = split(m, ":")[1]
    mt = split(m, ":")[2]
    pos = mt[2:end-1]
    AApos = gn*":"*pos
    return AApos
end
########################################################################################################################################################################
########################################################################################################################################################################
########################################################################################################################################################################
function mixed2nuc(mix_mut)
    mut = mix_mut
    qry_nuc = qry_nuc_comprehensive_dict[mut]
    ref_nuc = ref_nuc_comprehensive_dict[mut]
    nuc_int = nuc_mut_int_comprehensive_dict[mut]
    ref_n_pos = "$(ref_nuc)$(nuc_int)"
    if length(mix_mut) ≥ 4
        if qry_nuc == "T"
            mut = mix_mut
        elseif qry_nuc == "C"
            mut = mix_mut
        elseif qry_nuc == "A"
            mut = mix_mut
        elseif qry_nuc == "G"
            mut = mix_mut
        elseif qry_nuc == "N"
            mut = mix_mut
        end   
        if ref_nuc == "T"
            if qry_nuc == "Y"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "C"
            if qry_nuc == "Y"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "A"
            if qry_nuc == "R"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "G"
            if qry_nuc == "R"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return mut
end
#####################################################################################################################################
####################### Making gene AA & nuc references for all designated variants #################################################
#####################################################################################################################################
gene_AA_pango_dict = Dict{String, Dict{String, String}}()
nuc_genome_pango_dict = Dict{String, String}()
pango_consensus_set = Set{String}()
headers1a, seqs1a = read_fasta("___pango_consensus_sequences/pango_consensus_AA_ORF1a_2025_06_25_NNL.fasta")
for i in 1:length(headers1a)
    pango = headers1a[i]
    push!(pango_consensus_set, pango)
end
for pango in pango_consensus_set
    gene_AA_pango_dict[pango] = Dict{String, String}()
end
################################################################################################
for gene in gene_array
    aa_file = "___pango_consensus_sequences/pango_consensus_AA_$(gene)_2025_06_25_NNL.fasta"
    headers, seqs = read_fasta(aa_file)
    for i in 1:length(headers)
        pango = headers[i]
        aa_seq = seqs[i]
        gene_AA_pango_dict[pango][gene] = aa_seq
    end
end
nuc_file = "___pango_consensus_sequences/pango_consensus_sequences_genome_nuc_2025_06_25_NNL.fasta"
nuc_headers, nuc_seqs = read_fasta(nuc_file)
for i in 1:length(nuc_headers)
    pango = nuc_headers[i]
    nuc_seq = nuc_seqs[i]
    if length(nuc_seq) ≥ 28000
        nuc_genome_pango_dict[pango] = nuc_seq
    end
end
seqs_in_nuc_genome_pango_dict = length(nuc_genome_pango_dict)
println("seqs_in_nuc_genome_pango_dict = $(seqs_in_nuc_genome_pango_dict)")
######################################################################################################################################
######################################################################################################################################
gene_hydrophobic_dict = Dict{String, Float64}()
for (gene, aaseq) in gene_AA_dict
    gene_hydrophobe_sum = 0
    for aa in aaseq
        hydrophobe_score = hydrophobic_index_dict[string(aa)]
        gene_hydrophobe_sum += hydrophobe_score
    end
    gene_hydrophobe_score = gene_hydrophobe_sum/length(aaseq)
    gene_hydrophobic_dict[gene] = gene_hydrophobe_score
end 
######################################################################################################################################
AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*"])
AA_res_set_noDel = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "*"])
AA_res_pairs = Set{Tuple{String, String}}()
for res1 in AA_res_set_noDel
    for res2 in AA_res_set
        push!(AA_res_pairs, (res1, res2) )
    end
end
###########################################################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
###########################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
nonleap_month_day_dict = Dict{Int,Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int,Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
###################
index_to_tuple = Dict{Int, Tuple{Int,Int, Int}}()
tuple_to_index = Dict{Tuple{Int,Int, Int}, Int}()        
###########################################################
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
############################################################################################################################################################################
############################################################################################################################################################################
function mut_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{String, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{String, Int}()
        end
    end
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function nuc_mut_pos_only_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{Int,Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{Int,Int}()
        end
    end
    date1_to_date2_ct = Dict()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function ORF1abMut_to_NSP(mut::String)
    NSPmut = ""
#    NSP_muts = Dict("NSP$(i)" => Dict{String, Int}() for i in 1:16 if i ≠ 11)
    gene = aa_gene_comprehensive_dict[mut]
    pos = aa_pos_comprehensive_dict[mut]
    refAA = ref_AA_dict[mut]
    qryAA = qry_AA_dict[mut]
    if gene == "ORF1a"
        for (NSP, range) in NSP_ranges1a
            if pos in range
                NSPpos = pos - NSP1a_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    if gene == "ORF1b"
        for (NSP, range) in NSP_ranges1b
            if pos in range
                NSPpos = pos - NSP1b_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    return NSPmut
end
#####################################################################################################################################
function NSPmut_to_ORF1ab(NSPmut::String)
    ORFmut = ""
    ORFpos = ""
    NSP_num = parse(Int, split(NSPmut, ":")[1][4:end])
    NSP_pos = NSP_muts_pos_dict[NSPmut]
    refAA = NSP_ref_AA_dict[NSPmut]
    qryAA = NSP_qry_AA_dict[NSPmut]
    if NSPnum in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
        ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSPnum in [13, 14, 15, 16]
        ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
        ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSP_num == 12
        if NSP_pos in [1:8...]
            ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
            ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
        end
        if NSP_pos in [10:932...]
            ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
            ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
        end
    end
    ORF1pos = parse(Int, ORFpos)
    return ORFmut
end
#####################################################################################################################################
function NSP_range_to_ORF1ab_range(NSP_range::String)
    NSP = split(NSP_range, ":")[1]
    if length(NSP) < 3
        return ""
    end
    if NSP[1:3] ≠ "NSP"
        return ""
    end
    NSP_int = parse(Int, NSP[4:end])
    range = split(NSP_range, ":")[2]
    NSPpos1_int = parse(Int, split(range, "-")[1])
    NSPpos2_int = parse(Int, split(range, "-")[2])
    ORF_int1 = 0
    ORF_int2 = 0
    ORF1_AorB_pos1 = ""
    ORF1_AorB_pos2 = ""
    if NSP_int in [1:10...]
        ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1a"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int in [13:16...]
        ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1b"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int == 12
        if NSPpos1_int in [1:9...]
            ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos1 = "1a"
        else
            ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
            ORF1_AorB_pos1 = "ORF1b"
        end
        if NSPpos2_int in [1:9...]
            ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos2 = "1a"
        end
        if !(NSPpos2_int in [1:9...])
            ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
            if ORF1_AorB_pos1 == "1a"
                ORF1_AorB_pos2 = "1b:"
            else
            ORF1_AorB_pos2 = ""
            end
        end
    end
    ORF_int1_str = string(ORF_int1)
    ORF_int2_str = string(ORF_int2)
    ORF_range = ORF1_AorB_pos1*":"*ORF_int1_str*"-"*ORF1_AorB_pos2*ORF_int2_str
    return ORF_range
end
#####################################################################################################################################
function ORF1ab_range_to_NSP_range(ab_range::String)
    ab = split(ab_range, ":")[1]
    range = split(ab_range, ":")[2]
    pos1 = split(range, "-")[1]
    pos2 = split(range, "-")[2]
    pos1_int = parse(Int, pos1)
    pos2_int = parse(Int, pos2)
    NSPint1 = 0
    NSPint2 = 0
    NSPpos1 = ""
    NSPpos2 = ""
    NSPrange = ""
    NSPrange_pt1 = ""
    top_ct = 0
    if ab == "ORF1a"
        ct = 0
        for (NSP, rng) in NSP_ranges1a
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPint2 = pos2_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1a_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end
    if ab == "ORF1b"
        ct = 0
        for (NSP, rng) in NSP_ranges1b
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPint2 = pos2_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1b_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end 
    return NSPrange
end
######################################################################################################################################
function multiepi_to_epis(multi)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
######################################################################################################################################
function stringlist_to_strings(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    arr_of_strings1 = Vector{String}()
    arr_of_strings2 = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(arr_of_strings2, mseq)
            end
        else 
            push!(arr_of_strings2, seq)
        end
    end
    sort_arr_of_strings2 = sort(collect(arr_of_strings2), by = x -> epi_sortkey(x))    
    return sort_arr_of_strings2
end
######################################################################################################################################
function stringlist_to_strings_set(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end
    return set_of_strings
end  
######################################################################################################################################
function stringlist_to_set(txt::String)
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end   
    return set_of_strings
end
#######################################################################################################################################
function list_to_strings(list::String)
    string_vec = string.(split(list, ", "))
    return string_vec
end
##################################################
function list_to_strings_set(list::String)
    string_vec = string.(split(list, ", "))
    string_set = Set{String}()
    for str in string_vec
        push!(string_set, str)
    end
    return string_set
end
#####################################################################################################################################
function cross_check_set(old_seq_set::Set{String}, new_seq_set::Set{String})
    new_set2 = Set{String}()
    old_set2 = Set{String}()
    old_seq_set_len = length(old_seq_set)
    new_seq_set_len = length(new_seq_set)
    difference = new_seq_set_len - old_seq_set_len
    println("Number of Sequences in Old List = $(old_seq_set_len)")
    println("Number of Sequences in New List = $(new_seq_set_len)")
    println("Difference between Old & New Sets = $(difference)")
    println()
    for seq in new_seq_set
        if !(seq in old_seq_set)
            push!(new_set2, seq)
        end
        if seq in old_seq_set
            push!(old_set2, seq)
        end
    end
    new_len = length(new_set2)
    old_len = length(old_set2)
    if difference == new_len
        println("The numbers check out: difference between old & new sets = Number of new seqs")
    else
        println("The numbers don't check out: difference between old & new sets not equal to Number of new seqs")
    end
    println()
    println("Number of New Sequences = $(new_len)")
    println("Number of Old Sequences = $(old_len)")
    new_set2_sort = sort(collect(new_set2), by = x -> (length(x), x))
    return new_set2, new_set2_sort
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
cell1_runtime = round(digits=1, time() - cell1_start_time)
c1runtime1, c1runtime2 = seconds_to_hrs_min_sec(cell1_runtime)
println("Cell1 Runtime v0 = $(cell1_runtime) seconds")
println("Cell1 Runtime v1 = $(c1runtime1)")
println("Cell1 Runtime v2 = $(c1runtime2)"); println()
######################################################################################################################################

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.to

2026_03_07__557PM
/Users/ryhisner
/Users/ryhisner
Time to Load Packages = 0:00:19.87
Time to Load Packages = 0 hr, 0 min, 19.87 sec
seqs_in_nuc_genome_pango_dict = 3586
Cell1 Runtime v0 = 58.9 seconds
Cell1 Runtime v1 = 0:00:58.90
Cell1 Runtime v2 = 0 hr, 0 min, 58.90 sec



In [89]:
### Fx: load_all_seq_dicts, 2026_03_08  |  Loading HQCS Datasets  | many large dictionaries not needed & therefore commented out  
load_all_start = time();   date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function load_all_seq_dicts(date::String, fx_name::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int)
    start_all = time()
    filename = "$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)"
############################################################################################################################################################################    
    global seq_ct_by_year_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
############################################################################################################################################################################
    global avg_private_AA_per_circ_seq = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq.jld2", "avg_private_AA_per_circ_seq")
    global all_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", "all_seq_ct")
    global qualifying_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", "qualifying_seq_ct")
    global nuc_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
    global AA_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct.jld2", "AA_muts_ct")
    global AA_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
    global AA_muts_ct_pos_only_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_muts_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global gene_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density.jld2", "domain_mut_density")
    global nuc_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort.jld2", "nuc_muts_ct_sort")
    global nuc_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global nuc_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort.jld2", "nuc_muts_ct_no_dels_sort")
    global nuc_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort.jld2", "AA_muts_ct_sort")
    global AA_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_pos_only_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort.jld2", "AA_muts_ct_pos_only_sort")
    global AA_muts_ct_pos_only_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort.jld2", "AA_muts_ct_no_dels_sort")
    global AA_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort.jld2", "AA_muts_ct_pos_only_no_dels_sort")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")
    global gene_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global domain_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global domain_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global AA_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq")
    global nuc_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_dels_ct.jld2", "nuc_dels_ct")
############################################################################################################################################################################
    global AA_no_dels_sub_count_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_no_dels_sub_count.jld2", "AA_no_dels_sub_count")
    global total_AA_subs_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_total_AA_subs.jld2", "total_AA_subs")
#    global date_nuc_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
#    global date_nuc_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
#    global date_AA_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct")
#    global date_AA_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
#    global date_AA_mut_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
#    global seq_date_index_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
    global clade_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct")
############################################################################################################################################################################
######################################################## Below: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
## Changed mind about pango_date_index_ct--better to have high qc filters in place for those to weed out shitty misassigned seqs
#    global pango_date_index_ct = load("2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut7_maxRevs2_qcMax10_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global pango_date_index_ct = load("2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_pango_all = load("dictionaries/2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_seq_pango.jld2", "seq_pango")



    
#####################################################################################################################################
    global country_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_country_set.jld2", "country_set")
    global clade_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_clade_set.jld2", "clade_set")
    global pango_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_pango_set.jld2", "pango_set")
#    global seq_lab_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################
    
    
    
    
    #    date_temp = "2025-08-03"; max_AA_mut_temp = 5; revs_thresh_temp = 1; qc_max_temp = 5; ndjson_name_temp = "EPI_ISL_400000_20080000"
#    global country_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_set.jld2", "country_set")
#    global clade_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_set.jld2", "clade_set")
#    global pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_set.jld2", "pango_set")
#    global pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_set.jld2", "pango_unaliased_set")
#    global pango_to_pango_unaliased = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_to_pango_unaliased.jld2", "pango_to_pango_unaliased")
#    global clade_pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_set.jld2", "clade_pango_set")
#    global clade_pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_unaliased_set.jld2", "clade_pango_unaliased_set")
#####################################################################################################################################
#    global seq_country_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_country.jld2", "seq_country")
#    global seq_US_state_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_US_state.jld2", "seq_US_state")
#    global seq_clade_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_clade.jld2", "seq_clade")
############ Currently using low-qc seq_pango_all as it's more comprehensive ###########
#    global seq_pango_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango.jld2", "seq_pango")
#    global seq_pango_unaliased_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango_unaliased.jld2", "seq_pango_unaliased")
############ Currently using low-qc seq_date_index_all as it's more comprehensive ########################
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_collection_date_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_collection_date.jld2", "seq_collection_date")
#    global seq_lab_dict_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_dict.jld2", "seq_lab_dict")
#    global seq_nuc_del_ranges_ct_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_nuc_del_ranges_ct.jld2", "seq_nuc_del_ranges_ct")
#    global seq_lab_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################    
#    global pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
#    global country_clade_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct")
#    global country_pango_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct")
#    global country_pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct")
############################################################################################################################################################################
######################################################## Above: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
#    global ORF9b_CTD_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq.jld2", "ORF9b_CTD_muts_seq")
    global multi_ORF9b_CTD_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD.jld2", "multi_ORF9b_CTD")
    global multi_ORF9b_CTD_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct.jld2", "multi_ORF9b_CTD_ct")
#    global qualifying_ORF9b_double_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct.jld2", "qualifying_ORF9b_double_ct")
    
#    global ORF9b_CTD_muts_seq_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_ORF9b_CTD_muts_seq_relaxed_qc_10_1_30.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_ct_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_10_1_30")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_qualifying_ORF9b_double_ct_relaxed_qc_10_1_30.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_10_1_30")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_ORF9b_CTD_muts_seq_relaxed_qc_15_1_50.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_ct_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_50_1_15")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_qualifying_ORF9b_double_ct_relaxed_qc_15_1_50.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_50_1_15")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_ORF9b_CTD_muts_seq_relaxed_qc_25_3_200.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_ct_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_200_3_25")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_qualifying_ORF9b_double_ct_relaxed_qc_25_3_200.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_200_3_25")
#    global avg_private_AA_per_circ_seq2 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq2.jld2", "avg_private_AA_per_circ_seq2")
    ##################### Below: Dicts for all seqs using higher (less restrictive) QC filters ##########################################
#    global AA_muts_seq_all_8000qc_filter = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_AA_muts_seq.jld2", "AA_muts_seq")
#    global AA_muts_seq_all_999qc_filter = load("dictionaries/2025-08-24_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs4_qcMax999_AA_muts_seq.jld2", "AA_muts_seq")
#####################################################################################################################################
    println("Dictionaries loaded!");   println()
    finish_all = time() - start_all;    finish_all_rd = round(digits=1, finish_all)
    load_hms1, load_hms2 = seconds_to_hrs_min_sec(finish_all)
    println("Total Time to Load ALL Dictionaries = $(finish_all_rd) seconds")  # println("Total Time to Load ALL Dictionaries = $(load_hms1)")
    println("Total Time to Load ALL Dictionaries = $(load_hms2)")
end
######################################################################################################################################
date = "2026_02_08"
clade_pct_date_index = load("dictionaries/$(date)__clade_pct_date_index.jld2", "clade_pct_date_index")
clade_pct_date_tuple = load("dictionaries/$(date)__clade_pct_date_tuple.jld2", "clade_pct_date_tuple")
clade_pct_date_string = load("dictionaries/$(date)__clade_pct_date_string.jld2", "clade_pct_date_string")
pango_pct_date_index = load("dictionaries/$(date)__pango_pct_date_index.jld2", "pango_pct_date_index")
pango_pct_date_tuple = load("dictionaries/$(date)__pango_pct_date_tuple.jld2", "pango_pct_date_tuple")
pango_pct_date_string = load("dictionaries/$(date)__pango_pct_date_string.jld2", "pango_pct_date_string")
pango_unaliased_pct_date_index = load("dictionaries/$(date)__pango_unaliased_pct_date_index.jld2", "pango_unaliased_pct_date_index")
pango_unaliased_pct_date_tuple = load("dictionaries/$(date)__pango_unaliased_pct_date_tuple.jld2", "pango_unaliased_pct_date_tuple")
pango_unaliased_pct_date_string = load("dictionaries/$(date)__pango_unaliased_pct_date_string.jld2", "pango_unaliased_pct_date_string")
pango_to_pango_unaliased_v2 = load("dictionaries/$(date)__pango_to_pango_unaliased_v2.jld2", "pango_to_pango_unaliased_v2")
pango_unaliased_to_pango_prefix = load("dictionaries/$(date)__pango_unaliased_to_pango_prefix.jld2", "pango_unaliased_to_pango_prefix")
pango_unaliased_to_pango = load("dictionaries/$(date)__pango_unaliased_to_pango.jld2", "pango_unaliased_to_pango")
pango_unaliased_predecessor_meta_dict = load("dictionaries/$(date)__pango_unaliased_predecessor_meta_dict.jld2", "pango_unaliased_predecessor_meta_dict")
pango_predecessor_meta_dict = load("dictionaries/$(date)__pango_predecessor_meta_dict.jld2", "pango_predecessor_meta_dict")
pango_unaliased_set = load("dictionaries/$(date)__pango_unaliased_set.jld2", "pango_unaliased_set")
pango_unaliased_date_index_ct = load("dictionaries/$(date)__pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
clade_date_index_cumul = load("dictionaries/$(date)__clade_date_index_cumul.jld2", "clade_date_index_cumul")
pango_date_index_cumul = load("dictionaries/$(date)__pango_date_index_cumul.jld2", "pango_date_index_cumul")
pango_unaliased_date_index_cumul = load("dictionaries/$(date)__pango_unaliased_date_index_cumul.jld2", "pango_unaliased_date_index_cumul")
clade_total = load("dictionaries/$(date)__clade_total.jld2", "clade_total")
pango_total = load("dictionaries/$(date)__pango_total.jld2", "pango_total")
pango_unaliased_total = load("dictionaries/$(date)__pango_unaliased_total.jld2", "pango_unaliased_total")
AAmut_date_index_cumul = load("dictionaries/$(date)__AAmut_date_index_cumul.jld2", "AAmut_date_index_cumul")
nucmut_date_index_cumul = load("dictionaries/$(date)__nucmut_date_index_cumul.jld2", "nucmut_date_index_cumul")
pango_nuc_sub_WT = load("dictionaries/$(date)__pango_nuc_sub_WT.jld2", "pango_nuc_sub_WT")
pango_nuc_del_WT = load("dictionaries/$(date)__pango_nuc_del_WT.jld2", "pango_nuc_del_WT")
pango_nuc_sub_private = load("dictionaries/$(date)__pango_nuc_sub_private.jld2", "pango_nuc_sub_private")
pango_nuc_del_private = load("dictionaries/$(date)__pango_nuc_del_private.jld2", "pango_nuc_del_private")
pango_nuc_sub_revs = load("dictionaries/$(date)__pango_nuc_sub_revs.jld2", "pango_nuc_sub_revs")
pango_nuc_del_revs = load("dictionaries/$(date)__pango_nuc_del_revs.jld2", "pango_nuc_del_revs")
pango_AAsub_WT = load("dictionaries/$(date)__pango_AAsub_WT.jld2", "pango_AAsub_WT")
pango_AAsub_WT_pos_only = load("dictionaries/$(date)__pango_AAsub_WT_pos_only.jld2", "pango_AAsub_WT_pos_only")
pango_AAdel_WT = load("dictionaries/$(date)__pango_AAdel_WT.jld2", "pango_AAdel_WT")
pango_AAsub_private = load("dictionaries/$(date)__pango_AAsub_private.jld2", "pango_AAsub_private")
pango_AAdel_private = load("dictionaries/$(date)__pango_AAdel_private.jld2", "pango_AAdel_private")
pango_AAsub_revs = load("dictionaries/$(date)__pango_AAsub_revs.jld2", "pango_AAsub_revs")
pango_AAdel_revs = load("dictionaries/$(date)__pango_AAdel_revs.jld2", "pango_AAdel_revs")
pango_frameshifts_WT = load("dictionaries/$(date)__pango_frameshifts_WT.jld2", "pango_frameshifts_WT")
pango_designation_date = load("dictionaries/$(date)__pango_designation_date.jld2", "pango_designation_date")
clade_pango_set = load("dictionaries/$(date)__clade_pango_set.jld2", "clade_pango_set")
delete!(pango_AAsub_WT["B.55"], "")
delete!(pango_AAsub_WT["B"], "")
println("Done!")
load_all_fx_runtime = time() - load_all_start; load_all_fx_runtime_rd = round(digits=3, load_all_fx_runtime)
load_all_fx_hms1, load_all_fx_hms2 = seconds_to_hrs_min_sec(load_all_fx_runtime)
println("Total Time to Run load_all Fx = $(load_all_fx_runtime_rd) seconds")  # println("Total Time to Run load_all Fx = $(load_all_fx_hms1)") 
println("Total Time to Run load_all Fx = $(load_all_fx_hms2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_07__558PM
Done!
Total Time to Run load_all Fx = 8.431 seconds
Total Time to Run load_all Fx = 0 hr, 0 min, 08.43 sec


In [90]:
### Execute Load All Dicts Fx, EPI_ISL_400000_20300000 | 2026_02_08 | QC--5-1-5 | Runtime = 38 sec #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_load_all_dicts = time()
#                     date          fx_name            ndjson_name     max_AA_mut,revs_thresh,qc_max
HQCS_date = "2026_01_06"
HQCS_fx_name = "all_private_muts"
HQCS_ndjson_name = "EPI_ISL_400001_20300000"
HQCS_max_AA_mut = 5
HQCS_revs_thresh = 1
HQCS_qc_max = 5
HQCS_qc_string = "$(HQCS_max_AA_mut)_$(HQCS_revs_thresh)_$(HQCS_qc_max)"
load_all_seq_dicts(HQCS_date, HQCS_fx_name, HQCS_ndjson_name, HQCS_max_AA_mut, HQCS_revs_thresh, HQCS_qc_max)

gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

finish_load_all_dicts = time() - start_load_all_dicts
finish_load_all_dicts_rd = round(digits=1, finish_load_all_dicts)
println("Total Time to Load ALL Dictionaries = $(finish_load_all_dicts_rd)")
####################################################################################################################################
#################################### Create  date_index_seq_ct_all  Dictionary #####################################################
############################# Only needed for special calculations, so commented out here ##########################################
####################################################################################################################################
#date_index_seq_ct_all = Dict{Int,Int}()
#for d_index in values(seq_date_index_all)
#    date_index_seq_ct_all[d_index] = get(date_index_seq_ct_all, d_index, 0) + 1
#end
function date_index_range_seq_ct(date1::Int, date2::Int)
    date1_date2_seq_ct = 0
    for d_index in date1:date2
        date1_date2_seq_ct += date_index_seq_ct_all[d_index]
    end
    return date1_date2_seq_ct
end   
###################################################################################################################################
############################### Create  nuc_muts_ct_pos_only_no_dels_all  Dictionary ##############################################
###################################################################################################################################
nuc_sort_key(m) = parse(Int, m[2:end-1])
dict_make_start = time()
nuc_muts_ct_pos_only_no_dels_all = Dict{Int,Int}()
for i in 1:30000
    nuc_muts_ct_pos_only_no_dels_all[i] = 0
end
for (nuc_mut, count) in nuc_muts_ct_no_dels_all
    pos = nuc_sort_key(nuc_mut)
    nuc_muts_ct_pos_only_no_dels_all[pos] += count
end
#############################################__Create   pango_index_date_set    ###################################################
pango_index_date_set = Set{String}()
for pango in keys(pango_date_index_ct)
    push!(pango_index_date_set, pango)
end
println(length(pango_index_date_set))
####################################################################################################################################
#function pango_prefix_to_unaliased_fx(fx_pango::String)
#    prefix_unaliased = ""
#    dotpts_ct = count('.', fx_pango)
#    if dotpts_ct ≥ 1
#        dotsplits = split(fx_pango, ".")
#        prefix = string(dotsplits[1])
#        prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
#        last_pts = join(dotsplits[2:end], ".")
#        unaliased = "$(prefix_unaliased).$(last_pts)"
#        return unaliased, prefix_unaliased
#    else
#        return fx_pango, prefix_unaliased
#    end
#end
########################################################################################################################
#for pango in pango_set
#    unali, new_prfx = pango_prefix_to_unaliased_fx(pango)
#    pango_to_pango_unaliased_v2[pango] = unali
#end
#################################################################################################################################
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
dict_make_runtime = round(digits=2, time() - dict_make_start)
println("Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = $(dict_make_runtime) seconds"); print("\n"^1)
#################################################################################################################################
#################################################################################################################################

2026_03_07__558PM
Dictionaries loaded!

Total Time to Load ALL Dictionaries = 39.9 seconds
Total Time to Load ALL Dictionaries = 0 hr, 0 min, 39.91 sec
Total Time to Load ALL Dictionaries = 40.0
4297
length(keys(pango_to_pango_unaliased_v2) = 5657
Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = 0.2 seconds



In [105]:
### Fx: chronic_load_dicts2_DQ, 2026_03_01 version |  Loading EPCI datasets  |###############################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_chr_load_fx = time()
function chronic_load_dicts2_DQ(ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Int, abs_min_mut_thresh::Int)
    start_date = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(start_date)
######################################################################################################################################
#    global cumulative_AA_mut_ct_by_date_index
#        cumulative_AA_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_AA_mut_ct_by_date_index.jld2", "cumulative_AA_mut_ct_by_date_index")
#    global cumulative_nuc_mut_ct_by_date_index
#        cumulative_nuc_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_nuc_mut_ct_by_date_index.jld2", "cumulative_nuc_mut_ct_by_date_index")
######################################################################################################################################
    global seq_AA_insertions_WT
        seq_AA_insertions_WT = load("$(folder_name)/$(folder_name)__seq_AA_insertions_WT.jld2", "seq_AA_insertions_WT")
    global seq_nuc_insertions_WT
        seq_nuc_insertions_WT = load("$(folder_name)/$(folder_name)__seq_nuc_insertions_WT.jld2", "seq_nuc_insertions_WT")    
######################################################################################################################################
    global total_chr_AA_subs
        total_chr_AA_subs = load("$(folder_name)/$(folder_name)__total_chr_AA_subs.jld2", "total_chr_AA_subs")
######################################################################################################################################
    global total_nuc_revs_seq
        total_nuc_revs_seq = load("$(folder_name)/$(folder_name)__total_nuc_revs_seq.jld2", "total_nuc_revs_seq")
    global seq_nuc_total_revs
        seq_nuc_total_revs = load("$(folder_name)/$(folder_name)__seq_nuc_total_revs.jld2", "seq_nuc_total_revs")
    global total_AA_revs_seq
        total_AA_revs_seq = load("$(folder_name)/$(folder_name)__total_AA_revs_seq.jld2", "total_AA_revs_seq")
    global seq_AA_total_revs
        seq_AA_total_revs = load("$(folder_name)/$(folder_name)__seq_AA_total_revs.jld2", "seq_AA_total_revs")
    global seq_AA_revs
        seq_AA_revs = load("$(folder_name)/$(folder_name)__seq_AA_revs.jld2", "seq_AA_revs")
######################################################################################################################################
    global AA_muts_ct_chr_all_ratio
        AA_muts_ct_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio.jld2", "AA_muts_ct_chr_all_ratio")
    global AA_muts_ct_no_dels_chr_all_ratio
        AA_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio.jld2", "AA_muts_ct_no_dels_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio")
    global AA_muts_ct_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_no_dels_no_revs_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
    global nuc_muts_ct_no_dels_chr_all_ratio
        nuc_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio")
    global nuc_muts_ct_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_no_revs_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
####################################################################################################################################
    global AA_muts_ct_pos_only_adj_score
        AA_muts_ct_pos_only_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score.jld2", "AA_muts_ct_pos_only_adj_score")
    global AA_muts_ct_adj
        AA_muts_ct_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj.jld2", "AA_muts_ct_adj")
    global AA_muts_ct_pos_only_adj_score_no_dels
        AA_muts_ct_pos_only_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels.jld2", "AA_muts_ct_pos_only_adj_score_no_dels")
    global nuc_muts_ct_adj
        nuc_muts_ct_adj = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj.jld2", "nuc_muts_ct_adj")
    global AA_muts_ct_adj_score
        AA_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score.jld2", "AA_muts_ct_adj_score")
    global AA_muts_ct_adj_score_no_dels
        AA_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels.jld2", "AA_muts_ct_adj_score_no_dels")
    global AA_muts_ct_pos_only_adj
        AA_muts_ct_pos_only_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj.jld2", "AA_muts_ct_pos_only_adj")
    global AA_muts_ct_pos_only_adj_no_dels
        AA_muts_ct_pos_only_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels.jld2", "AA_muts_ct_pos_only_adj_no_dels")
    global nuc_muts_ct_adj_score_no_dels
        nuc_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels.jld2", "nuc_muts_ct_adj_score_no_dels")
    global nuc_muts_ct_adj_score
        nuc_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score.jld2", "nuc_muts_ct_adj_score")
    global nuc_muts_ct_adj_no_dels
        nuc_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels.jld2", "nuc_muts_ct_adj_no_dels")
    global AA_muts_ct_adj_no_dels
        AA_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels.jld2", "AA_muts_ct_adj_no_dels")
####################################################################################################################################
####################################################################################################################################
    global seq_ct_by_year
        seq_ct_by_year = load("$(folder_name)/$(folder_name)__seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month
        seq_ct_by_year_month = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day
        seq_ct_by_year_month_day = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global seq_date_tuple
        seq_date_tuple = load("$(folder_name)/$(folder_name)__seq_date_tuple.jld2", "seq_date_tuple")
    global date_nuc_mut_ct_no_dels
        date_nuc_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
    global date_AA_mut_ct_no_dels
        date_AA_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
    global date_AA_mut_ct
        date_AA_mut_ct = load("$(folder_name)/$(folder_name)__date_AA_mut_ct.jld2", "date_AA_mut_ct")
    global date_AA_mut_ct_pos_only_no_dels
        date_AA_mut_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
####################################################################################################################################
    global seq_collection_date
        seq_collection_date = load("$(folder_name)/$(folder_name)__seq_collection_date.jld2", "seq_collection_date")
    global seq_date_index
        seq_date_index = load("$(folder_name)/$(folder_name)__seq_date_index.jld2", "seq_date_index")
    global date_nuc_mut_ct
        date_nuc_mut_ct = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
####################################################################################################################################
    global seq_AA_dels
        seq_AA_dels = load("$(folder_name)/$(folder_name)__seq_AA_dels.jld2", "seq_AA_dels")
    global seq_AA_muts_pos_only_no_dels
        seq_AA_muts_pos_only_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only_no_dels.jld2", "seq_AA_muts_pos_only_no_dels")
    global seq_AA_muts_WT_pos_only
        seq_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT_pos_only.jld2", "seq_AA_muts_WT_pos_only")
    global seq_AA_dels_WT
        seq_AA_dels_WT = load("$(folder_name)/$(folder_name)__seq_AA_dels_WT.jld2", "seq_AA_dels_WT")
    global seq_AA_muts_WT
        seq_AA_muts_WT = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT.jld2", "seq_AA_muts_WT")
    global seq_AA_muts
        seq_AA_muts = load("$(folder_name)/$(folder_name)__seq_AA_muts.jld2", "seq_AA_muts")
    global seq_AA_muts_no_dels
        seq_AA_muts_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_no_dels.jld2", "seq_AA_muts_no_dels")
    global seq_AA_muts_pos_only
        seq_AA_muts_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only.jld2", "seq_AA_muts_pos_only")
    global seq_mixed_AA_muts
        seq_mixed_AA_muts = load("$(folder_name)/$(folder_name)__seq_mixed_AA_muts.jld2", "seq_mixed_AA_muts")
    global seq_unknown_AA
        seq_unknown_AA = load("$(folder_name)/$(folder_name)__seq_unknown_AA.jld2", "seq_unknown_AA")
    global seq_unknown_AA_ranges
        seq_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__seq_unknown_AA_ranges.jld2", "seq_unknown_AA_ranges")
####################################################################################################################################
    global seq_nuc_muts
        seq_nuc_muts = load("$(folder_name)/$(folder_name)__seq_nuc_muts.jld2", "seq_nuc_muts")
    global seq_nuc_muts_WT
        seq_nuc_muts_WT = load("$(folder_name)/$(folder_name)__seq_nuc_muts_WT.jld2", "seq_nuc_muts_WT")
    global seq_nuc_del_ranges_WT
        seq_nuc_del_ranges_WT = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges_WT.jld2", "seq_nuc_del_ranges_WT")
    global seq_nuc_dropout
        seq_nuc_dropout = load("$(folder_name)/$(folder_name)__seq_nuc_dropout.jld2", "seq_nuc_dropout")
    global seq_nuc_del_ranges
        seq_nuc_del_ranges = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges.jld2", "seq_nuc_del_ranges")
    global seq_mixed_nucs
        seq_mixed_nucs = load("$(folder_name)/$(folder_name)__seq_mixed_nucs.jld2", "seq_mixed_nucs")
###################################################################################################################################
    global seq_clade
        seq_clade = load("$(folder_name)/$(folder_name)__seq_clade.jld2", "seq_clade")
    global seq_pango
        seq_pango = load("$(folder_name)/$(folder_name)__seq_pango.jld2", "seq_pango")
    global seq_pango_unaliased
        seq_pango_unaliased = load("$(folder_name)/$(folder_name)__seq_pango_unaliased.jld2", "seq_pango_unaliased")    
    global seq_clade_display
        seq_clade_display = load("$(folder_name)/$(folder_name)__seq_clade_display.jld2", "seq_clade_display")
    global seq_clade_ct
        seq_clade_ct = load("$(folder_name)/$(folder_name)__seq_clade_ct.jld2", "seq_clade_ct")
    global seq_pango_ct
        seq_pango_ct = load("$(folder_name)/$(folder_name)__seq_pango_ct.jld2", "seq_pango_ct")
    global seq_pango_unaliased_ct
        seq_pango_unaliased_ct = load("$(folder_name)/$(folder_name)__seq_pango_unaliased_ct.jld2", "seq_pango_unaliased_ct")
####################################################################################################################################
    global seq_US_state
        seq_US_state = load("$(folder_name)/$(folder_name)__seq_US_state.jld2", "seq_US_state")
    global seq_country
        seq_country = load("$(folder_name)/$(folder_name)__seq_country.jld2", "seq_country")
    global seq_lab_dict
        seq_lab_dict = load("$(folder_name)/$(folder_name)__seq_lab_dict.jld2", "seq_lab_dict")
####################################################################################################################################
    global gene_mut_density
        gene_mut_density = load("$(folder_name)/$(folder_name)__gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density
        domain_mut_density = load("$(folder_name)/$(folder_name)__domain_mut_density.jld2", "domain_mut_density")
####################################################################################################################################
    global nuc_muts_ct
        nuc_muts_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_dels_ct
        nuc_dels_ct = load("$(folder_name)/$(folder_name)__nuc_dels_ct.jld2", "nuc_dels_ct")
    global nuc_muts_ct_no_dels
        nuc_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
######################
    global nuc_dels_seq
        nuc_dels_seq = load("$(folder_name)/$(folder_name)__nuc_dels_seq.jld2", "nuc_dels_seq")
    global nuc_muts_seq
        nuc_muts_seq = load("$(folder_name)/$(folder_name)__nuc_muts_seq.jld2", "nuc_muts_seq")
    global nuc_dels_seq_WT
        nuc_dels_seq_WT = load("$(folder_name)/$(folder_name)__nuc_dels_seq_WT.jld2", "nuc_dels_seq_WT")
    global nuc_muts_seq_WT
        nuc_muts_seq_WT = load("$(folder_name)/$(folder_name)__nuc_muts_seq_WT.jld2", "nuc_muts_seq_WT")
##################################################################
    global AA_muts_ct
        AA_muts_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct.jld2", "AA_muts_ct")
    global AA_muts_ct_no_dels_no_revs
        AA_muts_ct_no_dels_no_revs = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs.jld2", "AA_muts_ct_no_dels_no_revs")
    global AA_muts_ct_pos_only
        AA_muts_ct_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_dels_ct
        AA_dels_ct = load("$(folder_name)/$(folder_name)__AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_pos_only_no_dels
        AA_muts_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global AA_muts_ct_no_dels
        AA_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
############################## 
    global AA_muts_seq
        AA_muts_seq = load("$(folder_name)/$(folder_name)__AA_muts_seq.jld2", "AA_muts_seq")
    global AA_muts_seq_pos_only
        AA_muts_seq_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only.jld2", "AA_muts_seq_pos_only")
    global AA_muts_seq_pos_only_no_dels
        AA_muts_seq_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only_no_dels.jld2", "AA_muts_seq_pos_only_no_dels")
    global AA_dels_seq
        AA_dels_seq = load("$(folder_name)/$(folder_name)__AA_dels_seq.jld2", "AA_dels_seq")
##############################
    global AA_muts_seq_WT
        AA_muts_seq_WT = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT.jld2", "AA_muts_seq_WT")
    global AA_muts_seq_WT_pos_only
        AA_muts_seq_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT_pos_only.jld2", "AA_muts_seq_WT_pos_only")
    global AA_dels_seq_WT
        AA_dels_seq_WT = load("$(folder_name)/$(folder_name)__AA_dels_seq_WT.jld2", "AA_dels_seq_WT")
#####################################################################################################################################
#####################################################################################################################################
    global NSP_muts
        NSP_muts = load("$(folder_name)/$(folder_name)__NSP_muts.jld2", "NSP_muts")
    global NSP_muts_no_dels
        NSP_muts_no_dels = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels.jld2", "NSP_muts_no_dels")
#####################################################################################################################################
    global non_rep_seqs_AA
        non_rep_seqs_AA = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA.jld2", "non_rep_seqs_AA")
    global non_rep_seqs_AA_pos_only
        non_rep_seqs_AA_pos_only = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only.jld2", "non_rep_seqs_AA_pos_only")
    global non_rep_seqs_AA_pos_only_no_dels
        non_rep_seqs_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only_no_dels.jld2", "non_rep_seqs_AA_pos_only_no_dels")
#####################################################################################################################################
    global rep_seq_grps_AA_muts_WT
        rep_seq_grps_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT.jld2", "rep_seq_grps_AA_muts_WT")
    global rep_seq_grps_AA_muts_WT_pos_only
        rep_seq_grps_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT_pos_only.jld2", "rep_seq_grps_AA_muts_WT_pos_only")
    global rep_seq_grps_AA_dels_WT
        rep_seq_grps_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels_WT.jld2", "rep_seq_grps_AA_dels_WT")
    global rep_seq_grps_nuc_muts_WT
        rep_seq_grps_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_muts_WT.jld2", "rep_seq_grps_nuc_muts_WT")
    global rep_seq_grps_nuc_dels_WT
        rep_seq_grps_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dels_WT.jld2", "rep_seq_grps_nuc_dels_WT")
    global nuc_muts_rep_seq_grps_WT
        nuc_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_WT.jld2", "nuc_muts_rep_seq_grps_WT")
    global nuc_dels_rep_seq_grps_WT
        nuc_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps_WT.jld2", "nuc_dels_rep_seq_grps_WT")
    global AA_dels_rep_seq_grps_WT
        AA_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps_WT.jld2", "AA_dels_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT
        AA_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT.jld2", "AA_muts_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT_pos_only
        AA_muts_rep_seq_grps_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT_pos_only.jld2", "AA_muts_rep_seq_grps_WT_pos_only")
#####################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_maxmut_pango
        rep_seq_grps_maxmut_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango.jld2", "rep_seq_grps_maxmut_pango")
    global rep_seq_grps_maxmut_clade
        rep_seq_grps_maxmut_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_clade.jld2", "rep_seq_grps_maxmut_clade")
    global rep_seq_grps_maxmut_pango_unaliased
        rep_seq_grps_maxmut_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango_unaliased.jld2", "rep_seq_grps_maxmut_pango_unaliased")
#####################################################################################################################################
    global rep_seq_grps_maxmut_dels
        rep_seq_grps_maxmut_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_dels.jld2", "rep_seq_grps_maxmut_dels")
    global rep_seq_grps_maxmut_AA_pos_only_no_dels
        rep_seq_grps_maxmut_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only_no_dels.jld2", "rep_seq_grps_maxmut_AA_pos_only_no_dels")
    global rep_seq_grps_maxmut_nuc
        rep_seq_grps_maxmut_nuc = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc.jld2", "rep_seq_grps_maxmut_nuc")
    global rep_seq_grps_maxmut_AA_pos_only
        rep_seq_grps_maxmut_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only.jld2", "rep_seq_grps_maxmut_AA_pos_only")
    global rep_seq_grps_maxmut_nuc_dropout
        rep_seq_grps_maxmut_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dropout.jld2", "rep_seq_grps_maxmut_nuc_dropout")
    global rep_seq_grps_maxmut_nuc_no_dels
        rep_seq_grps_maxmut_nuc_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_no_dels.jld2", "rep_seq_grps_maxmut_nuc_no_dels")
    global rep_seq_grps_maxmut_mixed_nucs
        rep_seq_grps_maxmut_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_nucs.jld2", "rep_seq_grps_maxmut_mixed_nucs")
    global rep_seq_grps_maxmut_AA_dels
        rep_seq_grps_maxmut_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels.jld2", "rep_seq_grps_maxmut_AA_dels")
    global rep_seq_grps_maxmut_del_ranges_ct
        rep_seq_grps_maxmut_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_del_ranges_ct.jld2", "rep_seq_grps_maxmut_del_ranges_ct")
    global rep_seq_grps_maxmut_AA
        rep_seq_grps_maxmut_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA.jld2", "rep_seq_grps_maxmut_AA")
    global rep_seq_grps_maxmut_mixed_AA_muts
        rep_seq_grps_maxmut_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_AA_muts.jld2", "rep_seq_grps_maxmut_mixed_AA_muts")
    global rep_seq_grps_maxmut_unknown_AA
        rep_seq_grps_maxmut_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA.jld2", "rep_seq_grps_maxmut_unknown_AA")
    global rep_seq_grps_maxmut_unknown_AA_ranges
        rep_seq_grps_maxmut_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA_ranges.jld2", "rep_seq_grps_maxmut_unknown_AA_ranges")
    global rep_seq_grps_maxmut_seqs
        rep_seq_grps_maxmut_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_seqs.jld2", "rep_seq_grps_maxmut_seqs")
    global rep_seq_grps_maxmut_AA_no_dels
        rep_seq_grps_maxmut_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_no_dels.jld2", "rep_seq_grps_maxmut_AA_no_dels")
    global rep_seq_grps_maxmut_nuc_muts_WT
        rep_seq_grps_maxmut_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_muts_WT.jld2", "rep_seq_grps_maxmut_nuc_muts_WT")
    global rep_seq_grps_maxmut_nuc_dels_WT
        rep_seq_grps_maxmut_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dels_WT.jld2", "rep_seq_grps_maxmut_nuc_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT
        rep_seq_grps_maxmut_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT.jld2", "rep_seq_grps_maxmut_AA_muts_WT")
    global rep_seq_grps_maxmut_AA_dels_WT
        rep_seq_grps_maxmut_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels_WT.jld2", "rep_seq_grps_maxmut_AA_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT_pos_only
        rep_seq_grps_maxmut_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT_pos_only.jld2", "rep_seq_grps_maxmut_AA_muts_WT_pos_only")
################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_seqs
        rep_seq_grps_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_seqs.jld2", "rep_seq_grps_seqs")
    global rep_seq_grps_pango
        rep_seq_grps_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango.jld2", "rep_seq_grps_pango")
    global rep_seq_grps_clade
        rep_seq_grps_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_clade.jld2", "rep_seq_grps_clade")
    global rep_seq_grps_pango_unaliased
        rep_seq_grps_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango_unaliased.jld2", "rep_seq_grps_pango_unaliased")
    global rep_seq_grps_muts
        rep_seq_grps_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts.jld2", "rep_seq_grps_muts")
    global rep_seq_grps_mixed_nucs
        rep_seq_grps_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_nucs.jld2", "rep_seq_grps_mixed_nucs")
    global rep_seq_grps_muts_no_dels
        rep_seq_grps_muts_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts_no_dels.jld2", "rep_seq_grps_muts_no_dels")
    global rep_seq_grps_AA_pos_only
        rep_seq_grps_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only.jld2", "rep_seq_grps_AA_pos_only")
    global rep_seq_grps_AA_no_dels
        rep_seq_grps_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_no_dels.jld2", "rep_seq_grps_AA_no_dels")
    global rep_seq_grps_AA_dels
        rep_seq_grps_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels.jld2", "rep_seq_grps_AA_dels")
    global rep_seq_grps_dels
        rep_seq_grps_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_dels.jld2", "rep_seq_grps_dels")
    global rep_seq_grps_AA_pos_only_no_dels
        rep_seq_grps_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only_no_dels.jld2", "rep_seq_grps_AA_pos_only_no_dels")
    global rep_seq_grps_del_ranges_ct
        rep_seq_grps_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_del_ranges_ct.jld2", "rep_seq_grps_del_ranges_ct")
    global rep_seq_grps_AA
        rep_seq_grps_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA.jld2", "rep_seq_grps_AA")
    global rep_seq_grps_nuc_dropout
        rep_seq_grps_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dropout.jld2", "rep_seq_grps_nuc_dropout")
#    global rep_seq_grps_unknown_AA
#        rep_seq_grps_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_unknown_AA.jld2", "rep_seq_grps_unknown_AA")
#    global rep_seq_grps_mixed_AA_muts
#        rep_seq_grps_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_AA_muts.jld2", "rep_seq_grps_mixed_AA_muts")
    global AA_dels_rep_seq_grps
        AA_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps.jld2", "AA_dels_rep_seq_grps")
    global AA_muts_rep_seq_grps
        AA_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps.jld2", "AA_muts_rep_seq_grps")
    global AA_muts_rep_seq_grps_pos_only
        AA_muts_rep_seq_grps_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_pos_only.jld2", "AA_muts_rep_seq_grps_pos_only")
    global AA_muts_rep_seq_grps_no_dels
        AA_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_no_dels.jld2", "AA_muts_rep_seq_grps_no_dels")
    global nuc_muts_rep_seq_grps
        nuc_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps.jld2", "nuc_muts_rep_seq_grps")
    global nuc_muts_rep_seq_grps_no_dels
        nuc_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_no_dels.jld2", "nuc_muts_rep_seq_grps_no_dels")
    global nuc_dels_rep_seq_grps
        nuc_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps.jld2", "nuc_dels_rep_seq_grps")    
##########################################################################################################################################################################
##########################################################################################################################################################################
################################################ Below: Used to be in ungodly @save form, now changed ####################################################################
##########################################################################################################################################################################
##########################################################################################################################################################################
    global AA_muts_ct_pos_only_adj_sort_by_site
        AA_muts_ct_pos_only_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_sort_by_site")
    global nuc_muts_ct_adj_score_no_dels_sort_by_site
        nuc_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_no_dels_sort_by_site
        nuc_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_no_dels_sort_by_site")
    global nuc_muts_ct_adj_sort_by_seq_ct
        nuc_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_sort_by_seq_ct")
    global nuc_muts_ct_adj_score_sort_by_score
        nuc_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_score.jld2", "nuc_muts_ct_adj_score_sort_by_score")
    global AA_muts_ct_adj_score_sort_by_site
        AA_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_site.jld2", "AA_muts_ct_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_sort_by_site
        AA_muts_ct_pos_only_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_score_no_dels_sort_by_site
        AA_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_site")
    global AA_muts_ct_adj_sort_by_seq_ct
        AA_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_seq_ct.jld2", "AA_muts_ct_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_no_dels_sort_by_site
        AA_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_adj_score_sort_by_score
        AA_muts_ct_pos_only_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_score")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_no_dels_sort_by_seq_ct
        nuc_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_adj_sort_by_site
        AA_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_site.jld2", "AA_muts_ct_adj_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_site")
    global AA_muts_ct_adj_score_sort_by_score
        AA_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_score.jld2", "AA_muts_ct_adj_score_sort_by_score")
    global nuc_muts_ct_adj_score_no_dels_sort_by_score
        nuc_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_score.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_sort_by_site
        nuc_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_site.jld2", "nuc_muts_ct_adj_sort_by_site")
    global AA_muts_ct_adj_score_no_dels_sort_by_score
        AA_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_score")
    global AA_muts_ct_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_score_sort_by_site
        nuc_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_site.jld2", "nuc_muts_ct_adj_score_sort_by_site")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_sort_by_site
        AA_muts_ct_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_site.jld2", "AA_muts_ct_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_seq_ct
        AA_muts_ct_pos_only_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global nuc_muts_ct_sort_by_site
        nuc_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_site.jld2", "nuc_muts_ct_sort_by_site")
    global excluded_AA
        excluded_AA = load("$(folder_name)/$(folder_name)__excluded_AA.jld2", "excluded_AA")
    global chronic_search_muts
        chronic_search_muts = load("$(folder_name)/$(folder_name)__chronic_search_muts.jld2", "chronic_search_muts")
    global excluded_pos
        excluded_pos = load("$(folder_name)/$(folder_name)__excluded_pos.jld2", "excluded_pos")
    global rep_seqs
        rep_seqs = load("$(folder_name)/$(folder_name)__rep_seqs.jld2", "rep_seqs")
    global non_rep_seqs
        non_rep_seqs = load("$(folder_name)/$(folder_name)__non_rep_seqs.jld2", "non_rep_seqs")
    global all_seqs
        all_seqs = load("$(folder_name)/$(folder_name)__all_seqs.jld2", "all_seqs")
    global domain_mut_density_sort_by_gene
        domain_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density
        gene_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global nuc_muts_ct_sort_by_seq_ct
        nuc_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_sort_by_seq_ct
        AA_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_by_seq_ct
        AA_muts_ct_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_by_site
        AA_muts_ct_pos_only_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_site")
    global gene_mut_density_sort_by_gene
        gene_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global AA_muts_ct_sort_by_site
        AA_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_site.jld2", "AA_muts_ct_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_site
        AA_muts_ct_pos_only_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_site.jld2", "AA_muts_ct_pos_only_sort_by_site")
    global domain_mut_density_sort_by_density
        domain_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global too_many_reversions
        too_many_reversions = load("$(folder_name)/$(folder_name)__too_many_reversions.jld2", "too_many_reversions")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")

    global AA_muts_ct_chr_all_ratio_ct_sort
        AA_muts_ct_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_chr_all_ratio_ct_sort")
    global AA_muts_ct_chr_all_ratio_pos_sort
        AA_muts_ct_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_chr_all_ratio_pos_sort")    

    global AA_muts_ct_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global AA_muts_ct_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_pos_sort")
    
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    
    global avg_AA_subs_per_chr_seq
        avg_AA_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq.jld2", "avg_AA_subs_per_chr_seq")
    global avg_AA_subs_per_circ_seq
        avg_AA_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_circ_seq.jld2", "avg_AA_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global NSP_muts_sortByCt_Arr
        NSP_muts_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByCt_Arr.jld2", "NSP_muts_sortByCt_Arr")
    global NSP_muts_sortByPos_Arr
        NSP_muts_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByPos_Arr.jld2", "NSP_muts_sortByPos_Arr")
    global NSP_muts_no_dels_sortByCt_Arr
        NSP_muts_no_dels_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByCt_Arr.jld2", "NSP_muts_no_dels_sortByCt_Arr")
    global NSP_muts_no_dels_sortByPos_Arr
        NSP_muts_no_dels_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByPos_Arr.jld2", "NSP_muts_no_dels_sortByPos_Arr")
    global NSP1_muts_sortByCt
        NSP1_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByCt.jld2", "NSP1_muts_sortByCt")
    global NSP1_muts_sortByPos
        NSP1_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByPos.jld2", "NSP1_muts_sortByPos")
    global NSP2_muts_sortByCt
        NSP2_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByCt.jld2", "NSP2_muts_sortByCt")
    global NSP2_muts_sortByPos
        NSP2_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByPos.jld2", "NSP2_muts_sortByPos")
    global NSP3_muts_sortByCt
        NSP3_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByCt.jld2", "NSP3_muts_sortByCt")
    global NSP3_muts_sortByPos
        NSP3_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByPos.jld2", "NSP3_muts_sortByPos")
    global NSP4_muts_sortByCt
        NSP4_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByCt.jld2", "NSP4_muts_sortByCt")
    global NSP4_muts_sortByPos
        NSP4_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByPos.jld2", "NSP4_muts_sortByPos")
    global NSP5_muts_sortByCt
        NSP5_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByCt.jld2", "NSP5_muts_sortByCt")
    global NSP5_muts_sortByPos
        NSP5_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByPos.jld2", "NSP5_muts_sortByPos")
    global NSP6_muts_sortByCt
        NSP6_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByCt.jld2", "NSP6_muts_sortByCt")
    global NSP6_muts_sortByPos
        NSP6_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByPos.jld2", "NSP6_muts_sortByPos")
    global NSP7_muts_sortByCt
        NSP7_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByCt.jld2", "NSP7_muts_sortByCt")
    global NSP7_muts_sortByPos
        NSP7_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByPos.jld2", "NSP7_muts_sortByPos")
    global NSP8_muts_sortByCt
        NSP8_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByCt.jld2", "NSP8_muts_sortByCt")
    global NSP8_muts_sortByPos
        NSP8_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByPos.jld2", "NSP8_muts_sortByPos")
    global NSP9_muts_sortByCt
        NSP9_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByCt.jld2", "NSP9_muts_sortByCt")
    global NSP9_muts_sortByPos
        NSP9_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByPos.jld2", "NSP9_muts_sortByPos")
    global NSP10_muts_sortByCt
        NSP10_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByCt.jld2", "NSP10_muts_sortByCt")
    global NSP10_muts_sortByPos
        NSP10_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByPos.jld2", "NSP10_muts_sortByPos")
    global NSP11_muts_sortByCt
        NSP11_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByCt.jld2", "NSP11_muts_sortByCt")
    global NSP11_muts_sortByPos
        NSP11_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByPos.jld2", "NSP11_muts_sortByPos")
    global NSP12_muts_sortByCt
        NSP12_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByCt.jld2", "NSP12_muts_sortByCt")
    global NSP12_muts_sortByPos
        NSP12_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByPos.jld2", "NSP12_muts_sortByPos")
    global NSP13_muts_sortByCt
        NSP13_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByCt.jld2", "NSP13_muts_sortByCt")
    global NSP13_muts_sortByPos
        NSP13_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByPos.jld2", "NSP13_muts_sortByPos")
    global NSP14_muts_sortByCt
        NSP14_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByCt.jld2", "NSP14_muts_sortByCt")
    global NSP14_muts_sortByPos
        NSP14_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByPos.jld2", "NSP14_muts_sortByPos")
    global NSP15_muts_sortByCt
        NSP15_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByCt.jld2", "NSP15_muts_sortByCt")
    global NSP15_muts_sortByPos
        NSP15_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByPos.jld2", "NSP15_muts_sortByPos")
    global NSP16_muts_sortByCt
        NSP16_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByCt.jld2", "NSP16_muts_sortByCt")
    global NSP16_muts_sortByPos
        NSP16_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByPos.jld2", "NSP16_muts_sortByPos")
    global NSP1_muts_no_dels_sortByCt
        NSP1_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByCt.jld2", "NSP1_muts_no_dels_sortByCt")
    global NSP1_muts_no_dels_sortByPos
        NSP1_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByPos.jld2", "NSP1_muts_no_dels_sortByPos")
    global NSP2_muts_no_dels_sortByCt
        NSP2_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByCt.jld2", "NSP2_muts_no_dels_sortByCt")
    global NSP2_muts_no_dels_sortByPos
        NSP2_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByPos.jld2", "NSP2_muts_no_dels_sortByPos")
    global NSP3_muts_no_dels_sortByCt
        NSP3_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByCt.jld2", "NSP3_muts_no_dels_sortByCt")
    global NSP3_muts_no_dels_sortByPos
        NSP3_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByPos.jld2", "NSP3_muts_no_dels_sortByPos")
    global NSP4_muts_no_dels_sortByCt
        NSP4_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByCt.jld2", "NSP4_muts_no_dels_sortByCt")
    global NSP4_muts_no_dels_sortByPos
        NSP4_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByPos.jld2", "NSP4_muts_no_dels_sortByPos")
    global NSP5_muts_no_dels_sortByCt
        NSP5_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByCt.jld2", "NSP5_muts_no_dels_sortByCt")
    global NSP5_muts_no_dels_sortByPos
        NSP5_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByPos.jld2", "NSP5_muts_no_dels_sortByPos")
    global NSP6_muts_no_dels_sortByCt
        NSP6_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByCt.jld2", "NSP6_muts_no_dels_sortByCt")
    global NSP6_muts_no_dels_sortByPos
        NSP6_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByPos.jld2", "NSP6_muts_no_dels_sortByPos")
    global NSP7_muts_no_dels_sortByCt
        NSP7_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByCt.jld2", "NSP7_muts_no_dels_sortByCt")
    global NSP7_muts_no_dels_sortByPos
        NSP7_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByPos.jld2", "NSP7_muts_no_dels_sortByPos")
    global NSP8_muts_no_dels_sortByCt
        NSP8_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByCt.jld2", "NSP8_muts_no_dels_sortByCt")
    global NSP8_muts_no_dels_sortByPos
        NSP8_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByPos.jld2", "NSP8_muts_no_dels_sortByPos")
    global NSP9_muts_no_dels_sortByCt
        NSP9_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByCt.jld2", "NSP9_muts_no_dels_sortByCt")
    global NSP9_muts_no_dels_sortByPos
        NSP9_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByPos.jld2", "NSP9_muts_no_dels_sortByPos")
    global NSP10_muts_no_dels_sortByCt
        NSP10_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByCt.jld2", "NSP10_muts_no_dels_sortByCt")
    global NSP10_muts_no_dels_sortByPos
        NSP10_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByPos.jld2", "NSP10_muts_no_dels_sortByPos")
    global NSP11_muts_no_dels_sortByCt
        NSP11_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByCt.jld2", "NSP11_muts_no_dels_sortByCt")
    global NSP11_muts_no_dels_sortByPos
        NSP11_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByPos.jld2", "NSP11_muts_no_dels_sortByPos")
    global NSP12_muts_no_dels_sortByCt
        NSP12_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByCt.jld2", "NSP12_muts_no_dels_sortByCt")
    global NSP12_muts_no_dels_sortByPos
        NSP12_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByPos.jld2", "NSP12_muts_no_dels_sortByPos")
    global NSP13_muts_no_dels_sortByCt
        NSP13_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByCt.jld2", "NSP13_muts_no_dels_sortByCt")
    global NSP13_muts_no_dels_sortByPos
        NSP13_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByPos.jld2", "NSP13_muts_no_dels_sortByPos")
    global NSP14_muts_no_dels_sortByCt
        NSP14_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByCt.jld2", "NSP14_muts_no_dels_sortByCt")
    global NSP14_muts_no_dels_sortByPos
        NSP14_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByPos.jld2", "NSP14_muts_no_dels_sortByPos")
    global NSP15_muts_no_dels_sortByCt
        NSP15_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByCt.jld2", "NSP15_muts_no_dels_sortByCt")
    global NSP15_muts_no_dels_sortByPos
        NSP15_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByPos.jld2", "NSP15_muts_no_dels_sortByPos")
    global NSP16_muts_no_dels_sortByCt
        NSP16_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByCt.jld2", "NSP16_muts_no_dels_sortByCt")
    global NSP16_muts_no_dels_sortByPos
        NSP16_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByPos.jld2", "NSP16_muts_no_dels_sortByPos")
####################################################################################################
    global all_seqs_set
        all_seqs_set = load("$(folder_name)/$(folder_name)__all_seqs_set.jld2", "all_seqs_set")
    global all_qualifying_seqs
        all_qualifying_seqs = load("$(folder_name)/$(folder_name)__all_qualifying_seqs.jld2", "all_qualifying_seqs")
    global all_qualifying_seqs_set
        all_qualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_qualifying_seqs_set.jld2", "all_qualifying_seqs_set")
    global all_nonqualifying_seqs
        all_nonqualifying_seqs = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs.jld2", "all_nonqualifying_seqs")
    global all_nonqualifying_seqs_set
        all_nonqualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs_set.jld2", "all_nonqualifying_seqs_set")
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_no_revs_sort_by_site
        AA_muts_ct_no_dels_no_revs_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_site.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_site")
    global AA_muts_ct_no_dels_no_revs_sort_by_seq_ct
        AA_muts_ct_no_dels_no_revs_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_seq_ct")
    global chronic_search_muts_v2
        chronic_search_muts_v2 = load("$(folder_name)/$(folder_name)__chronic_search_muts_v2.jld2", "chronic_search_muts_v2")
    global avg_AA_subs_per_chr_seq_no_revs
        avg_AA_subs_per_chr_seq_no_revs = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq_no_revs.jld2", "avg_AA_subs_per_chr_seq_no_revs")
###########################################################################################################################################################################
    global nuc_muts_ct_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global nuc_muts_ct_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    global avg_nuc_subs_per_chr_seq
        avg_nuc_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_chr_seq.jld2", "avg_nuc_subs_per_chr_seq")
    global avg_nuc_subs_per_circ_seq
        avg_nuc_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_circ_seq.jld2", "avg_nuc_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    return date_nuc_mut_ct, date_nuc_mut_ct_no_dels, date_AA_mut_ct, date_AA_mut_ct_no_dels, date_AA_mut_ct_pos_only_no_dels, 
    seq_ct_by_year, seq_ct_by_year_month, seq_ct_by_year_month_day, 
    seq_collection_date, seq_date_index, seq_date_tuple, 
###################################################################################################################################
    seq_clade, seq_clade_display, seq_pango, 
    seq_pango_unaliased, seq_clade_ct, seq_pango_ct, seq_pango_unaliased_ct, 
###################################################################################################################################
    too_many_reversions, 
###################################################################################################################################
    seq_nuc_muts, seq_nuc_del_ranges, seq_nuc_dropout, seq_nuc_muts_WT, seq_nuc_del_ranges_WT, 
################################
    seq_AA_insertions_WT, seq_nuc_insertions_WT,
################################
    seq_AA_muts, seq_AA_muts_no_dels, seq_AA_muts_pos_only, seq_AA_dels, seq_mixed_AA_muts, 
    seq_AA_muts_WT, seq_AA_muts_WT_pos_only, seq_AA_dels_WT, seq_AA_muts_pos_only_no_dels, 
########################################################
    nuc_muts_seq, nuc_muts_seq_WT, 
    nuc_dels_seq, nuc_dels_seq_WT, 
    AA_muts_seq, AA_muts_seq_pos_only, AA_muts_seq_WT, AA_muts_seq_WT_pos_only, AA_muts_seq_pos_only_no_dels, 
    AA_dels_seq, AA_dels_seq_WT, 
######################################################## 
    seq_unknown_AA, seq_unknown_AA_ranges, seq_mixed_nucs, 
############################
    nuc_dels_ct, nuc_muts_ct, nuc_muts_ct_no_dels, 
###############
    AA_dels_ct, AA_muts_ct, AA_muts_ct_no_dels, AA_muts_ct_pos_only, AA_muts_ct_pos_only_no_dels, AA_muts_ct_no_dels_no_revs, 
############################
    nuc_muts_ct_sort_by_site, nuc_muts_ct_sort_by_seq_ct, 
############################ 
    AA_muts_ct_sort_by_site, AA_muts_ct_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_site, AA_muts_ct_no_dels_no_revs_sort_by_site, 
    AA_muts_ct_no_dels_no_revs_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_seq_ct,  AA_muts_ct_pos_only_sort_by_site, 
    AA_muts_ct_pos_only_sort_by_seq_ct, AA_muts_ct_pos_only_no_dels_sort_by_site, AA_muts_ct_pos_only_no_dels_sort_by_seq_ct, 
###################################################################################################################################
################################################################################################################################### 
    nuc_muts_ct_adj, nuc_muts_ct_adj_no_dels, nuc_muts_ct_adj_score, nuc_muts_ct_adj_score_no_dels,   
    nuc_muts_ct_adj_sort_by_seq_ct, nuc_muts_ct_adj_no_dels_sort_by_site, nuc_muts_ct_adj_no_dels_sort_by_seq_ct,
    nuc_muts_ct_adj_sort_by_site, nuc_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score, 
    nuc_muts_ct_adj_score_sort_by_site, nuc_muts_ct_adj_score_sort_by_score, nuc_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_adj, AA_muts_ct_adj_no_dels, AA_muts_ct_pos_only_adj, AA_muts_ct_pos_only_adj_no_dels, 
    AA_muts_ct_adj_score, AA_muts_ct_adj_score_no_dels, AA_muts_ct_pos_only_adj_score, AA_muts_ct_pos_only_adj_score_no_dels,
    AA_muts_ct_adj_sort_by_site, AA_muts_ct_adj_sort_by_seq_ct, AA_muts_ct_adj_no_dels_sort_by_site,
    AA_muts_ct_adj_no_dels_sort_by_seq_ct, AA_muts_ct_adj_score_sort_by_score, AA_muts_ct_adj_score_sort_by_site, 
    AA_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_sort_by_site, AA_muts_ct_pos_only_adj_sort_by_seq_ct, AA_muts_ct_pos_only_adj_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct, AA_muts_ct_pos_only_adj_score_sort_by_site, 
    AA_muts_ct_pos_only_adj_score_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site, 
####################################################################################################################################
    AA_muts_ct_chr_all_ratio, AA_muts_ct_no_dels_chr_all_ratio, AA_muts_ct_no_dels_no_revs_chr_all_ratio, AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
########################
    AA_muts_ct_chr_all_ratio_ct_sort, AA_muts_ct_chr_all_ratio_pos_sort, 
    AA_muts_ct_no_dels_chr_all_ratio_ct_sort, AA_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
###################################################### 
    nuc_muts_ct_no_dels_chr_all_ratio, nuc_muts_ct_no_dels_no_revs_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
#######################
    nuc_muts_ct_no_dels_chr_all_ratio_ct_sort, nuc_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
####################################################################################################################################
    NSP_muts, NSP_muts_sortByCt_Arr, NSP_muts_sortByPos_Arr, NSP_muts_no_dels_sortByCt_Arr, NSP_muts_no_dels_sortByPos_Arr, 
    NSP1_muts_sortByCt, NSP2_muts_sortByCt, NSP3_muts_sortByCt, NSP4_muts_sortByCt, NSP5_muts_sortByCt, NSP6_muts_sortByCt,
    NSP7_muts_sortByCt, NSP8_muts_sortByCt, NSP9_muts_sortByCt, NSP10_muts_sortByCt, NSP11_muts_sortByCt, NSP12_muts_sortByCt, 
    NSP13_muts_sortByCt, NSP14_muts_sortByCt, NSP15_muts_sortByCt, NSP16_muts_sortByCt, NSP1_muts_sortByPos, NSP2_muts_sortByPos, 
    NSP3_muts_sortByPos, NSP4_muts_sortByPos, NSP5_muts_sortByPos, NSP6_muts_sortByPos, NSP7_muts_sortByPos, NSP8_muts_sortByPos, 
    NSP9_muts_sortByPos, NSP10_muts_sortByPos, NSP11_muts_sortByPos, NSP12_muts_sortByPos, NSP13_muts_sortByPos, NSP14_muts_sortByPos, 
    NSP15_muts_sortByPos, NSP16_muts_sortByPos, NSP1_muts_no_dels_sortByCt, NSP2_muts_no_dels_sortByCt, NSP3_muts_no_dels_sortByCt, 
    NSP4_muts_no_dels_sortByCt, NSP5_muts_no_dels_sortByCt, NSP6_muts_no_dels_sortByCt, NSP7_muts_no_dels_sortByCt, 
    NSP8_muts_no_dels_sortByCt, NSP9_muts_no_dels_sortByCt, NSP10_muts_no_dels_sortByCt, NSP11_muts_no_dels_sortByCt, 
    NSP12_muts_no_dels_sortByCt, NSP13_muts_no_dels_sortByCt, NSP14_muts_no_dels_sortByCt, NSP15_muts_no_dels_sortByCt, 
    NSP16_muts_no_dels_sortByCt, NSP1_muts_no_dels_sortByPos, NSP2_muts_no_dels_sortByPos, NSP3_muts_no_dels_sortByPos, 
    NSP4_muts_no_dels_sortByPos, NSP5_muts_no_dels_sortByPos, NSP6_muts_no_dels_sortByPos, NSP7_muts_no_dels_sortByPos, 
    NSP8_muts_no_dels_sortByPos, NSP9_muts_no_dels_sortByPos, NSP10_muts_no_dels_sortByPos, NSP11_muts_no_dels_sortByPos, 
    NSP12_muts_no_dels_sortByPos, NSP13_muts_no_dels_sortByPos, NSP14_muts_no_dels_sortByPos, NSP15_muts_no_dels_sortByPos, 
    NSP16_muts_no_dels_sortByPos,
####################################################################################################################################
    chronic_search_muts, chronic_search_muts_v2, 
####################################################################################################################################
    avg_AA_subs_per_circ_seq, avg_nuc_subs_per_circ_seq, avg_AA_subs_per_chr_seq, avg_nuc_subs_per_chr_seq,
####################################################################################################################################
    nuc_muts_rep_seq_grps, nuc_dels_rep_seq_grps, 
    nuc_muts_rep_seq_grps_no_dels, rep_seq_grps_dels, rep_seq_grps_muts_no_dels, rep_seq_grps_del_ranges_ct, rep_seq_grps_nuc_dropout, 
    rep_seq_grps_mixed_nucs, rep_seq_grps_AA_dels, rep_seq_grps_AA_no_dels, AA_muts_rep_seq_grps, AA_dels_rep_seq_grps, 
    AA_muts_rep_seq_grps_no_dels, AA_muts_rep_seq_grps_pos_only,  
    rep_seq_grps_nuc_muts_WT, rep_seq_grps_nuc_dels_WT, rep_seq_grps_AA_muts_WT, rep_seq_grps_AA_dels_WT, nuc_muts_rep_seq_grps_WT, 
    nuc_dels_rep_seq_grps_WT, AA_muts_rep_seq_grps_WT, AA_dels_rep_seq_grps_WT, rep_seq_grps_AA_muts_WT_pos_only, 
    AA_muts_rep_seq_grps_WT_pos_only, rep_seq_grps_pango, rep_seq_grps_pango_unaliased, excluded_pos, excluded_AA, all_seqs, rep_seqs, 
    non_rep_seqs, rep_seq_grps_seqs, rep_seq_grps_muts, rep_seq_grps_clade, rep_seq_grps_AA, rep_seq_grps_AA_pos_only, 
    rep_seq_grps_AA_pos_only_no_dels, non_rep_seqs_AA, non_rep_seqs_AA_pos_only, non_rep_seqs_AA_pos_only_no_dels, 
####################################################################################################################################
    rep_seq_grps_maxmut_nuc_muts_WT, rep_seq_grps_maxmut_nuc_dels_WT, 
    rep_seq_grps_maxmut_AA_muts_WT, rep_seq_grps_maxmut_AA_dels_WT, rep_seq_grps_maxmut_AA_muts_WT_pos_only,
    rep_seq_grps_maxmut_seqs, rep_seq_grps_maxmut_nuc, rep_seq_grps_maxmut_nuc_no_dels, rep_seq_grps_maxmut_dels, 
    rep_seq_grps_maxmut_nuc_dropout, rep_seq_grps_maxmut_mixed_nucs, rep_seq_grps_maxmut_AA, rep_seq_grps_maxmut_AA_no_dels, 
    rep_seq_grps_maxmut_AA_dels, rep_seq_grps_maxmut_AA_pos_only, rep_seq_grps_maxmut_AA_pos_only_no_dels, 
    rep_seq_grps_maxmut_unknown_AA, rep_seq_grps_maxmut_unknown_AA_ranges, rep_seq_grps_maxmut_mixed_AA_muts, rep_seq_grps_maxmut_del_ranges_ct,
####################################################################################################################################
    gene_mut_density_sort_by_gene, gene_mut_density_sort_by_density, domain_mut_density_sort_by_gene, domain_mut_density_sort_by_density, 
    gene_mut_density, domain_mut_density, 
####################################################################################################################################
    all_seqs_set, all_qualifying_seqs, all_qualifying_seqs_set, all_nonqualifying_seqs, all_nonqualifying_seqs_set
####################################################################################################################################
    total_chr_AA_subs, total_nuc_revs_seq, seq_nuc_total_revs, total_AA_revs_seq, seq_AA_total_revs, seq_AA_revs
end
# REMOVED: rep_seq_grps_unknown_AA, rep_seq_grps_mixed_AA_muts,
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_chr(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    ORF1a_pos_ct = Dict{String, Int}()
    ORF1b_pos_ct = Dict{String, Int}()
    S_pos_ct = Dict{String, Int}()
    ORF3a_pos_ct = Dict{String, Int}()
    E_pos_ct = Dict{String, Int}()
    M_pos_ct = Dict{String, Int}()
    ORF6_pos_ct = Dict{String, Int}()
    ORF7a_pos_ct = Dict{String, Int}()
    ORF7b_pos_ct = Dict{String, Int}()
    ORF8_pos_ct = Dict{String, Int}()
    N_pos_ct = Dict{String, Int}()
    ORF9b_pos_ct = Dict{String, Int}()
    gene_pos_ct_dict_arr = [ORF1a_pos_ct, ORF1b_pos_ct, S_pos_ct, ORF3a_pos_ct, E_pos_ct, M_pos_ct, ORF6_pos_ct, ORF7a_pos_ct, ORF7b_pos_ct, ORF8_pos_ct, N_pos_ct, ORF9b_pos_ct]
######################################################
    ORF1a_pos_ct_v1 = Dict{Int,Int}()
    ORF1b_pos_ct_v1 = Dict{Int,Int}()
    S_pos_ct_v1 = Dict{Int,Int}()
    ORF3a_pos_ct_v1 = Dict{Int,Int}()
    E_pos_ct_v1 = Dict{Int,Int}()
    M_pos_ct_v1 = Dict{Int,Int}()
    ORF6_pos_ct_v1 = Dict{Int,Int}()
    ORF7a_pos_ct_v1 = Dict{Int,Int}()
    ORF7b_pos_ct_v1 = Dict{Int,Int}()
    ORF8_pos_ct_v1 = Dict{Int,Int}()
    N_pos_ct_v1 = Dict{Int,Int}()
    ORF9b_pos_ct_v1 = Dict{Int,Int}()
    gene_pos_ct_v1_dict_arr = [ORF1a_pos_ct_v1, ORF1b_pos_ct_v1, S_pos_ct_v1, ORF3a_pos_ct_v1, E_pos_ct_v1, M_pos_ct_v1, ORF6_pos_ct_v1, ORF7a_pos_ct_v1, ORF7b_pos_ct_v1, ORF8_pos_ct_v1, N_pos_ct_v1, ORF9b_pos_ct_v1]
######################################################
    for i in 1:length(gene_pos_ct_dict_arr)
        dict = gene_pos_ct_dict_arr[i]
        dict_v1 = gene_pos_ct_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct[mut] = ct
            ORF1a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct[mut] = ct
            ORF1b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct[mut] = ct
            S_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct[mut] = ct
            ORF3a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct[mut] = ct
            E_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct[mut] = ct
            M_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct[mut] = ct
            ORF6_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct[mut] = ct
            ORF7a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct[mut] = ct
            ORF7b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct[mut] = ct
            ORF8_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct[mut] = ct
            N_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct[mut] = ct
            ORF9b_pos_ct_v1[pos] = ct
        end
    end
######################### 
    global ORF1a_pos_ct_sort_by_pos = sort(collect(ORF1a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_sort_by_pos = sort(collect(ORF1b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_sort_by_pos = sort(collect(S_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_sort_by_pos = sort(collect(ORF3a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_sort_by_pos = sort(collect(E_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global M_pos_ct_sort_by_pos = sort(collect(M_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_sort_by_pos = sort(collect(ORF6_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_sort_by_pos = sort(collect(ORF7a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_sort_by_pos = sort(collect(ORF7b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_sort_by_pos = sort(collect(ORF8_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_sort_by_pos = sort(collect(N_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF9b_pos_ct_sort_by_pos = sort(collect(ORF9b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
#########################
    global ORF1a_pos_ct_sort_by_ct = sort(collect(ORF1a_pos_ct), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct = sort(collect(ORF1b_pos_ct), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct = sort(collect(S_pos_ct), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct = sort(collect(ORF3a_pos_ct), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct = sort(collect(E_pos_ct), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct = sort(collect(M_pos_ct), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct = sort(collect(ORF6_pos_ct), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct = sort(collect(ORF7a_pos_ct), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct = sort(collect(ORF7b_pos_ct), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct = sort(collect(ORF8_pos_ct), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct = sort(collect(N_pos_ct), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct = sort(collect(ORF9b_pos_ct), by = x -> x[2], rev=true)
#########################
    global ORF1a_pos_ct_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[1])
    global ORF1b_pos_ct_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[1])
    global S_pos_ct_sort_by_pos_v1 = sort(collect(S_pos_ct_v1), by = x -> x[1])
    global ORF3a_pos_ct_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[1])
    global E_pos_ct_sort_by_pos_v1 = sort(collect(E_pos_ct_v1), by = x -> x[1])
    global M_pos_ct_sort_by_pos_v1 = sort(collect(M_pos_ct_v1), by = x -> x[1])
    global ORF6_pos_ct_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[1])
    global ORF7a_pos_ct_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[1])
    global ORF7b_pos_ct_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[1])
    global ORF8_pos_ct_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[1])
    global N_pos_ct_sort_by_pos_v1 = sort(collect(N_pos_ct_v1), by = x -> x[1])
    global ORF9b_pos_ct_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[1])
#########################
    global ORF1a_pos_ct_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct_v1 = sort(collect(S_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct_v1 = sort(collect(E_pos_ct_v1), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct_v1 = sort(collect(M_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct_v1 = sort(collect(N_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[2], rev=true)
end
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
function gene_sub_ranks_chr(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    ORF1a_ct = Dict{String, Int}()
    ORF1b_ct = Dict{String, Int}()
    S_ct = Dict{String, Int}()
    ORF3a_ct = Dict{String, Int}()
    E_ct = Dict{String, Int}()
    M_ct = Dict{String, Int}()
    ORF6_ct = Dict{String, Int}()
    ORF7a_ct = Dict{String, Int}()
    ORF7b_ct = Dict{String, Int}()
    ORF8_ct = Dict{String, Int}()
    N_ct = Dict{String, Int}()
    ORF9b_ct = Dict{String, Int}()
#############################################################
    gene_ct_dict_arr = [ORF1a_ct, ORF1b_ct, S_ct, ORF3a_ct, E_ct, M_ct, ORF6_ct, ORF7a_ct, ORF7b_ct, ORF8_ct, N_ct, ORF9b_ct] 
#############################################################
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
########################################## 
    global ORF1a_ct_sort_by_pos = sort(collect(ORF1a_ct), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_sort_by_pos = sort(collect(ORF1b_ct), by = x -> fin_sortkey(x[1]))
    global S_ct_sort_by_pos = sort(collect(S_ct), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_sort_by_pos = sort(collect(ORF3a_ct), by = x -> fin_sortkey(x[1]))
    global E_ct_sort_by_pos = sort(collect(E_ct), by = x -> fin_sortkey(x[1]))
    global M_ct_sort_by_pos = sort(collect(M_ct), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_sort_by_pos = sort(collect(ORF6_ct), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_sort_by_pos = sort(collect(ORF7a_ct), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_sort_by_pos = sort(collect(ORF7b_ct), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_sort_by_pos = sort(collect(ORF8_ct), by = x -> fin_sortkey(x[1]))
    global N_ct_sort_by_pos = sort(collect(N_ct), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_sort_by_pos = sort(collect(ORF9b_ct), by = x -> fin_sortkey(x[1]))
##########################################
    global ORF1a_ct_sort_by_ct = sort(collect(ORF1a_ct), by = x -> x[2], rev=true)
    global ORF1b_ct_sort_by_ct = sort(collect(ORF1b_ct), by = x -> x[2], rev=true)
    global S_ct_sort_by_ct = sort(collect(S_ct), by = x -> x[2], rev=true)
    global ORF3a_ct_sort_by_ct = sort(collect(ORF3a_ct), by = x -> x[2], rev=true)
    global E_ct_sort_by_ct = sort(collect(E_ct), by = x -> x[2], rev=true)
    global M_ct_sort_by_ct = sort(collect(M_ct), by = x -> x[2], rev=true)
    global ORF6_ct_sort_by_ct = sort(collect(ORF6_ct), by = x -> x[2], rev=true)
    global ORF7a_ct_sort_by_ct = sort(collect(ORF7a_ct), by = x -> x[2], rev=true)
    global ORF7b_ct_sort_by_ct = sort(collect(ORF7b_ct), by = x -> x[2], rev=true)
    global ORF8_ct_sort_by_ct = sort(collect(ORF8_ct), by = x -> x[2], rev=true)
    global N_ct_sort_by_ct = sort(collect(N_ct), by = x -> x[2], rev=true)
    global ORF9b_ct_sort_by_ct = sort(collect(ORF9b_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = aa_pos_comprehensive_dict[sub]
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
#############################
    global NSP1_ct_sort_by_pos = sort(collect(NSP1_ct), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_sort_by_pos = sort(collect(NSP2_ct), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_sort_by_pos = sort(collect(NSP3_ct), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_sort_by_pos = sort(collect(NSP4_ct), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_sort_by_pos = sort(collect(NSP5_ct), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_sort_by_pos = sort(collect(NSP6_ct), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_sort_by_pos = sort(collect(NSP7_ct), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_sort_by_pos = sort(collect(NSP8_ct), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_sort_by_pos = sort(collect(NSP9_ct), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_sort_by_pos = sort(collect(NSP10_ct), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_sort_by_pos = sort(collect(NSP11_ct), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_sort_by_pos = sort(collect(NSP12_ct), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_sort_by_pos = sort(collect(NSP13_ct), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_sort_by_pos = sort(collect(NSP14_ct), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_sort_by_pos = sort(collect(NSP15_ct), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_sort_by_pos = sort(collect(NSP16_ct), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_sort_by_ct = sort(collect(NSP1_ct), by = x -> x[2], rev=true)
    global NSP2_ct_sort_by_ct = sort(collect(NSP2_ct), by = x -> x[2], rev=true)
    global NSP3_ct_sort_by_ct = sort(collect(NSP3_ct), by = x -> x[2], rev=true)
    global NSP4_ct_sort_by_ct = sort(collect(NSP4_ct), by = x -> x[2], rev=true)
    global NSP5_ct_sort_by_ct = sort(collect(NSP5_ct), by = x -> x[2], rev=true)
    global NSP6_ct_sort_by_ct = sort(collect(NSP6_ct), by = x -> x[2], rev=true)
    global NSP7_ct_sort_by_ct = sort(collect(NSP7_ct), by = x -> x[2], rev=true)
    global NSP8_ct_sort_by_ct = sort(collect(NSP8_ct), by = x -> x[2], rev=true)
    global NSP9_ct_sort_by_ct = sort(collect(NSP9_ct), by = x -> x[2], rev=true)
    global NSP10_ct_sort_by_ct = sort(collect(NSP10_ct), by = x -> x[2], rev=true)
    global NSP11_ct_sort_by_ct = sort(collect(NSP11_ct), by = x -> x[2], rev=true)
    global NSP12_ct_sort_by_ct = sort(collect(NSP12_ct), by = x -> x[2], rev=true)
    global NSP13_ct_sort_by_ct = sort(collect(NSP13_ct), by = x -> x[2], rev=true)
    global NSP14_ct_sort_by_ct = sort(collect(NSP14_ct), by = x -> x[2], rev=true)
    global NSP15_ct_sort_by_ct = sort(collect(NSP15_ct), by = x -> x[2], rev=true)
    global NSP16_ct_sort_by_ct = sort(collect(NSP16_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_pos_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    NSP1_pos_ct = Dict{String, Int}()
    NSP2_pos_ct = Dict{String, Int}()
    NSP3_pos_ct = Dict{String, Int}()
    NSP4_pos_ct = Dict{String, Int}()
    NSP5_pos_ct = Dict{String, Int}()
    NSP6_pos_ct = Dict{String, Int}()
    NSP7_pos_ct = Dict{String, Int}()
    NSP8_pos_ct = Dict{String, Int}()
    NSP9_pos_ct = Dict{String, Int}()
    NSP10_pos_ct = Dict{String, Int}()
    NSP11_pos_ct = Dict{String, Int}()
    NSP12_pos_ct = Dict{String, Int}()
    NSP13_pos_ct = Dict{String, Int}()
    NSP14_pos_ct = Dict{String, Int}()
    NSP15_pos_ct = Dict{String, Int}()
    NSP16_pos_ct = Dict{String, Int}()
    NSP_pos_ct_array = [NSP1_pos_ct, NSP2_pos_ct, NSP3_pos_ct, NSP4_pos_ct, NSP5_pos_ct, NSP6_pos_ct, NSP7_pos_ct, NSP8_pos_ct, NSP9_pos_ct, NSP10_pos_ct, NSP11_pos_ct, NSP12_pos_ct, NSP13_pos_ct, NSP14_pos_ct, NSP15_pos_ct, NSP16_pos_ct]
##########################################
    NSP1_pos_ct_v1 = Dict{Int,Int}()
    NSP2_pos_ct_v1 = Dict{Int,Int}()
    NSP3_pos_ct_v1 = Dict{Int,Int}()
    NSP4_pos_ct_v1 = Dict{Int,Int}()
    NSP5_pos_ct_v1 = Dict{Int,Int}()
    NSP6_pos_ct_v1 = Dict{Int,Int}()
    NSP7_pos_ct_v1 = Dict{Int,Int}()
    NSP8_pos_ct_v1 = Dict{Int,Int}()
    NSP9_pos_ct_v1 = Dict{Int,Int}()
    NSP10_pos_ct_v1 = Dict{Int,Int}()
    NSP11_pos_ct_v1 = Dict{Int,Int}()
    NSP12_pos_ct_v1 = Dict{Int,Int}()
    NSP13_pos_ct_v1 = Dict{Int,Int}()
    NSP14_pos_ct_v1 = Dict{Int,Int}()
    NSP15_pos_ct_v1 = Dict{Int,Int}()
    NSP16_pos_ct_v1 = Dict{Int,Int}()
    NSP_pos_ct_v1_array = [NSP1_pos_ct_v1, NSP2_pos_ct_v1, NSP3_pos_ct_v1, NSP4_pos_ct_v1, NSP5_pos_ct_v1, NSP6_pos_ct_v1, NSP7_pos_ct_v1, NSP8_pos_ct_v1, NSP9_pos_ct_v1, NSP10_pos_ct_v1, NSP11_pos_ct_v1, NSP12_pos_ct_v1, NSP13_pos_ct_v1, NSP14_pos_ct_v1, NSP15_pos_ct_v1, NSP16_pos_ct_v1]
#######################################################
    NSP_pos_ct = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_array)
        NSP_dict = NSP_pos_ct_array[i]
        NSP_dict_v1 = NSP_pos_ct_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_dict[NSP_pos] = 0
            NSP_dict_v1[j] = 0
            NSP_pos_ct[NSP_pos] = 0
        end
    end
##########################################################################################################################
##########################################################################################################################
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            NSP_num = parse(Int, split(NSP, "P")[2])
            NSP_dict = NSP_ct_dict_arr[NSP_num]
            NSP_dict[NSP_sub] = ct
        end
    end
##########################################################################################################################
##########################################################################################################################
    for i in 1:length(NSP_ct_dict_arr)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = NSP_muts_pos_dict[sub]
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct[sub_pos] += ct
        end
    end
    global NSP1_pos_ct_sort_by_pos = sort(collect(NSP1_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_sort_by_pos = sort(collect(NSP2_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_sort_by_pos = sort(collect(NSP3_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_sort_by_pos = sort(collect(NSP4_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_sort_by_pos = sort(collect(NSP5_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_sort_by_pos = sort(collect(NSP6_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_sort_by_pos = sort(collect(NSP7_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_sort_by_pos = sort(collect(NSP8_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_sort_by_pos = sort(collect(NSP9_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_sort_by_pos = sort(collect(NSP10_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_sort_by_pos = sort(collect(NSP11_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_sort_by_pos = sort(collect(NSP12_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_sort_by_pos = sort(collect(NSP13_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_sort_by_pos = sort(collect(NSP14_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_sort_by_pos = sort(collect(NSP15_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_sort_by_pos = sort(collect(NSP16_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
#########################
    global NSP1_pos_ct_sort_by_ct = sort(collect(NSP1_pos_ct), by = x -> x[2], rev=true)
    global NSP2_pos_ct_sort_by_ct = sort(collect(NSP2_pos_ct), by = x -> x[2], rev=true)
    global NSP3_pos_ct_sort_by_ct = sort(collect(NSP3_pos_ct), by = x -> x[2], rev=true)
    global NSP4_pos_ct_sort_by_ct = sort(collect(NSP4_pos_ct), by = x -> x[2], rev=true)
    global NSP5_pos_ct_sort_by_ct = sort(collect(NSP5_pos_ct), by = x -> x[2], rev=true)
    global NSP6_pos_ct_sort_by_ct = sort(collect(NSP6_pos_ct), by = x -> x[2], rev=true)
    global NSP7_pos_ct_sort_by_ct = sort(collect(NSP7_pos_ct), by = x -> x[2], rev=true)
    global NSP8_pos_ct_sort_by_ct = sort(collect(NSP8_pos_ct), by = x -> x[2], rev=true)
    global NSP9_pos_ct_sort_by_ct = sort(collect(NSP9_pos_ct), by = x -> x[2], rev=true)
    global NSP10_pos_ct_sort_by_ct = sort(collect(NSP10_pos_ct), by = x -> x[2], rev=true)
    global NSP11_pos_ct_sort_by_ct = sort(collect(NSP11_pos_ct), by = x -> x[2], rev=true)
    global NSP12_pos_ct_sort_by_ct = sort(collect(NSP12_pos_ct), by = x -> x[2], rev=true)
    global NSP13_pos_ct_sort_by_ct = sort(collect(NSP13_pos_ct), by = x -> x[2], rev=true)
    global NSP14_pos_ct_sort_by_ct = sort(collect(NSP14_pos_ct), by = x -> x[2], rev=true)
    global NSP15_pos_ct_sort_by_ct = sort(collect(NSP15_pos_ct), by = x -> x[2], rev=true)
    global NSP16_pos_ct_sort_by_ct = sort(collect(NSP16_pos_ct), by = x -> x[2], rev=true)
#########################
    global NSP1_pos_ct_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_v1), by = x -> x[1])
    global NSP2_pos_ct_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_v1), by = x -> x[1])
    global NSP3_pos_ct_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_v1), by = x -> x[1])
    global NSP4_pos_ct_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_v1), by = x -> x[1])
    global NSP5_pos_ct_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_v1), by = x -> x[1])
    global NSP6_pos_ct_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_v1), by = x -> x[1])
    global NSP7_pos_ct_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_v1), by = x -> x[1])
    global NSP8_pos_ct_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_v1), by = x -> x[1])
    global NSP9_pos_ct_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_v1), by = x -> x[1])
    global NSP10_pos_ct_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_v1), by = x -> x[1])
    global NSP11_pos_ct_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_v1), by = x -> x[1])
    global NSP12_pos_ct_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_v1), by = x -> x[1])
    global NSP13_pos_ct_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_v1), by = x -> x[1])
    global NSP14_pos_ct_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_v1), by = x -> x[1])
    global NSP15_pos_ct_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_v1), by = x -> x[1])
    global NSP16_pos_ct_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_v1), by = x -> x[1])
end
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_ranks_all(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    ORF1a_ct_all = Dict{String, Int}()
    ORF1b_ct_all = Dict{String, Int}()
    S_ct_all = Dict{String, Int}()
    ORF3a_ct_all = Dict{String, Int}()
    E_ct_all = Dict{String, Int}()
    M_ct_all = Dict{String, Int}()
    ORF6_ct_all = Dict{String, Int}()
    ORF7a_ct_all = Dict{String, Int}()
    ORF7b_ct_all = Dict{String, Int}()
    ORF8_ct_all = Dict{String, Int}()
    N_ct_all = Dict{String, Int}()
    ORF9b_ct_all = Dict{String, Int}()
#############################################################
    #                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    gene_ct_dict_all_arr = [ORF1a_ct_all, ORF1b_ct_all, S_ct_all, ORF3a_ct_all, E_ct_all, M_ct_all, ORF6_ct_all, ORF7a_ct_all, ORF7b_ct_all, ORF8_ct_all, N_ct_all, ORF9b_ct_all]
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_all_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels_all
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct_all[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
    global ORF1a_ct_all_sort_by_pos = sort(collect(ORF1a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_all_sort_by_pos = sort(collect(ORF1b_ct_all), by = x -> fin_sortkey(x[1]))
    global S_ct_all_sort_by_pos = sort(collect(S_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_all_sort_by_pos = sort(collect(ORF3a_ct_all), by = x -> fin_sortkey(x[1]))
    global E_ct_all_sort_by_pos = sort(collect(E_ct_all), by = x -> fin_sortkey(x[1]))
    global M_ct_all_sort_by_pos = sort(collect(M_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_all_sort_by_pos = sort(collect(ORF6_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_all_sort_by_pos = sort(collect(ORF7a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_all_sort_by_pos = sort(collect(ORF7b_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_all_sort_by_pos = sort(collect(ORF8_ct_all), by = x -> fin_sortkey(x[1]))
    global N_ct_all_sort_by_pos = sort(collect(N_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_all_sort_by_pos = sort(collect(ORF9b_ct_all), by = x -> fin_sortkey(x[1]))

    global ORF1b_ct_all_sort_by_ct = sort(collect(ORF1b_ct_all), by = x -> x[2], rev=true)
    global ORF1a_ct_all_sort_by_ct = sort(collect(ORF1a_ct_all), by = x -> x[2], rev=true)
    global S_ct_all_sort_by_ct = sort(collect(S_ct_all), by = x -> x[2], rev=true)
    global ORF3a_ct_all_sort_by_ct = sort(collect(ORF3a_ct_all), by = x -> x[2], rev=true)
    global E_ct_all_sort_by_ct = sort(collect(E_ct_all), by = x -> x[2], rev=true)
    global M_ct_all_sort_by_ct = sort(collect(M_ct_all), by = x -> x[2], rev=true)
    global ORF6_ct_all_sort_by_ct = sort(collect(ORF6_ct_all), by = x -> x[2], rev=true)
    global ORF7a_ct_all_sort_by_ct = sort(collect(ORF7a_ct_all), by = x -> x[2], rev=true)
    global ORF7b_ct_all_sort_by_ct = sort(collect(ORF7b_ct_all), by = x -> x[2], rev=true)
    global ORF8_ct_all_sort_by_ct = sort(collect(ORF8_ct_all), by = x -> x[2], rev=true)
    global N_ct_all_sort_by_ct = sort(collect(N_ct_all), by = x -> x[2], rev=true)
    global ORF9b_ct_all_sort_by_ct = sort(collect(ORF9b_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_all(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int},  AA_muts_ct_pos_only_no_dels_all::Dict{String, Int})
    ORF1a_pos_ct_all = Dict{String, Int}()
    ORF1b_pos_ct_all = Dict{String, Int}()
    S_pos_ct_all = Dict{String, Int}()
    ORF3a_pos_ct_all = Dict{String, Int}()
    E_pos_ct_all = Dict{String, Int}()
    M_pos_ct_all = Dict{String, Int}()
    ORF6_pos_ct_all = Dict{String, Int}()
    ORF7a_pos_ct_all = Dict{String, Int}()
    ORF7b_pos_ct_all = Dict{String, Int}()
    ORF8_pos_ct_all = Dict{String, Int}()
    N_pos_ct_all = Dict{String, Int}()
    ORF9b_pos_ct_all = Dict{String, Int}()
    gene_ct_pos_all_dict_arr = [ORF1a_pos_ct_all, ORF1b_pos_ct_all, S_pos_ct_all, ORF3a_pos_ct_all, E_pos_ct_all, M_pos_ct_all, ORF6_pos_ct_all, ORF7a_pos_ct_all, ORF7b_pos_ct_all, ORF8_pos_ct_all, N_pos_ct_all, ORF9b_pos_ct_all] 
###################################################### 
    ORF1a_pos_ct_all_v1 = Dict{Int,Int}()
    ORF1b_pos_ct_all_v1 = Dict{Int,Int}()
    S_pos_ct_all_v1 = Dict{Int,Int}()
    ORF3a_pos_ct_all_v1 = Dict{Int,Int}()
    E_pos_ct_all_v1 = Dict{Int,Int}()
    M_pos_ct_all_v1 = Dict{Int,Int}()
    ORF6_pos_ct_all_v1 = Dict{Int,Int}()
    ORF7a_pos_ct_all_v1 = Dict{Int,Int}()
    ORF7b_pos_ct_all_v1 = Dict{Int,Int}()
    ORF8_pos_ct_all_v1 = Dict{Int,Int}()
    N_pos_ct_all_v1 = Dict{Int,Int}()
    ORF9b_pos_ct_all_v1 = Dict{Int,Int}()
    gene_pos_ct_all_v1_dict_arr = [ORF1a_pos_ct_all_v1, ORF1b_pos_ct_all_v1, S_pos_ct_all_v1, ORF3a_pos_ct_all_v1, E_pos_ct_all_v1, M_pos_ct_all_v1, ORF6_pos_ct_all_v1, ORF7a_pos_ct_all_v1, ORF7b_pos_ct_all_v1, ORF8_pos_ct_all_v1, N_pos_ct_all_v1, ORF9b_pos_ct_all_v1]
###################################################### 
    for i in 1:length(gene_ct_pos_all_dict_arr)
        dict = gene_ct_pos_all_dict_arr[i]
        dict_v1 = gene_pos_ct_all_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels_all
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct_all[mut] = ct
            ORF1a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct_all[mut] = ct
            ORF1b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct_all[mut] = ct
            S_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct_all[mut] = ct
            ORF3a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct_all[mut] = ct
            E_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct_all[mut] = ct
            M_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct_all[mut] = ct
            ORF6_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct_all[mut] = ct
            ORF7a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct_all[mut] = ct
            ORF7b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct_all[mut] = ct
            ORF8_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct_all[mut] = ct
            N_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct_all[mut] = ct
            ORF9b_pos_ct_all_v1[pos] = ct
        end
    end
    global ORF1a_pos_ct_all_sort_by_pos = sort(collect(ORF1a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_all_sort_by_pos = sort(collect(ORF1b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_all_sort_by_pos = sort(collect(S_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_all_sort_by_pos = sort(collect(ORF3a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_all_sort_by_pos = sort(collect(E_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]]) 
    global M_pos_ct_all_sort_by_pos = sort(collect(M_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_all_sort_by_pos = sort(collect(ORF6_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_all_sort_by_pos = sort(collect(ORF7a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_all_sort_by_pos = sort(collect(ORF7b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_all_sort_by_pos = sort(collect(ORF8_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_all_sort_by_pos = sort(collect(N_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])   
    global ORF9b_pos_ct_all_sort_by_pos = sort(collect(ORF9b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct = sort(collect(ORF1a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct = sort(collect(ORF1b_pos_ct_all), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct = sort(collect(S_pos_ct_all), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct = sort(collect(ORF3a_pos_ct_all), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct = sort(collect(E_pos_ct_all), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct = sort(collect(M_pos_ct_all), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct = sort(collect(ORF6_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct = sort(collect(ORF7a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct = sort(collect(ORF7b_pos_ct_all), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct = sort(collect(ORF8_pos_ct_all), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct = sort(collect(N_pos_ct_all), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct = sort(collect(ORF9b_pos_ct_all), by = x -> x[2], rev=true)
##################################
    global ORF1a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[1])
    global ORF1b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[1])
    global S_pos_ct_all_sort_by_pos_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[1])
    global ORF3a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[1])
    global E_pos_ct_all_sort_by_pos_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[1]) 
    global M_pos_ct_all_sort_by_pos_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[1])
    global ORF6_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[1])
    global ORF7a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[1])
    global ORF7b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[1])
    global ORF8_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[1])
    global N_pos_ct_all_sort_by_pos_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[1])   
    global ORF9b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[1])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function NSP_sub_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
#############################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            gene = aa_gene_comprehensive_dict[sub]
            if gene == "ORF1a" || gene == "ORF1b"
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
#############################
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
    global NSP1_ct_all_sort_by_pos = sort(collect(NSP1_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_all_sort_by_pos = sort(collect(NSP2_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_all_sort_by_pos = sort(collect(NSP3_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_all_sort_by_pos = sort(collect(NSP4_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_all_sort_by_pos = sort(collect(NSP5_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_all_sort_by_pos = sort(collect(NSP6_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_all_sort_by_pos = sort(collect(NSP7_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_all_sort_by_pos = sort(collect(NSP8_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_all_sort_by_pos = sort(collect(NSP9_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_all_sort_by_pos = sort(collect(NSP10_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_all_sort_by_pos = sort(collect(NSP11_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_all_sort_by_pos = sort(collect(NSP12_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_all_sort_by_pos = sort(collect(NSP13_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_all_sort_by_pos = sort(collect(NSP14_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_all_sort_by_pos = sort(collect(NSP15_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_all_sort_by_pos = sort(collect(NSP16_ct_all), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_all_sort_by_ct = sort(collect(NSP1_ct_all), by = x -> x[2], rev=true)
    global NSP2_ct_all_sort_by_ct = sort(collect(NSP2_ct_all), by = x -> x[2], rev=true)
    global NSP3_ct_all_sort_by_ct = sort(collect(NSP3_ct_all), by = x -> x[2], rev=true)
    global NSP4_ct_all_sort_by_ct = sort(collect(NSP4_ct_all), by = x -> x[2], rev=true)
    global NSP5_ct_all_sort_by_ct = sort(collect(NSP5_ct_all), by = x -> x[2], rev=true)
    global NSP6_ct_all_sort_by_ct = sort(collect(NSP6_ct_all), by = x -> x[2], rev=true)
    global NSP7_ct_all_sort_by_ct = sort(collect(NSP7_ct_all), by = x -> x[2], rev=true)
    global NSP8_ct_all_sort_by_ct = sort(collect(NSP8_ct_all), by = x -> x[2], rev=true)
    global NSP9_ct_all_sort_by_ct = sort(collect(NSP9_ct_all), by = x -> x[2], rev=true)
    global NSP10_ct_all_sort_by_ct = sort(collect(NSP10_ct_all), by = x -> x[2], rev=true)
    global NSP11_ct_all_sort_by_ct = sort(collect(NSP11_ct_all), by = x -> x[2], rev=true)
    global NSP12_ct_all_sort_by_ct = sort(collect(NSP12_ct_all), by = x -> x[2], rev=true)
    global NSP13_ct_all_sort_by_ct = sort(collect(NSP13_ct_all), by = x -> x[2], rev=true)
    global NSP14_ct_all_sort_by_ct = sort(collect(NSP14_ct_all), by = x -> x[2], rev=true)
    global NSP15_ct_all_sort_by_ct = sort(collect(NSP15_ct_all), by = x -> x[2], rev=true)
    global NSP16_ct_all_sort_by_ct = sort(collect(NSP16_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
# Removed from parameters: NSP_AA_size::Dict{String, Int}
function NSP_sub_pos_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_pos_ct_all = Dict{String, Int}()
    NSP2_pos_ct_all = Dict{String, Int}()
    NSP3_pos_ct_all = Dict{String, Int}()
    NSP4_pos_ct_all = Dict{String, Int}()
    NSP5_pos_ct_all = Dict{String, Int}()
    NSP6_pos_ct_all = Dict{String, Int}()
    NSP7_pos_ct_all = Dict{String, Int}()
    NSP8_pos_ct_all = Dict{String, Int}()
    NSP9_pos_ct_all = Dict{String, Int}()
    NSP10_pos_ct_all = Dict{String, Int}()
    NSP11_pos_ct_all = Dict{String, Int}()
    NSP12_pos_ct_all = Dict{String, Int}()
    NSP13_pos_ct_all = Dict{String, Int}()
    NSP14_pos_ct_all = Dict{String, Int}()
    NSP15_pos_ct_all = Dict{String, Int}()
    NSP16_pos_ct_all = Dict{String, Int}()
    NSP_pos_ct_all_array = [NSP1_pos_ct_all, NSP2_pos_ct_all, NSP3_pos_ct_all, NSP4_pos_ct_all, NSP5_pos_ct_all, NSP6_pos_ct_all, NSP7_pos_ct_all, NSP8_pos_ct_all, NSP9_pos_ct_all, NSP10_pos_ct_all, NSP11_pos_ct_all, NSP12_pos_ct_all, NSP13_pos_ct_all, NSP14_pos_ct_all, NSP15_pos_ct_all, NSP16_pos_ct_all]
#######################################################
    NSP1_pos_ct_all_v1 = Dict{Int,Int}()
    NSP2_pos_ct_all_v1 = Dict{Int,Int}()
    NSP3_pos_ct_all_v1 = Dict{Int,Int}()
    NSP4_pos_ct_all_v1 = Dict{Int,Int}()
    NSP5_pos_ct_all_v1 = Dict{Int,Int}()
    NSP6_pos_ct_all_v1 = Dict{Int,Int}()
    NSP7_pos_ct_all_v1 = Dict{Int,Int}()
    NSP8_pos_ct_all_v1 = Dict{Int,Int}()
    NSP9_pos_ct_all_v1 = Dict{Int,Int}()
    NSP10_pos_ct_all_v1 = Dict{Int,Int}()
    NSP11_pos_ct_all_v1 = Dict{Int,Int}()
    NSP12_pos_ct_all_v1 = Dict{Int,Int}()
    NSP13_pos_ct_all_v1 = Dict{Int,Int}()
    NSP14_pos_ct_all_v1 = Dict{Int,Int}()
    NSP15_pos_ct_all_v1 = Dict{Int,Int}()
    NSP16_pos_ct_all_v1 = Dict{Int,Int}()
    NSP_pos_ct_all_v1_array = [NSP1_pos_ct_all_v1, NSP2_pos_ct_all_v1, NSP3_pos_ct_all_v1, NSP4_pos_ct_all_v1, NSP5_pos_ct_all_v1, NSP6_pos_ct_all_v1, NSP7_pos_ct_all_v1, NSP8_pos_ct_all_v1, NSP9_pos_ct_all_v1, NSP10_pos_ct_all_v1, NSP11_pos_ct_all_v1, NSP12_pos_ct_all_v1, NSP13_pos_ct_all_v1, NSP14_pos_ct_all_v1, NSP15_pos_ct_all_v1, NSP16_pos_ct_all_v1]
#######################################################
    NSP_pos_ct_all = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_all_array)
        NSP_all_dict = NSP_pos_ct_all_array[i]
        NSP_all_dict_v1 = NSP_pos_ct_all_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_all_dict[NSP_pos] = 0
            NSP_pos_ct_all[NSP_pos] = 0
            NSP_all_dict_v1[j] = 0
        end
    end
############################################################
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
##################################################################################################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = parse(Int, split(sub, ":")[2][2:end-1])
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
###################################################################################################################################
    for i in 1:length(NSP_pos_ct_all_array)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_all_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_all_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_all_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = parse(Int, split(sub, ":")[2][2:end-1])
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct_all[sub_pos] += ct
        end
    end
##############################################################
    global NSP1_pos_ct_all_sort_by_pos = sort(collect(NSP1_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_all_sort_by_pos = sort(collect(NSP2_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_all_sort_by_pos = sort(collect(NSP3_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_all_sort_by_pos = sort(collect(NSP4_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_all_sort_by_pos = sort(collect(NSP5_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_all_sort_by_pos = sort(collect(NSP6_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_all_sort_by_pos = sort(collect(NSP7_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_all_sort_by_pos = sort(collect(NSP8_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_all_sort_by_pos = sort(collect(NSP9_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_all_sort_by_pos = sort(collect(NSP10_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_all_sort_by_pos = sort(collect(NSP11_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_all_sort_by_pos = sort(collect(NSP12_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_all_sort_by_pos = sort(collect(NSP13_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_all_sort_by_pos = sort(collect(NSP14_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_all_sort_by_pos = sort(collect(NSP15_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_all_sort_by_pos = sort(collect(NSP16_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
#############################
    global NSP1_pos_ct_all_sort_by_ct = sort(collect(NSP1_pos_ct_all), by = x -> x[2], rev=true)
    global NSP2_pos_ct_all_sort_by_ct = sort(collect(NSP2_pos_ct_all), by = x -> x[2], rev=true)
    global NSP3_pos_ct_all_sort_by_ct = sort(collect(NSP3_pos_ct_all), by = x -> x[2], rev=true)
    global NSP4_pos_ct_all_sort_by_ct = sort(collect(NSP4_pos_ct_all), by = x -> x[2], rev=true)
    global NSP5_pos_ct_all_sort_by_ct = sort(collect(NSP5_pos_ct_all), by = x -> x[2], rev=true)
    global NSP6_pos_ct_all_sort_by_ct = sort(collect(NSP6_pos_ct_all), by = x -> x[2], rev=true)
    global NSP7_pos_ct_all_sort_by_ct = sort(collect(NSP7_pos_ct_all), by = x -> x[2], rev=true)
    global NSP8_pos_ct_all_sort_by_ct = sort(collect(NSP8_pos_ct_all), by = x -> x[2], rev=true)
    global NSP9_pos_ct_all_sort_by_ct = sort(collect(NSP9_pos_ct_all), by = x -> x[2], rev=true)
    global NSP10_pos_ct_all_sort_by_ct = sort(collect(NSP10_pos_ct_all), by = x -> x[2], rev=true)
    global NSP11_pos_ct_all_sort_by_ct = sort(collect(NSP11_pos_ct_all), by = x -> x[2], rev=true)
    global NSP12_pos_ct_all_sort_by_ct = sort(collect(NSP12_pos_ct_all), by = x -> x[2], rev=true)
    global NSP13_pos_ct_all_sort_by_ct = sort(collect(NSP13_pos_ct_all), by = x -> x[2], rev=true)
    global NSP14_pos_ct_all_sort_by_ct = sort(collect(NSP14_pos_ct_all), by = x -> x[2], rev=true)
    global NSP15_pos_ct_all_sort_by_ct = sort(collect(NSP15_pos_ct_all), by = x -> x[2], rev=true)
    global NSP16_pos_ct_all_sort_by_ct = sort(collect(NSP16_pos_ct_all), by = x -> x[2], rev=true)
###############################
    global NSP1_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_all_v1), by = x -> x[1])
    global NSP2_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_all_v1), by = x -> x[1])
    global NSP3_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_all_v1), by = x -> x[1])
    global NSP4_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_all_v1), by = x -> x[1])
    global NSP5_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_all_v1), by = x -> x[1])
    global NSP6_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_all_v1), by = x -> x[1])
    global NSP7_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_all_v1), by = x -> x[1])
    global NSP8_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_all_v1), by = x -> x[1])
    global NSP9_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_all_v1), by = x -> x[1])
    global NSP10_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_all_v1), by = x -> x[1])
    global NSP11_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_all_v1), by = x -> x[1])
    global NSP12_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_all_v1), by = x -> x[1])
    global NSP13_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_all_v1), by = x -> x[1])
    global NSP14_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_all_v1), by = x -> x[1])
    global NSP15_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_all_v1), by = x -> x[1])
    global NSP16_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_all_v1), by = x -> x[1])
end
######################################################################################################################################
runtime = time() - start_chr_load_fx
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime); print("\n"^1)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_07__640PM

Runtime v0 = 0.9297239780426025 seconds
Runtime v2 = 0 hr, 0 min, 00.93 sec



In [106]:
### Execute Load Chronic Dicts DQ Fx | 2026_02_22 version | chronics_2026_02_01__6153seq | 95—8—6 | Runtime: 1 min, 00.26 sec | 2026_2_13
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
chr_dict_load_start = time()

abs_min_mut_thresh = 8
DQ_mut_thresh = 10
date_pct_DQ_thresh = 95

rep_thresh = 5
revs_thresh = 6
HQCS_qc_str = HQCS_qc_string
EPCI_qc_str = "$(abs_min_mut_thresh)_$(DQ_mut_thresh)_$(date_pct_DQ_thresh)"

print_ct_thresh = 1
#date = "2026_02_21"
date = "2026_02_20"
ndjson_name = "chronics_2026_02_01__6153seq_v2"
folder_name = "$(date)__$(ndjson_name)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"

println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_str = $(HQCS_qc_str)")

chronic_load_dicts2_DQ(ndjson_name, folder_name, date, rep_thresh, revs_thresh, print_ct_thresh, DQ_mut_thresh, date_pct_DQ_thresh, abs_min_mut_thresh)
# ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Float64, abs_min_mut_thresh::Int
######################################################################################################################################
seq_AA_muts["EPI_ISL_8725398"] = Set{String}();             seq_AA_muts["EPI_ISL_949208"] = Set{String}();   seq_pango["EPI_ISL_6281381"] = "AY.4"
######################################################################################################################################
### Create  all_unique_chr_seqs  &  all_unique_chr_seqs_set 
rep_seq_grps_maxmut_seqs_arr = collect(values(rep_seq_grps_maxmut_seqs))
all_unique_chr_seqs = union(rep_seq_grps_maxmut_seqs_arr, non_rep_seqs)
all_unique_chr_seqs_ct = length(all_unique_chr_seqs)
all_unique_chr_seqs_set = Set{String}()
for seq in all_unique_chr_seqs
    push!(all_unique_chr_seqs_set, seq)
end
total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
println("length(all_unique_chr_seqs) = $(length(all_unique_chr_seqs))")
println("length(all_unique_chr_seqs_set)  $(length(all_unique_chr_seqs_set))"); print("\n"^2)
########################################################################################################################################
for seq in all_unique_chr_seqs_set 
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
#############################################################################################################################################################################
#### Removing clearly artifactual reversions that occur directly next to dropout (and are almost always from labs that frequently have this problem)
FCS_fake_revs = Set(["S:K679N", "S:H681P", "S:R681P"])
FCS_fake_revs_pos = Set(["S:679", "S:681"])
for seq in all_qualifying_seqs_set
    if "S:683" in seq_unknown_AA[seq] || "S:676" in seq_unknown_AA[seq]
        if seq in all_unique_chr_seqs_set
            for mut in FCS_fake_revs
                if mut in seq_AA_muts[seq]
                    mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
                    AA_muts_ct[mut] -= 1
                    AA_muts_ct_no_dels[mut] -= 1
                    AA_muts_ct_pos_only[mut_pos] -= 1
                    AA_muts_ct_pos_only_no_dels[mut_pos] -= 1
                end
            end
        end
        setdiff!(seq_AA_muts[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_no_dels[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_pos_only[seq], FCS_fake_revs_pos)
        setdiff!(seq_AA_muts_pos_only_no_dels[seq], FCS_fake_revs_pos)
    end
end
######################################################################################################################################
if haskey(AA_muts_ct_pos_only_no_dels, "")
    delete!(AA_muts_ct_pos_only_no_dels, "")
end
if haskey(AA_muts_ct_no_dels, "")
    delete!(AA_muts_ct_no_dels, "")
end
######################################################################################################################################
## Deletions royally screw up any attempt to find correlated mutations since they frequently occur in bunches, which are, of course, highly correlated, but only in the most trivial way. 
## The code below is a preliminary attempt to include common deletions by only allowing the inclusion of a single AA deletion, though most contain multiple consecutive 
## AA deletions. Also included are mutations that only occur as 1-AA deletions, such as E:I13- and E:V14-. It can be removed or inserted without substantially changing the results 
deletion_exceptions = list_to_strings_set("E:I13-, E:V14-, M:G6-, S:C15-, S:C136-, S:D138-, S:A243-, S:F371-, S:F375-, S:V483-, S:A484-, S:E484-, S:D1257-, ORF3a:T14-, ORF3a:V255-, ORF3a:V256-, ORF3a:N257-, ORF7a:I103-, ORF7a:*122-, ORF1a:N2081-, ORF1a:D2136-, ORF1a:S4398-")
pos_only_deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
del_to_del_pos = Dict{String, String}("E:I13-"=>"E:13", "E:V14-"=>"E:14", "M:G6-"=>"M:6", "S:C15-"=>"S:15", "S:C136-"=>"S:136", "S:D138-"=>"S:138", "S:A243-"=>"S:243", "S:F371-"=>"S:371", "S:F375-"=>"S:375", "S:V483-"=>"S:483", "S:A484-"=>"S:484", "S:E484-"=>"S:484", "S:D1257-"=>"S:1257", "ORF3a:T14-"=>"ORF3a:14", "ORF3a:V255-"=>"ORF3a:255", "ORF3a:V256-"=>"ORF3a:256", "ORF3a:N257-"=>"ORF3a:257", "ORF7a:I103-"=>"ORF7a:103", "ORF7a:*122-"=>"ORF7a:122", "ORF1a:N2081-"=>"ORF1a:2081", "ORF1a:D2136-"=>"ORF1a:2136", "ORF1a:S4398-"=>"ORF1a:4398")
pos_only_exception_count::Int = 0
for seq in all_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            del_pos = del_to_del_pos[del]
            push!(seq_AA_muts_pos_only_no_dels[seq], del_pos)
            pos_only_exception_count += 1
            pos_only_deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    del_pos = del_to_del_pos[del]
    if !haskey(AA_muts_ct_pos_only_no_dels, del_pos)
        AA_muts_ct_pos_only_no_dels[del_pos] = 0
    end
    AA_muts_ct_pos_only_no_dels[del_pos] += pos_only_deletion_exception_ct_dict[del]
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[del_pos] = 999
end
######################################################################################################################################
deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
exception_count::Int = 0
for seq in all_unique_chr_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            push!(seq_AA_muts_no_dels[seq], del)
            exception_count += 1
            deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    AA_muts_ct_no_dels[del] = get(AA_muts_ct_no_dels, del, 0)
    AA_muts_ct_no_dels[del] += deletion_exception_ct_dict[del]
    AA_muts_ct_chr_all_ratio[del] = 999
end
######################################################################################################################################
blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
for blnk in blank_mutstrings
    AA_muts_ct_chr_all_ratio[blnk] = 0
    AA_muts_ct_no_dels_chr_all_ratio[blnk] = 0
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[blnk] = 0
end
#######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
######################################################################################################################################
mp_AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_sortKey2(n) = (n[2], mp_AA_ct_sortKey1(n))
#########
mp_AA_gene_pos_only_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_pos_only_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_pos_only_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_pos_only_sortKey2(n) = (n[2], mp_AA_ct_pos_only_sortKey1(n))
#################################
function mp_AApos_sort_key(n)
    if haskey(aa_pos_comprehensive_dict, n)
        return (aa_pos_comprehensive_dict[n], n)
    else
        return (9999, n)
    end
end
function sel_muts_pt1_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_sortKey_2(string(split(n, ", ")[1]))
    end
end
##################################################################
println("Number of Deletion Exceptions Made = $(exception_count)")
deletion_exception_ct_dict_sort = sort(collect(deletion_exception_ct_dict), by = x -> x[2])
for del__ct in deletion_exception_ct_dict_sort
    del = del__ct[1]
    ct = del__ct[2]
    println("$(del) = $(ct)")
end
###########################################################################################
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seqAAlen = length(AAsubvec)
    seq_privAA_len[seq] = seqAAlen
end
############################################################################################################################################################################
###########################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
############################################################################################################################################################################
corrected_count = 0
noncorrected_ct = 0
for seq in all_seqs_set
    if seq_date_index[seq] > 4000
#        println("seq_date_tuple = $(seq_date_tuple[seq]); seq_date_tuple[seq][1] = $(seq_date_tuple[seq][1]); seq_date_tuple[seq][2] = $(seq_date_tuple[seq][2]); seq_date_tuple[seq][3] = $(seq_date_tuple[seq][3])")
        if seq_date_tuple[seq][1] ≠ 0 && seq_date_tuple[seq][2] ≠ 0
            new_date_tuple = (seq_date_tuple[seq][1], seq_date_tuple[seq][2], 15)
            seq_date_tuple[seq] = new_date_tuple
            seq_date_index[seq] = tuple_to_index[new_date_tuple]
            corrected_count += 1
        elseif seq_date_tuple[seq][2] == 0
            noncorrected_ct += 1
        end
    end
end
total_baddie_ct = corrected_count + noncorrected_ct
println("seq_date_index corrected = $(corrected_count)")
println("seq_date_index not corrected = $(noncorrected_ct)")
println("total_baddie_ct = $(total_baddie_ct)")
######################################################################################################################################################
######################################################################################################################################  
chr_load_runtime = time() - chr_dict_load_start
chr_load_runtime_rd = round(digits=1, chr_load_runtime)
chr_load_hms1, chr_load_hms2 = seconds_to_hrs_min_sec(chr_load_runtime)
println("Total Time to Load Chronic Dictionaries = $(chr_load_runtime_rd)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms1)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms2)"); print("\n"^1)
println("Finished!")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_07__640PM
EPCI_qc_str = 8_10_95 | HQCS_qc_str = 5_1_5
2026_03_07__640PM
length(all_unique_chr_seqs) = 3297
length(all_unique_chr_seqs_set)  3297


Number of Deletion Exceptions Made = 838
S:E484- = 1
ORF1a:D2136- = 2
S:F371- = 4
ORF1a:N2081- = 6
ORF1a:S4398- = 7
E:I13- = 8
S:D1257- = 8
S:V483- = 9
ORF7a:I103- = 10
M:G6- = 12
ORF7a:*122- = 16
S:A484- = 23
S:F375- = 24
S:C136- = 32
ORF3a:V256- = 38
ORF3a:T14- = 40
ORF3a:N257- = 43
ORF3a:V255- = 51
S:D138- = 89
E:V14- = 90
S:C15- = 105
S:A243- = 220
seq_date_index corrected = 70
seq_date_index not corrected = 52
total_baddie_ct = 122
Total Time to Load Chronic Dictionaries = 16.6
Total Time to Load Chronic Dictionaries = 0:00:16.57
Total Time to Load Chronic Dictionaries = 0 hr, 0 min, 16.57 sec

Finished!
2026_03_07__641PM



In [93]:
### NEW | 2026_02_22 | Many, many functions (mixed_nuc, nuc_to_AA, etc) | Runtime: 1 min 33 sec 
### Timing Template
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
start = time()
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
function list_to_strings_no_spaces(list::String)
    string_vec = string.(split(list, ","))
    return string_vec
end
######################################################################################################################################
function stringlist_to_strings_nonEPI(txt::String)
    arr_of_strings = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        push!(arr_of_strings, seq)
    end
    sort_arr_of_strings = sort(collect(arr_of_strings), by = x -> nuc_mut_int_comprehensive_dict[x])  
    return sort_arr_of_strings
end
######################################################################################################################################
function list_to_string_array(txt::String) # similar to stringlist_to_strings but not for EPIs
    no_newlines = replace(txt, "\n" =>" ")
    string_array = string.(split(no_newlines, ", "))
    return string_array
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function get_ref_pango_nucseq_and_geneseqs(ref_pango::String)
    ref_seq = "..."
    refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
    refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
    refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
    refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
    refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
    refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
    refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
    refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
    refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
    refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
    refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
    refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
    if ref_pango ≠ "Wuhan"
        if haskey(nuc_genome_pango_dict, ref_pango)
            ref_seq = nuc_genome_pango_dict[ref_pango]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        end
##########################################################################################
        if haskey(gene_AA_pango_dict, ref_pango)
            refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
            refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
            refAA_S = gene_AA_pango_dict[ref_pango]["S"]
            refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
            refAA_E = gene_AA_pango_dict[ref_pango]["E"]
            refAA_M = gene_AA_pango_dict[ref_pango]["M"]
            refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
            refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
            refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
            refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
            refAA_N = gene_AA_pango_dict[ref_pango]["N"]
            refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        end
    end
    return ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_nucs_filter(mut_arr)
    mixed_nuc_arr = Vector{String}()
    mixed_nuc_res_list = Set(["Y", "R", "K", "M", "W", "S"])
    for mut in mut_arr
        if string(mut[end]) in mixed_nuc_res_list
            push!(mixed_nuc_arr, mut)
        end
    end
    return mixed_nuc_arr
end
#####################################################################################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
#####################################################################################################################################
function mixed2nuc(mix_mut::String)
    nuc_mut = mix_mut
    qrynuc = qry_nuc_comprehensive_dict[mix_mut]
    refnuc = ref_nuc_comprehensive_dict[mix_mut]
    pos_str = nuc_mut_int_string_comprehensive_dict[mix_mut]
    ref_n_pos = refnuc*pos_str
    if length(mix_mut) ≥ 4
        if qrynuc == "T"
            nuc_mut = mix_mut
        elseif qrynuc == "C"
            nuc_mut = mix_mut
        elseif qrynuc == "A"
            nuc_mut = mix_mut
        elseif qrynuc == "G"
            nuc_mut = mix_mut
        elseif qrynuc == "N"
            nuc_mut = mix_mut
        end   
        if refnuc == "T"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if refnuc == "A"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if refnuc == "G"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return nuc_mut
end
######################################################################################################################################
function muts_to_strings(mut_list_in_string_form::String)
    mut_arr = split(mut_list_in_string_form, ", ")
    mut_array = Vector{String}()
    for mut in mut_arr
        if string(mut[end]) ≠ "-"
            mutstr = string(mut)
            push!(mut_array, mutstr)
        end
    end
    sortKey(n) = (length(n), nuc_mut_int_comprehensive_dict[n])  ## Fx ##
    mut_array_sort = sort(mut_array, by = x -> sortKey(x))
#    mixed_mut_arr = mixed_nucs_filter(mut_arr)
    return mut_array_sort
end
######################################################################################################################################
function mixed_mut_to_regular_mut(mixed_nuc_muts)            ### New, 2025-1-26 (entire function)
    ct = 0
    mixed_regular_muts = Vector{String}()
    for i in 1:length(mixed_nuc_muts)
        mut = mixed_nuc_muts[i]
        qrynuc = qry_nuc_comprehensive_dict[mut]
        refnuc = ref_nuc_comprehensive_dict[mut]
        pos_str = nuc_mut_int_string_comprehensive_dict[mut]
        ref_n_pos = refnuc*pos_str
        if refnuc == "T"
            if qrynuc == "Y"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
        if refnuc == "A"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "G"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
    end
    return mixed_regular_muts
end
######################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#######################################################################################################################################
function nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts_filtered = filter(!isempty, muts)
#    all_muts_sort = sort(muts_filtered, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int,Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int,Int}()
    nuc_gene_num_9b = Dict{Int,Int}()
    nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
    nonsynonymous_nuc_to_context_dict = Dict{String, String}()
    all_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_context_dict = Dict{String, String}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
################################################        
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nuc_pos in gene_nuc_arr[i]
            nuc_gene_num[nuc_pos] = i-1
        end
    end
    for nuc_pos in gene_nuc_arr[end]
        nuc_gene_num_9b[nuc_pos] = 11
    end

    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
    
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                                          ### FUNCTION #####################
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)                  ### FUNCTION #####################
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                                            ### FUNCTION #####################
    nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA   ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
    
    nuc_codon_pos_dict = Dict{Int,Int}()
    for nuc_pos in coding_ranges
        gene_number = nuc_gene_num[nuc_pos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict[nuc_pos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int,Int}()
    for nuc_pos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nuc_pos] = codon_num
    end
#######################################################################################################################
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ",")[1])
            mut2 = string(split(mut, ",")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>6), muts)
        end
    end
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
            mut = mixed2nuc(nuc_mut)  
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end
            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            push!(AA_mut_set, AA_mut)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            all_nuc_to_AA_dict[mut] = AA_mut
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
                nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
#                push!(nonsynonymous_nuc_muts, mut)
            end
            all_nuc_to_context_dict[mut] = ref2qry_context
###################################
            for nuc_mut2 in muts
                mut2 = mixed2nuc(nuc_mut2)
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        push!(AA_mut_set, AA_mut2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        all_nuc_to_AA_dict[mut2] = AA_mut2
                        all_nuc_to_AA_dict[mut] = AA_mut2
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context 
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut2] = ref2qry_context
                            nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
                        end
                        all_nuc_to_context_dict[mut] = ref2qry_context
                        all_nuc_to_context_dict[mut2] = ref2qry_context
                    end
                end
            end
        else                  
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
            ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            full_nc_context = ref_nc_nuc_context*"|"*qry_nc_nuc_context
            noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
            for (start_end, place) in noncoding_range_dict
                frst = start_end[1]
                last = start_end[2]
                if npos ≥ frst && npos ≤ last
                    noncoding_to_noncoding_region_dict[nuc_mut] = place
                    mut_vec = [nuc_mut, place]
                    push!(noncoding_nuc_vector, nuc_mut)
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
            mut_9b = mixed2nuc(nuc_mut)
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])  *string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            push!(AA_mut_set, AA_mut_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            end
            if refAA_9b ≠ qryAA_9b
                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
#                push!(nonsynonymous_nuc_muts, mut_9b)
            end
            all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
###################################
            for nuc_mut2_9b in muts
                mut2_9b = mixed2nuc(nuc_mut2_9b)
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        push!(AA_mut_set, AA_mut2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        end
                        all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        all_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                    end
                end
            end
        end
    end
#################################################################################
    mut_vec_dict = Dict{String, Vector{String}}()
    for gene in gene_print_array
        mut_vec_dict[gene] = Vector{String}()
    end
    for mut in AA_mut_set
        jeen = aa_gene_comprehensive_dict[mut]
        mut_only = string(split(mut, ":")[2])
        push!(mut_vec_dict[jeen], mut_only) 
    end
    for gene in keys(mut_vec_dict)
        if !isempty(mut_vec_dict[gene])
            sort!(mut_vec_dict[gene], by = x -> x[2:end-1])
        end
    end
#####################################################################################################################################
    aasortkeyhere(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
    AA_sort = sort(collect(AA_mut_set), by = x -> aasortkeyhere(x))
    AA_sort2 = Vector{String}()
    for mut in AA_sort
        refAA = refAA_comprehensive_dict[mut]
        qryAA = qryAA_comprehensive_dict[mut]
        if !(refAA == qryAA)
            push!(AA_sort2, mut)
        end
    end
#    print("\n"^2)
#    println("##################### Amino Acid Mutations ######################")
#    for i in 1:length(AA_sort2)
#        mut = AA_sort2[i]
#        gene = aa_gene_comprehensive_dict[mut]
#        non_gene = string(split(mut, ":")[2])
#        if i == 1
#            print("               ", AA_sort2[i])
#        elseif i > 1
#            lastmut = AA_sort2[i-1]
#            last_gene = aa_gene_comprehensive_dict[lastmut]
#            last_non_gene = string(split(lastmut, ":")[2])
#            if gene == last_gene
#                print(", $(non_gene)")
#            else
#                println()
#                print("               $(mut)")
#            end
#        end
#    end
#    print("\n"^2)
#####################################################################################################################################
    all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    all_nuc_to_context_dict_sort = sort(collect(all_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    nonsynonymous_nuc_to_context_dict_sort = sort(collect(nonsynonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
####################################################################################################################################
    nc_len = length(noncoding_to_noncoding_region_dict_sort)
#    println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
    if !isempty(noncoding_to_noncoding_region_dict_sort)
        for i in 1:length(noncoding_to_noncoding_region_dict_sort)
            nc_nuc = noncoding_to_noncoding_region_dict_sort[i][1]
            nc_nuc_pad = rpad(nc_nuc, 7)
            nc_region = noncoding_to_noncoding_region_dict_sort[i][2]
            nc_region_len = length(nc_region)
            ncpad1 = (13 - nc_region_len)÷2
            ncpad12 = " "^ncpad1*nc_region
            nc_region_pad2 = rpad(ncpad12, 13)
            nc_context = noncoding_nuc_to_context_dict_sort[i][2]
            premut_context = ""
            postmut_context = ""
            context = split(nc_context, "-->")
            if !isempty(context)
                if length(context) >1
                    premut_context = split(nc_context, "-->")[1]
                    postmut_context = split(nc_context, "-->")[2]
                end
            end    
            postpad = lpad(postmut_context, 39)
#            println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#            println(postpad)
#                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#                println(g, postpad)
        end
#        println("#"^94); println() 
#            println(g, "#"^94); println(g)
    end
######################################################################################################
    total_syn = length(synonymous_nuc_to_AA_dict_sort)
#    println("SYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(synonymous_nuc_to_AA_dict_sort)
        synnuc = synonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = synonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = synonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
#################################################################
    total_syn = length(nonsynonymous_nuc_to_AA_dict_sort)
#    println("NONSYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(nonsynonymous_nuc_to_AA_dict_sort)
        synnuc = nonsynonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = nonsynonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = nonsynonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
###################################################################
    nonsynonymous_nuc_total = length(nonsynonymous_nuc_sort)
#    println("        Total Number of Non-synonymous Nuc Muts = $(nonsynonymous_nuc_total)")
    nonsynonymous_nuc_sort_join = join(nonsynonymous_nuc_sort, ", ")
#    println("################ Nonsynonymous Nuc Mutations ################")
#    println(nonsynonymous_nuc_sort_join)
#    print("\n"^1)
    for i in 1:length(nonsynonymous_nuc_sort)
#        println("               $(nonsynonymous_nuc_to_AA_dict_sort[i][1]) | $(nonsynonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        println("                 $(premut_nucseq)")
#        println("                 $(postmut_nucseq)")
    end
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mixed_muts = mixed_nucs_filter(mut_strings)             ### New, 2025-1-26
    mixed_muts_regular = mixed_mut_to_regular_mut(mixed_muts)    ### New, 2025-1-26
    ct = 0
    total_mixed_muts = length(mixed_muts)
#    println("Total Mixed Nucs = $(total_mixed_muts)")
#    print("\n"^2)
    for i in 1:length(mixed_muts)
        if ct == 0
#            print(mixed_muts[i])
            ct = 1
        else
#            print(", ", mixed_muts[i])
        end
    end
    ct2 = 0
#    print("\n"^2)
    for i in 1:length(mixed_muts_regular)
        if ct2 == 0
#            print(mixed_muts_regular[i])
            ct2 = 1
#        else
#            print(", ", mixed_muts_regular[i])
        end
    end
#    println()
    syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort = nuc_to_AA(ref_pango, mixed_muts_regular)
    AA__AAprintlen__vec = []
    nonsynnuc__nonsynprintlen__vec = []
    for nuc___AA in nonsyn_nuc_to_AA_dict_sort
        nucmut = nuc___AA[1]
        nucmutlen = length(nucmut)
        AAmut = nuc___AA[2]
        AAmutlen = length(AAmut)
        push!(AA__AAprintlen__vec, [AAmut, AAmutlen])
        push!(nonsynnuc__nonsynprintlen__vec, [nucmut, nucmutlen])
    end
    aa_pad_vec = String[]
    nuc_pad_vec = String[]
    for i in 1:length(AA__AAprintlen__vec)
        aa = AA__AAprintlen__vec[i][1]
        nuc = nonsynnuc__nonsynprintlen__vec[i][1]
        aapad = AA__AAprintlen__vec[i][2]
        nucpad = nonsynnuc__nonsynprintlen__vec[i][2]
        pads = [nucpad, aapad]
        pad = maximum(pads)
        push!(aa_pad_vec, rpad(aa, pad))
        push!(nuc_pad_vec, rpad(nuc, pad))
    end
    aapad_join = join(aa_pad_vec, ", ")
    nucpad_join = join(nuc_pad_vec, ", ")
#    println("\n"^1)
#    println(aapad_join)
#    println(nucpad_join)
#    println("\n"^1)
    return syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
######################################################################################################################################
function count_nuc_mut_types(mut_strings::Vector{String})
    mut_types_arr = ["TC", "TA", "TG", "CT", "CA", "CG", "AT", "AC", "AG", "GT", "GC", "GA"]
    mut_type_cts = Dict{String, Int}(mut_nuc_type=>0 for mut_nuc_type in mut_types_arr)
    for nuc_mut in mut_strings
        ref = ref_nuc_comprehensive_dict[nuc_mut]
        qry = ref_nuc_comprehensive_dict[nuc_mut]
        if ref == "T"
            if qry == "C"
                mut_type_cts["TC"] += 1
            elseif qry == "A"
                mut_type_cts["TA"] += 1
            elseif qry == "G"
                mut_type_cts["TG"] += 1
            end
        end
        if ref == "C"
            if qry == "T"
                mut_type_cts["CT"] += 1
            elseif qry == "A"
                mut_type_cts["CA"] += 1
            elseif qry == "G"
                mut_type_cts["CG"] += 1
            end
        end 
        if ref == "A"
            if qry == "T"
                mut_type_cts["AT"] += 1
            elseif qry == "C"
                mut_type_cts["AC"] += 1
            elseif qry == "G"
                mut_type_cts["AG"] += 1
            end
        end   
        if ref == "G"
            if qry == "T"
                mut_type_cts["GT"] += 1
            elseif qry == "C"
                mut_type_cts["GC"] += 1
            elseif qry == "A"
                mut_type_cts["GA"] += 1
            end
        end
    end
    mut_type_cts_sort_by_type = sort(collect(mut_type_cts), by = x -> x[1])
    mut_type_cts_sort_by_count = sort(collect(mut_type_cts), by = x -> x[2], rev=true)
    return mut_type_cts_sort_by_count
end
######################################################################################################################################
function AA_triple(pos::Int, rem0::BitSet, rem1::BitSet, rem2::BitSet, mut::String, ref_pango::String)
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ref_triple = ""
    qry_triple = ""
    if pos in rem0 && pos%3 == 0
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem1 && pos%3 == 1
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem2 && pos%3 == 2
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem0 && pos%3 == 1
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem1 && pos%3 == 2
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem2 && pos%3 == 0
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem0 && pos%3 == 2
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem1 && pos%3 == 0
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem2 && pos%3 == 1
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    end
    return ref_triple, qry_triple
end
######################################################################################################################################
### Fx: SIMPLER_nuc_to_AA (no context)
function SIMPLER_nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts = filter(!isempty, muts)
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ###############################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int,Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int,Int}()
        nuc_gene_num_9b = Dict{Int,Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
        all_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int,Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int,Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
        N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut_set = Set{String}()
        AA_mut = ""
        for nuc_mut in muts
            pos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos in coding_ranges
                mut = mixed2nuc(nuc_mut)  
                gene_number = gene_num(mut)
                ref_triple = ""
                qry_triple = ""
                if nuc_codon_pos_dict[pos] == 1
                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                elseif nuc_codon_pos_dict[pos] == 2
                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                elseif nuc_codon_pos_dict[pos] == 3
                    ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                    qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                end
                refAA = AA_triplets[ref_triple]
                qryAA = AA_triplets[qry_triple]
                AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                push!(AA_mut_set, AA_mut)
                all_nuc_to_AA_dict[mut] = AA_mut
                if refAA == qryAA && !(pos in rem9b)
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                else
                    nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
#                    push!(nonsynonymous_nuc_muts, mut)
                end
###################################
                for nuc_mut2 in muts
                    mut2 = mixed2nuc(nuc_mut2)
                    pos2 = nuc_mut_int_comprehensive_dict[mut2]
                    if pos2 in coding_ranges
                        gene_number2 = gene_num(mut2)
                        if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                            if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                            end
                            refAA2 = AA_triplets[ref_triple]
                            qryAA2 = AA_triplets[qry_triple]
                            AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                            push!(AA_mut_set, AA_mut2)
                            delete!(AA_mut_set, AA_mut)
                            all_nuc_to_AA_dict[mut2] = AA_mut2
                            all_nuc_to_AA_dict[mut] = AA_mut2
                            if refAA2 == qryAA2 && !(pos2 in rem9b)
                                synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            end
                        end
                    end
                end
            elseif !isempty(nuc_mut)
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if pos ≥ frst && pos ≤ last
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                push!(AA_mut_set, AA_mut_9b)
                all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
                if refAA_9b ≠ qryAA_9b
                    nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
#                    push!(nonsynonymous_nuc_muts, mut_9b)
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            push!(AA_mut_set, AA_mut2_9b)
                            delete!(AA_mut_set, AA_mut_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
        AA_sort2 = Vector{String}()
        for aa in AA_sort
            if !(refAA_comprehensive_dict[aa] == qryAA_comprehensive_dict)
                push!(AA_sort2, aa)
            end
        end
#        print("\n"^1)
#        ct = 0
#        println("############# AA Mutations #############")
#        for i in 1:length(AA_sort2)
#            mut = AA_sort2[i]
#            gene = string(split(mut, ":")[1])
#            non_gene = string(split(mut, ":")[2])
#            if i == 1
#                print("        ", AA_sort2[i])
#            elseif i > 1
#                lastmut = AA_sort2[i-1]
#                last_gene = string(split(lastmut, ":")[1])
#                last_non_gene = string(split(lastmut, ":")[2])
#                if gene == last_gene
#                    print(", $(non_gene)")
#                else
#                    print("        $(mut)")
#                end
#            end
#        end
#####################################################################################################################################
        all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
        for i in 1:length(synonymous_nuc_sort)
            nucpad = rpad(synonymous_nuc_to_AA_dict_sort[i][1], 10)
        end
#        return synonymous_nuc_to_AA_dict, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
        return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
    end
    if isempty(muts)
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
###########################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
function pango_minus_1_fx(pango::String)
    if '.' in pango
        dot_ct = count(".", pango)
        dotsplits = split(pango, ".")
        minus_1 = join(dotsplits(1:dot_ct-1), ".")
        return minus_1
    else
        return pango
    end
end
############################################################################################################################################################################
### Fx: SIMPLER_syn_noncoding_nonsyn_nuc (no context)
function SIMPLER_syn_noncoding_nuc(ref_pango::String, muts::Set{String})
    B_1_1_list = ["B.1.1.53", "B.1.1.273"]
    if ref_pango in B_1_1_list
        ref_pango = "B.1.1"
    end
    if ref_pango == "XBB.1.5.82"
         ref_pango = "XBB.1.5"
    end
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
###################################################################################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int,Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int,Int}()
        nuc_gene_num_9b = Dict{Int,Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int,Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int,Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut = ""
        for nuc_mut in muts
            if !isempty(muts) && !isempty(ref_seq) && nuc_mut ≠ ""
                pos = nuc_mut_int_comprehensive_dict[nuc_mut]
                if pos in coding_ranges
                    mut = mixed2nuc(nuc_mut)
                    if isempty(mut)
                        println(nuc_mut)
                        println(mut)
                        println("$(pango)")
                    end
                    try
                        nuc_mut_int_comprehensive_dict[mut]
                    catch e
                        println("Problematic mutation string: ", repr(mut))
                        println("Length: ", length(mut))
                        println("Extracted substring: ", repr(mut[2:end-1]))
                        rethrow(e)
                    end
                    gene_number = nuc_gene_num[pos]
                    ref_triple = ""
                    qry_triple = ""
                    if nuc_codon_pos_dict[pos] == 1
                        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    elseif nuc_codon_pos_dict[pos] == 2
                        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    elseif nuc_codon_pos_dict[pos] == 3
                        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                    end
                    refAA = AA_triplets[ref_triple]
                    qryAA = AA_triplets[qry_triple]
                    AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                    if refAA == qryAA && !(pos in rem9b)
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    end
###################################
                    for nuc_mut2 in muts
                        mut2 = mixed2nuc(nuc_mut2)
                        pos2 = nuc_mut_int_comprehensive_dict[mut2]
                        if pos2 in coding_ranges
                            gene_number2 = gene_num(mut2)
                            if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                                if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                                elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                                end
                                refAA2 = AA_triplets[ref_triple]
                                qryAA2 = AA_triplets[qry_triple]
                                AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                                if refAA2 == qryAA2 && !(pos2 in rem9b)
                                    synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                end
                            end
                        end
                    end
                else
                    qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                    for (start_end, place) in noncoding_range_dict
                        frst = start_end[1]
                        last = start_end[2]
                        if pos ≥ frst && pos ≤ last
                            mut_vec = [nuc_mut, place]
                            push!(noncoding_nuc_vector, nuc_mut)
                        end
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b && !isempty(ref_seq) && nuc_mut ≠ ""
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
#        for i in 1:length(synonymous_nuc_sort)
#            nucpad = rpad(synonymous_nuc_to_AA_dict_sort[i][1], 10)
#        end
        return synonymous_nuc_sort, noncoding_nuc_vector_sort
    else
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
##############################################################################################################################
###################################################################################################################
index_to_date_str = Dict{Int, String}()
date_str_to_index = Dict{String, Int}()
function convert_date_to_date_index(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    date_index = tuple_to_index[date_tuple]
    return date_index
end
print("\n"^1)
println("Done Loading Functions (line #1618 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##############################################################################################################################################
##############################################################################################################################################
for tuple in keys(tuple_to_index)
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    date_str_to_index[date_string] = convert_date_to_date_index(date_string)
end
for (index, tuple) in index_to_tuple
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    index_to_date_str[index] = date_string
end
println("Done Making date_str_to_index & index_to_date_str (line #1641 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################
date_to_tuple = Dict{String, Tuple{Int,Int, Int}}()
tuple_to_date = Dict{Tuple{Int,Int, Int}, String}()
function convert_date_to_date_tuple(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    return date_tuple
end
for date in keys(date_str_to_index)
    date_to_tuple[date] = convert_date_to_date_tuple(date)
end
for (date, date_tuple) in date_to_tuple
    tuple_to_date[date_tuple] = date
end
println("Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################################################################
function find_X_pct_date(clade_pango::String, pct::Float64, cp_date_cumul_dict::Dict{String, Dict{Int,Int}}, cp_total_dict::Dict{String, Int})
    cp_total = cp_total_dict[clade_pango]
    pct_date_index = 0
    pct_date_tuple = nothing
    for date_index in 1:2500
        cumulative_ct = cp_date_cumul_dict[clade_pango][date_index]
        if 100*cumulative_ct/cp_total ≥ pct
            pct_date_index = date_index
            pct_date_tuple = index_to_tuple[date_index]
            break
        end
    end
    return pct_date_index, pct_date_tuple
end
##################################################################################################################
### Truly necessary stuff from old megacell | 2026_02_08
############################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
all_chr_seqs_inherited_pos_only = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
            if haskey(pango_predecessor_meta_dict[pango], 2)
                pango = pango_predecessor_meta_dict[pango][2]
                if !haskey(pango_AAsub_WT, pango)
                    if haskey(pango_predecessor_meta_dict, pango)
                        if haskey(pango_predecessor_meta_dict[pango], 3)
                            pango = pango_predecessor_meta_dict[pango][3]
                        end
                    end
                end
            end
        end
    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_AAsub_WT, pango)
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_predecessor_meta_dict, pango)
        if haskey(pango_predecessor_meta_dict[pango], 1)
            pango_pre1 = pango_predecessor_meta_dict[pango][1]
            for mut in pango_AAsub_WT[pango_pre1]
                all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
            end
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
############################################################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
println("Finished!"); print("\n"^2)

2026_03_07__559PM
5:59.19_PM

Done Loading Functions (line #1618 as of 2022_02_22)!
5:59.19_PM

Done Making date_str_to_index & index_to_date_str (line #1641 as of 2022_02_22)!
5:59.19_PM

Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!
5:59.20_PM

Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!
5:59.20_PM

Done Filling seq_syn_nucs, seq_noncoding_nucs!
6:00.34_PM

2026_03_07__600PM
Runtime v0 = 74.89579606056213 seconds
Runtime v2 = 0 hr, 1 min, 14.90 sec
Finished!




In [ ]:
### Making pango_to_pango_unaliased_v2 & pango consensus Dicts + MANY Others | Runtime = 154 seconds
####### Making pango_to_pango_unaliased_v2 —— using pango_variant_alias_key_2026_01_03_update.json from https://github.com/cov-lineages/pango-designation/blob/master/pango_designation/alias_key.json
start = time(); date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
println("nuc_genome_pango_dict length = $(length(nuc_genome_pango_dict))")

missing_AA_dict_keys = Set{String}()
############################################################################################################################################################################
#################### Making/filling index_to_tuple, tuple_to_index, index_to_date_str, date_str_to_index...etc dictionaries ################################################
############################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
###################################
chronic_pango_set = Set{String}()
chronic_pango_unaliased_set = Set{String}()
chronic_clade_set = Set{String}()
for seq in all_qualifying_seqs_set
    push!(chronic_pango_set, seq_pango[seq])
    push!(chronic_pango_unaliased_set, seq_pango_unaliased[seq])
    push!(chronic_clade_set, seq_clade[seq])
end
chronic_pango_set_len = length(chronic_pango_set)
chronic_pango_unaliased_set_len = length(chronic_pango_unaliased_set)
chronic_clade_set_len = length(chronic_clade_set)
println("Number of Different Pango Lineages in Chronics = $(chronic_pango_set_len)")
println("Number of Different Pango_Unaliased Lineages in Chronics = $(chronic_pango_unaliased_set_len)")
println("Number of Different Clades in Chronics = $(chronic_clade_set_len)")
println()
#####################################################################################################################################
all_fucking_pangos = union(pango_set, pango_index_date_set, chronic_pango_set)
push!(all_fucking_pangos, "B.1.640")
push!(all_fucking_pangos, "B.1.640.1")
push!(all_fucking_pangos, "B.1.640.2")
push!(all_fucking_pangos, "B.1.258.2")
push!(all_fucking_pangos, "G.1")
#####################################################################################################################################
nonleap_month_day_dict = Dict{Int,Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int,Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
###################
index_to_tuple = Dict{Int, Tuple{Int,Int, Int}}()
tuple_to_index = Dict{Tuple{Int,Int, Int}, Int}()        
###########################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
###################
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
###################################################################################################################
############################################################################################################################################################################
pango_to_pango_unaliased_v2["G.1"] = "B.1.258.2.1"
pgct = 0
for (pngo, unal) in pango_to_pango_unaliased_v2
    pgct += 1
    if pgct < 15
        println("$(pngo) = $(unal)")
    end
end
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
########################################################################################################################################################################
########################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
missing_keys = String[]
ndjson = "___pango_consensus_sequences/pango_consensus_sequences_summary_2025_06_25.ndjson"
for line in eachline(ndjson)
    j = JSON3.read(line)
    pango = j.lineage
    if !(pango in pango_set)
        push!(missing_keys, pango)
    end
    push!(pango_set, pango)
end
for pango in keys(pango_date_index_ct)
    if !(pango in keys(pango_to_pango_unaliased_v2))
        push!(pango_set, pango)
    end
end
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
#pango_to_pango_unaliased = Dict{String, String}()
clade_pango_set = Dict{String, Set{String}}()
#################################################################################
pango_nuc_sub_WT = Dict{String, Set{String}}()
pango_nuc_del_WT = Dict{String, Set{String}}()
pango_nuc_sub_private = Dict{String, Set{String}}()
pango_nuc_del_private = Dict{String, Set{String}}()
pango_nuc_sub_revs = Dict{String, Set{String}}()
pango_nuc_del_revs = Dict{String, Set{String}}()

pango_AAsub_WT = Dict{String, Set{String}}()
pango_AAsub_WT_pos_only = Dict{String, Set{String}}()
pango_AAdel_WT = Dict{String, Set{String}}()
pango_AAsub_private = Dict{String, Set{String}}()
pango_AAdel_private = Dict{String, Set{String}}()
pango_AAsub_revs = Dict{String, Set{String}}()
pango_AAdel_revs = Dict{String, Set{String}}()

pango_frameshifts_WT = Dict{String, Set{String}}()
pango_designation_date = Dict{String, String}()
#####################################################
for pango in pango_set
    pango_nuc_sub_WT[pango] = Set{String}()
    pango_nuc_del_WT[pango] = Set{String}()
    pango_nuc_sub_private[pango] = Set{String}()
    pango_nuc_del_private[pango] = Set{String}()
    pango_nuc_sub_revs[pango] = Set{String}()
    pango_nuc_del_revs[pango] = Set{String}()
########
    pango_AAsub_WT[pango] = Set{String}()
    pango_AAsub_WT_pos_only[pango] = Set{String}()
    pango_AAdel_WT[pango] = Set{String}()
    pango_AAsub_private[pango] = Set{String}()
    pango_AAdel_private[pango] = Set{String}()
    pango_AAsub_revs[pango] = Set{String}()
    pango_AAdel_revs[pango] = Set{String}()
#########
    pango_frameshifts_WT[pango] = Set{String}()
end
#################################################################################
for clade in clade_set
    clade_pango_set[clade] = Set{String}()
end
ndjson = "___pango_consensus_sequences/pango_consensus_sequences_summary_2025_06_25.ndjson"
missing_keys_for_WT = Set{String}()
###################################################################
#pango_consensus_nuc_subs_WT = 
#pango_consensus_nuc_dels_WT = 
#pango_consensus_nuc_subs = 
#pango_consensus_nuc_dels = 
#pango_consensus_nuc_revs = 
#pango_consensus_nuc_del_revs = 
#pango_consensus_AA_subs_WT = 
#pango_consensus_AA_dels_WT = 
#pango_consensus_AA_subs = 
#pango_consensus_AA_dels = 
#pango_consensus_AA_revs = 
#pango_consensus_AA_del_revs = 
#pango_consensus_
###################################################################
for line in eachline(ndjson)
    j = JSON3.read(line)
    pango = j.lineage
    unaliased = j.unaliased
    clade = j.nextstrainClade
    push!(clade_pango_set[clade], pango)
#    pango_to_pango_unaliased[pango] = unaliased
####################
    parent = j.parent
    children = j.children
####################
    nuc_subs_WT = j.nucSubstitutions
    nuc_dels_WT = j.nucDeletions
    nuc_subs = j.nucSubstitutionsNew
    nuc_dels = j.nucDeletionsNew
    nuc_revs = j.nucSubstitutionsReverted
    nuc_del_revs = j.nucDeletionsReverted
#####################
    for nuc_sub in nuc_subs_WT
        push!(pango_nuc_sub_WT[pango], nuc_sub)
    end
    for ndwt in nuc_dels_WT
        push!(pango_nuc_del_WT[pango], ndwt)
    end
    for n in nuc_subs
        push!(pango_nuc_sub_private[pango], n)
    end
    for n in nuc_dels
        push!(pango_nuc_del_private[pango], n)
    end
    for n in nuc_revs
        push!(pango_nuc_sub_revs[pango], n)
    end
    for n in nuc_del_revs
        push!(pango_nuc_del_revs[pango], n)
    end
########################################################
    AA_subs_WT = j.aaSubstitutions
    AA_dels_WT = Set(j.aaDeletions)
    AA_subs = j.aaSubstitutionsNew
    AA_dels = j.aaDeletionsNew
    AA_revs = j.aaSubstitutionsReverted
    AA_del_revs = j.aaDeletionsReverted
##################################################
    AA_subs_WT_pos_only = Set{String}()
    delete!(AA_dels_WT, "")
    for sub in AA_subs_WT
        if !haskey(aa_gene_and_pos_comprehensive_dict, sub)
            push!(missing_keys_for_WT, sub)
            continue
        end
        sub_po = aa_gene_and_pos_comprehensive_dict[sub]
        push!(AA_subs_WT_pos_only, sub_po)
    end 
    for AA_sub_po in AA_subs_WT_pos_only
        push!(pango_AAsub_WT_pos_only[pango], AA_sub_po)
    end
##################################################
    for AAsub in AA_subs_WT
        push!(pango_AAsub_WT[pango], AAsub)
    end
    for AAdel in AA_dels_WT
        push!(pango_AAdel_WT[pango], AAdel)
    end
    for AAsub in AA_subs
        push!(pango_AAsub_private[pango], AAsub)
    end
    for AAdel in AA_dels
        push!(pango_AAdel_private[pango], AAdel)
    end
    for AArev in AA_revs
        push!(pango_AAsub_revs[pango], AArev)
    end
    for AAdelrev in AA_del_revs
        push!(pango_AAdel_revs[pango], AAdelrev)
    end
    frameshiftsWT = j.frameShifts
    for fs in frameshiftsWT
        push!(pango_frameshifts_WT[pango], fs)
    end
    designation_date = j.designationDate
    pango_designation_date[pango] = designation_date
end
#####################################################################################################################################################################
#####################################################################################################################################################################
push!(pango_nuc_del_WT["B.1.617.2"], "28271")
push!(pango_nuc_del_private["B.1.617.2"], "28271")
#####################################################################################################################################################################
#####################################################################################################################################################################
println("Done Making Pango Consensus Sequences! (Part 1)"); print("\n"^1)
#####################################################################################################################################################################
println("Total Missing Keys for AA_pos_only_gene_and_pos_dict = $(length(missing_keys_for_WT))")
#missing_keys_for_WT_sort = sort(collect(missing_keys_for_WT), by = x -> mp_AA_gene_sortKey_2(x))
missing_keys_for_WT_join = join(collect(missing_keys_for_WT), ", ")
println("missing_keys_for_WT_join = ", "|", missing_keys_for_WT_join, "|"); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################# NEW SECTION (2026-01-03)   #################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### Making consensus muts for new sequences (using Angie Hinrichs' tree mut TSV file) —— MUST RUN load_all_seq_dicts FIRST
df_tree_muts = CSV.read("___pango_consensus_sequences/pango_clade_mutations_from_Angie_Hinrichs_2026_01_03.tsv", DataFrame, delim='\t')
##################synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = nuc_to_AA(pango, new_muts)########################################################################################################################################
total_rows = nrow(df_tree_muts)
tree_pangos = df_tree_muts[!, 1]
tree_muts = df_tree_muts[!, 2]
#####################################################################################################################
#  Example of 1st, then 2nd column (i.e. tree_pangos & tree_muts) ——  AY.112.2 | AY.112 > G23012C > G4162T > G27516A
#####################################################################################################################
tree_pangos_set = Set(tree_pangos)
tree_muts_set = Set(tree_muts)
################################################
bad_pangos = Vector{String}()
################################################
OG_tree_pangos_set_len = length(tree_pangos_set)
println("OG_tree_pangos_set_len = $(OG_tree_pangos_set_len)")
excluded_bits = ["misc", "proposed", '_']
for tpango in tree_pangos_set
    for exclude in excluded_bits
        if occursin(exclude, tpango)
            push!(bad_pangos, tpango)
        end
    end
end
for baddy in bad_pangos
    delete!(tree_pangos_set, baddy)
end
NEW_tree_pangos_set_len = length(tree_pangos_set)
println("NEW_tree_pangos_set_len = $(NEW_tree_pangos_set_len)")
println("bad_pangos length = $(length(bad_pangos))")
bad_pangos_sort = sort(bad_pangos)
bad_pangos_sort_join = join(bad_pangos_sort, ", ")
bad_pangos_sort_join_newlines = join(bad_pangos_sort, "\n")
###########
println(bad_pangos_sort_join)
############################################################################################################################################################################
pango_unaliased_to_pango_prefix = Dict{String, String}()
pango_to_pango_unaliased_v2 = Dict{String, String}()
pango_unaliased_to_pango = Dict{String, String}()
##################################################################
pango_unaliased_predecessor_meta_dict = Dict{String, Dict{Int, String}}()
pango_predecessor_meta_dict = Dict{String, Dict{Int, String}}()
##################################################################
for (prefix, unaliased) in pango_prefix_to_pango_unaliased
    pango_unaliased_to_pango_prefix[unaliased] = prefix
end
for pango in tree_pangos_set
    dotpts_ct = count('.', pango)
    if dotpts_ct ≥ 1
        dotsplits = split(pango, ".")
        prefix = string(dotsplits[1])
        prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
        last_pts = join(dotsplits[2:end], ".")
        pango_to_pango_unaliased_v2[pango] = "$(prefix_unaliased).$(last_pts)"
    else
        pango_to_pango_unaliased_v2[pango] = pango
    end
end
#####################################################################
pango_to_pango_unaliased_v2_key_set = Set(collect(keys(pango_to_pango_unaliased_v2)))
no_key_pangos = Set(["C.1.2", "B.1.1.70", "B.1.177.46", "B.1.36", "B.1.551", "B.1.1.329", "B.1.1.170", "C.26", "B.1.214.3", "B.1.609", "B.55", "A.2.5.3", "B.1.1.117", "C.13", "B.1.1.523", "B.1.470", "B.1.1.67", "B.39", "B.1.36.31", "B.1.36.22", "B.4", "B.1.1.500", "B.1.258.17", "B.1.1.101", "B.1.1.141", "B.1.260", "B.1.1.360", "AE.1", "B.1.404", "B.1.351.1", "AE.8", "B.1.1.10", "B.1.36.19", "B.1.587", "B.1.465", "B.1.177.52", "B.1.1.404", "B.1.436", "A.5", "B.1.1.221", "B.1.396", "B.1.428", "C.4", "B.1.214.2", "B.1.568", "B.1.147", "B.1.1.442", "B.1.1.47", "B.1.575.1", "B.1.177.4", "B.1.527", "B.1.1.354", "B.1.530", "B.1.361", "B.1.36.36", "B.1.126", "B.1.1.297", "B.1.420", "B.1.1.406", "B.1.36.29", "B.1.560", "B.1.214.1", "W.1", "B.1.177.12", "B.1.629", "B.1.1.37", "B.1.1.192", "N.6", "A.23", "B.1.177.54", "A.2.5.1", "A.3", "B.1.1.263", "B.1.379", "B.1.1.376", "B.1.240.1", "B.1.466.1", "B.1.1.241", "B.1.1.368", "B.1.177.32", "B.1.1.420", "B.1.1.413", "B.1.236", "B.1.2", "B.23", "B.1.1.418", "B.1.416.1", "B.1.1.54", "B.1.540", "AZ.5", "B.1.389", "B.1.433", "B.1.1.205", "B.1.1.434", "B.1.1.359", "C.2", "B.1.600", "B.1.9", "B.1.575", "B.1.177.57", "B.1.627", "B.1.36.35", "A.19", "B.1.371", "B.1.462", "B.1.22", "B.1.111", "B.1.258", "B.1.427", "B.1.177.53", "B.1.626", "B.1.1.294", "B.1.36.38", "B.1.1.332", "B.1.1.417", "B.1.1.521", "B.1.558", "B.1.36.18", "B.1.544", "B.1.1.375", "B.1.1.412", "B.1.1.485", "B.1.1.326", "B.1.375", "B.1.1.243", "B.1.1.369", "B.1.336", "B.1.577", "B.1.409", "B.1.219", "B.1.1.301", "B.1.1.433", "B.1.128", "B.1.179", "B.1.1.519", "B.1.1.307", "B.1.537", "B.1.1.265", "C.38", "B.1.177.44", "B.1.620", "AD.2", "C.23", "B.1.189", "B.1.349", "B.1.567", "B.1.398", "AZ.2", "B.1.1.25", "B.1.1.163", "B.1.497", "B.1.480", "B.1.1.198", "B.1.320", "B.1.1.121", "B.1.402", "B.6.6", "B.1.8", "B.1.298", "B.1.177.8", "B.1.1.39", "B.1.324", "B.1.1.351", "B.1.1.161", "B.1.1.279", "B.1.1.46", "B.1.177.83", "B.1.177.15", "B.1.346", "B.1.566", "B.1.78", "B.1.239", "A", "B.1.1.318", "B.1.397", "B.1.177.10", "B.1.1.416", "B.1.1.27", "AZ.3", "B.1.319", "B.1.619", "B.1.177.7", "B.1.258.3", "B.1.1.317", "B.1.605", "B.1.146", "B.1.1.61", "A.25", "C.40", "B.3", "B.1.438", "B.1.1.33", "B.1.192", "B.1.468", "C.11", "B.1.177.18", "B.1.471", "B.1.1.52", "B.1.1.171", "C.1", "B.1.1.348", "B.1.1.507", "B.1.177.82", "B.1.533", "B.1.1.306", "B.1.524", "B.1.149", "B.1.619.1", "B.1.36.9", "B.1.570", "B.1.1.50", "B.1.517", "B.1.499", "B.1.177.60", "B.1.1.378", "B.1.177.62", "C.16", "C.36.3", "B.1.1.459", "B.1.177.75", "A.27", "B.1.1.207", "B.1.618", "B.1.391", "B.1.1.115", "A.29", "B.1.1.1", "L.3", "A.28", "B.1.1.302", "B.1.3", "C.2.1", "B.1.1.135", "B.1.1.316", "B.1.393", "B.1.177.87", "R.1", "B.1.1.176", "C.32", "B.1.636", "B.1.1.133", "B.1.1.462", "N.3", "B.1.399", "B.1.1.487", "B.1.1.311", "B.1.606", "B.1.617.1", "B.1.265", "B.1.177.86", "N.9", "B.1.1.374", "B.1.1.228", "B.1.177", "B.1.177.5", "A.2.5.2", "B.1.1.57", "C.33", "B.1.538", "B.1.258.14", "B.1.1.305", "B.1.505", "B.1.1.397", "A.1", "B.1.565", "AT.1", "B.1.1.214", "B.1.177.16", "C.35", "B.6", "B.1.356", "B.1.177.21", "B.1.1.153", "B.1.1.432", "B.1.1.181", "B.1.1.312", "B.1.1.216", "B.1.1.526", "A.21", "B.1.380", "B.1.210", "B.1.1.398", "B.1.232", "B.1.1.220", "C.36", "B.1.438.1", "B.1.503", "B.1.36.7", "B.1.426", "B.1.1.159", "B.1.1.231", "A.18", "B.1.177.77", "B.1.1.99", "B.1.1.298", "B.1.453", "B.1.1.253", "N.5", "B.1.367", "B.1.549", "C.14", "B.1.400", "B.1.1.284", "AZ.4", "B.1.459", "B.1.243", "C.36.3.1", "B.1.474", "B.1.221", "B.1.630", "C.29", "B.1.1.528", "B.1.1.203", "B.1.1.269", "B.1.340", "B.1.634", "B.1.1.174", "B.1.596.1", "B.1.311", "B.1.36.24", "B.1.509", "B.1.36.39", "B.1.1.189", "B.1.36.16", "B.1.441", "B.1.91", "B.1.416", "B.1.1.56", "B.1.258.11", "B.1.417", "B.1.525", "B.1.561", "B.1.1.525", "B.1.1.277", "B.1.177.35", "B.1.240", "B.1.1.382", "B.1.234", "B.1.1.273", "B.1.1.291", "N.4", "B.1.617.3", "A.2", "B.1.523", "B.1.279", "B.1.1.63", "B.1.1.51", "B.1.1.157", "B.1.415.1", "B.1.212", "B.1.177.73", "B.1.1.371", "A.2.5", "B.1.1.44", "B.1.1.464", "B.1.362", "D.2", "B.1.241", "B.1.1.486", "B.1.377", "B.1.1.53", "B.1.588", "B.1.1.391", "B.1.1.186", "B.1.1.274", "B.1.623", "B.1.36.1", "B.1.195", "B.1.237", "B.1.110", "B.1.1.111", "AZ.1", "B.1.395", "B.1.381", "C.39", "B.1.206", "B.1.177.81", "B.1.214", "B.40", "B.1.110.3", "B.1.1.83", "B.1.411", "B.1.625", "B.1.350"])
for pango in all_fucking_pangos
    if !haskey(pango_to_pango_unaliased_v2, pango)
        dotpts_ct = count('.', pango)
        if dotpts_ct ≥ 1
            dotsplits = split(pango, ".")
            prefix = string(dotsplits[1])
            prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
            last_pts = join(dotsplits[2:end], ".")
            pango_to_pango_unaliased_v2[pango] = "$(prefix_unaliased).$(last_pts)"
        end
    end
end
pango_to_pango_unaliased_v2["B.1.1.529"] = "B.1.1.529"
pango_to_pango_unaliased_v2["LF.3"] = "B.1.1.529.2.86.1.1.16.1.3"
pango_to_pango_unaliased_v2["LF.3.1"] = "B.1.1.529.2.86.1.1.16.1.3.1"
pango_to_pango_unaliased_v2["B.1.469"] = "B.1.469"
pango_to_pango_unaliased_v2["A"] = "A"
pango_to_pango_unaliased_v2["B"] = "B"
pango_to_pango_unaliased_v2["B.1.1.482"] = "B.1.1.482"
pango_to_pango_unaliased_v2["B.1.1.370"] = "B.1.1.370"
pango_to_pango_unaliased_v2["B.1.596"] = "B.1.596"
pango_to_pango_unaliased_v2["B.1.415"] = "B.1.415"
####################################################################
pango_unaliased_set = Set{String}()
for (pango, unaliased) in pango_to_pango_unaliased_v2
    pango_unaliased_to_pango[unaliased] = pango
    push!(pango_unaliased_set, unaliased)
end
####################################################################
all_fucking_pangos = union(all_fucking_pangos, pango_to_pango_unaliased_v2_key_set)
for pango in all_fucking_pangos
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_predecessor_meta_dict[unaliased] = Dict{Int, String}()
#    println(pango)
    dotpts_ct = count('.', unaliased)
    if dotpts_ct ≥ 1
        dotsplits = split(unaliased, ".")
        for i in 1:dotpts_ct
            predecessor_i_unaliased = join(dotsplits[1:end-i], ".")
            pango_unaliased_predecessor_meta_dict[unaliased][i] = predecessor_i_unaliased
#            println("$(pango), $(unaliased), $(i) = $(predecessor_i_unaliased)")
        end
    end
end
pango_unaliased_predecessor_meta_dict["B.1.1.370"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.370"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.370"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.370"][3] = "B"
pango_unaliased_to_pango["B.1.1.370"] = "B.1.1.370"

pango_unaliased_predecessor_meta_dict["B.1.1.482"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.482"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.482"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.482"][3] = "B"
pango_unaliased_to_pango["B.1.1.482"] = "B.1.1.482"

pango_unaliased_predecessor_meta_dict["B.1.596"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.596"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.596"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.596"][3] = "B"
pango_unaliased_to_pango["B.1.596"] = "B.1.596"

pango_unaliased_predecessor_meta_dict["B.1.415"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.415"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.415"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.415"][3] = "B"
pango_unaliased_to_pango["B.1.415"] = "B.1.415"

for pango in all_fucking_pangos
    pango_predecessor_meta_dict[pango] = Dict{Int, String}()
    unaliased = pango_to_pango_unaliased_v2[pango]
    for (i, predecessor_i) in pango_unaliased_predecessor_meta_dict[unaliased]
        predecessor_i_pango = pango_unaliased_to_pango[predecessor_i]
        pango_predecessor_meta_dict[pango][i] = predecessor_i_pango
    end
end
#######################################################
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][1] = "B.1.1.529.2.86.1.1.16.1.3"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][2] = "B.1.1.529.2.86.1.1.16.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][3] = "B.1.1.529.2.86.1.1.16"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][4] = "B.1.1.529.2.86.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][5] = "B.1.1.529.2.86.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][6] = "B.1.1.529.2.86"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][7] = "B.1.1.529.2"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][8] = "B.1.1.529"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][9] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][10] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][11] = "B"
pango_predecessor_meta_dict["LF.3.1"] = Dict{Int, String}()
pango_predecessor_meta_dict["LF.3.1"][1] = "LF.3"
pango_predecessor_meta_dict["LF.3.1"][2] = "JN.1.16.1"
pango_predecessor_meta_dict["LF.3.1"][3] = "JN.1.16"
pango_predecessor_meta_dict["LF.3.1"][4] = "JN.1"
pango_predecessor_meta_dict["LF.3.1"][5] = "BA.2.86.1"
pango_predecessor_meta_dict["LF.3.1"][6] = "BA.2.86"
pango_predecessor_meta_dict["LF.3.1"][7] = "BA.2"
pango_predecessor_meta_dict["LF.3.1"][8] = "B.1.1.529"
pango_predecessor_meta_dict["LF.3.1"][9] = "B.1.1"
pango_predecessor_meta_dict["LF.3.1"][10] = "B.1"
pango_predecessor_meta_dict["LF.3.1"][11] = "B"
pango_unaliased_predecessor_meta_dict["B.1.1.529"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.529"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529"][3] = "B"
pango_predecessor_meta_dict["B.1.1.529"] = Dict{Int, String}()
pango_predecessor_meta_dict["B.1.1.529"][1] = "B.1.1"
pango_predecessor_meta_dict["B.1.1.529"][2] = "B.1"
pango_predecessor_meta_dict["B.1.1.529"][3] = "B"
############################################################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
#######################################################################################################################################
forbidden_pangos = Set(["XCD", "XCT", "XEA", "XFR", "XFS", "XFT", "XFT.1", "XFU", "XGD"])
#######################################################################################################################################
pango_new_nuc_muts = Dict{String, Set{String}}()
pango_new_nuc_muts_WT = Dict{String, Set{String}}()
pango_new_nuc_muts_pos_only = Dict{String, Set{Int}}()
###########################
pango_new_AA_muts = Dict{String, Set{String}}()
pango_new_AA_muts_WT = Dict{String, Set{String}}()
pango_new_AA_muts_pos_only = Dict{String, Set{String}}()
###########################
nuc_mut_all_and_chr_set_missing = Set{String}()
new_pango_ct = 0
empty_string_pangos = Set{String}()
for i in 1:length(tree_muts)
    if tree_pangos[i] in tree_pangos_set
        pango = convert(String, tree_pangos[i])
        unaliased = pango_to_pango_unaliased_v2[pango]
        if ismissing(tree_muts[i])
            println("missing tree muts: $(pango)")
            continue
        end
        new_muts_pre0 = replace(tree_muts[i], r"        "=>",")
        new_muts_pre1 = replace(new_muts_pre0, r"[TCAGN] [TCAGN]"=>",")
        new_muts_pre2 = replace(new_muts_pre1, r"\s"=>"")
        new_muts_pre3 = replace(new_muts_pre2, ">"=>",")
        new_muts_pre4 = replace(new_muts_pre3, ",,"=>",")
#        println("new_muts_pre3 = $(new_muts_pre3)")
        comma_splits = string.(split(new_muts_pre3, ","))
        predecessor_vec = String[]
        predecessor = string(comma_splits[1])
        if '_' in predecessor
            predecessor = string(split(predecessor, "_")[1])
        end
#        predecessor_unaliased = pango_to_pango_unaliased_v2[predecessor]
        push!(predecessor_vec, predecessor)
#        println(predecessor)
        for j in 1:length(pango_unaliased_predecessor_meta_dict[unaliased])
            predecessor_j = pango_unaliased_predecessor_meta_dict[unaliased][j]
            predecessor_j_pango = pango_unaliased_to_pango[predecessor_j]
            push!(predecessor_vec, predecessor_j_pango)
        end
        new_muts_pre = comma_splits[2:end]
        new_muts = String[]
        for mut in new_muts_pre
            if !('N' in mut)
                push!(new_muts, mut)
            end
        end
#        println("new_muts = $(new_muts)")
#        if !haskey(pango_nuc_sub_private, pango) && !haskey(pango_AAsub_private, pango)
#            pango_nuc_sub_WT[pango] = Set{String}()
#            pango_nuc_del_WT[pango] = Set{String}()
#            pango_nuc_sub_private[pango] = Set{String}()
#            pango_nuc_del_private[pango] = Set{String}()
#            pango_AAsub_WT[pango] = Set{String}()
#            pango_AAsub_WT_pos_only[pango] = Set{String}()
#            pango_AAdel_WT[pango] = Set{String}()
#            pango_AAsub_private[pango] = Set{String}()
#            pango_AAdel_private[pango] = Set{String}()
#        end
#        new_muts = filter!(
        if !isempty(new_muts)
            for xmut in new_muts
                if xmut == ""
                    push!(empty_string_pangos, pango)
#                    println("empty string in $(pango)")
                    continue
                end
                if !(xmut in nuc_mut_all_and_chr_set)
                    push!(nuc_mut_all_and_chr_set_missing, xmut)
                    pos_int = split_nuc_pos_int(xmut)
                    refnuc = string(xmut[1])
                    qrynuc = string(xmut[end])
                    nuc_mut_int_comprehensive_dict[xmut] = pos_int
                    ref_nuc_comprehensive_dict[xmut] = refnuc
                    qry_nuc_comprehensive_dict[xmut] = qrynuc
                    nuc_mut_int_string_comprehensive_dict[xmut] = string(pos_int)
                end
            end
        end
        precedessor_vec_join = join(predecessor_vec, ", ")
        if pango == "BA.3.2"
            println("BA.3.2 new_muts = $(new_muts)")
            println("BA.3.2 precedessor_vec_join = $(precedessor_vec_join)")
        end
#        println(precedessor_vec_join)
        for predecessor_j in predecessor_vec
            if (!haskey(nuc_genome_pango_dict, pango) || !haskey(pango_AAsub_private, pango)) && haskey(nuc_genome_pango_dict, predecessor_j) && length(nuc_genome_pango_dict[predecessor_j]) ≥ 28600 # && predecessor_j in pango_consensus_set && !(pango in forbidden_pangos)
                new_pango_ct += 1
                if new_pango_ct%100 == 0
                    println("################## NEW PANGO COUNT = $(new_pango_ct) #######################")
                end
                gene_AA_pango_dict[pango] = Dict{String, String}()
                for gene in gene_array
                    gene_AA_pango_dict[pango][gene] = gene_AA_pango_dict[predecessor_j][gene]
                end
####################################################################################################################################################
#                println(new_muts)
                synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = SIMPLER_nuc_to_AA(predecessor_j, new_muts)
####################################################################################################################################################
                sorted_nuc_muts = String[]
                if !isempty(all_nuc_AA_sorted_dict)
                    pango_nuc_sub_private[pango] = Set{String}()
                    pango_nuc_sub_WT[pango] = Set{String}()
                    pango_new_nuc_muts_pos_only[pango] = Set{String}()
                    sorted_nuc_muts = [all_nuc_AA_sorted_dict[i][1] for i in 1:length(all_nuc_AA_sorted_dict)]
                    for mut in sorted_nuc_muts
                        nuc_mut_int = nuc_mut_int_comprehensive_dict[mut]
                        WT_ref_nuc = string(wuhan_ref_seq[nuc_mut_int])
#                        predecessor_ref_nuc = ref_nuc_comprehensive_dict[mut]
                        predecessor_ref_nuc = ref_nuc_comprehensive_dict[mut]
#                        qry_nuc = qry_nuc_comprehensive_dict[mut]
                        qry_nuc = qry_nuc_comprehensive_dict[mut]
                        nuc_mut_WT = ""
                        if WT_ref_nuc == predecessor_ref_nuc
                            nuc_mut_WT = mut
                        else
                            nuc_mut_WT = "$(WT_ref_nuc)$(nuc_mut_int)$(qry_nuc)"
                        end
                        push!(pango_nuc_sub_private[pango], mut)
                        push!(pango_nuc_sub_WT[pango], nuc_mut_WT)
                        push!(pango_new_nuc_muts_pos_only[pango], nuc_mut_int)
                    end
                end
###################################################################
                sorted_AA_muts = String[]
                if !isempty(nonsynonymous_nuc_AA_sorted_dict)
                    pango_AAsub_private[pango] = Set{String}()
                    pango_AAsub_WT[pango] = Set{String}()
                    pango_new_AA_muts_pos_only[pango] = Set{String}()
                    sorted_AA_muts = [nonsynonymous_nuc_AA_sorted_dict[i][2] for i in 1:length(nonsynonymous_nuc_AA_sorted_dict)]
                    for mut in sorted_AA_muts
                        mutgeen = aa_gene_comprehensive_dict[mut]
                        AAmut_int = aa_pos_comprehensive_dict[mut]
                        AAmut_pos_only = aa_gene_and_pos_comprehensive_dict[mut]
                        refAA = refAA_comprehensive_dict[mut]
                        qryAA = qryAA_comprehensive_dict[mut]
                        new_gene_AA_seq = gene_AA_pango_dict[pango][mutgeen][1:AAmut_int-1]*qryAA*gene_AA_pango_dict[pango][mutgeen][AAmut_int+1:end]
                        gene_AA_pango_dict[pango][mutgeen] = new_gene_AA_seq
#                        pango_new_AA_muts[pango] = Set{String}()
#                        pango_new_AA_muts_WT[pango] = Set{String}()
#                        pango_new_AA_muts_pos_only[pango] = Set{String}()
                        WT_ref_AA = string(gene_AA_pango_dict[predecessor_j][mutgeen][AAmut_int])
                        AA_mut_WT = ""
                        if WT_ref_AA == refAA
                            AA_mut_WT = mut
                        else
                            AA_mut_WT = "$(mutgeen):$(WT_ref_AA)$(AAmut_int)$(qryAA)"
                        end 
                        push!(pango_AAsub_private[pango], mut)
                        push!(pango_AAsub_WT[pango], AA_mut_WT)
                        push!(pango_new_AA_muts_pos_only[pango], AAmut_pos_only)
                    end
                end
##################################################################
                nuc_genome_pango_dict[pango] = nuc_genome_pango_dict[predecessor_j]
                if !isempty(new_muts)
                    for mut in new_muts
                        if mut ≠ ""
                            qryNuc = qry_nuc_comprehensive_dict[mut]
                            nuc_int = nuc_mut_int_comprehensive_dict[mut]
                            new_nuc_seq = nuc_genome_pango_dict[pango][1:nuc_int-1]*qryNuc*nuc_genome_pango_dict[pango][nuc_int+1:end]
                            nuc_genome_pango_dict[pango] = new_nuc_seq
                        end
                    end
                    push!(pango_consensus_set, pango)
                end
                break
            end
        end
    end
end
#for pango in all_fucking_pangos
#    if !haskey(nuc_genome_pango_dict, pango)
#        unaliased = pango_to_pango_unaliased_v2[pango]
#        println("No nuc_genome_pango_dict for $(pango) | $(unaliased)")
#    end
#end
#for pango in all_fucking_pangos
#    if !haskey(nuc_genome_pango_dict, pango)
#        dot_ct = count('.', unaliased)
#        if dot_ct ≥ 1
#            for i in 1:dot_ct
#                predecessor_i = pango_unaliased_predecessor_meta_dict[pango][i]
#                if haskey(nuc_genome_pango_dict, predecessor_i)
#                    nuc_genome_pango_dict[pango] = nuc_genome_pango_dict[predecessor_i]
#                    break
#                end
#            end
#        end
#    end
#end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
#########################################################__END NEW SECTION (2026-01-03)    #################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################### 2025-11-13 | Filling pango/clade/unaliased date/ct/country dicts  Runtime = 41.57 seconds ##################################################
############################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
index_len = length(values(tuple_to_index))
println("tuple_to_index_len = $(index_len)"); println("index_to_tuple[1] = $(index_to_tuple[1])"); println("index_to_tuple[2000] = $(index_to_tuple[2000])")
###########################################################################################################
pango_unaliased_date_index_ct = Dict{String, Dict{Int64, Int64}}()    
#                          clade     date_index  cumulative_count
clade_date_index_cumul = Dict{String, Dict{Int,Int}}()
pango_date_index_cumul = Dict{String, Dict{Int,Int}}()
pango_unaliased_date_index_cumul = Dict{String, Dict{Int,Int}}()
#                                country       clade    date_index  cumulative_count  
#country_clade_date_index_cumul = Dict{String, Dict{String, Dict{Int,Int}}}()
country_pango_date_index_cumul = Dict{String, Dict{String, Dict{Int,Int}}}()
country_pango_unaliased_date_index_cumul = Dict{String, Dict{String, Dict{Int,Int}}}()
#
clade_total = Dict{String, Int}()
pango_total = Dict{String, Int}()
pango_unaliased_total = Dict{String, Int}()
####################################################################################################
AAmut_date_index_cumul = Dict{String, Dict{Int,Int}}()
nucmut_date_index_cumul = Dict{String, Dict{Int,Int}}()
###################################################################################################################################
for clade in chronic_clade_set
    clade_date_index_cumul[clade] = Dict{Int,Int}()
    cumulative_clade_ct = 0
    for date_index in 1:2500
        if !haskey(clade_date_index_ct[clade], date_index)
            clade_date_index_cumul[clade][date_index] = cumulative_clade_ct
        end
        if haskey(clade_date_index_ct[clade], date_index)
            cumulative_clade_ct += clade_date_index_ct[clade][date_index]
            clade_date_index_cumul[clade][date_index] = cumulative_clade_ct
        end
    end
    clade_total[clade] = cumulative_clade_ct
end
################################################################################################################################################################
missing_pangos = Set{String}()
for pango in all_fucking_pangos
    if !haskey(pango_date_index_ct, pango)
        push!(missing_pangos, pango)
    end
end     
################################################################################################################################################################
for pango in all_fucking_pangos
    if pango in missing_pangos
        continue
    end
    pango_date_index_cumul[pango] = Dict{Int,Int}()
    cumulative_pango_ct = 0
    for date_index in 1:2500
        if !haskey(pango_date_index_ct[pango], date_index)
            pango_date_index_cumul[pango][date_index] = cumulative_pango_ct
        end
        if haskey(pango_date_index_ct[pango], date_index)
            cumulative_pango_ct += pango_date_index_ct[pango][date_index]
            pango_date_index_cumul[pango][date_index] = cumulative_pango_ct
        end
    end
    pango_total[pango] = cumulative_pango_ct
end
################################################################################
for pango in keys(pango_date_index_cumul)
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_date_index_cumul[unaliased] = Dict{Int,Int}()
    for date_index in 1:2500
        pango_unaliased_date_index_cumul[unaliased][date_index] = pango_date_index_cumul[pango][date_index]
    end
    pango_unaliased_total[unaliased] = pango_total[pango]
end
println("Done Filling Pango Date Index Cts!")
################################################################################
clade_pct_date_index = Dict{String, Dict{Float64,Int}}()
clade_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int,Int, Int}}}()
clade_pct_date_string = Dict{String, Dict{Float64, String}}()
#####
pango_pct_date_index = Dict{String, Dict{Float64,Int}}()
pango_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int,Int, Int}}}()
pango_pct_date_string = Dict{String, Dict{Float64, String}}()
#####
pango_unaliased_pct_date_index = Dict{String, Dict{Float64,Int}}()
pango_unaliased_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int,Int, Int}}}()
pango_unaliased_pct_date_string = Dict{String, Dict{Float64, String}}()
######################################################################################################################
for clade in chronic_clade_set
    clade_pct_date_index[clade] = Dict{Float64,Int}()
    clade_pct_date_tuple[clade] = Dict{Float64, Tuple{Int,Int, Int}}()
    clade_pct_date_string[clade] = Dict{Float64, String}()
    cp_total = clade_total[clade]
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in 950:999
#        pct = pct/10
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = clade_date_index_cumul[clade][date_index]
            if 100*cumulative_ct/cp_total ≥ pct
                clade_pct_date_index[clade][pct] = date_index
                clade_pct_date_tuple[clade][pct] = index_to_tuple[date_index]
                clade_pct_date_string[clade][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
##########################################################
for pango in all_fucking_pangos
    if pango in missing_pangos
        continue
    end
    pango_pct_date_index[pango] = Dict{Float64,Int}()
    pango_pct_date_tuple[pango] = Dict{Float64, Tuple{Int,Int, Int}}()
    pango_pct_date_string[pango] = Dict{Float64, String}()
    cp_total = get(pango_total, pango, 0)
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in [10.0, 25.0, 50.0, 75.0, 90.0, 95.0, 98.0, 99.0, 99.5, 99.7, 99.8, 99.9]
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = 0
            cumulative_ct = pango_date_index_cumul[pango][date_index]
            if 100*cumulative_ct/cp_total ≥ pct
                pango_pct_date_index[pango][pct] = date_index
                pango_pct_date_tuple[pango][pct] = index_to_tuple[date_index]
                pango_pct_date_string[pango][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
println("Done Filling pango_pct_date_index/tuple/string!")
##########################################################
for pango in keys(pango_pct_date_index)
    if pango in missing_pangos
        continue
    end
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_pct_date_index[unaliased] = Dict{Float64,Int}()
    pango_unaliased_pct_date_tuple[unaliased] = Dict{Float64, Tuple{Int,Int, Int}}()
    pango_unaliased_pct_date_string[unaliased] = Dict{Float64, String}()
    cp_total = get(pango_total, pango, 0)
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in 950:999
#    for pct in [10.0, 25.0, 50.0, 75.0, 90.0, 95.0, 98.0, 99.0, 99.5, 99.7, 99.8, 99.9]
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = 0
            if pango == "B.1.1.529"
                cumulative_ct = pango_date_index_cumul["BA.2"][date_index]
            elseif pango == "LF.3.1"
                cumulative_ct = pango_date_index_cumul["LF.3"][date_index]
            else
                cumulative_ct = pango_date_index_cumul[pango][date_index]
            end
            if 100*cumulative_ct/cp_total ≥ pct
                pango_unaliased_pct_date_index[unaliased][pct] = date_index
                pango_unaliased_pct_date_tuple[unaliased][pct] = index_to_tuple[date_index]
                pango_unaliased_pct_date_string[unaliased][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
println("Done Filling pango_unaliased_pct_date_index/tuple/string!")
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
            if haskey(pango_predecessor_meta_dict[pango], 2)
                pango = pango_predecessor_meta_dict[pango][2]
                if !haskey(pango_AAsub_WT, pango)
                    if haskey(pango_predecessor_meta_dict, pango)
                        if haskey(pango_predecessor_meta_dict[pango], 3)
                            pango = pango_predecessor_meta_dict[pango][3]
                        end
                    end
                end
            end
        end
    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    else
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
############################################################################################################################################################################
############################################################################################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
############################################################################################################################################################################
nuc_genome_pango_dict["B.1.1.273"] = nuc_genome_pango_dict["B.1.1"]
nuc_genome_pango_dict["B.1.1.53"] = nuc_genome_pango_dict["B.1.1"]
############################################################################################################################################################################
finish = time() - start
finish_rd = round(digits=2, finish)
println("nuc_genome_pango_dict length = $(length(nuc_genome_pango_dict))")
println("Length all_fucking_pangos = $(length(all_fucking_pangos))")
print("\n"^1)
println("Total time to finish = $(finish_rd)"); print("\n"^1)
println("Finished!")
print("\n"^1)
temp_nuc_sort_key(n) = (length(n), n[2:end-1])
nuc_mut_all_and_chr_set_missing_sort = sort(collect(nuc_mut_all_and_chr_set_missing), by = x -> temp_nuc_sort_key(x))
nuc_mut_all_and_chr_set_missing_sort_join = join(nuc_mut_all_and_chr_set_missing_sort, ", ")
missing_AA_dict_keys_sort = sort(collect(missing_AA_dict_keys), by = x -> AA_gene_sortKey_2(x))
missing_AA_dict_keys_sort_join = join(missing_AA_dict_keys_sort, ", ")
empty_string_pangos_sort = sort(collect(empty_string_pangos))
empty_string_pangos_sort_join = join(empty_string_pangos_sort, ", ")
open("missing_nuc_dict_keys_$(date_now).txt", "w") do g
    println(g, "nuc_mut_all_and_chr_set_missing_sort_join = $(nuc_mut_all_and_chr_set_missing_sort_join)")
    println("nuc_mut_all_and_chr_set_missing_sort_join = $(nuc_mut_all_and_chr_set_missing_sort_join)")
    print("\n"^1)
    print(g, "\n"^1)
    println(g, "missing_AA_dict_keys_sort_join = $(missing_AA_dict_keys_sort_join)")
    println("missing_AA_dict_keys_sort_join = $(missing_AA_dict_keys_sort_join)")
    print("\n"^1)
    print(g, "\n"^1)
    println(g, "empty_string_pangos_sort_join = $(empty_string_pangos_sort_join)")
    println("empty_string_pangos_sort_join = $(empty_string_pangos_sort_join)")
end
print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################

In [94]:
### print_all_seq_info Fx + All gene/AA/nuc/key + synonymous_nuc_to_AA_and_noncoding_to_context #######################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
for seq in all_unique_chr_seqs_set 
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
mut_gene_Dict2 = Dict{String, Int}("ORF1a"=>11, "ORF1b"=>12, "S"=>1, "E"=>2, "M"=>3, "N"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10)
##############################################################################
nuc_del_range_sortkey(n) = parse(Int, split(n, "-")[1])
###########
AA_del_num(n) = parse(Int, split(split(n, "-")[1], ":")[2])
unknown_AA_first_pos(n) = string(split(n, "-")[1])
gene___AAnum_sortkey(n) = [mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAnum_sortkey2(n) = [mut_gene_Dict2[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAdel_sortkey2(n) = [mut_gene_Dict2[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAposonly_sortkey(n) = [mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
unknown_AA_ranges_sort(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[unknown_AA_first_pos(n)]], aa_pos_comprehensive_dict[unknown_AA_first_pos(n)])
##############################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
##############################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
##############################################################################
seq_nuc_muts["EPI_ISL_6281381"] = Set{String}()
seq_nuc_muts_no_dels = Dict{String, Set{String}}()
for (seq, mut_set) in seq_nuc_muts
    seq_nuc_muts_no_dels[seq] = Set{String}()
    for mut in mut_set
        if qry_nuc_comprehensive_dict[mut] ≠ "-"
            push!(seq_nuc_muts_no_dels[seq], mut)
        end
    end
end      
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#####################################################################################################################################
function print_all_seq_info(seq::String, txt_out::String)
    open(txt_out, "a") do g
        println("#"^94); println(g, "#"^94)
        cntry = seq_country[seq]
        pango = seq_pango[seq]
        USA_state = seq_US_state[seq] 
        coll_date = seq_collection_date[seq]
        date_index = seq_date_index[seq]
        lab = seq_lab_dict[seq]
#################################################################################
        aa_muts = seq_AA_muts_no_dels[seq]
        aa_del_ranges = seq_AA_dels[seq]
        nuc_muts = seq_nuc_muts[seq]
        nuc_muts_no_dels = seq_nuc_muts_no_dels[seq]
        nuc_del_ranges = seq_nuc_del_ranges[seq]
        aa_muts_sort = sort(collect(aa_muts), by = x -> gene___AAnum_sortkey(x))
        aa_muts_sort2 = sort(collect(aa_muts), by = x -> gene___AAnum_sortkey2(x))
        aa_del_ranges_sort = sort(collect(aa_del_ranges), by = x -> gene___AAdel_sortkey2(x))
        nuc_muts_sort = sort(collect(nuc_muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        nuc_muts_no_dels_sort = sort(collect(nuc_muts_no_dels), by = x -> nuc_mut_int_comprehensive_dict[x])
        nuc_del_ranges_sort = sort(collect(nuc_del_ranges), by = x -> nuc_del_range_sortkey(x))
        seq_mixed_AA_muts_sort = sort(collect(seq_mixed_AA_muts[seq]), by = x -> gene___AAnum_sortkey(x))
        seq_mixed_nucs_sort = sort(collect(seq_mixed_nucs[seq]), by = x -> nuc_mut_int_comprehensive_dict[x])
        seq_unknown_AA_ranges_sort = sort(collect(seq_unknown_AA_ranges[seq]), by = x -> unknown_AA_ranges_sort(x))
        total_AA_subs_plus_del_ranges = length(aa_muts) + length(aa_del_ranges)
#################################################################################
        mut_vec_dict = Dict{String, Vector{String}}()
        for gene in gene_print_array
            mut_vec_dict[gene] = Vector{String}()
        end
        for mut in aa_muts
            gene = aa_gene_comprehensive_dict[mut]
            mut_only = string(split(mut, ":")[2])
            push!(mut_vec_dict[gene], mut_only) 
        end
        mutonly_sortkey(n) = parse(Int, n[2:end-1])
        for gene in keys(mut_vec_dict)
            if !isempty(mut_vec_dict[gene])
                sort!(mut_vec_dict[gene], by = x -> mutonly_sortkey(x))
            end
        end
#################################################################################
        if seq in rep_seqs
            for (grp_num, seq_set) in rep_seq_grps_seqs
                grp_size = length(seq_set)
                if seq in seq_set
                    seq_set_sort = sort(collect(seq_set), by = x -> (length(x), x))
                    seq_set_sort_join = join(seq_set_sort, ", ")
                    seq_set_sort_join_len = length(seq_set_sort_join)
#                    title_len = length("Group #$(grp_num), $(grp_size) related seqs--")
                        println("Group #$(grp_num), $(grp_size) related sequences")
                    println(g, "Group #$(grp_num), $(grp_size) related sequences")
                    if seq_set_sort_join_len ≤ 84
                           print("       ")
                        print(g, "       ")
                           println(seq_set_sort_join)
                        println(g, seq_set_sort_join)
                    elseif seq_set_sort_join_len > 84
                        start = 0
                        for z in 1:10
                            if seq_set_sort_join_len - start > 84
                                for i in start+85:-1:start+65
                                    if string(seq_set_sort_join[i]) == " "
                                           print("       ")
                                        print(g, "       ")
                                           println(seq_set_sort_join[start+1:i-1])
                                        println(g, seq_set_sort_join[start+1:i-1])
                                        start = i
                                        break
                                    end
                                end
                            end
                            if seq_set_sort_join_len - start ≤ 84
                                   print("       ")
                                print(g, "       ")
                                   println(seq_set_sort_join[start+1:end])
                                println(g, seq_set_sort_join[start+1:end])
                                break
                            end
                        end
                    end
                end
            end     
        else
            println("Singlet")
            println(g, "Singlet")
        end
########################################################################
           println("$(pango)|$(seq)|$(coll_date) | $(cntry)|$(USA_state)|$(lab) | AAsubs+DelRanges = $(total_AA_subs_plus_del_ranges)")
        println(g, "$(pango)|$(seq)|$(coll_date) | $(cntry)|$(USA_state)|$(lab) | AAsubs+DelRanges = $(total_AA_subs_plus_del_ranges)")
           println("----------------------------------- AA Substitutions -------------------------------------")
        println(g, "----------------------------------- AA Substitutions -------------------------------------")
################# Next bit is to figure out how many muts are in a given gene & how much print space they take up
        gene_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_mut_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_set = Set{String}()
        for mut in aa_muts_sort2
            gene = split(mut, ":")[1]
            push!(gene_set, gene)
            gene_mut_ct[gene] += 1
            gene_print_length[gene] += length(mut) + 2
        end
        for gene in gene_set
            ## +4 is for the "--", the -2 b/c the last mut doesn't have ", " after it, & the +10 for 10 spaces btwn gene muts
            gene_print_length[gene] += length(gene) + 4 - 2
        end
################ Same thing as above but for deletions
        gene_del_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_AA_del_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_del_set = Set{String}()
        for del in aa_del_ranges_sort
            gene = split(del, ":")[1]
            push!(gene_del_set, gene)
            del_rng = split(del, ":")[2]
            gene_del_print_length[gene] += length(del_rng) + 2
        end
        for gene in gene_del_set
            gene_del_print_length[gene] += length(gene) + 4 - 2
        end
############### Same thing as above but for unknown AA
        gene_unk_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_AA_unk_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_unk_set = Set{String}()
        for unk in seq_unknown_AA_ranges_sort
            gene = split(unk, ":")[1]
            push!(gene_unk_set, gene)
            unk_rng = split(unk, ":")[2]
            gene_unk_print_length[gene] += length(unk_rng) + 2
        end
        for gene in gene_unk_set
            gene_unk_print_length[gene] += length(gene) + 4 - 2
        end
#####################################################################################################################################
        line_vec_dict = Dict{Int,Int}(1=>0, 2=>0, 3=>0, 4=>0, 5=>0, 6=>0, 7=>0, 8=>0, 9=>0, 10=>0, 11=>0, 12=>0, 13=>0, 14=>0, 15=>0, 16=>0)
        line_length = 0
        previous_gene = "S"
        midline = "no"
        for v in 1:length(gene_print_array)
            gene = gene_print_array[v]
            skip = "no"
            if !isempty(mut_vec_dict[gene])
                muts_joined = join(mut_vec_dict[gene], ", ")
                muts_line_empty = "$(gene)--$(muts_joined)"
                muts_line_nonempty = "     $(gene)--$(muts_joined)"
                muts_line_empty_len = length(muts_line_empty)
                muts_line_nonempty_len = length(muts_line_nonempty)
                if length(previous_gene) < 3 && length(gene) ≥ 4
                    skip = "yes"
                end
                previous_gene = gene
                if line_length > 0 && muts_line_nonempty_len ≤ (90-line_length) && skip == "yes"
                    println(); println(g)
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = muts_line_empty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len ≤ (90-line_length) && skip == "no"
                    print(muts_line_nonempty); print(g, muts_line_nonempty)
                    line_length = line_length + muts_line_nonempty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len > (90-line_length) && muts_line_empty_len ≤ 90
                    println(); println(g)
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = line_length + muts_line_nonempty_len
                    midline = "yes"
                    continue
                elseif line_length == 0 && muts_line_empty_len ≤ 90
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = muts_line_empty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len > (90-line_length) && muts_line_empty_len > 90
                    println(); println(g)
                    line_length = 0
                    midline = "no"
                    for i in 77:91
                        if string(muts_line_empty[i]) == " "
                            println(muts_line_empty[1:i-1]); println(g, muts_line_empty[1:i-1])
                            line_vec_dict[1] = i
                            print("      "); print(g, "      ") 
                            break
                        end
                    end
                    current_pos = line_vec_dict[1]
                    for j in 2:length(line_vec_dict)
                        line = line_vec_dict[j]
                        if muts_line_empty_len - current_pos ≤ 84
                            println(muts_line_empty[current_pos+1:end]); println(g, muts_line_empty[current_pos+1:end])
                            line_length = 0
                            midline = "no"
                            break
                        end
                        if muts_line_empty_len - current_pos > 84
                            for i in current_pos+71:current_pos+85
                                if string(muts_line_empty[i]) == " "
                                    println(muts_line_empty[current_pos+1:i-1]); println(g, muts_line_empty[current_pos+1:i-1])
                                    midline = "no"
                                    if v < length(gene_print_array) && !all(isempty(mut_vec_dict[gene_print_array[q]]) for q in v+1:length(gene_print_array))
                                        print("      "); print(g, "      ")
                                    end
                                    current_pos = i
                                    break
                                end
                            end
                        end
                    end
                elseif line_length == 0 && muts_line_empty_len > 90
                    for i in 77:91
                        if string(muts_line_empty[i]) == " "
                            println(muts_line_empty[1:i-1]); println(g, muts_line_empty[1:i-1])
                            line_vec_dict[1] = i
                            print("      "); print(g, "      ") 
                            break
                        end
                    end
                    current_pos = line_vec_dict[1]
                    for j in 2:length(line_vec_dict)
                        line = line_vec_dict[j]
                        if muts_line_empty_len - current_pos ≤ 84
                            println(muts_line_empty[current_pos+1:end]); println(g, muts_line_empty[current_pos+1:end])
                            line_length = 0
                            midline = "no"
                            break
                        end
                        if muts_line_empty_len - current_pos > 84
                            for i in current_pos+71:current_pos+85
                                if string(muts_line_empty[i]) == " "
                                    println(muts_line_empty[current_pos+1:i-1]); println(g, muts_line_empty[current_pos+1:i-1])
                                    midline = "no"
                                    if v < length(gene_print_array) && !all(isempty(mut_vec_dict[gene_print_array[q]]) for q in v+1:length(gene_print_array))
                                        print("      "); print(g, "      ")
                                    end
                                    current_pos = i
                                    break
                                end
                            end
                        end
                    end
                end
            end
        end
        if midline == "yes"
            println(); println(g)
        end
           println("------------------------------------------------------------------------------------------")
        println(g, "------------------------------------------------------------------------------------------")
##########################################################################################
#        print("\n"^1); print(g, "\n"^1)
        if !isempty(aa_del_ranges_sort)
            aadel_caps_pad = rpad("AA DELETIONS", 14)
            print("$(aadel_caps_pad)| ")
            print(g, "$(aadel_caps_pad)| ")
            line_AA_del_print_ct = 20
            for i in 1:length(aa_del_ranges_sort)
                del = aa_del_ranges_sort[i]
                gene = split(del, ":")[1]
                del2 = split(del, ":")[2]
                if length(aa_del_ranges_sort) == 1
                    println("$(gene)--$(del2)")
                    println(g, "$(gene)--$(del2)")
                end
                if length(aa_del_ranges_sort) > 1
                    if i == 1
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            print("$(gene)--$(del2), ")
                            print(g, "$(gene)--$(del2), ")
                            line_AA_del_print_ct = length(gene) + 4 + length(del2) + 2
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            print("$(gene)--$(del2)")
                            print(g, "$(gene)--$(del2)")
                            line_AA_del_print_ct = length(gene) + 4 + length(del2)
                        end
                    end
                    if i ≠ 1 && i ≠ length(aa_del_ranges_sort)
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            line_AA_del_print_ct += length(del2) + 2
                            if line_AA_del_print_ct > 102 
                                println(); println(g)
                                print("          $(del2), ")
                                print(g, "          $(del2), ")
                                line_AA_del_print_ct = 10 + length(del2) + 2
                            else
                                print("$(del2), ")
                                print(g, "$(del2), ")
                            end
                        elseif mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            line_AA_del_print_ct += length(del2)
                            if line_AA_del_print_ct > 102
                                println(); println(g)
                                println("          $(del2)")
                                println(g, "          $(del2)")
                                line_AA_del_print_ct = length(del2)
                            else
                                print("$(del2)")
                                print(g, "$(del2)")
                            end
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            if line_AA_del_print_ct + gene_del_print_length[gene]+ 10 > 102
                                println(); println(g)
                                print("          $(gene)--$(del2), ")
                                print(g, "          $(gene)--$(del2), ")
                                line_AA_del_print_ct = 10 + length(gene) + 4 + length(del2) + 2
                            else
                                print("          $(gene)--$(del2), ")
                                print(g, "          $(gene)--$(del2), ")
                                line_AA_del_print_ct += 10 + length(gene) + 4 + length(del2) + 2
                            end
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            if line_AA_del_print_ct + gene_del_print_length[gene]+ 10 > 102
                                println(); println(g)
                                print("          $(gene)--$(del2)")
                                print(g, "          $(gene)--$(del2)")
                                line_AA_del_print_ct = 10 + length(gene) + 4 + length(del2)
                            else
                                print("          $(gene)--$(del2)")
                                print(g, "          $(gene)--$(del2)")
                                line_AA_del_print_ct += length(gene) + 4 + length(del2) + 10
                            end
                        end
                    end
                    if i == length(aa_del_ranges_sort)
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1])
                            println("$(del2)")
                            println(g, "$(del2)")
                        elseif line_AA_del_print_ct + gene_del_print_length[gene] > 102
                            println(); println(g)
                            println("          $(gene)--$(del2)")
                            println(g, "          $(gene)--$(del2)")
                        else
                            println("          $(gene)--$(del2)")
                            println(g, "          $(gene)--$(del2)")
                        end
                    end
                end
            end
        end
#        println(); println(g)
##############################################################################################
        if !isempty(seq_mixed_AA_muts_sort)
            mixed_caps_pad = rpad("MIXED AA MUTS", 14)
            print("$(mixed_caps_pad)| ")
            print(g, "$(mixed_caps_pad)| ")
            mixed_print_ct = 21
            for i in 1:length(seq_mixed_AA_muts_sort)
                mix = seq_mixed_AA_muts_sort[i]
                mix1 = split(mix, ":")[1]
                mix2 = split(mix, ":")[2]
                if length(seq_mixed_AA_muts_sort) == 1
                    println("$(mix1)--$(mix2)")
                    println(g, "$(mix1)--$(mix2)")
                end
                if length(seq_mixed_AA_muts_sort) > 1
                    if i == 1
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix1)--$(mix2), ")
                            print(g, "$(mix1)--$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix1)--$(mix2)")
                            print(g, "$(mix1)--$(mix2)")
                        end
                    end
                    if i ≠ 1 && i ≠ length(seq_mixed_AA_muts_sort)
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix2), ")
                            print(g, "$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            println("$(mix2)")
                            println(g, "$(mix2)")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("    $(mix1)--$(mix2), ")
                            print(g, "    $(mix1)--$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            println("    $(mix1)--$(mix2)")
                            println(g, "    $(mix1)--$(mix2)")
                        end
                    end
                    if i == length(seq_mixed_AA_muts_sort)
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]]
                            println("$(mix2)")
                            println(g, "$(mix2)")
                        else
                            println("    $(mix1)--$(mix2)   ")
                            println(g, "    $(mix1)--$(mix2)   ")
                        end
                    end
                end
            end
        end
##############################################################################################
        if !isempty(seq_unknown_AA_ranges_sort)
            aadrop_caps_pad = rpad("AA DROPOUT", 14)
            print("$(aadrop_caps_pad)| ")
            print(g, "$(aadrop_caps_pad)| ")
            line_unk_print_ct = 18
            for i in 1:length(seq_unknown_AA_ranges_sort)
                unk = seq_unknown_AA_ranges_sort[i]
                gene = split(unk, ":")[1]
                unk2 = split(unk, ":")[2]
                if length(seq_unknown_AA_ranges_sort) == 1
                    println("$(gene)--$(unk2)")
                    println(g, "$(gene)--$(unk2)")
                end
                if length(seq_unknown_AA_ranges_sort) > 1
                    if i == 1
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            print("$(gene)--$(unk2), ")
                            print(g, "$(gene)--$(unk2), ")
                            line_unk_print_ct = length(gene) + 4 + length(unk2) + 2
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            print("$(gene)--$(unk2)")
                            print(g, "$(gene)--$(unk2)")
                            line_unk_print_ct = length(gene) + 4 + length(unk2)
                        end
                    end
                    if i ≠ 1 && i ≠ length(seq_unknown_AA_ranges_sort)
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            line_unk_print_ct += length(unk2) + 2
                            if line_unk_print_ct > 102 
                                println(); println(g)
                                print("      $(unk2), ")
                                print(g, "      $(unk2), ")
                                line_unk_print_ct = 10 + length(unk2) + 2
                            else
                                print("$(unk2), ")
                                print(g, "$(unk2), ")
                            end
                        elseif mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            line_unk_print_ct += length(unk2)
                            if line_unk_print_ct > 102
                                println(); println(g)
                                println("      $(unk2)")
                                println(g, "      $(unk2)")
                                line_unk_print_ct = 0
                            else
                                print("$(unk2)")
                                print(g, "$(unk2)")
                            end
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            if line_unk_print_ct + gene_unk_print_length[gene] + 10 > 102
                                println();  println(g)
                                print("      $(gene)--$(unk2), ")
                                print(g, "      $(gene)--$(unk2), ")
                                line_unk_print_ct = 10 + length(gene) + 4 + length(unk2) + 2
                            else
                                print("      $(gene)--$(unk2), ")
                                print(g, "      $(gene)--$(unk2), ")
                                line_unk_print_ct += length(gene) + 4 + length(unk2) + 2 + 10
                            end
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            if line_unk_print_ct + gene_unk_print_length[gene] + 10 > 102
                                println(); println(g)
                                print("      $(gene)--$(unk2)")
                                print(g, "      $(gene)--$(unk2)")
                                line_unk_print_ct = 10 + length(gene) + 4 + length(unk2)
                            else
                                print("      $(gene)--$(unk2)")
                                print(g, "      $(gene)--$(unk2)")
                                line_unk_print_ct += length(gene) + 4 + length(unk2) + 10
                            end
                        end
                    end
                    if i == length(seq_unknown_AA_ranges_sort)
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1])
                            println("$(unk2)")
                            println(g, "$(unk2)")
                        elseif line_unk_print_ct + gene_unk_print_length[gene] > 102
                            println(); println(g)
                            println("    $(gene)--$(unk2)")
                            println(g, "    $(gene)--$(unk2)")
                        else
                            println("      $(gene)--$(unk2)")
                            println(g, "      $(gene)--$(unk2)")
                        end
                    end
                end
            end
            println(); println(g)
        end
###############################################################################################################################
        print("NUC MUTATIONS--")
        print(g, "NUC MUTATIONS--")
        nuc_muts_join = join(nuc_muts_no_dels_sort, ", ")
        nuc_mut_len = length(nuc_muts_join)
        line_length = 0
#        line1 = 0; line2 = 0; line3 = 0; line4 = 0; line5 = 0; line6 = 0; line7 = 0; line8 = 0
#        line9 = 0; line10 = 0; line11 = 0; line12 = 0; line13 = 0; line14 = 0; line15 = 0; line16 = 0
#        line_vec = [line1, line2, line3, line4, line5, line6, line7, line8, line9, line10, line11, line12, line13, line14, line15, line16]
        line_vec_dict = Dict{Int,Int}(1=>0, 2=>0, 3=>0, 4=>0, 5=>0, 6=>0, 7=>0, 8=>0, 9=>0, 10=>0, 11=>0, 12=>0, 13=>0, 14=>0, 15=>0, 16=>0)
        done = "no"
        if nuc_mut_len ≤ 75
               println(nuc_muts_join)
            println(g, nuc_muts_join)
            done = "yes"
        end
        if nuc_mut_len > 75 && done == "no"
            for j in 69:77
                if string(nuc_muts_join[j]) == " "
                       println(nuc_muts_join[1:(j-1)])
                    println(g, nuc_muts_join[1:(j-1)])
                    line_vec_dict[1] = j
                    break
                end
            end
        end
        for i in 1:length(line_vec_dict)
            line = line_vec_dict[i]
            if nuc_mut_len - line ≤ 84 && done == "no"
                   print("      ")
                print(g, "      ")
                   println(nuc_muts_join[(line+1):end])
                println(g, nuc_muts_join[(line+1):end])
                done = "yes"
                break
            end
            if nuc_mut_len - line > 84 && done == "no"
                   print("      ")
                print(g, "      ")
                for j in line+77:line+85
                    if string(nuc_muts_join[j]) == " "
                           println(nuc_muts_join[line+1:j-1])
                        println(g, nuc_muts_join[line+1:j-1])
                        if i ≠ length(line_vec_dict)
                            line_vec_dict[i+1] = j
                        end
                    end
                end
            end
        end
##############################################################################################
        if !isempty(nuc_del_ranges_sort)
            nucdel_caps = "NUC DELETIONS--"
            print("$(nucdel_caps)")
            print(g, "$(nucdel_caps)")
            print_length = 16
            for i in 1:length(nuc_del_ranges_sort)
                nuc_del = nuc_del_ranges_sort[i]
                print_length += length(nuc_del) + 2
                if print_length > 102
                    if i == 1
                        println("$(nuc_del), ")
                        println(g, "$(nuc_del), ")
                        print_length += 8
                    end
                    if i ≠ length(nuc_del_ranges_sort) && !(i == 1)
                        println("        $(nuc_del), ")
                        println(g, "        $(nuc_del), ")
                        print_length += 8
                    elseif !(i == 1)
                        println("        $(nuc_del)")
                        println(g, "        $(nuc_del)")
                        print_length += 8
                    end
                else
                    if i ≠ length(nuc_del_ranges_sort)
                        print("$(nuc_del), ")
                        print(g, "$(nuc_del), ")
                    else
                        println("$(nuc_del)")
                        println(g, "$(nuc_del)")
                    end
                end
            end
        end
        seqnucmutsnodels = collect(seq_nuc_muts_no_dels[seq])
        synonymous_nuc__AA_sort, synonymous_nuc__context_sort, noncoding__noncoding_region_sort, noncoding_nuc__context_sort = synonymous_nuc_to_AA_and_noncoding_to_context(seq_pango[seq], seqnucmutsnodels)
#        synonymous_nucs_vec = String[]
#        synonymous_AA_vec = String[]
#        synonymous_context_vec = String[]
######################################################################################################
        if !isempty(seq_mixed_nucs_sort)
            mixed_nucs_pad = rpad("MIXED NUCS", 14)
               print("$(mixed_nucs_pad)| ")
            print(g, "$(mixed_nucs_pad)| ")
            seq_mixed_nucs_sort_join = join(seq_mixed_nucs_sort, ", ")
               println(seq_mixed_nucs_sort_join)
            println(g, seq_mixed_nucs_sort_join)
            println(); println(g)
        end
##############################################################################################
############# Below = SHORT version of syn nuc print, WITHOUT AA positions and nuc context
        if !isempty(synonymous_nuc__AA_sort)
            total_syn = length(synonymous_nuc__AA_sort)
            println("SYNONYMOUS NUC MUTATIONS - Total:$(total_syn)")
            syn_nuc_mut_vec = String[]
            for i in 1:length(synonymous_nuc__AA_sort)
                synnuc = synonymous_nuc__AA_sort[i][1]
                push!(syn_nuc_mut_vec, synnuc)
            end
            syn_nuc_string = join(syn_nuc_mut_vec, ", ")
            println("     $syn_nuc_string")
############# Below = LONG version of syn nuc print, WITH AA positions & nuc context
#            for i in 1:length(synonymous_nuc__AA_sort)
#                synnuc = synonymous_nuc__AA_sort[i][1]
#                synnuc_pad = rpad(synnuc, 7)
#                synAA = synonymous_nuc__AA_sort[i][2]
#                AAlen = length(synAA)
#                pad1 = (14 - AAlen)÷2
#                pad12 = " "^pad1*synAA
#                synAA_pad = rpad(pad12, 14)
#                syncontext = synonymous_nuc__context_sort[i][2]
#                premut_context = split(syncontext, "-->")[1]
#                postmut_context = split(syncontext, "-->")[2]
#                postpad = lpad(postmut_context, 38)
#                println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#                println(postpad)
#                println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#                println(g, postpad)
#            end
            println(); println(g)
        end
######################################################################################################
        if !isempty(noncoding__noncoding_region_sort)
            nc_len = length(noncoding__noncoding_region_sort)
            println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
            for i in 1:length(noncoding__noncoding_region_sort)
                nc_nuc = noncoding__noncoding_region_sort[i][1]
                nc_nuc_pad = rpad(nc_nuc, 7)
                nc_region = noncoding__noncoding_region_sort[i][2]
                nc_region_len = length(nc_region)
                ncpad1 = (13 - nc_region_len)÷2
                ncpad12 = " "^ncpad1*nc_region
                nc_region_pad2 = rpad(ncpad12, 13)
                nc_context = noncoding_nuc__context_sort[i][2]
                premut_context = split(nc_context, "-->")[1]
                postmut_context = split(nc_context, "-->")[2]
                postpad = lpad(postmut_context, 39)
                println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
                println(postpad)
                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
                println(g, postpad)
            end
        end
        println("#"^94); println(g, "#"^94)
##############################################################################################
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### function synonymous_nuc_to_AA_and_noncoding_to_context #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function synonymous_nuc_to_AA_and_noncoding_to_context(ref_pango::String, muts::Vector{String})
#    all_muts_sort = sort(muts, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
#########################################################################################################################
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int,Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int,Int}()
    nuc_gene_num_9b = Dict{Int,Int}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
#########################################################################################################################
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nucpos in gene_nuc_arr[i]
            nuc_gene_num[nucpos] = i-1
        end
    end
    for nucpos in gene_nuc_arr[end]
        nuc_gene_num_9b[nucpos] = 11
    end
    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
######################################################################################################################################
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                           ## Fx ##
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)   ## Fx ##
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                             ## Fx ##
    nuc2AA_ORF1a(nuc_mut,refAA,qryAA) = "$(gene_strings[gene_num(nuc_mut)]):$(refAA*nuc_to_AA_pos(nuc_mut))$qryAA" ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:$(refAA)$(nuc_to_AA_pos_9b(nuc_mut))$(qryAA)"   ## Fx ##
######################################################################################################################################
    nuc_codon_pos_dict = Dict{Int,Int}()
    for nucpos in coding_ranges
        gene_number = nuc_gene_num[nucpos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nucpos-gene_start)%3 + 1
        nuc_codon_pos_dict[nucpos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int,Int}()
    for nucpos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nucpos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nucpos] = codon_num
    end
#########################################################################
#    muts_v2 = Vector{String}()
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ", ")[1])
            mut2 = string(split(mut, ", ")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>9), muts)
        end
    end
######################################################################### 
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
#            mut = mixed2nuc(nuc_mut)  
            mut = nuc_mut
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end

            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                push!(AA_mut_set, AA_mut)
            end
###################################
            for nuc_mut2 in muts
                mut2 = nuc_mut2
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context
                        else
                            push!(AA_mut_set, AA_mut2)
                        end
                    end
                end
            end
        else
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if npos < 29890
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
                qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
                full_nc_context = ref_nc_nuc_context*"-->"*qry_nc_nuc_context
                noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if npos ≥ frst && npos ≤ last
                        noncoding_to_noncoding_region_dict[nuc_mut] = place
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
#            mut_9b = mixed2nuc(nuc_mut)
            mut_9b = nuc_mut
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            else
                push!(AA_mut_set, AA_mut_9b)
            end
#############################################################################
            for nuc_mut2_9b in muts
#                mut2_9b = mixed2nuc(nuc_mut2_9b)
                mut2_9b = nuc_mut2_9b
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            push!(AA_mut_set, AA_mut2_9b)
                        end
                    end
                end
            end
        end
    end
#####################################################################################################################################
#    for mut in AA_mut_set
#        if !(mut in aa_mut_all_and_chr_set)
#            push!(missing_gene_pos_AA_keys, mut)
#        end
#    end
#    AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
#    AA_sort2 = Vector{String}()
#    for mut in AA_sort
#        refAA = ref_AA_dict[mut]
#        qryAA = qry_AA_dict[mut]
#        if !(refAA == qryAA)
#            push!(AA_sort2, mut)
#        end
#    end
#####################################################################################################################################
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
############################################################
#    println("################## Noncoding Nuc Mutations ##################")
#    noncoding_nuc_total = length(keys(noncoding_nuc_to_context_dict))
#    println("    Total Number of Noncoding Nucs = $(noncoding_nuc_total)")
#    for nc_mut in noncoding_nuc_to_context_dict_sort
#        print("     $(nc_mut[1]) ($(nc_mut[2])), ")
#    end
#    println()
#    for i in 1:length(noncoding_nuc_to_context_dict_sort)
#        println("$(noncoding_nuc_to_context_dict_sort[i][1]) | $(noncoding_nuc_to_context_dict_sort[i][2])")
#        premut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[1]
#        postmut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[2]
#        println("    $(premut_nucseq)")
#        println("    $(postmut_nucseq)")
#    end
#    println("################# Synonymous Nuc Mutations #################")
#    synonymous_nuc_total = length(synonymous_nuc_sort)
#    println("  Total Number of Synonymous Nuc Muts = $(synonymous_nuc_total)")
#    synonymous_nuc_sort_join = join(synonymous_nuc_sort, ", ")
#    println("  ", synonymous_nuc_sort_join)
#    print("\n"^1)
#    for i in 1:length(synonymous_nuc_sort)
#        premut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
#        postmut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        postmut_cntxt_pad = lpad(postmut_nucseq, 13)
#        nucpad = rpad(synonymous_nuc_to_AA_dict_sort[i][1], 7)
#        println("     $(nucpad)|$(premut_nucseq)")
#        println("$(postmut_cntxt_pad)")
#    end
#    print("\n"^1)
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, synonymous_nuc_to_context_dict_sort, noncoding_to_noncoding_region_dict_sort, noncoding_nuc_to_context_dict_sort
end
#############################################################################

2026_03_07__600PM
2026_03_07__600PM


synonymous_nuc_to_AA_and_noncoding_to_context (generic function with 1 method)

In [8]:
#### Finds avg # of AA subs + AA deletion ranges per EPCI sequence
tot_AA_subs = sum(values(AA_muts_ct_no_dels))
total_seqs = length(non_rep_seqs) + length(rep_seq_grps_maxmut_seqs)
avg_AA_subs_per_seq_all_chronics = round(digits=2, tot_AA_subs/total_seqs)
println("Avg AA Subs per Chronic Sequence = $(avg_AA_subs_per_seq_all_chronics)")

Avg AA Subs per Chronic Sequence = 18.01


In [ ]:
### Fx: Disulfide counts for any given mutation
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
####################################################
unknown_forbidden = list_to_set("S:9, S:12, S:13, S:15, S:64, S:136, S:138, S:139, S:142, S:144, S:145, S:151, S:152, S:258")
disulfide_list = list_to_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15F, S:C15G, S:C15L, S:C15R, S:C15S, S:C15Y, S:R21I, S:W64C, S:S71C, S:C136-, S:C136F, S:C136G, S:C136H, S:C136R, S:C136Y, S:D138-, S:P139S, S:P139T, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:W258C, S:G261C")
C136_muts = list_to_set("S:C136-, S:C136F, S:C136G, S:C136H, S:C136R, S:C136Y")
non_C136_muts = list_to_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64C, S:S71C, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:W258C, S:G261C")
####################################################
function disulfide_counts_for_seqs_with_any_given_mutation(sel_mut::String)
    ds_mut_cts = Dict{String, Int}()
    sel_mut_seqs = Set{String}()
    sel_mut_seq_total = 0
    second_ds_mut_seq_total = 0
    ds_mut_total = 0
    no_ds_but_ds_dropout_seq_ct = 0
    ds_muts_minus_selmut = setdiff(disulfide_list, Set([sel_mut]))
    selmut_seq_spike_muts = Dict{String, String}()
    spikevec_sortkey(n) = parse(Int, n[2:end-1])
    for seq in all_unique_chr_seqs_set
        if sel_mut in seq_AA_muts[seq]
            sel_mut_seq_total += 1
            push!(sel_mut_seqs, seq)
            spike_vec = String[]
            for mut in seq_AA_muts[seq]
                if aa_gene_comprehensive_dict[mut] == "S"
                    mutonly = mut[3:end]
                    push!(spike_vec, mutonly)
                end
            end
            spike_vec_sort = sort(spike_vec, by = x -> spikevec_sortkey(x))
            spike_vec_sort_join = join(spike_vec_sort, ", ")
            selmut_seq_spike_muts[seq] = spike_vec_sort_join
            if !isempty(intersect(seq_AA_muts[seq], ds_muts_minus_selmut))
                second_ds_mut_seq_total += 1
                if "S:C136-" in seq_AA_muts[seq] && "S:D138-" in seq_AA_muts[seq]
                    second_ds_mut_seq_total -= 1
                end
            end
            if isempty(intersect(seq_AA_muts[seq], ds_muts_minus_selmut))
                if !isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
                    no_ds_but_ds_dropout_seq_ct += 1
                end
            end
            for ds_mut in ds_muts_minus_selmut
                if ds_mut in seq_AA_muts[seq]
                    ds_mut_total += 1
                    ds_mut_cts[ds_mut] = get(ds_mut_cts, ds_mut, 0) + 1
                end
            end
        end
    end
    ds_mut_cts_pos_sort = sort(collect(ds_mut_cts), by = x -> aa_pos_comprehensive_dict[x[1]])
    ds_mut_cts_count_sort = sort(collect(ds_mut_cts), by = x -> x[2], rev=true)
    selmut_seq_ct_minus_dropout_seqs = sel_mut_seq_total - no_ds_but_ds_dropout_seq_ct
    open("disulfide_mut_cts_for_seqs_with_$(sel_mut)__$(date_now).txt", "w") do g
        print("\n"^2); print(g, "\n"^2)
        println("Total $(sel_mut) Sequences = $(sel_mut_seq_total)")
        println(g, "Total $(sel_mut) Sequences = $(sel_mut_seq_total)")
        println("Total Disulfide Muts in $(sel_mut) Seqs = $(ds_mut_total)")
        println(g, "Total Disulfide Muts in $(sel_mut) Seqs = $(ds_mut_total)")
        println("Total $(sel_mut) Seqs w/ ≥1 other disulfide mut = $(second_ds_mut_seq_total)")
        println(g, "Total $(sel_mut) Seqs w/ ≥1 other disulfide mut = $(second_ds_mut_seq_total)")
        println("Total $(sel_mut) Seqs (minus those w/dropout in ds regions) = $(selmut_seq_ct_minus_dropout_seqs)")
        println(g, "Total $(sel_mut) Seqs (minus those w/dropout in ds regions) = $(selmut_seq_ct_minus_dropout_seqs)")
        print("\n"^1); print(g, "\n"^1)
        avg_ds_muts_per_seq = round(digits=2, ds_mut_total/selmut_seq_ct_minus_dropout_seqs)
        pct_seqs_with_at_least_one_ds_mut = round(digits=2, 100*second_ds_mut_seq_total/selmut_seq_ct_minus_dropout_seqs)
        println("Avg Disulfide Muts per $(sel_mut) Seq = $(avg_ds_muts_per_seq)")
        println(g, "Avg Disulfide Muts per $(sel_mut) Seq = $(avg_ds_muts_per_seq)")
        println("Pct of $(sel_mut) seqs w/ ≥ 1 other disulfide mut (and no disulfide-region dropout) = $(pct_seqs_with_at_least_one_ds_mut)%")
        println(g, "Pct of $(sel_mut) seqs w/ ≥ 1 other disulfide mut (and no disulfide-region dropout) = $(pct_seqs_with_at_least_one_ds_mut)%")
        print("\n"^2); print(g, "\n"^2)
        println("##### Disulfide Cts for seqs with $(sel_mut), sorted by count #####")
        println(g, "##### Disulfide Cts for seqs with $(sel_mut), sorted by count #####")
        for mut___ct in ds_mut_cts_count_sort
            mutpad = rpad(mut___ct[1], 7)
            ct_pad = lpad(mut___ct[2], 2)
            println("$(mutpad) = $(ct_pad)")
            println(g, "$(mutpad) = $(ct_pad)")
        end
        print("\n"^2); print(g, "\n"^2)
        println("##### Disulfide Cts for seqs with $(sel_mut), sorted by position #####")
        println(g, "##### Disulfide Cts for seqs with $(sel_mut), sorted by position #####")
        for mut___ct in ds_mut_cts_pos_sort
            mutpad = rpad(mut___ct[1], 7)
            ct_pad = lpad(mut___ct[2], 2)
            println("$(mutpad) = $(ct_pad)")
            println(g, "$(mutpad) = $(ct_pad)")
        end
        print("\n"^3)
        print(g, "\n"^3)
        for (seq, spikestring) in selmut_seq_spike_muts
            println("$(seq) | $(seq_pango[seq]) | $(spikestring)")     
        end
        for (seq, spikestring) in selmut_seq_spike_muts
            print_all_seq_info(seq, "temp_seq_info.txt")
        end
    end
end
########################################################################################################
########################################################################################################

In [ ]:
### Disulfide EPCI seq ct (0, 1, 2, 3+) with D138- ###
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
####################################################
unknown_forbidden = list_to_set("S:9, S:12, S:13, S:15, S:64, S:136, S:138, S:139, S:142, S:144, S:145, S:151, S:152, S:258")
disulfide_list = list_to_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15F, S:C15G, S:C15L, S:C15R, S:C15S, S:C15Y, S:R21I, S:W64C, S:S71C, S:C136-, S:C136F, S:C136G, S:C136H, S:C136R, S:C136Y, S:D138-, S:P139S, S:P139T, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:W258C, S:G261C")
C136_muts = list_to_set("S:C136-, S:C136F, S:C136G, S:C136H, S:C136R, S:C136Y")
non_C136_muts = list_to_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64C, S:S71C, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:W258C, S:G261C")
####################################################
zero_disulfide = 0
exactly_one_disulfide = 0
exactly_one_disulfide_plus_V143del_WT = 0
one_plus_dropout = 0
two_disulfide = 0
two_plus_disulfide = 0
C136_singlets_ct = 0
C136_ct = 0
####################################################
exactly_one_disulfide_seqs = Set{String}()
one_plus_dropout_seqs = Set{String}()
two_disulfide_seqs = Set{String}()
two_plus_disulfide_seqs = Set{String}()
C136_singlets = Set{String}()
C136_compensatory_mut_dict = Dict{String, Int}()
####################################################
exactly_one_disulfide_muts_dict = Dict{String, Int}()
for mut in disulfide_list
    exactly_one_disulfide_muts_dict[mut] = 0
end
####################################################
for seq in all_unique_chr_seqs_set
    disulfide_mut_ct = 0
    if "S:C136-" in seq_AA_muts[seq] && "S:D138-" in seq_AA_muts[seq]
        disulfide_mut_ct -= 1
    end
    disulfides = intersect(seq_AA_muts[seq], disulfide_list)
    C136_disulfides = intersect(seq_AA_muts[seq], C136_muts)
    non_C136_disulfides = intersect(seq_AA_muts[seq], non_C136_muts)
    disulfide_seq_total = length(disulfides)
    disulfide_mut_ct += disulfide_seq_total
    if disulfide_mut_ct == 0
        zero_disulfide += 1
    elseif disulfide_mut_ct == 1 && isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
        exactly_one_disulfide += 1
        push!(exactly_one_disulfide_seqs, seq)
        for mut in disulfides
            exactly_one_disulfide_muts_dict[mut] += 1
        end
        if "S:V143-" in seq_AA_muts_WT[seq] && !("S:D138-" in seq_AA_muts_WT[seq])
            exactly_one_disulfide_plus_V143del_WT += 1
        end
    elseif disulfide_mut_ct == 1 && !isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
        one_plus_dropout += 1
    elseif disulfide_mut_ct ≥ 2
        two_disulfide += 1
    end
    if disulfide_mut_ct ≥ 3 
        two_plus_disulfide += 1
    end
    if !isempty(intersect(C136_muts, seq_AA_muts[seq])) && isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
        C136_ct += 1
        for mut in non_C136_disulfides
            C136_compensatory_mut_dict[mut] = get(C136_compensatory_mut_dict, mut, 0) + 1
        end
        if seq in exactly_one_disulfide_seqs
            push!(C136_singlets, seq)
            C136_singlets_ct += 1
        end
    end
end
total_one_plus_disulfide = exactly_one_disulfide + two_disulfide
one_of_pct = round(digits=1, 100*exactly_one_disulfide/total_one_plus_disulfide)
two_of_pct = round(digits=1, 100*two_disulfide/total_one_plus_disulfide)
EPCI_total_nondropout = length(all_unique_chr_seqs_set) - one_plus_dropout
one_plus_disulfide_EPCI_pct = round(digits=1, 100*(exactly_one_disulfide + two_disulfide)/EPCI_total_nondropout)
###########################
C136_singlets_sort = sort(collect(C136_singlets), by = x -> (length(x), x))
C136_singlets_sort_join = join(C136_singlets_sort, ", ")
exactly_one_disulfide_muts_dict_sort = sort(collect(exactly_one_disulfide_muts_dict), by = x -> x[2], rev=true)
###########################
exactly_one_disulfide_seqs_sort = sort(collect(exactly_one_disulfide_seqs), by = x -> (length(x), x))
exactly_one_disulfide_seqs_sort_join = join(exactly_one_disulfide_seqs_sort, ", ")
###########################
C136_compensatory_mut_dict_pos_sort = sort(collect(C136_compensatory_mut_dict), by = x -> aa_pos_comprehensive_dict[x[1]])
C136_compensatory_mut_dict_count_sort = sort(collect(C136_compensatory_mut_dict), by = x -> x[2], rev=true)
###########################
open("EPCI_Disulfide_Stats__EPCI_$(EPCI_qc_str)_$(date_now).txt", "w") do g
    print("\n"^1)
    println("######## EPCI Disulfide Stats EPCI $(EPCI_qc_str) ###########")
    println("zero_disulfide = $(zero_disulfide)")
    println("exactly_one_disulfide = $(exactly_one_disulfide)")
    println("exactly_one_disulfide_plus_V143del_WT = $(exactly_one_disulfide_plus_V143del_WT)")
    println("one_plus_dropout = $(one_plus_dropout)")
    println("two_disulfide = $(two_disulfide)")
    println("two_plus_disulfide = $(two_plus_disulfide)")
    print("\n"^1)
    println("Exactly-1 disulfide pct = $(one_of_pct)")
    println("Two-plus disulfide pct = $(two_of_pct)")
    println("1+ disulfide EPCI pct = $(one_plus_disulfide_EPCI_pct)")
    print("\n"^1)
    println("Total C136 mut count = $(C136_ct)")
    print("\n"^1)
    println("C136 Mut + No Major Others Count = $(C136_singlets_ct)")
    print("\n"^2)
    println("####### Exactly-one Disulfide Mut Counts #######")
    for mut___ct in exactly_one_disulfide_muts_dict_sort
        mut = mut___ct[1]
        ct = mut___ct[2]
        mutpad = rpad(mut, 12)
        ctpad = lpad(ct, 2)
        println("$(mutpad) = $(ctpad)")
    end
    print("\n"^2)
    println("##### C136 Mut + No Major Others #####")
    println(C136_singlets_sort_join)
    print("\n"^2)
    println("##### C136 Compensatory Mut Cts #####")
    for mut___ct in C136_compensatory_mut_dict_count_sort
        mutpad = rpad(mut___ct[1], 7)
        ct_pad = lpad(mut___ct[2], 2)
        println("$(mutpad) = $(ct_pad)")
    end
    print("\n"^2)
    println("####### Exactly-one Disulfide Mut Sequences #######")
    println("Total Count: $(length(exactly_one_disulfide_seqs))")
    println(exactly_one_disulfide_seqs_sort_join)
    print("\n"^2)
######################################
    print(g, "\n"^1)
    println(g, "######## EPCI Disulfide Stats EPCI $(EPCI_qc_str) ###########")
    println(g, "zero_disulfide = $(zero_disulfide)")
    println(g, "exactly_one_disulfide = $(exactly_one_disulfide)")
    println(g, "one_plus_dropout = $(one_plus_dropout)")
    println(g, "two_disulfide = $(two_disulfide)")
    println(g, "two_plus_disulfide = $(two_plus_disulfide)")
    print(g, "\n"^1)
    println(g, "Exactly-1 disulfide pct = $(one_of_pct)")
    println(g, "Two-plus disulfide pct = $(two_of_pct)")
    println(g, "1+ disulfide EPCI pct = $(one_plus_disulfide_EPCI_pct)")
    print(g, "\n"^1)
    println(g, "C136 Mut + No Major Others Count = $(C136_singlets_ct)")
    print(g, "\n"^1)
    println(g, "Total C136 mut count = $(C136_ct)")
    print(g, "\n"^2)
    println(g, "####### Exactly-one Disulfide Mut Counts #######")
    for mut___ct in exactly_one_disulfide_muts_dict_sort
        mut = mut___ct[1]
        ct = mut___ct[2]
        mutpad = rpad(mut, 12)
        ctpad = lpad(ct, 2)
        println(g, "$(mutpad) = $(ctpad)")
    end
    print(g, "\n"^2)
    println(g, "##### C136 Mut + No Major Others #####")
    println(g, C136_singlets_sort_join)
    print(g, "\n"^2)
    println(g, "##### C136 Compensatory Mut Cts #####")
    for mut___ct in C136_compensatory_mut_dict_count_sort
        mutpad = rpad(mut___ct[1], 7)
        ct_pad = lpad(mut___ct[2], 2)
        println(g, "$(mutpad) = $(ct_pad)")
    end
    print(g, "\n"^2)
    println(g, "####### Exactly-one Disulfide Mut Sequences #######")
    println(g, "Total Count: $(length(exactly_one_disulfide_seqs))")
    println(g, exactly_one_disulfide_seqs_sort_join)
    print(g, "\n"^2)
end
########################################################################################
########################################################################################

In [ ]:
### Disulfide EPCI seq ct (0, 1, 2, 3+) without D138-
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
####################################################
unknown_forbidden = list_to_set("S:9, S:12, S:13, S:15, S:64, S:136, S:138, S:139, S:142, S:144, S:145, S:151, S:152, S:258")
disulfide_list = list_to_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15F, S:C15G, S:C15L, S:C15R, S:C15S, S:C15Y, S:R21I, S:W64C, S:S71C, S:C136-, S:C136F, S:C136G, S:C136H, S:C136R, S:C136Y, S:P139S, S:P139T, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:W258C, S:G261C")
C136_muts = list_to_set("S:C136-, S:C136F, S:C136G, S:C136H, S:C136R, S:C136Y")
non_C136_muts = list_to_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64C, S:S71C, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:W258C, S:G261C")
####################################################
zero_disulfide = 0
exactly_one_disulfide = 0
exactly_one_disulfide_plus_V143del_WT = 0
one_plus_dropout = 0
two_disulfide = 0
two_plus_disulfide = 0
C136_singlets_ct = 0
C136_ct = 0
####################################################
exactly_one_disulfide_seqs = Set{String}()
one_plus_dropout_seqs = Set{String}()
two_disulfide_seqs = Set{String}()
two_plus_disulfide_seqs = Set{String}()
C136_singlets = Set{String}()
C136_compensatory_mut_dict = Dict{String, Int}()
####################################################
exactly_one_disulfide_muts_dict = Dict{String, Int}()
for mut in disulfide_list
    exactly_one_disulfide_muts_dict[mut] = 0
end
####################################################
for seq in all_unique_chr_seqs_set
    disulfide_mut_ct = 0
    disulfides = intersect(seq_AA_muts[seq], disulfide_list)
    C136_disulfides = intersect(seq_AA_muts[seq], C136_muts)
    non_C136_disulfides = intersect(seq_AA_muts[seq], non_C136_muts)
    disulfide_seq_total = length(disulfides)
    disulfide_mut_ct += disulfide_seq_total
    if disulfide_mut_ct == 0
        zero_disulfide += 1
    elseif disulfide_mut_ct == 1 && isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
        exactly_one_disulfide += 1
        push!(exactly_one_disulfide_seqs, seq)
        for mut in disulfides
            exactly_one_disulfide_muts_dict[mut] += 1
        end
        if "S:V143-" in seq_AA_muts_WT[seq] && !("S:D138-" in seq_AA_muts_WT[seq])
            exactly_one_disulfide_plus_V143del_WT += 1
        end
    elseif disulfide_mut_ct == 1 && !isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
        one_plus_dropout += 1
    elseif disulfide_mut_ct ≥ 2
        two_disulfide += 1
    end
    if disulfide_mut_ct ≥ 3 
        two_plus_disulfide += 1
    end
    if !isempty(intersect(C136_muts, seq_AA_muts[seq])) && isempty(intersect(seq_unknown_AA[seq], unknown_forbidden))
        C136_ct += 1
        for mut in non_C136_disulfides
            C136_compensatory_mut_dict[mut] = get(C136_compensatory_mut_dict, mut, 0) + 1
        end
        if seq in exactly_one_disulfide_seqs
            push!(C136_singlets, seq)
            C136_singlets_ct += 1
        end
    end
end
total_one_plus_disulfide = exactly_one_disulfide + two_disulfide
one_of_pct = round(digits=1, 100*exactly_one_disulfide/total_one_plus_disulfide)
two_of_pct = round(digits=1, 100*two_disulfide/total_one_plus_disulfide)
EPCI_total_nondropout = length(all_unique_chr_seqs_set) - one_plus_dropout
one_plus_disulfide_EPCI_pct = round(digits=1, 100*(exactly_one_disulfide + two_disulfide)/EPCI_total_nondropout)
###########################
C136_singlets_sort = sort(collect(C136_singlets), by = x -> (length(x), x))
C136_singlets_sort_join = join(C136_singlets_sort, ", ")
exactly_one_disulfide_muts_dict_sort = sort(collect(exactly_one_disulfide_muts_dict), by = x -> x[2], rev=true)
###########################
exactly_one_disulfide_seqs_sort = sort(collect(exactly_one_disulfide_seqs), by = x -> (length(x), x))
exactly_one_disulfide_seqs_sort_join = join(exactly_one_disulfide_seqs_sort, ", ")
###########################
C136_compensatory_mut_dict_pos_sort = sort(collect(C136_compensatory_mut_dict), by = x -> aa_pos_comprehensive_dict[x[1]])
C136_compensatory_mut_dict_count_sort = sort(collect(C136_compensatory_mut_dict), by = x -> x[2], rev=true)
###########################
open("EPCI_Disulfide_Stats__EPCI_$(EPCI_qc_str)_$(date_now).txt", "w") do g
    print("\n"^1)
    println("######## EPCI Disulfide Stats EPCI $(EPCI_qc_str) (no D138-) ###########")
    println("zero_disulfide = $(zero_disulfide)")
    println("exactly_one_disulfide = $(exactly_one_disulfide)")
    println("exactly_one_disulfide_plus_V143del_WT = $(exactly_one_disulfide_plus_V143del_WT)")
    println("one_plus_dropout = $(one_plus_dropout)")
    println("two_disulfide = $(two_disulfide)")
    println("two_plus_disulfide = $(two_plus_disulfide)")
    print("\n"^1)
    println("Exactly-1 disulfide pct = $(one_of_pct)")
    println("Two-plus disulfide pct = $(two_of_pct)")
    println("1+ disulfide EPCI pct = $(one_plus_disulfide_EPCI_pct)")
    print("\n"^1)
    println("Total C136 mut count = $(C136_ct)")
    print("\n"^1)
    println("C136 Mut + No Major Others Count = $(C136_singlets_ct)")
    print("\n"^2)
    println("####### Exactly-one Disulfide Mut Counts (no D138-) #######")
    for mut___ct in exactly_one_disulfide_muts_dict_sort
        mut = mut___ct[1]
        ct = mut___ct[2]
        mutpad = rpad(mut, 12)
        ctpad = lpad(ct, 2)
        println("$(mutpad) = $(ctpad)")
    end
    print("\n"^2)
    println("##### C136 Mut + No Major Others (no D138-) #####")
    println(C136_singlets_sort_join)
    print("\n"^2)
    println("##### C136 Compensatory Mut Cts (no D138-) #####")
    for mut___ct in C136_compensatory_mut_dict_count_sort
        mutpad = rpad(mut___ct[1], 7)
        ct_pad = lpad(mut___ct[2], 2)
        println("$(mutpad) = $(ct_pad)")
    end
    print("\n"^2)
    println("####### Exactly-one Disulfide Mut Sequences (no D138-) #######")
    println("Total Count: $(length(exactly_one_disulfide_seqs))")
    println(exactly_one_disulfide_seqs_sort_join)
    print("\n"^2)
######################################
    print(g, "\n"^1)
    println(g, "######## EPCI Disulfide Stats EPCI $(EPCI_qc_str) (no D138-) ###########")
    println(g, "zero_disulfide = $(zero_disulfide)")
    println(g, "exactly_one_disulfide = $(exactly_one_disulfide)")
    println(g, "one_plus_dropout = $(one_plus_dropout)")
    println(g, "two_disulfide = $(two_disulfide)")
    println(g, "two_plus_disulfide = $(two_plus_disulfide)")
    print(g, "\n"^1)
    println(g, "Exactly-1 disulfide pct = $(one_of_pct)")
    println(g, "Two-plus disulfide pct = $(two_of_pct)")
    println(g, "1+ disulfide EPCI pct = $(one_plus_disulfide_EPCI_pct)")
    print(g, "\n"^1)
    println(g, "C136 Mut + No Major Others Count = $(C136_singlets_ct)")
    print(g, "\n"^1)
    println(g, "Total C136 mut count = $(C136_ct)")
    print(g, "\n"^2)
    println(g, "####### Exactly-one Disulfide Mut Counts (no D138-) #######")
    for mut___ct in exactly_one_disulfide_muts_dict_sort
        mut = mut___ct[1]
        ct = mut___ct[2]
        mutpad = rpad(mut, 12)
        ctpad = lpad(ct, 2)
        println(g, "$(mutpad) = $(ctpad)")
    end
    print(g, "\n"^2)
    println(g, "##### C136 Mut + No Major Others (no D138-) #####")
    println(g, C136_singlets_sort_join)
    print(g, "\n"^2)
    println(g, "##### C136 Compensatory Mut Cts (no D138-) #####")
    for mut___ct in C136_compensatory_mut_dict_count_sort
        mutpad = rpad(mut___ct[1], 7)
        ct_pad = lpad(mut___ct[2], 2)
        println(g, "$(mutpad) = $(ct_pad)")
    end
    print(g, "\n"^2)
    println(g, "####### Exactly-one Disulfide Mut Sequences (no D138-) #######")
    println(g, "Total Count: $(length(exactly_one_disulfide_seqs))")
    println(g, exactly_one_disulfide_seqs_sort_join)
    print(g, "\n"^2)
end
########################################################################################
########################################################################################

In [96]:
### Getting RBM/RBD muts + ORF9b/N Doubles + artifactual_private_muts_subs + subs/pos_only + sub2pos/pos2sub | Runtime = 9.1 sec
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
                                                        none_0___sub2pos_1___pos2sub_2 = 0
                                                        sub_0__posonly_1__mixed_2 = 0
                                                        normal_0__spikeonly_1 = 0
                                                        noBAL_0__withBAL_1 = 1
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
if none_0___sub2pos_1___pos2sub_2 == 0 && sub_0__posonly_1__mixed_2 == 2
    println("WARNING: INCOMPATIBLE none_0___sub2pos_1___pos2sub_2 and sub_0__posonly_1__mixed_2 parameters!!!"^5)
elseif none_0___sub2pos_1___pos2sub_2 ≠ 0 && sub_0__posonly_1__mixed_2 ≠ 2
    println("WARNING: INCOMPATIBLE none_0___sub2pos_1___pos2sub_2 and sub_0__posonly_1__mixed_2 parameters!!!"^5)
end
########################################################################################################################################################################### 
############################################################# Begin EXCLUDED MUTS SECTION #################################################################################
########################################################################################################################################################################### 
##### The mutations added below are ones that have been found to produce artifactual correlations. For example, 18/19 EPCI sequences 
####      with ORF1a:M2606I are B.1.2 (the other is likely a B.1.2 miscategorized as a B.1), and seven of these have N:D377Y, indicating
####      that these are both inherited (not private) mutations, which can be confirmed by viewing the Usher Tree for these sequences. 
####      Nextclade miscategorizes both of these as private mutations. Similarly, 5/6 sequences with ORF1a:E3764D are B.1.258, and all five also 
####      have ORF3a:A72S. Both are inherited and mischaracterized as private by Nextclade. 
global artifactual_private_muts_subs = Set{String}()
#push!(artifactual_private_muts_subs, "ORF1a:T2501I") ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_subs, "ORF1a:M2606I") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "N:D377Y")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "ORF3a:A72S")   ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:E3764D") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:I2283T") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF1a:A599T")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "S:F59S")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF8:L60F")    ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "ORF1a:H2125Y") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "S:Q146K")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
push!(artifactual_private_muts_subs, "N:M203K")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:G204R")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:R204G")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_subs, "ORF1b:M1156I") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_subs, "ORF:L61D")     ## Artifactual 3-nuc reversion
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## Omicron mut misattributed as private in 3 recombinants
push!(artifactual_private_muts_subs, "M:A30T")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_subs, "ORF1a:S1612L") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:E39*")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:F18L")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_subs, "ORF1a:A2554V") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1b:H1087Y") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:M2796T") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:P1803S") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:P104S")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "N:A208S")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF8:K68*")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1975S") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:V2178F") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:I97V")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A2589T") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF9b:T83I")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T842I")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF3a:V48F")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:G49C")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A1204T") ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:A376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:T376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:F375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:S375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:D339G")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:N417K")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:K440N")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "ORF1b:T1555I") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A4285V") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:N2361K") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:K1191N")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:R158G")      ## Artifactual Delta reversion
push!(artifactual_private_muts_subs, "ORF1a:I1023-") ## Dumb mistake I made that I have to undo with this.
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF7b:I2V")    ## Corresponds to ORF7a:*122W
push!(artifactual_private_muts_subs, "ORF1b:P218L")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G2118D") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_subs, "ORF1a:A1298V") ## Inherited BA.1 mut, miscategorized by Nextclade as private


push!(artifactual_private_muts_subs, "ORF3a:V112F")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2695I") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:I35K")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
######################################################################################################################################
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
#### This list was created in one of the above cells (the one with the ORF9b_N_nucmuts_to_AAres function) & then copied & pasted here.
global Double_N_ORF9b_muts = Set(["N:F33S", "N:L13-", "N:L13I", "N:L13H", "N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:G63D", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])   
# global  Double_N_ORF9b_muts = Set(["ORF9b:M1L", "ORF9b:M1L", "ORF9b:M1V", "ORF9b:M1K", "ORF9b:M1R", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:D2Y", "ORF9b:D2H", "ORF9b:D2N", "ORF9b:D2E", "ORF9b:D2E", "ORF9b:P3S", "ORF9b:P3T", "ORF9b:P3A", "ORF9b:K4*", "ORF9b:K4Q", "ORF9b:K4E", "ORF9b:K4I", "ORF9b:K4T", "ORF9b:K4N", "ORF9b:K4N", "ORF9b:I5F", "ORF9b:I5L", "ORF9b:I5V", "ORF9b:I5N", "ORF9b:I5S", "ORF9b:I5M", "ORF9b:S6C", "ORF9b:S6R", "ORF9b:S6G", "ORF9b:S6I", "ORF9b:S6T", "ORF9b:S6R", "ORF9b:E7*", "ORF9b:E7Q", "ORF9b:E7K", "ORF9b:E7D", "ORF9b:E7D", "ORF9b:M8L", "ORF9b:M8L", "ORF9b:M8V", "ORF9b:M8K", "ORF9b:M8R", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:H9Y", "ORF9b:H9N", "ORF9b:H9D", "ORF9b:H9Q", "ORF9b:P10S", "ORF9b:P10T", "ORF9b:P10A", "ORF9b:A11S", "ORF9b:A11P", "ORF9b:A11T", "ORF9b:L12I", "ORF9b:L12V", "ORF9b:L12*", "ORF9b:L12F", "ORF9b:L12F", "ORF9b:R13C", "ORF9b:R13S", "ORF9b:R13G", "ORF9b:L14M", "ORF9b:L14V", "ORF9b:L14*", "ORF9b:L14W", "ORF9b:L14F", "ORF9b:L14F", "ORF9b:V15L", "ORF9b:V15L", "ORF9b:V15M", "ORF9b:D16Y", "ORF9b:D16H", "ORF9b:D16N", "ORF9b:D16E", "ORF9b:D16E", "ORF9b:P17S", "ORF9b:P17T", "ORF9b:P17A", "ORF9b:Q18*", "ORF9b:Q18K", "ORF9b:Q18E", "ORF9b:Q18H", "ORF9b:Q18H", "ORF9b:I19F", "ORF9b:I19L", "ORF9b:I19V", "ORF9b:I19N", "ORF9b:I19S", "ORF9b:I19M", "ORF9b:Q20*", "ORF9b:Q20K", "ORF9b:Q20E", "ORF9b:Q20H", "ORF9b:Q20H", "ORF9b:L21M", "ORF9b:L21V", "ORF9b:A22S", "ORF9b:A22P", "ORF9b:A22T", "ORF9b:V23L", "ORF9b:V23L", "ORF9b:V23I", "ORF9b:V23E", "ORF9b:V23G", "ORF9b:T24S", "ORF9b:T24P", "ORF9b:T24A", "ORF9b:T24N", "ORF9b:T24S", "ORF9b:R25*", "ORF9b:R25G", "ORF9b:R25I", "ORF9b:R25T", "ORF9b:R25S", "ORF9b:R25S", "ORF9b:M26L", "ORF9b:M26L", "ORF9b:M26V", "ORF9b:M26K", "ORF9b:M26R", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:E27*", "ORF9b:E27Q", "ORF9b:E27K", "ORF9b:E27D", "ORF9b:E27D", "ORF9b:N28Y", "ORF9b:N28H", "ORF9b:N28D", "ORF9b:N28I", "ORF9b:N28T", "ORF9b:N28K", "ORF9b:N28K", "ORF9b:A29S", "ORF9b:A29P", "ORF9b:A29T", "ORF9b:V30L", "ORF9b:V30L", "ORF9b:V30M", "ORF9b:V30E", "ORF9b:V30G", "ORF9b:G31W", "ORF9b:G31R", "ORF9b:G31R", "ORF9b:R32C", "ORF9b:R32S", "ORF9b:R32G", "ORF9b:D33Y", "ORF9b:D33H", "ORF9b:D33N", "ORF9b:D33E", "ORF9b:D33E", "ORF9b:Q34*", "ORF9b:Q34K", "ORF9b:Q34E", "ORF9b:Q34H", "ORF9b:Q34H", "ORF9b:N35Y", "ORF9b:N35H", "ORF9b:N35D", "ORF9b:N35I", "ORF9b:N35T", "ORF9b:N35K", "ORF9b:N35K", "ORF9b:N36Y", "ORF9b:N36H", "ORF9b:N36D", "ORF9b:N36I", "ORF9b:N36T", "ORF9b:N36K", "ORF9b:N36K", "ORF9b:V37F", "ORF9b:V37L", "ORF9b:V37I", "ORF9b:G38C", "ORF9b:G38R", "ORF9b:G38S", "ORF9b:P39S", "ORF9b:P39T", "ORF9b:P39A", "ORF9b:K40*", "ORF9b:K40Q", "ORF9b:K40E", "ORF9b:K40M", "ORF9b:K40T", "ORF9b:K40N", "ORF9b:K40N", "ORF9b:V41F", "ORF9b:V41L", "ORF9b:V41I", "ORF9b:Y42H", "ORF9b:Y42N", "ORF9b:Y42D", "ORF9b:Y42F", "ORF9b:Y42S", "ORF9b:Y42*", "ORF9b:Y42*", "ORF9b:P43S", "ORF9b:P43T", "ORF9b:P43A", "ORF9b:I44L", "ORF9b:I44L", "ORF9b:I44V", "ORF9b:I44K", "ORF9b:I44R", "ORF9b:I44M", "ORF9b:I45L", "ORF9b:I45L", "ORF9b:I45V", "ORF9b:I45K", "ORF9b:I45R", "ORF9b:I45M", "ORF9b:L46M", "ORF9b:L46V", "ORF9b:R47C", "ORF9b:R47S", "ORF9b:R47G", "ORF9b:L48F", "ORF9b:L48I", "ORF9b:L48V", "ORF9b:G49C", "ORF9b:G49R", "ORF9b:G49S", "ORF9b:G49V", "ORF9b:G49A", "ORF9b:G49D", "ORF9b:S50P", "ORF9b:S50T", "ORF9b:S50A", "ORF9b:S50*", "ORF9b:S50*", "ORF9b:P51S", "ORF9b:P51T", "ORF9b:P51A", "ORF9b:L52F", "ORF9b:L52I", "ORF9b:L52V", "ORF9b:S53P", "ORF9b:S53T", "ORF9b:S53A", "ORF9b:L54F", "ORF9b:L54I", "ORF9b:L54V", "ORF9b:N55Y", "ORF9b:N55H", "ORF9b:N55D", "ORF9b:N55I", "ORF9b:N55T", "ORF9b:N55K", "ORF9b:N55K", "ORF9b:M56L", "ORF9b:M56L", "ORF9b:M56V", "ORF9b:M56K", "ORF9b:M56R", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:A57S", "ORF9b:A57P", "ORF9b:A57T", "ORF9b:R58W", "ORF9b:R58G", "ORF9b:R58M", "ORF9b:R58T", "ORF9b:R58S", "ORF9b:R58S", "ORF9b:K59*", "ORF9b:K59Q", "ORF9b:K59E", "ORF9b:K59M", "ORF9b:K59T", "ORF9b:K59N", "ORF9b:K59N", "ORF9b:T60S", "ORF9b:T60P", "ORF9b:T60A", "ORF9b:T60N", "ORF9b:T60S", "ORF9b:L61I", "ORF9b:L61V", "ORF9b:L61F", "ORF9b:L61F", "ORF9b:N62Y", "ORF9b:N62H", "ORF9b:N62D", "ORF9b:N62I", "ORF9b:N62T", "ORF9b:N62K", "ORF9b:N62K", "ORF9b:S63P", "ORF9b:S63T", "ORF9b:S63A", "ORF9b:S63Y", "ORF9b:S63C", "ORF9b:L64F", "ORF9b:L64I", "ORF9b:L64V", "ORF9b:E65*", "ORF9b:E65Q", "ORF9b:E65K", "ORF9b:E65D", "ORF9b:E65D", "ORF9b:D66Y", "ORF9b:D66H", "ORF9b:D66N", "ORF9b:D66E", "ORF9b:D66E", "ORF9b:K67*", "ORF9b:K67Q", "ORF9b:K67E", "ORF9b:K67M", "ORF9b:K67T", "ORF9b:K67N", "ORF9b:K67N", "ORF9b:A68S", "ORF9b:A68P", "ORF9b:A68T", "ORF9b:F69L", "ORF9b:F69I", "ORF9b:F69V", "ORF9b:F69L", "ORF9b:F69L", "ORF9b:Q70*", "ORF9b:Q70K", "ORF9b:Q70E", "ORF9b:Q70H", "ORF9b:Q70H", "ORF9b:L71I", "ORF9b:L71V", "ORF9b:L71*", "ORF9b:L71F", "ORF9b:L71F", "ORF9b:T72S", "ORF9b:T72P", "ORF9b:T72A", "ORF9b:T72K", "ORF9b:T72R", "ORF9b:P73S", "ORF9b:P73T", "ORF9b:P73A", "ORF9b:I74L", "ORF9b:I74L", "ORF9b:I74V", "ORF9b:I74K", "ORF9b:I74R", "ORF9b:I74M", "ORF9b:A75S", "ORF9b:A75P", "ORF9b:A75T", "ORF9b:A75E", "ORF9b:A75G", "ORF9b:V76F", "ORF9b:V76L", "ORF9b:V76I", "ORF9b:V76D", "ORF9b:V76G", "ORF9b:Q77*", "ORF9b:Q77K", "ORF9b:Q77E", "ORF9b:Q77H", "ORF9b:Q77H", "ORF9b:M78L", "ORF9b:M78L", "ORF9b:M78V", "ORF9b:M78K", "ORF9b:M78R", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:T79S", "ORF9b:T79P", "ORF9b:T79A", "ORF9b:T79N", "ORF9b:T79S", "ORF9b:K80*", "ORF9b:K80Q", "ORF9b:K80E", "ORF9b:K80I", "ORF9b:K80T", "ORF9b:K80N", "ORF9b:K80N", "ORF9b:L81M", "ORF9b:L81V", "ORF9b:L81W", "ORF9b:L81F", "ORF9b:L81F", "ORF9b:A82S", "ORF9b:A82P", "ORF9b:A82T", "ORF9b:T83S", "ORF9b:T83P", "ORF9b:T83A", "ORF9b:T83N", "ORF9b:T83S", "ORF9b:T84S", "ORF9b:T84P", "ORF9b:T84A", "ORF9b:T84N", "ORF9b:T84S", "ORF9b:E85*", "ORF9b:E85Q", "ORF9b:E85K", "ORF9b:E85D", "ORF9b:E86*", "ORF9b:E86Q", "ORF9b:E86K", "ORF9b:E86V", "ORF9b:E86A", "ORF9b:E86D", "ORF9b:E86D", "ORF9b:L87I", "ORF9b:L87V", "ORF9b:P88S", "ORF9b:P88T", "ORF9b:P88A", "ORF9b:D89Y", "ORF9b:D89H", "ORF9b:D89N", "ORF9b:D89V", "ORF9b:D89A", "ORF9b:D89E", "ORF9b:E90*", "ORF9b:E90Q", "ORF9b:E90K", "ORF9b:E90D", "ORF9b:E90D", "ORF9b:F91L", "ORF9b:F91I", "ORF9b:F91V", "ORF9b:F91C", "ORF9b:F91L", "ORF9b:F91L", "ORF9b:V92L", "ORF9b:V92L", "ORF9b:V92M", "ORF9b:V93L", "ORF9b:V93L", "ORF9b:V93M", "ORF9b:V94L", "ORF9b:V94L", "ORF9b:V94M", "ORF9b:T95S", "ORF9b:T95P", "ORF9b:T95A", "ORF9b:T95K", "ORF9b:T95R", "ORF9b:V96L", "ORF9b:V96L", "ORF9b:V96I", "ORF9b:K97*", "ORF9b:K97Q", "ORF9b:K97E", "ORF9b:K97I", "ORF9b:K97T", "ORF9b:K97N", "ORF9b:K97N", "ORF9b:*98R", "ORF9b:*98R", "ORF9b:*98G", "ORF9b:*98L", "ORF9b:*98S", "ORF9b:*98C", "ORF9b:*98C", "ORF9b:*98W"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_OG = list_to_strings_set("ORF1a:T2183I, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:F374L, S:F375-, S:A376V, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659P, S:A668V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:L1186F, E:V5A, E:V5F, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:F23S, E:T30I, E:A36V, E:Y42C, M:A2V, M:S4P, M:V10I, M:F28S, M:T77N, M:S94R, M:S94N, M:H125Y, M:H148R, M:A188T, N:P80L, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*") 
artifactual_private_muts_pos_only = Set{String}()
#push!(artifactual_private_muts_pos_only, "ORF1a:2501) ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_pos_only, "ORF1a:2606") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "N:377")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "ORF3a:72")   ## Inherited B.1.258 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:3764") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:2283") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF1a:599")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "S:59")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF8:L60")    ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "ORF1a:2125") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "S:146")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
#push!(artifactual_private_muts_pos_only, "N:203")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_pos_only, "ORF1b:1156") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_pos_only, "ORF:61")     ## Artifactual 3-nuc reversion
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## Omicron mut misattributed as private in 3 recombinants
#push!(artifactual_private_muts_pos_only, "M:30")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_pos_only, "ORF1a:1612") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:39")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:18")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_pos_only, "ORF1a:2554") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1b:1087") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:2796") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:1803") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:104")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "N:208")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF8:68")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1975") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2178") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:97")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2589") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF9b:83")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:842")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF3a:48")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:49")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1204") ## Inherited BE.1 mutation miscategorized by Nextclade as private
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:339")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:417")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:440")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_pos_only, "ORF1b:1555") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:4285") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2361") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:1191")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:44")   ## Deletions here that maintain the stop codon get misinterpreted as substitutions
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF7b:1")    ## Necessarily overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF7b:2")    ## Overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF1b:218")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2118") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_pos_only, "ORF1a:1298") ## Inherited BA.1 mut, miscategorized by Nextclade as private


push!(artifactual_private_muts_pos_only, "ORF3a:112")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2695") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:35")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
global Double_N_ORF9b_muts_pos_only = Set(["N:4", "N:5", "N:6", "N:7", "N:8", "N:9", "N:10", "N:11", "N:12", "N:13", "N:14", "N:15", "N:16", "N:17", "N:18", "N:19", "N:20", "N:21", "N:22", "N:23", "N:24", "N:25", "N:26", "N:27", "N:28", "N:29", "N:30", "N:31", "N:32", "N:33", "N:34", "N:35", "N:36", "N:37", "N:38", "N:39", "N:40", "N:41", "N:42", "N:43", "N:44", "N:45", "N:46", "N:47", "N:48", "N:49", "N:50", "N:51", "N:52", "N:53", "N:54", "N:55", "N:56", "N:57", "N:58", "N:59", "N:60", "N:61", "N:62", "N:63", "N:64", "N:65", "N:66", "N:67", "N:68", "N:69", "N:70", "N:71", "N:72", "N:73", "N:74", "N:75", "N:76", "N:77", "N:78", "N:79", "N:80", "N:81", "N:82", "N:83", "N:84", "N:85", "N:86", "N:87", "N:88", "N:89", "N:90", "N:91", "N:92", "N:93", "N:94", "N:95", "N:96", "N:97", "N:98", "N:99", "N:100", "N:101"])
# global  Double_N_ORF9b_muts = Set(["ORF9b:1", "ORF9b:2", "ORF9b:3", "ORF9b:4", "ORF9b:5", "ORF9b:6", "ORF9b:7", "ORF9b:8", "ORF9b:9", "ORF9b:10", "ORF9b:11", "ORF9b:12", "ORF9b:13", "ORF9b:14", "ORF9b:15", "ORF9b:16", "ORF9b:17", "ORF9b:18", "ORF9b:19", "ORF9b:20", "ORF9b:21", "ORF9b:22", "ORF9b:23", "ORF9b:24", "ORF9b:25", "ORF9b:26", "ORF9b:27", "ORF9b:28", "ORF9b:29", "ORF9b:30", "ORF9b:31", "ORF9b:32", "ORF9b:33", "ORF9b:34", "ORF9b:35", "ORF9b:36", "ORF9b:37", "ORF9b:38", "ORF9b:39", "ORF9b:40", "ORF9b:41", "ORF9b:42", "ORF9b:43", "ORF9b:44", "ORF9b:45", "ORF9b:46", "ORF9b:47", "ORF9b:48", "ORF9b:49", "ORF9b:50", "ORF9b:51", "ORF9b:52", "ORF9b:53", "ORF9b:54", "ORF9b:55", "ORF9b:56", "ORF9b:57", "ORF9b:58", "ORF9b:59", "ORF9b:60", "ORF9b:61", "ORF9b:62", "ORF9b:63", "ORF9b:64", "ORF9b:65", "ORF9b:66", "ORF9b:67", "ORF9b:68", "ORF9b:69", "ORF9b:70", "ORF9b:71", "ORF9b:72", "ORF9b:73", "ORF9b:74", "ORF9b:75", "ORF9b:76", "ORF9b:77", "ORF9b:78", "ORF9b:79", "ORF9b:80", "ORF9b:81", "ORF9b:82", "ORF9b:83", "ORF9b:84", "ORF9b:85", "ORF9b:86", "ORF9b:87", "ORF9b:88", "ORF9b:89", "ORF9b:90", "ORF9b:91", "ORF9b:92", "ORF9b:93", "ORF9b:94", "ORF9b:95", "ORF9b:96", "ORF9b:97", "ORF9b:98"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_pos_only = list_to_strings_set("ORF1a:2183, ORF1a:2972, ORF1a:2972, ORF1a:3049, ORF1a:3058, ORF1a:3070, ORF1a:3072, ORF1a:3076, ORF1a:3123, ORF1a:3195, ORF1a:3359, ORF1a:3454, ORF1a:3456, ORF1a:4110, ORF1a:4175, ORF1a:4197, ORF1a:4205, ORF1a:4207, ORF1a:4387, ORF1a:314, ORF1b:1181, ORF1b:1247, ORF1b:1424, ORF1b:2339, ORF1b:2340, ORF1b:2537, S:50, S:330, S:354, S:367, S:374, S:375-, S:376, S:405, S:508, S:551, S:573, S:647, S:653, S:657, S:659, S:668, S:859, S:944, S:958, S:978, S:1169, S:1179, S:1186, E:5, E:5, E:9, E:10, E:10, E:13-, E:14-, E:16, E:23, E:30, E:36, E:42, M:2, M:4, M:10, M:28, M:77, M:94, M:94, M:125, M:148, M:188, N:80, N:416, N:417, ORF7a:115, ORF9b:1, ORF9b:77")
###########################################################################################################################################################################
#####################################################   BEGIN Sub/pos_only Section   ######################################################################################
###########################################################################################################################################################################
pango_AAsub_WT_universal = Dict{String, String}()
artifactual_private_muts = union(artifactual_private_muts_subs, artifactual_private_muts_pos_only)
pango_AAsub_WT_universal = pango_AAsub_WT
Double_N_ORF9b_muts = union(Double_N_ORF9b_muts, Double_N_ORF9b_muts_pos_only)
BAL_muts = union(BAL_muts_OG, BAL_muts_pos_only)
mp_chr_all_ratio_universal = Dict{String, Float64}()
for (mut, ratio) in AA_muts_ct_chr_all_ratio
    mp_chr_all_ratio_universal[mut] = ratio
end
for (mut, ratio) in AA_muts_ct_pos_only_no_dels_chr_all_ratio
    mp_chr_all_ratio_universal[mut] = ratio
end
seq_AA_muts_sub2pos_pos2sub_ref = Dict{String, Set{String}}()
seq_AA_muts_sub2pos_pos2sub_qry = Dict{String, Set{String}}()
#####################################################
global mp_folder_universal = ""
if sub_0__posonly_1__mixed_2 == 0 && none_0___sub2pos_1___pos2sub_2 == 0
    seq_AA_muts_sub2pos_pos2sub_ref = seq_AA_muts
    seq_AA_muts_sub2pos_pos2sub_qry = seq_AA_muts
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_subs_SIMPLE_$(date_hour)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_subs_SIMPLE_spike_only_$(date_hour)"
    end
elseif sub_0__posonly_1__mixed_2 == 1 && none_0___sub2pos_1___pos2sub_2 == 0
    seq_AA_muts_sub2pos_pos2sub_ref = seq_AA_muts_pos_only
    seq_AA_muts_sub2pos_pos2sub_qry = seq_AA_muts_pos_only
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_pos_only_SIMPLE_$(date_hour)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_pos_only_SIMPLE_spike_only_$(date_hour)"
    end
elseif sub_0__posonly_1__mixed_2 == 2 && none_0___sub2pos_1___pos2sub_2 == 1
    seq_AA_muts_sub2pos_pos2sub_ref = seq_AA_muts
    seq_AA_muts_sub2pos_pos2sub_qry = seq_AA_muts_pos_only
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_sub2pos_SIMPLE_$(date_hour)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_sub2pos_SIMPLE_spike_only_$(date_hour)"
    end    
elseif sub_0__posonly_1__mixed_2 == 2 && none_0___sub2pos_1___pos2sub_2 == 2
    seq_AA_muts_sub2pos_pos2sub_ref = seq_AA_muts_pos_only
    seq_AA_muts_sub2pos_pos2sub_qry = seq_AA_muts
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_pos2sub_SIMPLE_$(date_hour)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_pos2sub_SIMPLE_spike_only_$(date_hour)"
    end
end
mkpath(mp_folder_universal)
#################################################################################################
function aapos_universal_fx(mut::String)
    aapos = 0
    mjeen = split(mut, ":")[1]
    refposqry = split(mut, ":")[2]
    if isdigit(refposqry[1])
        aapos = AA_pos_only_pos_dict[mut]
    else
        aapos = aa_pos_comprehensive_dict[mut]
    end
    return aapos
end
function jeen_universal_fx(mut::String)
    mjeen = split(mut, ":")[1]
    return mjeen
end
function jeen_n_pos_universal_fx(mut::String)
    refposqry = split(mut, ":")[2]
    aapos_n_gene = "jeen_n_pos_universal_fx ERROR"
    if isdigit(refposqry[1])
        aapos_n_gene = AA_pos_only_gene_and_pos_dict[mut]
    else
        aapos_n_gene = aa_gene_and_pos_comprehensive_dict[mut]
    end
    return aapos_n_gene
end
#################################################################################################
function mp_AApos_sort_key_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return mp_AApos_sort_key(n)
    elseif sub_0__posonly_1__mixed_2 == 1
        return mp_AApos_pos_only_sort_key(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return mp_AApos_pos_only_sort_key(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return return mp_AApos_sort_key(n)
    end
end
############################################################
function sel_muts_pt1_sort_key_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return sel_muts_pt1_sort_key(n)
    elseif sub_0__posonly_1__mixed_2 == 1
        return sel_muts_pt1_pos_only_sort_key(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return sel_muts_pt1_sort_key(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 2
    end
end
############################################################
function mp_AAsub_gene_number_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
    elseif sub_0__posonly_1__mixed_2 == 1
        return [AA_pos_only_gene_dict[n], AA_pos_only_pos_dict[n]]
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return [AA_pos_only_gene_dict[n], AA_pos_only_pos_dict[n]]
    end
end      
############################################################
function mp_AA_gene_sortKey_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return mp_AA_gene_sortKey(n)
    elseif sub_0__posonly_1__mixed_2 == 1
        return mp_AA_gene_pos_only_sortKey(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return mp_AA_gene_sortKey(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return mp_AA_gene_pos_only_sortKey(n)
    end
end    
############################################################
function mp_AA_gene_sortKey_2_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return mp_AA_gene_sortKey_2(n)
    elseif sub_0__posonly_1__mixed_2 == 1
        return mp_AA_gene_pos_only_sortKey_2(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return mp_AA_gene_sortKey_2(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return mp_AA_gene_pos_only_sortKey_2(n)
    end
end    
############################################################
function mp_AA_ct_sortKey1_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return mp_AA_ct_sortKey1(n)
    elseif sub_0__posonly_1__mixed_2 == 1
        return mp_AA_ct_pos_only_sortKey1(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return mp_AA_ct_sortKey1(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return mp_AA_ct_pos_only_sortKey1(n)
    end
end    
############################################################
function mp_AA_ct_sortKey2_universal(n, sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return mp_AA_ct_sortKey2(n)
    elseif sub_0__posonly_1__mixed_2 == 1
        return mp_AA_ct_pos_only_sortKey2(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return mp_AA_ct_sortKey2(n)
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return mp_AA_ct_pos_only_sortKey2(n)
    end
end
############################################################
function AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return AA_muts_ct
    end
    if sub_0__posonly_1__mixed_2 == 1
        return AA_muts_ct_pos_only
    end
    if none_0___sub2pos_1___pos2sub_2 == 1
        return AA_muts_ct
    end
    if none_0___sub2pos_1___pos2sub_2 == 2
        return AA_muts_ct_pos_only
    end
end
AA_muts_ct_no_dels_universal = AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1__mixed_2, none_0___sub2pos_1___pos2sub_2)
if AA_muts_ct_no_dels_universal == AA_muts_ct
    println("AA_muts_ct_no_dels_universal == AA_muts_ct")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct")
end
if AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_pos_only")
end
############################################################
function seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1__mixed_2::Int, none_0___sub2pos_1___pos2sub_2::Int)
    if sub_0__posonly_1__mixed_2 == 0
        return seq_AA_muts_no_dels
    elseif sub_0__posonly_1__mixed_2 == 1
        return seq_AA_muts_pos_only_no_dels
    elseif none_0___sub2pos_1___pos2sub_2 == 1
        return seq_AA_muts_no_dels
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        return seq_AA_muts_pos_only_no_dels
    end
end
seq_AA_muts_no_dels_universal = seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1__mixed_2, none_0___sub2pos_1___pos2sub_2)
seq_AA_muts_no_dels_combined = Dict{String, Set{String}}()
for seq in all_unique_chr_seqs_set
    seq_AA_muts_no_dels_combined[seq] = union(seq_AA_muts_no_dels[seq], seq_AA_muts_pos_only_no_dels[seq])
end
####################################################################################
artifacts_ORF9bdoubles = union(artifactual_private_muts, Double_N_ORF9b_muts)
###########################################################################################################################################################################
######################################################   END Sub/pos_only Section   #######################################################################################
###########################################################################################################################################################################
global RBM_muts = Set{String}()
global RBD_muts = Set{String}()
global RBD_not_RBM_muts = Set{String}()
global spike_not_RBD_muts = Set{String}()
global spike_muts = Set{String}()
RBD_sites = BitSet(335:528)
RBD_no_RBM_1 = BitSet(335:437)
RBD_no_RBM_2 = BitSet(507:528)
RBD_not_RBM_sites = union(RBD_no_RBM_1, RBD_no_RBM_2)
RBM_sites = BitSet(438:506)
spike_not_RBD1 = BitSet(1:334)
spike_not_RBD2 = BitSet(529:1273)
spike_not_RBD_sites = union(spike_not_RBD1, spike_not_RBD2)
############################################################################
RBM_pos_only_set = Set(("S:$(i)" for i in 438:506))
RBD_pos_only_set = Set(("S:$(i)" for i in 335:528))
RBD_no_RBM_pos_only_1_set = Set(("S:$(i)" for i in 335:437))
RBD_no_RBM_pos_only_2_set = Set(("S:$(i)" for i in 507:528))
RBD_no_RBM_pos_only_set = union(RBD_no_RBM_pos_only_1_set, RBD_no_RBM_pos_only_2_set)
spike_pos_only_set = Set(("S:$(i)" for i in 1:1274))
spike_no_RBD_pos_only_set = setdiff(spike_pos_only_set, RBD_pos_only_set)
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
RBM_muts = Set{String}()
RBD_muts = Set{String}()
RBD_not_RBM_muts = Set{String}()
spike_not_RBD_muts = Set{String}()
for aa1 in AA_residues
    for aa2 in AA_residues
        for i in 438:506
            rbmmut = "S:$(aa1)$(i)$(aa2)"
            push!(RBM_muts, rbmmut)
        end
    end
end
for aa1 in AA_residues
    for aa2 in AA_residues
        for i in 335:528
            rbdmut = "S:$(aa1)$(i)$(aa2)"
            push!(RBD_muts, rbdmut)
        end
    end
end
for aa1 in AA_residues
    for aa2 in AA_residues
        for i in RBD_not_RBM_sites
            rbd_not_rbm_mut = "S:$(aa1)$(i)$(aa2)"
            push!(RBD_not_RBM_muts, rbd_not_rbm_mut)
        end
    end
end
for aa1 in AA_residues
    for aa2 in AA_residues
        for i in spike_not_RBD_sites
            spike_not_rbd_mut = "S:$(aa1)$(i)$(aa2)"
            push!(spike_not_RBD_muts, spike_not_rbd_mut)
        end
    end
end
spike_muts = union(RBD_muts, spike_not_RBD_muts)

RBM_muts = union(RBM_muts, RBM_pos_only_set)
RBD_muts = union(RBD_muts, RBD_pos_only_set)
RBD_not_RBM_muts = union(RBD_not_RBM_muts, RBD_no_RBM_pos_only_set)
spike_not_RBD_muts = union(spike_not_RBD_muts, spike_no_RBD_pos_only_set)
spike_muts = union(spike_muts, spike_pos_only_set)
######################################################################################################################################
global purespikebanned_0__purespikeallowed_1 = 0
global nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 0
global all_excluded_muts = Set{String}()
if normal_0__spikeonly_1 == 0
    purespikebanned_0__purespikeallowed_1 = 0
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 0
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, BAL_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts)
    end
elseif normal_0__spikeonly_1 == 1
    purespikebanned_0__purespikeallowed_1 = 1
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 3
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, BAL_muts, nonspike_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, nonspike_muts)
    end
end
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
RBD_not_RBM_muts_sort = sort(collect(RBD_not_RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_not_RBM_tot = length(RBD_not_RBM_muts_sort)
println("Total Different RBD-not-RBM Muts = $(RBD_not_RBM_tot)")
RBD_not_RBM_muts_sort_join = join(RBD_not_RBM_muts_sort, ", ")
########################
RBM_muts_sort = sort(collect(RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBM_tot = length(RBM_muts_sort)
println("Total Different RBM Muts = $(RBM_tot)")
RBM_muts_sort_join = join(RBM_muts_sort, ", ")
########################
RBD_muts_sort = sort(collect(RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_tot = length(RBD_muts_sort)
println("Total Different RBD Muts = $(RBD_tot)")
RBD_muts_sort_join = join(RBD_muts_sort, ", ")
########################
spike_not_RBD_muts_sort = sort(collect(spike_not_RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
spike_not_RBD_tot = length(spike_not_RBD_muts_sort)
println("Total Different spike_not_RBD Muts = $(spike_not_RBD_tot)")
spike_not_RBD_muts_sort_join = join(spike_not_RBD_muts_sort, ", ")
#####################################################################################################################################
ORF9bNdoubles_artifacts = union(Double_N_ORF9b_muts, artifactual_private_muts)
tot_grp_ct = length(rep_seq_grps_maxmut_seqs)
tot_single_ct = length(non_rep_seqs)
tot_chr_seq_ct = tot_grp_ct + tot_single_ct
total_chronic_AA_ct = 0
total_chronic_AA_ct_nonRBM = 0
total_chronic_AA_ct_nonRBD = 0
total_chronic_AA_ct_spike_nonRBD = 0
total_chronic_AA_ct_nonspike = 0
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in artifacts_ORF9bdoubles)
        total_chronic_AA_ct += ct
        if aa_gene_comprehensive_dict[mut] ≠ "S"
            total_chronic_AA_ct_nonspike += ct
            total_chronic_AA_ct_nonRBM += ct
            total_chronic_AA_ct_nonRBD += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
            total_chronic_AA_ct_nonRBM += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
            total_chronic_AA_ct_nonRBD += ct
            total_chronic_AA_ct_spike_nonRBD += ct
        end
    end
end
#####################################################################################################################
avg_AA_sub_ct_per_chronic_seq = round(digits=2, total_chronic_AA_ct/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq = $(avg_AA_sub_ct_per_chronic_seq)")
avg_AA_sub_ct_per_chronic_seq_nonspike = round(digits=2, total_chronic_AA_ct_nonspike/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonspike = $(avg_AA_sub_ct_per_chronic_seq_nonspike)")
avg_AA_sub_ct_per_chronic_seq_nonRBM = round(digits=2, total_chronic_AA_ct_nonRBM/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_nonRBD = round(digits=2, total_chronic_AA_ct_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = round(digits=2, total_chronic_AA_ct_spike_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_spike_nonRBD)")
######################################################################################################################################
#seq_privAA_nonRBD_len = Dict{String, Int}()
#for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
#    seq_nonRBD_ct = 0
#    for sub in AAsubvec
#        if !(sub in RBD_muts)
#            seq_nonRBD_ct += 1
#        end
#    end
#    seq_privAA_nonRBD_len[seq] = seq_nonRBD_ct
#end
######################################################################################################################################
#seq_privAA_nonRBD_len_relative = Dict{String, Float64}()
#for (seq, ct) in seq_privAA_nonRBD_len
#    relative_nonRBD = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_nonRBD)
#    seq_privAA_nonRBD_len_relative[seq] = relative_nonRBD
#end
#####################################################################################################################
global avg_AA_sub_ct_per_chronic_seq_for_main_fx = 0.0
#####################################################################################################################
if normal_0__spikeonly_1 == 0
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_nonRBD
elseif normal_0__spikeonly_1 == 1
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_spike_nonRBD
end
#####################################################################################################################
# all_excluded_muts
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seq_mut_ct = 0
    for sub in AAsubvec
        if !(sub in all_excluded_muts)
            seq_mut_ct += 1
        end
    end
    seq_privAA_len[seq] = seq_mut_ct
end
#####################################################################################################################
seq_privAA_len_relative = Dict{String, Float64}()
for (seq, ct) in seq_privAA_len
    relative_muts = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_for_main_fx)
    seq_privAA_len_relative[seq] = relative_muts
end
#####################################################################################################################
total_chronic_AA_ct_v2 = 0
total_chronic_AA_ct_v2_nonRBM = 0
total_chronic_AA_ct_v2_nonRBD = 0
total_chronic_AA_ct_v2_spike_nonRBD = 0
total_chronic_AA_ct_v2_nonspike = 0
for seq in all_unique_chr_seqs_set
    for mut in seq_AA_muts_no_dels_universal[seq]
        if !(mut in artifacts_ORF9bdoubles)
            total_chronic_AA_ct_v2 += 1
            if aa_gene_comprehensive_dict[mut] ≠ "S"
                total_chronic_AA_ct_v2_nonspike += 1
                total_chronic_AA_ct_v2_nonRBM += 1
                total_chronic_AA_ct_v2_nonRBD += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
                total_chronic_AA_ct_v2_nonRBM += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
                total_chronic_AA_ct_v2_nonRBD += 1
                total_chronic_AA_ct_v2_spike_nonRBD += 1
            end
        end
    end
end
###################################################
seq_AA_muts_combined = Dict{String, Set{String}}()
for seq in all_unique_chr_seqs_set
    seq_AA_muts_combined[seq] = union(seq_AA_muts[seq], seq_AA_muts_pos_only[seq])
end
AA_muts_ct_combined = Dict{String, Int}()
for (mut, ct) in AA_muts_ct
    AA_muts_ct_combined[mut] = ct
end
for (mut, ct) in AA_muts_ct_pos_only
    AA_muts_ct_combined[mut] = ct
end
####################################################################################################################################
avg_AA_sub_ct_per_chronic_seq_v2 = round(digits=2, total_chronic_AA_ct_v2/total_EPCI_seq_ct)  #### Double checking the first count
println("avg_AA_sub_ct_per_chronic_seq_v2 = $(avg_AA_sub_ct_per_chronic_seq_v2)")
println("total_chronic_AA_ct = $(total_chronic_AA_ct)")
println("total_chronic_AA_ct_v2 = $(total_chronic_AA_ct_v2)")
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = round(digits=2, total_chronic_AA_ct_v2_nonspike/total_EPCI_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonspike = $(avg_AA_sub_ct_per_chronic_seq_v2_nonspike)")
println("total_chronic_AA_ct_nonspike = $(total_chronic_AA_ct_nonspike)")
println("total_chronic_AA_ct_v2_nonspike = $(total_chronic_AA_ct_v2_nonspike)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = round(digits=2, total_chronic_AA_ct_v2_nonRBM/total_EPCI_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBM)")
println("total_chronic_AA_ct_nonRBM = $(total_chronic_AA_ct_nonRBM)")
println("total_chronic_AA_ct_v2_nonRBM = $(total_chronic_AA_ct_v2_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = round(digits=2, total_chronic_AA_ct_v2_nonRBD/total_EPCI_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBD)")
println("total_chronic_AA_ct_nonRBD = $(total_chronic_AA_ct_nonRBD)")
println("total_chronic_AA_ct_v2_nonRBD = $(total_chronic_AA_ct_v2_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = round(digits=2, total_chronic_AA_ct_v2_spike_nonRBD/total_EPCI_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD)")
println("total_chronic_AA_ct_spike_nonRBD = $(total_chronic_AA_ct_spike_nonRBD)")
println("total_chronic_AA_ct_v2_spike_nonRBD = $(total_chronic_AA_ct_v2_spike_nonRBD)"); print("\n"^1)
####################################################################################################################################
####################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
println("Finished!"); print("\n"^1)
####################################################################################################################################
####################################################################################################################################
###################################################################################################################################

2026_03_07__601PM
AA_muts_ct_no_dels_universal == AA_muts_ct
AA_muts_ct_no_dels_universal ≠ AA_muts_ct_pos_only
Total Different RBD-not-RBM Muts = 66250
Total Different RBM Muts = 36570
Total Different RBD Muts = 102820
Total Different spike_not_RBD Muts = 571871
avg_AA_sub_ct_per_chronic_seq = 20.18
avg_AA_sub_ct_per_chronic_seq_nonspike = 13.1
avg_AA_sub_ct_per_chronic_seq_nonRBM = 18.79
avg_AA_sub_ct_per_chronic_seq_nonRBD = 17.86
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = 4.76
avg_AA_sub_ct_per_chronic_seq_v2 = 17.64
total_chronic_AA_ct = 66542
total_chronic_AA_ct_v2 = 58153
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = 11.81
total_chronic_AA_ct_nonspike = 43207
total_chronic_AA_ct_v2_nonspike = 38932
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = 16.26
total_chronic_AA_ct_nonRBM = 61943
total_chronic_AA_ct_v2_nonRBM = 53601
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = 15.37
total_chronic_AA_ct_nonRBD = 58896
total_chronic_AA_ct_v2_nonRBD = 50687
avg_AA_sub_ct_per_chronic_seq_v2_spike_nonR

In [109]:
### MAIN FUNCTION —— New sub2pos & pos2sub versions 
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
NTD_disulfide_variations = Set(["NTD:disulfide", "NTD_disulfide", "S:NTD_disulfide", "NTDdisulfide", "NTD:Disulfide"])
tot_grp_ct = length(rep_seq_grps_maxmut_seqs) # Method #1 for finding avg AA subs + del ranges/seq in chronic list
tot_single_ct = length(non_rep_seqs)
tot_chr_seq_ct = tot_grp_ct + tot_single_ct
total_chronic_AA_ct = 0
for (mut, ct) in AA_muts_ct_no_dels
    total_chronic_AA_ct += ct
end
Double_N_ORF9b_muts_sub2pos = Set(["N:N29-", "N:G30-", "N:E31-", "N:R32-", "N:S33-", "N:L13-", "N:L13I", "N:L13H", "N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:G63D", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
Double_N_ORF9b_muts_po_sub2pos = list_to_strings_set("N:4, N:5, N:6, N:7, N:8, N:9, N:10, N:11, N:12, N:13, N:14, N:15, N:16, N:17, N:18, N:19, N:20, N:21, N:22, N:23, N:24, N:25, N:26, N:27, N:28, N:29, N:30, N:31, N:32, N:33, N:34, N:35, N:36, N:37, N:38, N:39, N:40, N:41, N:42, N:43, N:44, N:45, N:46, N:47, N:48, N:49, N:50, N:51, N:52, N:53, N:54, N:55, N:56, N:57, N:58, N:59, N:60, N:61, N:62, N:63, N:64, N:65, N:66, N:67, N:68, N:69, N:70, N:71, N:72, N:73, N:74, N:75, N:76, N:77, N:78, N:79, N:80, N:81, N:82, N:83, N:84, N:85, N:86, N:87, N:88, N:89, N:90, N:91, N:92, N:93, N:94, N:95, N:96, N:97, N:98, N:99, N:100, N:101, N:102")
Double_N_ORF9b_combined_sub2pos = union(Double_N_ORF9b_muts_sub2pos, Double_N_ORF9b_muts_po_sub2pos)
ORF9b_all_muts_set_sub2pos = Set(["ORF9b:M1R","ORF9b:M1K","ORF9b:M1T","ORF9b:M1L","ORF9b:M1V","ORF9b:M1-","ORF9b:M1I","ORF9b:G2S","ORF9b:D2D","ORF9b:D2H","ORF9b:D2G","ORF9b:D2Y","ORF9b:D2V","ORF9b:D2E","ORF9b:D2N","ORF9b:D2-","ORF9b:D2A","ORF9b:P3S","ORF9b:P3F","ORF9b:H3L","ORF9b:P3-","ORF9b:P3H","ORF9b:H3N","ORF9b:H3D","ORF9b:H3Q","ORF9b:P3P","ORF9b:H3P","ORF9b:H3Y","ORF9b:P3R","ORF9b:P3T","ORF9b:P3L","ORF9b:P3A","ORF9b:H3R","ORF9b:K4R","ORF9b:K4K","ORF9b:K4I","ORF9b:K4-","ORF9b:K4T","ORF9b:K4E","ORF9b:K4Y","ORF9b:K4Q","ORF9b:K4N","ORF9b:I5L","ORF9b:T5N","ORF9b:I5F","ORF9b:I5V","ORF9b:T5P","ORF9b:T5V","ORF9b:I5-","ORF9b:T5I","ORF9b:I5N","ORF9b:T5A","ORF9b:I5T","ORF9b:I5I","ORF9b:I5S","ORF9b:I5M","ORF9b:T5T","ORF9b:I5C","ORF9b:S6T","ORF9b:S6N","ORF9b:S6G","ORF9b:S6I","ORF9b:S6-","ORF9b:S6C","ORF9b:S6P","ORF9b:S6R","ORF9b:C6-","ORF9b:C6F","ORF9b:C6R","ORF9b:E7*","ORF9b:E7K","ORF9b:E7S","ORF9b:E7E","ORF9b:E7V","ORF9b:E7Q","ORF9b:E7A","ORF9b:E7D","ORF9b:E7G","ORF9b:E7-","ORF9b:M8K","ORF9b:M8-","ORF9b:M8T","ORF9b:M8L","ORF9b:M8I","ORF9b:M8R","ORF9b:M8V","ORF9b:H9N","ORF9b:H9P","ORF9b:H9L","ORF9b:D9Y","ORF9b:H9H","ORF9b:D9E","ORF9b:H9-","ORF9b:H9Q","ORF9b:H9Y","ORF9b:H9D","ORF9b:H9R","ORF9b:F10C","ORF9b:P10-","ORF9b:P10T","ORF9b:P10R","ORF9b:F10P","ORF9b:F10Y","ORF9b:F10L","ORF9b:F10I","ORF9b:S10S","ORF9b:S10L","ORF9b:P10L","ORF9b:P10S","ORF9b:S10P","ORF9b:P10F","ORF9b:F10S","ORF9b:S10H","ORF9b:S10Y","ORF9b:S10A","ORF9b:S10C","ORF9b:S10-","ORF9b:P10H","ORF9b:F10-","ORF9b:S10T","ORF9b:P10A","ORF9b:S10F","ORF9b:P10D","ORF9b:A11S","ORF9b:A11P","ORF9b:V11I","ORF9b:A11T","ORF9b:A11G","ORF9b:A11I","ORF9b:A11V","ORF9b:A11E","ORF9b:A11-","ORF9b:L12I","ORF9b:L12-","ORF9b:L12L","ORF9b:L12V","ORF9b:L12S","ORF9b:L12Q","ORF9b:L12F","ORF9b:R13C","ORF9b:R13S","ORF9b:R13F","ORF9b:R13H","ORF9b:R13-","ORF9b:R13G","ORF9b:R13L","ORF9b:R13P","ORF9b:L14M","ORF9b:L14V","ORF9b:L14L","ORF9b:L14W","ORF9b:L14I","ORF9b:L14S","ORF9b:L14-","ORF9b:L14F","ORF9b:V15L","ORF9b:V15-","ORF9b:V15V","ORF9b:V15A","ORF9b:L15V","ORF9b:V15M","ORF9b:V15E","ORF9b:G16A","ORF9b:D16N","ORF9b:G16V","ORF9b:D16A","ORF9b:D16E","ORF9b:G16-","ORF9b:G16C","ORF9b:D16V","ORF9b:G16D","ORF9b:D16D","ORF9b:D16Y","ORF9b:G16S","ORF9b:D16-","ORF9b:D16H","ORF9b:D16G","ORF9b:P17S","ORF9b:P17T","ORF9b:P17A","ORF9b:P17L","ORF9b:P17-","ORF9b:P17R","ORF9b:P17P","ORF9b:P17H","ORF9b:Q18S","ORF9b:Q18H","ORF9b:Q18L","ORF9b:Q18R","ORF9b:Q18*","ORF9b:Q18-","ORF9b:Q18P","ORF9b:I19F","ORF9b:I19L","ORF9b:I19-","ORF9b:I19M","ORF9b:I19N","ORF9b:I19T","ORF9b:I19S","ORF9b:I19V","ORF9b:I19I","ORF9b:Q20L","ORF9b:Q20H","ORF9b:Q20*","ORF9b:Q20P","ORF9b:Q20R","ORF9b:Q20-","ORF9b:L21-","ORF9b:L21Q","ORF9b:L21I","ORF9b:L21L","ORF9b:L21R","ORF9b:L21M","ORF9b:L21V","ORF9b:L21P","ORF9b:A22-","ORF9b:A22S","ORF9b:A22A","ORF9b:A22E","ORF9b:A22T","ORF9b:A22P","ORF9b:A22L","ORF9b:A22V","ORF9b:V23A","ORF9b:V23-","ORF9b:V23L","ORF9b:V23I","ORF9b:V23E","ORF9b:T24N","ORF9b:T24I","ORF9b:T24A","ORF9b:T24S","ORF9b:T24P","ORF9b:T24D","ORF9b:T24-","ORF9b:R25C","ORF9b:R25G","ORF9b:R25T","ORF9b:R25K","ORF9b:R25-","ORF9b:R25S","ORF9b:R25I","ORF9b:R25M","ORF9b:M26T","ORF9b:M26I","ORF9b:M26V","ORF9b:M26L","ORF9b:M26K","ORF9b:M26-","ORF9b:M26R","ORF9b:E27K","ORF9b:E27D","ORF9b:E27Q","ORF9b:E27A","ORF9b:E27V","ORF9b:E27-","ORF9b:E27G","ORF9b:E27W","ORF9b:N28Y","ORF9b:N28T","ORF9b:N28D","ORF9b:N28S","ORF9b:N28I","ORF9b:N28K","ORF9b:N28H","ORF9b:N28-","ORF9b:A29G","ORF9b:A29L","ORF9b:A29P","ORF9b:A29I","ORF9b:I29A","ORF9b:I29N","ORF9b:I29S","ORF9b:I29-","ORF9b:A29V","ORF9b:I29T","ORF9b:A29T","ORF9b:A29E","ORF9b:A29S","ORF9b:I29V","ORF9b:A29-","ORF9b:L30V","ORF9b:L30F","ORF9b:V30-","ORF9b:L30-","ORF9b:L30S","ORF9b:V30L","ORF9b:V30E","ORF9b:V30G","ORF9b:V30F","ORF9b:L30I","ORF9b:A30-","ORF9b:V30M","ORF9b:V30A","ORF9b:G31G","ORF9b:E31-","ORF9b:G31-","ORF9b:R31G","ORF9b:G31V","ORF9b:G31A","ORF9b:G31W","ORF9b:G31E","ORF9b:G31T","ORF9b:G31R","ORF9b:R32I","ORF9b:R32P","ORF9b:R32-","ORF9b:R32C","ORF9b:R32L","ORF9b:R32S","ORF9b:R32N","ORF9b:R32F","ORF9b:R32G","ORF9b:R32H","ORF9b:D33G","ORF9b:D33Y","ORF9b:D33A","ORF9b:D33N","ORF9b:D33D","ORF9b:D33V","ORF9b:D33-","ORF9b:D33E","ORF9b:D33H","ORF9b:Q34H","ORF9b:Q34P","ORF9b:Q34L","ORF9b:Q34Q","ORF9b:Q34R","ORF9b:Q34-","ORF9b:Q34*","ORF9b:Q34E","ORF9b:N35-","ORF9b:N35H","ORF9b:N35Y","ORF9b:N35I","ORF9b:N35L","ORF9b:N35T","ORF9b:N35K","ORF9b:N35D","ORF9b:N35S","ORF9b:N36I","ORF9b:N36N","ORF9b:N36K","ORF9b:N36-","ORF9b:N36Y","ORF9b:N36D","ORF9b:N36T","ORF9b:N36H","ORF9b:N36S","ORF9b:V37I","ORF9b:V37L","ORF9b:V37F","ORF9b:V37-","ORF9b:V37G","ORF9b:V37A","ORF9b:V37D","ORF9b:G38A","ORF9b:G38R","ORF9b:G38-","ORF9b:G38F","ORF9b:G38D","ORF9b:G38S","ORF9b:G38C","ORF9b:D38N","ORF9b:G38V","ORF9b:P39R","ORF9b:P39A","ORF9b:P39L","ORF9b:P39H","ORF9b:P39S","ORF9b:P39-","ORF9b:K40T","ORF9b:R40M","ORF9b:K40Q","ORF9b:K40M","ORF9b:K40*","ORF9b:K40R","ORF9b:K40N","ORF9b:K40E","ORF9b:K40-","ORF9b:V41I","ORF9b:V41-","ORF9b:V41A","ORF9b:V41F","ORF9b:V41L","ORF9b:V41D","ORF9b:V41G","ORF9b:Y42H","ORF9b:Y42D","ORF9b:Y42C","ORF9b:Y42-","ORF9b:Y42N","ORF9b:P43R","ORF9b:P43L","ORF9b:P43-","ORF9b:P43S","ORF9b:P43A","ORF9b:P43T","ORF9b:P43P","ORF9b:P43Q","ORF9b:I44V","ORF9b:I44-","ORF9b:I44L","ORF9b:I44T","ORF9b:I44M","ORF9b:I45M","ORF9b:I45-","ORF9b:I45K","ORF9b:I45T","ORF9b:I45L","ORF9b:I45V","ORF9b:L46P","ORF9b:L46V","ORF9b:L46R","ORF9b:L46M","ORF9b:L46F","ORF9b:L46L","ORF9b:L46-","ORF9b:L46I","ORF9b:L46Q","ORF9b:R47G","ORF9b:R47L","ORF9b:R47P","ORF9b:R47C","ORF9b:R47H","ORF9b:R47F","ORF9b:R47-","ORF9b:L48R","ORF9b:L48H","ORF9b:L48P","ORF9b:L48F","ORF9b:L48-","ORF9b:G49-","ORF9b:G49C","ORF9b:G49D","ORF9b:G49V","ORF9b:S50L","ORF9b:S50P","ORF9b:S50T","ORF9b:S50-","ORF9b:L50I","ORF9b:P51S","ORF9b:P51R","ORF9b:P51A","ORF9b:P51Q","ORF9b:P51-","ORF9b:P51L","ORF9b:P51P","ORF9b:P51T","ORF9b:L52R","ORF9b:L52V","ORF9b:L52P","ORF9b:L52-","ORF9b:L52H","ORF9b:S53L","ORF9b:S53T","ORF9b:S53-","ORF9b:L54V","ORF9b:L54H","ORF9b:L54R","ORF9b:L54-","ORF9b:L54P","ORF9b:N55-","ORF9b:S55N","ORF9b:N55S","ORF9b:N55T","ORF9b:N55D","ORF9b:N55Y","ORF9b:M56-","ORF9b:M56T","ORF9b:M56I","ORF9b:M56V","ORF9b:A57-","ORF9b:A57P","ORF9b:A57V","ORF9b:A57S","ORF9b:A57E","ORF9b:A57G","ORF9b:A57T","ORF9b:R58K","ORF9b:R58G","ORF9b:R58-","ORF9b:R58E","ORF9b:R58T","ORF9b:R58W","ORF9b:R58S","ORF9b:R58M","ORF9b:K59R","ORF9b:K59E","ORF9b:K59Q","ORF9b:K59T","ORF9b:K59N","ORF9b:K59M","ORF9b:K59-","ORF9b:K59K","ORF9b:T60A","ORF9b:A60S","ORF9b:T60-","ORF9b:A60V","ORF9b:T60S","ORF9b:T60P","ORF9b:T60N","ORF9b:A60-","ORF9b:A60T","ORF9b:T60I","ORF9b:I60T","ORF9b:A60D","ORF9b:L61-","ORF9b:L61S","ORF9b:L61F","ORF9b:N62H","ORF9b:N62S","ORF9b:N62T","ORF9b:N62I","ORF9b:N62K","ORF9b:N62V","ORF9b:N62-","ORF9b:N62Y","ORF9b:N62D","ORF9b:S63A","ORF9b:S63T","ORF9b:S63F","ORF9b:S63Y","ORF9b:S63-","ORF9b:S63S","ORF9b:L64V","ORF9b:L64P","ORF9b:L64-","ORF9b:L64S","ORF9b:L64L","ORF9b:L64H","ORF9b:L64I","ORF9b:L64R","ORF9b:L64F","ORF9b:E65N","ORF9b:E65K","ORF9b:E65G","ORF9b:E65*","ORF9b:E65Q","ORF9b:E65A","ORF9b:E65S","ORF9b:E65D","ORF9b:E65-","ORF9b:E65V","ORF9b:D66Y","ORF9b:D66G","ORF9b:D66V","ORF9b:D66N","ORF9b:D66A","ORF9b:D66-","ORF9b:D66H","ORF9b:D66E","ORF9b:K67A","ORF9b:K67L","ORF9b:K67T","ORF9b:K67E","ORF9b:K67R","ORF9b:K67Q","ORF9b:K67-","ORF9b:K67M","ORF9b:A68D","ORF9b:A68-","ORF9b:A68E","ORF9b:A68V","ORF9b:A68S","ORF9b:F69C","ORF9b:F69S","ORF9b:F69Y","ORF9b:F69-","ORF9b:F69L","ORF9b:F69I","ORF9b:Q70L","ORF9b:Q70C","ORF9b:Q70H","ORF9b:Q70P","ORF9b:Q70-","ORF9b:Q70R","ORF9b:L71P","ORF9b:L71-","ORF9b:L71V","ORF9b:L71K","ORF9b:L71R","ORF9b:L71A","ORF9b:L71I","ORF9b:L71E","ORF9b:L71S","ORF9b:T72A","ORF9b:T72-","ORF9b:T72P","ORF9b:T72R","ORF9b:T72I","ORF9b:P73-","ORF9b:P73S","ORF9b:P73T","ORF9b:P73L","ORF9b:P73R","ORF9b:P73Q","ORF9b:I74V","ORF9b:I74L","ORF9b:I74-","ORF9b:I74T","ORF9b:I74M","ORF9b:A75-","ORF9b:A75T","ORF9b:A75K","ORF9b:A75S","ORF9b:A75V","ORF9b:A75P","ORF9b:A75A","ORF9b:V76L","ORF9b:V76D","ORF9b:V76C","ORF9b:V76N","ORF9b:V76S","ORF9b:V76H","ORF9b:V76F","ORF9b:V76A","ORF9b:V76-","ORF9b:V76G","ORF9b:V76V","ORF9b:V76I","ORF9b:Q77L","ORF9b:Q77-","ORF9b:E77*","ORF9b:Q77E","ORF9b:E77A","ORF9b:Q77H","ORF9b:Q77R","ORF9b:Q77P","ORF9b:E77G","ORF9b:E77D","ORF9b:Q77Y","ORF9b:Q77*","ORF9b:Q77K","ORF9b:Q77S","ORF9b:M78T","ORF9b:M78-","ORF9b:M78V","ORF9b:M78I","ORF9b:M78L","ORF9b:M78K","ORF9b:T79N","ORF9b:T79T","ORF9b:T79-","ORF9b:T79I","ORF9b:K80R","ORF9b:K80T","ORF9b:K80E","ORF9b:K80-","ORF9b:K80I","ORF9b:K80K","ORF9b:K80N","ORF9b:L81-","ORF9b:L81S","ORF9b:L81F","ORF9b:L81W","ORF9b:A82D","ORF9b:A82-","ORF9b:A82V","ORF9b:A82G","ORF9b:T83-","ORF9b:T83I","ORF9b:T84S","ORF9b:T84-","ORF9b:T84I","ORF9b:E85-","ORF9b:E85A","ORF9b:E85V","ORF9b:E85D","ORF9b:E85K","ORF9b:E85G","ORF9b:E86V","ORF9b:E86E","ORF9b:E86K","ORF9b:E86P","ORF9b:E86*","ORF9b:E86Q","ORF9b:E86Y","ORF9b:E86A","ORF9b:E86G","ORF9b:E86-","ORF9b:E86D","ORF9b:L87-","ORF9b:L87Q","ORF9b:L87I","ORF9b:L87P","ORF9b:L87L","ORF9b:L87V","ORF9b:P88L","ORF9b:P88-","ORF9b:P88Q","ORF9b:P88S","ORF9b:P88T","ORF9b:P88R","ORF9b:P88A","ORF9b:D89-","ORF9b:D89G","ORF9b:D89E","ORF9b:D89H","ORF9b:D89N","ORF9b:D89V","ORF9b:D89A","ORF9b:D89Y","ORF9b:E90A","ORF9b:E90K","ORF9b:E90D","ORF9b:E90-","ORF9b:E90V","ORF9b:E90Q","ORF9b:E90G","ORF9b:F91Y","ORF9b:F91C","ORF9b:F91S","ORF9b:F91P","ORF9b:F91L","ORF9b:F91-","ORF9b:F91V","ORF9b:V92A","ORF9b:V92L","ORF9b:V92V","ORF9b:V92I","ORF9b:V92M","ORF9b:V92E","ORF9b:V92-","ORF9b:V93-","ORF9b:V93F","ORF9b:V93L","ORF9b:V93E","ORF9b:V93V","ORF9b:V93M","ORF9b:V93A","ORF9b:V94V","ORF9b:V94L","ORF9b:V94-","ORF9b:V94M","ORF9b:V94A","ORF9b:T95P","ORF9b:T95V","ORF9b:T95E","ORF9b:T95M","ORF9b:T95-","ORF9b:M95T","ORF9b:T95R","ORF9b:T95A","ORF9b:T95K","ORF9b:V96G","ORF9b:V96T","ORF9b:V96-","ORF9b:V96I","ORF9b:V96A","ORF9b:V96L","ORF9b:V96E","ORF9b:K97Q","ORF9b:K97T","ORF9b:K97-","ORF9b:K97E","ORF9b:K97R","ORF9b:K97N","ORF9b:K97I","ORF9b:*98R","ORF9b:*98G","ORF9b:*98L","ORF9b:*98S","ORF9b:*98C","ORF9b:*98-","ORF9b:*98*"])
ORF9b_all_muts_set_po_sub2pos = list_to_strings_set("ORF9b:1, ORF9b:2, ORF9b:3, ORF9b:4, ORF9b:5, ORF9b:6, ORF9b:7, ORF9b:8, ORF9b:9, ORF9b:10, ORF9b:11, ORF9b:12, ORF9b:13, ORF9b:14, ORF9b:15, ORF9b:16, ORF9b:17, ORF9b:18, ORF9b:19, ORF9b:20, ORF9b:21, ORF9b:22, ORF9b:23, ORF9b:24, ORF9b:25, ORF9b:26, ORF9b:27, ORF9b:28, ORF9b:29, ORF9b:30, ORF9b:31, ORF9b:32, ORF9b:33, ORF9b:34, ORF9b:35, ORF9b:36, ORF9b:37, ORF9b:38, ORF9b:39, ORF9b:40, ORF9b:41, ORF9b:42, ORF9b:43, ORF9b:44, ORF9b:45, ORF9b:46, ORF9b:47, ORF9b:48, ORF9b:49, ORF9b:50, ORF9b:51, ORF9b:52, ORF9b:53, ORF9b:54, ORF9b:55, ORF9b:56, ORF9b:57, ORF9b:58, ORF9b:59, ORF9b:60, ORF9b:61, ORF9b:62, ORF9b:63, ORF9b:64, ORF9b:65, ORF9b:66, ORF9b:67, ORF9b:68, ORF9b:69, ORF9b:70, ORF9b:71, ORF9b:72, ORF9b:73, ORF9b:74, ORF9b:75, ORF9b:76, ORF9b:77, ORF9b:78, ORF9b:79, ORF9b:80, ORF9b:81, ORF9b:82, ORF9b:83, ORF9b:84, ORF9b:85, ORF9b:86, ORF9b:87, ORF9b:88, ORF9b:89, ORF9b:90, ORF9b:91, ORF9b:92, ORF9b:93, ORF9b:94, ORF9b:95, ORF9b:96, ORF9b:97, ORF9b:98")
ORF9b_all_muts_combined_sub2pos = union(ORF9b_all_muts_set_sub2pos, ORF9b_all_muts_set_po_sub2pos)
correlated_muts_sub2pos = list_to_strings_set("ORF1a:I1367V, ORF1a:R1318G, ORF1a:N3443Y, ORF1a:T1567I, ORF1a:A2325V, ORF1a:N3443K, ORF1b:Y822H, ORF1a:P1220S, ORF1a:A1397V, ORF1a:H110Y, ORF1a:I114T, ORF1a:R232L, ORF1a:E233D, ORF1a:E233K, ORF1a:E233A, ORF1a:E233G, ORF1a:K958R, ORF1a:P959L, ORF1a:P959S, ORF1a:P1220H, ORF1a:P1220L, ORF1a:P1220S, ORF1a:S1221P, ORF1a:E1223K, ORF1a:Q1224R, ORF1a:S1272G, ORF1a:D1273N, ORF1a:D1273G, ORF1a:I1276T, ORF1a:T1322A, ORF1a:T1322I, ORF1a:D1323N, ORF1a:D1323G, ORF1a:I1326V, ORF1a:T1328N, ORF1a:S1361P, ORF1a:Q1365R, ORF1a:Q1365P, ORF1a:I1367L, ORF1a:L1368I, ORF1a:G1369R, ORF1a:S1372A, ORF1a:S1372C, ORF1a:E1394D, ORF1a:T1395N, ORF1a:T1395I, ORF1a:H1500Y, ORF1a:T1504I, ORF1a:I1505L, ORF1a:L1507F, ORF1a:T1542I, ORF1a:T1543I, ORF1a:R1566M, ORF1a:V1570A, ORF1a:V1570M, ORF1a:V1570L, ORF1a:T1572I, ORF1a:T1638A, ORF1a:T1638I, ORF1a:T1638N, ORF1a:D1639N, ORF1a:E1706D, ORF1a:N1709S, ORF1a:N1709T, ORF1a:I1714V, ORF1a:I1714T, ORF1a:I1714A, ORF1a:M1769V, ORF1a:M1771V, ORF1a:T1773I, ORF1a:F1779L, ORF1a:K1795Q, ORF1a:P2134S, ORF1a:P2134L, ORF1a:P2134H, ORF1a:D2136N, ORF1a:D2136G, ORF1a:I2138V, ORF1a:C2165F, ORF1a:T2166I, ORF1a:F2209L, ORF1a:S2214L, ORF1a:F2215L, ORF1a:T2274I, ORF1a:A2325T, ORF1a:A2325V, ORF1a:F2328V, ORF1a:F2328L, ORF1a:L2329F, ORF1a:S2466G, ORF1a:D2467E, ORF1a:E2468D, ORF1a:V2469I, ORF1a:A2470S, ORF1a:R2471G, ORF1a:R2471S, ORF1a:R2471K, ORF1a:D2472E, ORF1a:L2475I, ORF1a:A3648V, ORF1a:V3653A, ORF1a:A3657V, ORF1a:R3802S, ORF1a:R3802H, ORF1a:R3802C, ORF1a:Y3803H, ORF1a:L3808F, ORF1a:L3951F, ORF1a:P3952S, ORF1a:T4087A, ORF1a:T4087I, ORF1a:T4088I, ORF1a:T4090I, ORF1a:T4090A, ORF1a:T4164N, ORF1a:T4164I, ORF1a:D4165N, ORF1a:D4165V, ORF1a:D4165G, ORF1a:D4165A, ORF1a:D4166N, ORF1a:D4166Y, ORF1a:D4166G, ORF1a:N4167S, ORF1a:A4396V, ORF1a:A4396T, ORF1a:S4398L, ORF1b:Q72R, ORF1b:E75K, ORF1b:T76I, ORF1b:T730I, ORF1b:L820F, ORF1b:P821S, ORF1b:D824N, ORF1b:D824E, ORF1b:S826A, ORF1b:L1220F, N:V270L, N:T271I, N:P326H, N:P326L, ORF3a:Q213K, ORF7a:E22D, ORF7a:T39I, ORF9b:S10P, ORF9b:R13C, ORF9b:L64P, ORF9b:D66E, ORF9b:A68E, ORF9b:A68V, ORF9b:V93M, ORF9b:V93L, ORF9b:T95M, ORF9b:T95A, ORF9b:V96A")
correlated_muts_po_sub2pos = list_to_strings_set("ORF1a:85, ORF1a:110, ORF1a:111, ORF1a:114, ORF1a:233, ORF1a:235, ORF1a:958, ORF1a:959, ORF1a:1218, ORF1a:1220, ORF1a:1221, ORF1a:1223, ORF1a:1224, ORF1a:1225, ORF1a:1272, ORF1a:1273, ORF1a:1274, ORF1a:1276, ORF1a:1322, ORF1a:1323, ORF1a:1326, ORF1a:1365, ORF1a:1367, ORF1a:1368, ORF1a:1369, ORF1a:1372, ORF1a:1394, ORF1a:1395, ORF1a:1397, ORF1a:1500, ORF1a:1504, ORF1a:1505, ORF1a:1507, ORF1a:1542, ORF1a:1543, ORF1a:1566, ORF1a:1570, ORF1a:1572, ORF1a:1638, ORF1a:1639, ORF1a:1709, ORF1a:1714, ORF1a:1716, ORF1a:1769, ORF1a:1773, ORF1a:2134, ORF1a:2136, ORF1a:2138, ORF1a:2166, ORF1a:2209, ORF1a:2214, ORF1a:2215, ORF1a:2274, ORF1a:2324, ORF1a:2325, ORF1a:2328, ORF1a:2345, ORF1a:2346, ORF1a:2349, ORF1a:2351, ORF1a:2353, ORF1a:2466, ORF1a:2467, ORF1a:2468, ORF1a:2469, ORF1a:2470, ORF1a:2471, ORF1a:2472, ORF1a:2475, ORF1a:3443, ORF1a:3648, ORF1a:3650, ORF1a:3652, ORF1a:3653, ORF1a:3657, ORF1a:3802, ORF1a:3803, ORF1a:3808, ORF1a:3949, ORF1a:3951, ORF1a:3952, ORF1a:4083, ORF1a:4087, ORF1a:4088, ORF1a:4090, ORF1a:4097, ORF1a:4102, ORF1a:4164, ORF1a:4165, ORF1a:4166, ORF1a:4167, ORF1a:4396, ORF1a:4397, ORF1a:4398, ORF1b:72, ORF1b:75, ORF1b:76, ORF1b:662, ORF1b:730, ORF1b:820, ORF1b:821, ORF1b:822, ORF1b:824, ORF1b:826, ORF1b:1220, N:270, N:271, N:326, ORF3a:213, ORF7a:22, ORF7a:39, ORF9b:93, ORF9b:95")
correlated_muts_combined_sub2pos = union(correlated_muts_sub2pos, correlated_muts_po_sub2pos)
BAL_muts_sub2pos = list_to_strings_set("ORF1a:K4176R, ORF1a:F3089L, ORF1a:T3032I, E:A32V, E:S68F, ORF1a:T2183I, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:F374L, S:F375-, S:A376V, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659P, S:A668V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:L1186F, E:V5A, E:V5F, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:F23S, E:T30I, E:A36V, E:Y42C, M:A2V, M:S4P, M:V10I, M:F28S, M:T77N, M:S94R, M:S94N, M:H125Y, M:H148R, M:A188T, N:P80L, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*")
BAL_muts_po_sub2pos = Set(["ORF1a:2183","ORF1a:2972","ORF1a:3049","ORF1a:3058","ORF1a:3070","ORF1a:3072","ORF1a:3076","ORF1a:3123","ORF1a:3195","ORF1a:3359","ORF1a:3454","ORF1a:3456","ORF1a:4110","ORF1a:4175","ORF1a:4197","ORF1a:4205","ORF1a:4207","ORF1a:4387","ORF1b:314","ORF1b:1181","ORF1b:1247","ORF1b:1424","ORF1b:2339","ORF1b:2340","ORF1b:2537","S:50","S:330","S:354","S:367","S:374","S:375","S:376","S:405","S:508","S:551","S:573","S:647","S:653","S:657","S:659","S:668","S:859","S:944","S:958","S:978","S:1169","S:1179","S:1186","E:5","E:5","E:9","E:10","E:10","E:13","E:14","E:16","E:23","E:30","E:36","E:42","M:2","M:4","M:10","M:28","M:77","M:94","M:94","M:125","M:148","M:188","N:80","N:416","N:417","ORF7a:115","ORF9b:1","ORF9b:77"])
BAL_muts_combined_sub2pos = union(BAL_muts_sub2pos, BAL_muts_po_sub2pos)
Others_sub2pos = Set(["S:Y248F", "ORF1a:L2084I", "ORF1a:S2083-", "S:N137M", "ORF9b:V30-", "ORF7b:*44Y", "ORF7b:M1T", "S:V213P", "S:R214E", "ORF1a:S142T", "ORF1a:F143M", "S:Y144D", "S:V143D", "ORF7b:I2V", "ORF7b:M1T", "ORF3a:L15V", "ORF9b:P10S", "S:T19R", "S:A27S", "S:V213G", "ORF3a:N257D", "M:I8F", "S:D145Y", "S:I212L", "ORF9b:G16D", "S:L18F", "ORF8:V117L", "ORF8:L118V", "ORF8:F120V", "ORF8:F120L", "ORF8:I121L", "ORF7a:A82V", "S:I212L", "S:L212I", "S:N211V", "S:R493Q", "ORF7b:M1I", "ORF7b:I2N", "ORF1b:R2613N", "ORF1b:P2612L", "ORF1a:L3606F", "S:H655Y", "N:I84V"])
Others_po_sub2pos = Set(["ORF9b:27", "ORF9b:28", "ORF9b:29", "ORF9b:30", "ORF1a:3674", "S:248", "M:3", "ORF1a:2084", "ORF1a:2083", "ORF1a:142", "ORF1a:143", "ORF1a:3606", "ORF1b:2612", "ORF1b:2613", "S:18", "S:19", "S:24", "S:25", "S:26", "S:27", "S:137", "S:137", "S:139", "S:141", "S:142", "S:142", "S:143", "S:143", "S:144", "S:144", "S:145", "S:152", "S:153", "S:154", "S:155", "S:156", "S:211", "S:212", "S:212", "S:213", "S:213", "S:214", "S:242", "S:493", "S:655", "M:8", "N:84", "ORF3a:15", "ORF3a:257", "ORF7a:82", "ORF7b:1", "ORF7b:1", "ORF7b:2", "ORF7b:2", "ORF7b:44", "ORF8:117", "ORF8:118", "ORF8:120", "ORF8:120", "ORF8:121", "ORF9b:10", "ORF9b:16", "ORF9b:30"])
Others_combined_sub2pos = union(Others_sub2pos, Others_po_sub2pos)
Temp_Others_sub2pos = Set(["N:T205I", "ORF1b:P1000L", "S:T19R"])
Temp_Others_po_sub2pos = Set(["N:205", "ORF1b:1000", "S:19"])
Temp_Others_combined_sub2pos = union(Temp_Others_sub2pos, Temp_Others_po_sub2pos)
Temp_Others2_sub2pos = Set(["S:R493Q", "S:E484K"])  
Temp_Others2_po_sub2pos = Set(["S:493", "S:484"])
Temp_Others2_combined_sub2pos = union(Temp_Others2_sub2pos, Temp_Others2_po_sub2pos)

ORF3a_dels = list_to_strings_set("ORF3a:M1-, ORF3a:D2-, ORF3a:L3-, ORF3a:F4-, ORF3a:M5-, ORF3a:R6-, ORF3a:I7-, ORF3a:F8-, ORF3a:T9-, ORF3a:I10-, ORF3a:G11-, ORF3a:T12-, ORF3a:V13-, ORF3a:L15-, ORF3a:K16-, ORF3a:Q17-, ORF3a:G18-, ORF3a:E19-, ORF3a:I20-, ORF3a:K21-, ORF3a:D22-, ORF3a:A23-, ORF3a:T24-, ORF3a:P25-, ORF3a:S26-, ORF3a:D27-, ORF3a:F28-, ORF3a:V29-, ORF3a:R30-, ORF3a:A31-, ORF3a:T32-, ORF3a:A33-, ORF3a:T34-, ORF3a:I35-, ORF3a:P36-, ORF3a:I37-, ORF3a:Q38-, ORF3a:A39-, ORF3a:S40-, ORF3a:L41-, ORF3a:P42-, ORF3a:F43-, ORF3a:G44-, ORF3a:W45-, ORF3a:L46-, ORF3a:I47-, ORF3a:V48-, ORF3a:G49-, ORF3a:V50-, ORF3a:A51-, ORF3a:L52-, ORF3a:L53-, ORF3a:A54-, ORF3a:V55-, ORF3a:F56-, ORF3a:Q57-, ORF3a:S58-, ORF3a:A59-, ORF3a:S60-, ORF3a:K61-, ORF3a:I62-, ORF3a:I63-, ORF3a:T64-, ORF3a:L65-, ORF3a:K66-, ORF3a:K67-, ORF3a:R68-, ORF3a:W69-, ORF3a:P258-, ORF3a:V259-, ORF3a:M260-, ORF3a:E261-, ORF3a:P262-, ORF3a:I263-")
ORF6_dels = list_to_strings_set("ORF6:M1-, ORF6:F2-, ORF6:H3-, ORF6:L4-, ORF6:V5-, ORF6:D6-, ORF6:F7-, ORF6:Q8-, ORF6:V9-, ORF6:T10-, ORF6:I11-, ORF6:A12-, ORF6:E13-, ORF6:I14-, ORF6:L15-, ORF6:L16-, ORF6:I17-, ORF6:I18-, ORF6:M19-, ORF6:R20-, ORF6:T21-, ORF6:F22-, ORF6:K23-, ORF6:V24-, ORF6:S25-, ORF6:I26-, ORF6:W27-, ORF6:N28-, ORF6:L29-, ORF6:D30-, ORF6:Y31-, ORF6:I32-, ORF6:I33-, ORF6:N34-, ORF6:L35-, ORF6:I36-, ORF6:I37-, ORF6:K38-, ORF6:N39-, ORF6:L40-, ORF6:S41-, ORF6:K42-, ORF6:S43-, ORF6:L44-, ORF6:T45-, ORF6:E46-, ORF6:N47-, ORF6:K48-, ORF6:Y49-, ORF6:S50-, ORF6:Q51-, ORF6:L52-, ORF6:D53-, ORF6:E54-, ORF6:E55-, ORF6:Q56-, ORF6:P57-, ORF6:M58-, ORF6:E59-, ORF6:I60-, ORF6:D61-, ORF6:*62-")
ORF7_dels = list_to_strings_set("ORF7a:M1-, ORF7a:K2-, ORF7a:I3-, ORF7a:I4-, ORF7a:L5-, ORF7a:F6-, ORF7a:L7-, ORF7a:A8-, ORF7a:L9-, ORF7a:I10-, ORF7a:T11-, ORF7a:L12-, ORF7a:A13-, ORF7a:T14-, ORF7a:C15-, ORF7a:E16-, ORF7a:L17-, ORF7a:Y18-, ORF7a:H19-, ORF7a:Y20-, ORF7a:Q21-, ORF7a:E22-, ORF7a:C23-, ORF7a:V24-, ORF7a:R25-, ORF7a:G26-, ORF7a:T27-, ORF7a:T28-, ORF7a:V29-, ORF7a:L30-, ORF7a:L31-, ORF7a:K32-, ORF7a:E33-, ORF7a:P34-, ORF7a:C35-, ORF7a:S36-, ORF7a:S37-, ORF7a:G38-, ORF7a:T39-, ORF7a:Y40-, ORF7a:E41-, ORF7a:G42-, ORF7a:N43-, ORF7a:S44-, ORF7a:P45-, ORF7a:F46-, ORF7a:H47-, ORF7a:P48-, ORF7a:L49-, ORF7a:A50-, ORF7a:D51-, ORF7a:N52-, ORF7a:K53-, ORF7a:F54-, ORF7a:A55-, ORF7a:L56-, ORF7a:T57-, ORF7a:C58-, ORF7a:F59-, ORF7a:S60-, ORF7a:T61-, ORF7a:Q62-, ORF7a:F63-, ORF7a:A64-, ORF7a:F65-, ORF7a:A66-, ORF7a:C67-, ORF7a:P68-, ORF7a:D69-, ORF7a:G70-, ORF7a:V71-, ORF7a:K72-, ORF7a:H73-, ORF7a:V74-, ORF7a:Y75-, ORF7a:Q76-, ORF7a:L77-, ORF7a:R78-, ORF7a:A79-, ORF7a:R80-, ORF7a:S81-, ORF7a:V82-, ORF7a:S83-, ORF7a:P84-, ORF7a:K85-, ORF7a:L86-, ORF7a:F87-, ORF7a:I88-, ORF7a:R89-, ORF7a:Q90-, ORF7a:E91-, ORF7a:E92-, ORF7a:V93-, ORF7a:Q94-, ORF7a:E95-, ORF7a:L96-, ORF7a:Y97-, ORF7a:S98-, ORF7a:P99-, ORF7a:I100-, ORF7a:F101-, ORF7a:L102-, ORF7a:I103-, ORF7a:V104-, ORF7a:A105-, ORF7a:A106-, ORF7a:I107-, ORF7a:V108-, ORF7a:F109-, ORF7a:I110-, ORF7a:T111-, ORF7a:L112-, ORF7a:C113-, ORF7a:F114-, ORF7a:T115-, ORF7b:M1-, ORF7b:I2-, ORF7b:E3-, ORF7b:L4-, ORF7b:S5-, ORF7b:L6-, ORF7b:I7-, ORF7b:D8-, ORF7b:F9-, ORF7b:Y10-, ORF7b:L11-, ORF7b:C12-, ORF7b:F13-, ORF7b:L14-, ORF7b:A15-, ORF7b:F16-, ORF7b:L17-, ORF7b:L18-, ORF7b:F19-, ORF7b:L20-, ORF7b:V21-, ORF7b:L22-, ORF7b:I23-, ORF7b:M24-, ORF7b:L25-, ORF7b:I26-, ORF7b:I27-, ORF7b:F28-, ORF7b:W29-, ORF7b:F30-, ORF7b:S31-, ORF7b:L32-, ORF7b:E33-, ORF7b:L34-, ORF7b:Q35-, ORF7b:D36-, ORF7b:H37-, ORF7b:N38-, ORF7b:E39-, ORF7b:T40-, ORF7b:C41-, ORF7b:H42-, ORF7b:A43-, ORF7b:*44-")
ORF8_dels = list_to_strings_set("ORF8:M1-, ORF8:K2-, ORF8:F3-, ORF8:L4-, ORF8:V5-, ORF8:F6-, ORF8:L7-, ORF8:G8-, ORF8:I9-, ORF8:I10-, ORF8:T11-, ORF8:T12-, ORF8:V13-, ORF8:A14-, ORF8:A15-, ORF8:F16-, ORF8:H17-, ORF8:Q18-, ORF8:E19-, ORF8:C20-, ORF8:S21-, ORF8:L22-, ORF8:Q23-, ORF8:S24-, ORF8:C25-, ORF8:T26-, ORF8:Q27-, ORF8:H28-, ORF8:Q29-, ORF8:P30-, ORF8:Y31-, ORF8:V32-, ORF8:V33-, ORF8:D34-, ORF8:D35-, ORF8:P36-, ORF8:C37-, ORF8:P38-, ORF8:I39-, ORF8:H40-, ORF8:F41-, ORF8:Y42-, ORF8:S43-, ORF8:K44-, ORF8:W45-, ORF8:Y46-, ORF8:I47-, ORF8:R48-, ORF8:V49-, ORF8:G50-, ORF8:A51-, ORF8:R52-, ORF8:K53-, ORF8:S54-, ORF8:A55-, ORF8:P56-, ORF8:L57-, ORF8:I58-, ORF8:E59-, ORF8:L60-, ORF8:C61-, ORF8:V62-, ORF8:D63-, ORF8:E64-, ORF8:A65-, ORF8:G66-, ORF8:S67-, ORF8:K68-, ORF8:S69-, ORF8:P70-, ORF8:I71-, ORF8:Q72-, ORF8:Y73-, ORF8:I74-, ORF8:D75-, ORF8:I76-, ORF8:G77-, ORF8:N78-, ORF8:Y79-, ORF8:T80-, ORF8:V81-, ORF8:S82-, ORF8:C83-, ORF8:L84-, ORF8:P85-, ORF8:F86-, ORF8:T87-, ORF8:I88-, ORF8:N89-, ORF8:C90-, ORF8:Q91-, ORF8:E92-, ORF8:P93-, ORF8:K94-, ORF8:L95-, ORF8:G96-, ORF8:S97-, ORF8:L98-, ORF8:V99-, ORF8:V100-, ORF8:R101-, ORF8:C102-, ORF8:S103-, ORF8:F104-, ORF8:Y105-, ORF8:E106-, ORF8:D107-, ORF8:F108-, ORF8:L109-, ORF8:E110-, ORF8:Y111-, ORF8:H112-, ORF8:D113-, ORF8:V114-, ORF8:R115-, ORF8:V116-, ORF8:V117-, ORF8:L118-, ORF8:D119-, ORF8:F120-")
ORF9b_N_dels = list_to_strings_set("N:Q28-, N:N29-, N:G30-, N:E31-, N:R32-, N:S33-, N:G34-, ORF9b:T24-, ORF9b:R25-, ORF9b:M26-, ORF9b:E27-, ORF9b:N28-, ORF9b:I29-, ORF9b:V30-, ORF9b:L30-, ORF9b:G31-")
ORF1a_dels = list_to_strings_set("ORF1a:H81-, ORF1a:G82-, ORF1a:H83-, ORF1a:V84-, ORF1a:M85-, ORF1a:V86-, ORF1a:E87-, ORF1a:I1005-, ORF1a:V1006-, ORF1a:E1007-, ORF1a:L3674-")
S_dels = list_to_strings_set("S:L7-, S:L8-, S:P9-, S:L10-, S:V11-, S:S12-, S:S13-, S:Q14-, S:V16-, S:N17-, S:L18-, S:T19-, S:I19-, S:R19-, S:T20-, S:T21-, S:R21-, S:T22-, S:N22-, S:Q23-, S:S24-, S:L24-, S:P25-, S:P26-, S:A27-, S:S27-, S:Y28-, S:N61-, S:V62-, S:T63-, S:W64-, S:F65-, S:H66-, S:A67-, S:I68-, S:S71-, S:G72-, S:T73-, S:N74-, S:G75-, S:T76-, S:K77-, S:F135-, S:N137-, S:P139-, S:F140-, S:L141-, S:G142-, S:D142-, S:Y145-, S:D145-, S:H146-, S:Q146-, S:K147-, S:N148-, S:N149-, S:K150-, S:M153-, S:E154-, S:S155-, S:E156-, S:F157-, S:S157-, S:L157-, S:R158-, S:L176-, S:N188-, S:L179-, S:E180-, S:S256-, S:G257-, S:W258-, S:T259-, S:A260-, S:G261-, S:A262-, S:A263-, S:A264-")
M_dels = list_to_strings_set("M:N5-, M:G6-, M:T7-")
COMBINED_dels = union(ORF3a_dels, ORF6_dels, ORF7_dels, ORF8_dels, ORF9b_N_dels, ORF1a_dels, S_dels, M_dels)

Doubtful_Revs_sub2pos = Set(["ORF9b:A60T, N:G63D, S:F18L", "ORF1a:G1307S", "ORF1a:R856K", "ORF6:L61D", "N:R413S", "S:G158R", "ORF7b:I40T", "N:R204G", "S:F981L", "S:D339G", "S:V67A", "S:I95T"])
Doubtful_Revs_po_sub2pos = Set(["ORF9b:60, N:63, S:18", "ORF1a:1307", "ORF1a:856", "ORF6:61", "N:413", "S:158", "ORF7b:40", "N:204", "S:981", "S:339", "S:67"])
Doubtful_Revs_combined_sub2pos = union(Doubtful_Revs_sub2pos, Doubtful_Revs_po_sub2pos)
Cryptics_sub2pos = Set(["ORF3a:W45R", "ORF1b:F2314L", "S:K417T", "S:V1176F", "S:L828F", "ORF3a:Q213L", "ORF3a:H182D", "S:K1266N", "S:L828F", "S:T19K", "S:H655Y", "S:T859N", "ORF1a:V38A", "N:S37P", "ORF1b:F2314V", "S:D1153A", "ORF1a:L3606V", "ORF1b:E2311K"])
Cryptics_po_sub2pos = Set(["ORF3a:45", "ORF1b:2314", "S:417", "S:1176", "S:828", "ORF3a:213", "ORF3a:182", "S:1266", "S:828", "S:19", "S:655", "S:859", "ORF1a:38", "N:37", "ORF1b:2314", "S:1153", "ORF1a:3606", "ORF1b:2311"])
Cryptics_combined_sub2pos = union(Cryptics_sub2pos, Cryptics_po_sub2pos)
#################### Note: commenting out lines below retains them from `excluded_muts`; NOT commenting them out excludes them from `excluded_muts` ########################
            correlated_muts_combined_sub2pos = Set{String}()
            BAL_muts_combined_sub2pos = Set{String}()
            Cryptics_combined_sub2pos = Set{String}()
#            Others_combined_sub2pos = Set{String}()
#            Temp_Others2_combined_sub2pos = Set{String}()
#            Double_N_ORF9b_combined_sub2pos = Set{String}()
            ORF9b_all_muts_combined_sub2pos = Set{String}()
#             = Set{String}()
###########################################################################################################################################################################
###########################################################################################################################################################################
excluded_muts_sub2pos = union(COMBINED_dels, Double_N_ORF9b_combined_sub2pos, ORF9b_all_muts_combined_sub2pos, Cryptics_combined_sub2pos, BAL_muts_combined_sub2pos, Others_combined_sub2pos, Temp_Others_combined_sub2pos, Doubtful_Revs_combined_sub2pos, artifactual_private_muts, RBM_muts) #, RBD_not_RBM_muts) #,BAL_muts)
###########################################################################################################################################################################
###########################################################################################################################################################################
rep_seq_grps_maxmut_seqs_set = collect(values(rep_seq_grps_maxmut_seqs))
all_unique_chr_seqs_set = union(rep_seq_grps_maxmut_seqs_set, non_rep_seqs)
total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
#####################################################################################################################
avg_AA_sub_ct_per_chronic_seq = round(digits=2, total_chronic_AA_ct/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq = $(avg_AA_sub_ct_per_chronic_seq)")
#####################################################################################################################
total_EPCI_seq_ct = length(all_unique_chr_seqs_set) # Method #2 for finding avg AA subs + del ranges/seq in chronic list
total_chronic_AA_ct_v2 = 0
for seq in all_unique_chr_seqs_set
    for AA_mut in seq_AA_muts_no_dels[seq]
        total_chronic_AA_ct_v2 += 1
    end
end
#####################################################################################################################
avg_AA_sub_ct_per_chronic_seq_v2 = round(digits=2, total_chronic_AA_ct_v2/total_EPCI_seq_ct)  #### Double checking the first count
println("avg_AA_sub_ct_per_chronic_seq_v2 = $(avg_AA_sub_ct_per_chronic_seq_v2)")
###########################################################################################################################################################################
####### NOTE: Mutations in exc_muts & exc_muts_po exclude any SEQUENCE with one of those mutations from the analysis. 
function AA_2plus__mut2pos_pos2mut__2026_02_12(min_mut_ct::Int, min_chi_sq::Int, min_log_pv_fish::Float64, print_min::Int, exc_muts::Vector{String}, exc_muts_po::Vector{String}, sel_muts::Vector{String}, seq_factor_OFF_HALF_ON__0_1_2::Int, nonmulti_0___multi_1::Int, with_D138del_0____without_D138del_1::Int, all_muts_0___disulfide_only_1___disulfide_chk_2::Int)
#    date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
    date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
    start_time = time()
    global folder_date
###################################################################
    max_log_pv_fish = 0.0
###################################################################
df_mut_corr = DataFrame(
    Mutation = String[],
    MatchMut = String[],
    Match_Ct = Int[],
    Mut_Ct = Int[],
    match_EPCI_Ct = Int[],
    logpvFish = Float64[],
    Fold_Incr = Union{Float64, String}[],
    EPCI_pct = Float64[],
    MP_pct = Float64[],
    non_MP_pct = Float64[],
    pvFish = Float64[])
df_mut_corr_gene_sort = DataFrame(
    Mutation = String[],
    MatchMut = String[],
    Match_Ct = Int[],
    Mut_Ct = Int[],
    match_EPCI_Ct = Int[],
    logpvFish = Float64[],
    Fold_Incr = Union{Float64, String}[],
    EPCI_pct = Float64[],
    MP_pct = Float64[],
    non_MP_pct = Float64[],
    pvFish = Float64[])
df_mut_corr_unfiltered = DataFrame(
    Mutation = String[],
    MatchMut = String[],
    Match_Ct = Int[],
    Mut_Ct = Int[],
    match_EPCI_Ct = Int[],
    logpvFish = Float64[],
    Fold_Incr = Union{Float64, String}[],
    EPCI_pct = Float64[],
    MP_pct = Float64[],
    non_MP_pct = Float64[],
    pvFish = Float64[])
###################################################################
    mut_str = join(sel_muts, "_")
    mut_str_name = replace(mut_str, ":"=>"_")
    if length(mut_str_name) > 200
        mut_str_name = "P9LT_S12P_S13I_C15del_C15YSFGLR_R21I_W64C_S71C_C136del_C136FGRY_P139ST_G142C_D142C_Y144C_Y145C_S151C_W152C_S247C_Y248C_W258C_G261C"
    end
#    mut_str_colon_split = split(mut_str1, ":")
#    mut_str = mut_str_under_join
#    mut_str = replace(":", mut_str1, "_")
    folder = ""
#    date_now = Dates.format(now(), "yyyy_mm_dd__IMMp")
    if nonmulti_0___multi_1 == 1
        if none_0___sub2pos_1___pos2sub_2 == 1
            folder = "sub2pos/$(folder_date)/sub2pos__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now)"
        elseif none_0___sub2pos_1___pos2sub_2 == 2
            folder = "pos2sub/$(folder_date)/pos2sub__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now)"
        elseif none_0___sub2pos_1___pos2sub_2 == 0
            if sub_0__posonly_1__mixed_2 == 0
                folder = "sub_corr/$(folder_date)/sub__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now)"
            elseif sub_0__posonly_1__mixed_2 == 1
                folder = "pos_only_corr/$(folder_date)/pos_only__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__$(HQCS_qc_string)__$(date_now)"
            end
        end
        mkdir(folder)
    elseif nonmulti_0___multi_1 == 0
        if none_0___sub2pos_1___pos2sub_2 == 1
            folder = "sub2pos/sub2pos__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now)"
        elseif none_0___sub2pos_1___pos2sub_2 == 2
            folder = "pos2sub/pos2sub__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now)"
        elseif none_0___sub2pos_1___pos2sub_2 == 0
            if sub_0__posonly_1__mixed_2 == 0
                folder = "sub_corr/sub__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now)"
            elseif sub_0__posonly_1__mixed_2 == 1
                folder = "pos_only_corr/pos_only__$(mut_str_name)__correlated_muts__EPCI_$(EPCI_qc_str)__$(HQCS_qc_string)__$(date_now)"
            end
        end
        mkdir(folder)
    end 
#                                                        none_0___sub2pos_1___pos2sub_2 = 1
#                                                        sub_0__posonly_1__mixed_2 = 2
#                                                        normal_0__spikeonly_1 = 0
#                                                        noBAL_0__withBAL_1 = 1
#### Note: nd = no deletions;  po = position only
    total_mut_ct = 0
    total_AA_mut_ct_nd = 0
######################################################################################################################################
    total_group_ct = length(rep_seq_grps_maxmut_seqs)
    total_singlet_ct = length(non_rep_seqs)
    total_chronic_ct = total_group_ct + total_singlet_ct
################################ Below: New stuff for dealing with unknown AA (2025-3-24) ############################################
#### coAAmut_ct_unknown counts the number of timesa given AA position is "unknown," either due to dropout or a mixed nucleotide for
#       the sequences that qualify as matching a given mutational pattern 
#                      unknown_AA_pos  count
    coAAmut_ct_unknown = Dict{String, Int}()      ### Total unknown count for each AA position in each qualifying sequence (i.e. each seq w/ ≥ min_ct sel_muts)
    total_AApos_ct_unknown = Dict{String, Int}()  ### Total unknown count for each AA position in each independent chronic sequence
    for seq in all_unique_chr_seqs_set
        for pos in seq_unknown_AA[seq]
            total_AApos_ct_unknown[pos] = get(total_AApos_ct_unknown, pos, 0) + 1
        end
    end
######################################################################################################################################
### MP_seqs_unknown_region_ct tallies the number of mutational regions with at least one unknown AA
#                                         EPI_ISL  total_regions_with_unknown_AA_count_in_seq
    MP_seqs_unknown_region_ct = Dict{String, Int}()
    total_unknown_region_ct = 0
######################################################################################################################################
## coAAmut_ct = number of sequences meeting specified criteria (i.e. that have ≥X AA muts from a given mutational pattern) that possess a given mutation. 
#                     AA_mut   Count  
    coAAmut_ct = Dict{String, Int}()                  
    coAAmut_ct_nonlist = Dict{String, Int}()          ## nonlist = muts/positions in mutation-pattern list excluded from count
###################################
    mut_groups_ln = length(sel_muts)                  ## Total number of mutation regions searched for (= # of mutations if all indicated individually, e.g. "E:16, E:30, E:31"=1 region, i.e. length=1; "E:16", "E:30", "E:31"=3 mutations, i.e. length=3)
###################################
### Each mut_group is assigned a number, and the value for each group number is a vector containing all the mutations in that group.
### If every mutation is listed as a separate string in the `sel_muts` parameter, then each mutation is its own mut_group.
### po = position-only; nonlist = mutations listed in sel_muts are excluded
    mut_groups = Dict{Int, Vector{String}}(i => Vector{String}() for i in 1:mut_groups_ln)  ## 
#########################################################################################################################################################
### This section is to determine the number of sequences with mutations that are clearly associated with C15-C136 NTD disulfide deletion or rearrangement.
### Later, this number os used to calculate the degree to which seqs w/the searched-for mut correlate with seqs w/NTD-disulfide-changing mutations
#    NTD_C15_disulfide_muts = list_to_strings_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R")
#    NTD_C136_disulfide_muts = list_to_strings_set("S:W64C, S:S71C, S:C136-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:Y248C, S:W258C, S:G261C")
##### Added to core OG_NTD_C136_disulfide_muts due to log10pvfish ≥ 4 for NTD_disulfide or major disulfide mut (like P9L): S:W64C, S:P139S, S:P139T, S:S151C, (S:D142C, S:G142C, S:Y144C, S:Y145C——tested & added collectively), 
#   comprehensive_NTD_disulfide_muts = list_to_strings_set(""S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:C136-, S:D138-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:W64C, S:S71C, S:G75C, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:S247C, S:Y248C, S:S255C, S:G257C, S:W258C, S:G261C")
    NTD_disulfide_filter_positions = list_to_strings_set("S:9, S:13, S:15, S:136, S:137, S:139, S:151, S:152")
    NTD_disulfide_positions_combined = list_to_strings_set("S:9, S:13, S:15, S:136")
#    NTD_disulfide_muts_combined = list_to_strings_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:W64C, S:S71C, S:C136-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:Y248C, S:W258C, S:G261C")
#    NTD_disulfide_all_combined = union(NTD_disulfide_muts_combined, NTD_disulfide_positions_combined)
    
#############################################################################################################################################################################
#############################################################################################################################################################################
    disulfide_comprehensive_for_corr_muts_with_D138del = list_to_strings_set("NTD:Disulfide, NTD_disulfide, NTD:disulfide, S:P9S, S:P9L, S:P9T, S:P9Q, S:S12P, S:S13I, S:S13C, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64G, S:W64C, S:S71C, S:C136-, S:D138-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:R158I, S:Y248C, S:W258C, S:G261C")
    disulfide_comprehensive_for_corr_muts_without_D138del = list_to_strings_set("NTD:Disulfide, NTD_disulfide, NTD:disulfide, S:P9S, S:P9L, S:P9T, S:P9Q, S:S12P, S:S13I, S:S13C, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64G, S:W64C, S:S71C, S:C136-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:R158I, S:Y248C, S:W258C, S:G261C")
    disulfide_comprehensive_for_corr_muts = Set{String}()
    if with_D138del_0____without_D138del_1 == 0
        disulfide_comprehensive_for_corr_muts = disulfide_comprehensive_for_corr_muts_with_D138del
    else
        disulfide_comprehensive_for_corr_muts = disulfide_comprehensive_for_corr_muts_without_D138del
    end
    NTD_disulfide_muts_combined = disulfide_comprehensive_for_corr_muts
#############################################################################################################################################################################
#############################################################################################################################################################################
### all_sel_muts & all_sel_muts_po are lists of all individual mutations & mutation-positions in sel_muts and sel_muts_po
### The only reason to create a list of these is so that the mutations contained in the list can be excluded from the results in e.g., in coAAmut_ct_nonlist and coAAmut_ct_nonlist_po2
    all_sel_muts = Set{String}()
    all_sel_muts_po = Set{String}()
### sel_muts = selected mutations/mut groups. If mutations are listed individually (e.g. "E:F4L", "E:V5I", "E:I9T", etc) then each mutation
###      is a "group." If all the mutations in a given region are grouped together (e.g. "E:F4L, E:V5I, E:I9T") each group is a group.
    mut_grp_ct = 0
    for mut_grp in sel_muts
        mut_grp_ct += 1
        muts = split(mut_grp, ", ")
        for mut in muts
            if !(mut in excluded_muts_sub2pos)
                push!(mut_groups[mut_grp_ct], mut)
                push!(all_sel_muts, mut)
                push!(all_sel_muts_po, aa_gene_and_pos_comprehensive_dict[mut])
            end
        end
    end
    all_sel_muts_combined = Set(union(all_sel_muts, all_sel_muts_po))
    if "" in all_sel_muts_combined
        delete!(all_sel_muts_combined, "")
    end
#    println("all_sel_muts_combined = $(all_sel_muts_combined)")
######################################################################################################################################
    MP_seqs = Set{String}()  ## MP_seqs = all seqs that meet criteria
    MP_seqs_pangos = Dict{String, Int}()  ## MP_seqs_pangos = Count for Pango lineages for all seqs that meet criteria
    MP_seqs_inherited = Dict{String, Int}()  ## MP_seqs_inherited = Counts for inherited mutations among sequences on list. 
    MP_seqs_priv_AA_adjustment_factors = Set{Float64}()  ## factors used to adjust for # of priv AA muts in a sequence
######################################################################################################################################
##  rep_seq_grps_maxmut_seqs is a dictionary containing one sequence that represents each group of related chronic sequences. 
##  Each representative sequence is the one with the most private AA muts in the group. Its keys are the group's number (which are arbitrary) and its values are the EPI_ISL number for the sequence.
##  exc_muts are forbidden mutations. If a sequence has one, it is excluded from consideration.  
##  non_rep_seqs are chronic singlets (i.e. seqs with no closely related sequences from the same patient)
##  MP_seqs is a set containing all sequences that meet the qualifying criteria (e.g. that have ≥2 mutations in sel_muts)
    for seq in all_unique_chr_seqs_set
        if isempty(intersect(exc_muts, seq_AA_muts_combined[seq]))
##############################################
            seq_priv_AA_mut_total = length(seq_AA_muts_no_dels[seq])
            seq_factor = (avg_AA_sub_ct_per_chronic_seq/seq_priv_AA_mut_total)
            half_seq_factor = (seq_factor+1)/2
            if seq_factor_OFF_HALF_ON__0_1_2 == 0
                seq_factor = 1
            end
            if seq_factor_OFF_HALF_ON__0_1_2 == 1
                seq_factor = half_seq_factor
            end
##############################################
            if length(sel_muts) == 1
                for mut in sel_muts
                    if isempty(intersect(exc_muts, sel_muts)) && !isempty(intersect(seq_AA_muts_sub2pos_pos2sub_ref[seq], sel_muts))
                        push!(MP_seqs, seq)
                    end
                end
            else
                seq_grp_mut_ct = 0
                for (group, muts) in mut_groups
                    group_ct = 0
                    if !isempty(intersect(seq_AA_muts_sub2pos_pos2sub_ref[seq], muts))
                        seq_grp_mut_ct += 1
                    end
                end
                if seq_grp_mut_ct ≥ min_mut_ct
                    push!(MP_seqs, seq)
                end
            end
        end
    end
#### Next 11 lines are to determine the AA-adjustment factor for each sequence. THe more private AA muts a sequence has, the more
####  the co-occurring mutations in it are depreciated. 
    for seq in MP_seqs
        seq_priv_AA_mut_total = length(seq_AA_muts_no_dels[seq])
        seq_factor = (avg_AA_sub_ct_per_chronic_seq/seq_priv_AA_mut_total)
        half_seq_factor = (seq_factor+1)/2
        if seq_factor_OFF_HALF_ON__0_1_2 == 0
            seq_factor = 1
        end
        if seq_factor_OFF_HALF_ON__0_1_2 == 1
            seq_factor = half_seq_factor
        end
#### For each qualifying sequence, all of its private mutations are tallied in the coAAmut_ct dictionary (keys = mutations, values = counts)
        for mut in seq_AA_muts_sub2pos_pos2sub_qry[seq]
            if !(mut in excluded_muts_sub2pos)
                coAAmut_ct[mut] = get(coAAmut_ct, mut, 0) + 1
                total_mut_ct += 1
#### Same as above but excluding all sel_muts
                if !(mut in all_sel_muts) && !(mut in all_sel_muts_po)
                    coAAmut_ct_nonlist[mut] = get(coAAmut_ct_nonlist, mut, 0) + 1
                end
            end
        end
    end
    for seq in MP_seqs
        seq_priv_AA_mut_total = length(seq_AA_muts_no_dels[seq])
        total_AA_mut_ct_nd += seq_priv_AA_mut_total
    end
#    println("length_MP_seqs = $(length(MP_seqs))")
######################################################################################################################################
#### This counts the pango lineages for each sequence on the list, then counts all the mutations inherited by that pango lineage.
#### Similar to the "unknown" dicts below, the purpose here is to remove from the denominator sequences that could not possibly have had a given 
####    private mutation (because you can't acquire a mutation you already have). For example, let's say we're testing whether the private mutation 
####    S:H655Y correlates with other mutations in a list of 100 sequences. If 70 of those sequences are Omicron, all of which inherited S:H655Y,
####    then only the 30 non-Omicron sequences will be used in calculating the Fisher's exact test (as well as all other calculations).
    for seq in MP_seqs
        pango = seq_pango[seq]
        MP_seqs_pangos[pango] = get(MP_seqs_pangos, pango, 0) + 1
        if !haskey(pango_AAsub_WT, pango)
            pango = pango_predecessor_meta_dict[pango][1]
            if !haskey(pango_AAsub_WT, pango)
                pango = pango_predecessor_meta_dict[pango][2]
            end
        end
        for mut in pango_AAsub_WT[pango]
            MP_seqs_inherited[mut] = get(MP_seqs_inherited, mut, 0) + 1
        end
    end
######################################################################################################################################
######################################## Below: Stuff for dealing with unknown AA ####################################################
    for seq in MP_seqs
        for mut in seq_unknown_AA[seq]
            coAAmut_ct_unknown[mut] = get(coAAmut_ct_unknown, mut, 0) + 1
        end
    end
######################################### Unknown AA adjustments for seqs w/dropout at known NTD-disulfide-mutation positions ##############################################
#### Counting disulfide-mut seqs. Any seq with ≥1 known disulfide mut (excluding all searched-for muts) counts. 
    NTD_disulfide_pos_only_non_search = setdiff(NTD_disulfide_positions_combined, all_sel_muts_combined)
    NTD_disulfide_sub_non_search = setdiff(NTD_disulfide_muts_combined, all_sel_muts_combined)
##################################
    NTD_disulfide_EPCI_pos_only_seqs = Set{String}()
    NTD_disulfide_EPCI_pos_only_seq_ct = 0
    for seq in all_unique_chr_seqs_set
        if !isempty(intersect(NTD_disulfide_pos_only_non_search, seq_AA_muts_pos_only[seq]))
            NTD_disulfide_EPCI_pos_only_seq_ct += 1
            push!(NTD_disulfide_EPCI_pos_only_seqs, seq)
        end
    end
    AA_muts_ct_pos_only["NTD:disulfide"] = NTD_disulfide_EPCI_pos_only_seq_ct
    NTD_disulfide_EPCI_sub_seqs = Set{String}()
    NTD_disulfide_EPCI_sub_seq_ct = 0
    for seq in all_unique_chr_seqs_set
        if !isempty(intersect(NTD_disulfide_sub_non_search, seq_AA_muts[seq]))
            NTD_disulfide_EPCI_sub_seq_ct += 1
            push!(NTD_disulfide_EPCI_sub_seqs, seq)
        end
    end
    AA_muts_ct["NTD:disulfide"] = NTD_disulfide_EPCI_sub_seq_ct
    AA_muts_ct_combined["NTD:disulfide"] = NTD_disulfide_EPCI_sub_seq_ct
    NTD_disulfide_EPCI_pos_only_dropout_seqs = Set{String}()
    NTD_disulfide_EPCI_pos_only_dropout_seq_ct = 0
    for seq in all_unique_chr_seqs_set
        if !(seq in NTD_disulfide_EPCI_pos_only_seqs)
            dropout_seq_check = intersect(seq_unknown_AA[seq], NTD_disulfide_filter_positions)
            if length(dropout_seq_check) ≥ 1
#            if !isempty(intersect(seq_unknown_AA[seq], NTD_disulfide_filter_positions))
                push!(NTD_disulfide_EPCI_pos_only_dropout_seqs, seq)
                NTD_disulfide_EPCI_pos_only_dropout_seq_ct += 1
            end
        end
    end
    NTD_disulfide_EPCI_sub_dropout_seqs = Set{String}()
    NTD_disulfide_EPCI_sub_dropout_seq_ct = 0
    for seq in all_unique_chr_seqs_set
        if !(seq in NTD_disulfide_EPCI_sub_seqs)
            if !isempty(intersect(seq_unknown_AA[seq], NTD_disulfide_filter_positions))
                push!(NTD_disulfide_EPCI_sub_dropout_seqs, seq)
                NTD_disulfide_EPCI_sub_dropout_seq_ct += 1
            end
        end
    end
############################################################
    NTD_disulfide_EPCI_seqs = Set{String}()
    NTD_disulfide_EPCI_dropout_seqs = Set{String}()
    NTD_disulfide_EPCI_total_seq_ct = 0
    NTD_disulfide_EPCI_dropout_seq_ct = 0
############################################################
    if none_0___sub2pos_1___pos2sub_2 == 1 || sub_0__posonly_1__mixed_2 == 1
        NTD_disulfide_EPCI_seqs = NTD_disulfide_EPCI_pos_only_seqs
        NTD_disulfide_EPCI_dropout_seqs = NTD_disulfide_EPCI_pos_only_dropout_seqs
        NTD_disulfide_EPCI_total_seq_ct = NTD_disulfide_EPCI_pos_only_seq_ct
        NTD_disulfide_EPCI_dropout_seq_ct = NTD_disulfide_EPCI_pos_only_dropout_seq_ct
    elseif none_0___sub2pos_1___pos2sub_2 == 2 || sub_0__posonly_1__mixed_2 == 0
        NTD_disulfide_EPCI_seqs = NTD_disulfide_EPCI_sub_seqs
        NTD_disulfide_EPCI_dropout_seqs = NTD_disulfide_EPCI_sub_dropout_seqs
        NTD_disulfide_EPCI_total_seq_ct = NTD_disulfide_EPCI_sub_seq_ct
        NTD_disulfide_EPCI_dropout_seq_ct = NTD_disulfide_EPCI_sub_dropout_seq_ct
    end
####################### Now doing the same for qualifying sequences (ones with ≥ min_ct mutations of the mutations searched for)
    NTD_disulfide_MP_seqs_pos_only_seqs = intersect(MP_seqs, NTD_disulfide_EPCI_pos_only_seqs)
    NTD_disulfide_MP_seqs_pos_only_seq_ct = length(NTD_disulfide_MP_seqs_pos_only_seqs)
    NTD_disulfide_MP_seqs_pos_only_dropout_seqs = intersect(MP_seqs, NTD_disulfide_EPCI_pos_only_dropout_seqs)
    NTD_disulfide_MP_seqs_pos_only_dropout_seq_ct = length(NTD_disulfide_MP_seqs_pos_only_dropout_seqs)
    
    NTD_disulfide_MP_seqs_sub_seqs = intersect(MP_seqs, NTD_disulfide_EPCI_sub_seqs)
    NTD_disulfide_MP_seqs_sub_seq_ct = length(NTD_disulfide_MP_seqs_sub_seqs)
    NTD_disulfide_MP_seqs_sub_dropout_seqs = intersect(MP_seqs, NTD_disulfide_EPCI_sub_dropout_seqs)
    NTD_disulfide_MP_seqs_sub_dropout_seq_ct = length(NTD_disulfide_MP_seqs_sub_dropout_seqs)
############################################################
    NTD_disulfide_MP_seqs = Set{String}
    NTD_disulfide_MP_total_seq_ct = 0
    NTD_disulfide_MP_seqs_dropout_total_seq_ct = 0
############################################################
    if none_0___sub2pos_1___pos2sub_2 == 1 || sub_0__posonly_1__mixed_2 == 1
        NTD_disulfide_MP_seqs = NTD_disulfide_MP_seqs_pos_only_seqs
        NTD_disulfide_MP_total_seq_ct = NTD_disulfide_MP_seqs_pos_only_seq_ct
        NTD_disulfide_MP_seqs_dropout_total_seq_ct = NTD_disulfide_MP_seqs_pos_only_dropout_seq_ct
    elseif none_0___sub2pos_1___pos2sub_2 == 2 || sub_0__posonly_1__mixed_2 == 0
        NTD_disulfide_MP_seqs = NTD_disulfide_MP_seqs_sub_seqs
        NTD_disulfide_MP_total_seq_ct = NTD_disulfide_MP_seqs_sub_seq_ct
        NTD_disulfide_MP_seqs_dropout_total_seq_ct = NTD_disulfide_MP_seqs_sub_dropout_seq_ct
    end
#    println("EPCI_qc_string = $(EPCI_qc_str) | HQCS_qc_string = $(HQCS_qc_str)")
#    println("NTD_disulfide_EPCI_total_seq_ct = $(NTD_disulfide_EPCI_total_seq_ct)")
#    println("NTD_disulfide_EPCI_dropout_seq_ct = $(NTD_disulfide_EPCI_dropout_seq_ct)")
#    println("NTD_disulfide_MP_total_seq_ct = $(NTD_disulfide_MP_total_seq_ct)")
#    println("NTD_disulfide_MP_seqs_dropout_total_seq_ct = $(NTD_disulfide_MP_seqs_dropout_total_seq_ct)")
#    println(" = $()")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
    MP_seq_ct = length(MP_seqs)
#    println("MP_seq_ct = $(MP_seq_ct)")
    avg_mutations_per_seq_no_dels = round(digits=2, total_AA_mut_ct_nd/MP_seq_ct)
    ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics = avg_mutations_per_seq_no_dels/avg_AA_sub_ct_per_chronic_seq
    ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_rd = round(digits = 2, ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics)
    MP_seqs_sort = sort(collect(MP_seqs), by = x -> (length(x), x) )
############################################################################################################################################################################
#                                                         Float64   Float64      Float64       Int        Float64    Float64    Float64     Float64       Int               Int
#### for prop_dict, keys = mutations; values = tuple of (EPCI_prop, MP_prop, non_MP_prop, co_mut_count, chi_squared, pv_fish, log_pv_fish, fold_incr, EPCI_mut_ct, adjusted_MP_seq_ct)
    prop_dict = Dict{String, Tuple{Float64,Float64,Float64,Int,Float64,Float64,Float64,Union{Float64, String},Int,Int} }()
    prop_dict_nonlist = Dict{String, Tuple{Float64,Float64,Float64,Int,Float64,Float64,Float64,Union{Float64, String},Int,Int} }()
#### prop_dict_unknown: keys = mutations, values = percentage of qualifying sequences that have an unknown AA at that position
    prop_dict_unknown = Dict{String, Float64}()
    ###### EPCI_prop = proportion of all EPCI seqs with a given AA mut that appear in seqs meeting the criteria (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40)
    ###### MP_prop = proportion of all MP seqs that have the mutation being considered (e.g. if 100 seqs meet the criteria & 4 of those have E:T30I, MP_prop = 0.04)
    ###### EPCI_pct = same as qry prop but in % form
    ###### MP_pct = same as qry prop but in % form
    ###### mut_prop_of_EPCI = proportion of all EPCI seqs that possess given AA mut
    ###### MP_prop_of_EPCI = proportion of all EPCI seqs that meet criteria
    ###### fold_inc = fold-change enrichment of selected
    for (mut, ct) in coAAmut_ct_unknown
        MP_pct = 100*ct/MP_seq_ct
        prop_dict_unknown[mut] = MP_pct
    end
###########################################################################################################################################################################
###########################################################################################################################################################################
#    for (mut, co_mut_count) in coAAmut_ct
#        mut_po = aa_gene_and_pos_comprehensive_dict[mut]
#        unknown = get(coAAmut_ct_unknown, mut_po, 0)             ### = total number of qualifying sequences w/an unknown AA at the mut_po position
#        total_unknown = get(total_AApos_ct_unknown, mut_po, 0)   ### = total number of sequences in entire chronics list w/an unknown AA at the mut_po position
#        inherited = get(MP_seqs_inherited, mut, 0)
#        all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
#        adjusted_MP_seq_ct = MP_seq_ct - unknown - inherited
#        adjusted_EPCI_seq_ct = total_chronic_ct - total_unknown - all_chr_inherited
#######################################
#        EPCI_mut_ct = AA_muts_ct_combined[mut]                   ### AA_muts_ct_combined[mut] = total AA mut count in all independent chronic seqs/lineages
#        EPCI_prop = co_mut_count/EPCI_mut_ct                     ### EPCI_prop = proportion of the total EPCI count of a mut that appears in the mp sequence list
#        MP_prop = co_mut_count/adjusted_MP_seq_ct                ### MP_prop = proportion of all qualifying mp sequences that contain the given mutation
#        non_MP_prop = (EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
#        EPCI_pct = 100*EPCI_prop 
#        MP_pct = 100*MP_prop                                     ### MP_seq_ct = All qualifying seqs (which have ≥ min_mut_ct muts from current mut pattern list)
#        non_MP_pct = 100*non_MP_prop
#        mut_prop_of_EPCI = EPCI_mut_ct/adjusted_EPCI_seq_ct      ### adjusted_EPCI_seq_ct = total # of independent chronic seqs in entire list
#        MP_prop_of_EPCI = adjusted_MP_seq_ct/adjusted_EPCI_seq_ct       
#        fold_incr = 0.0
#        if non_MP_prop ≠ 0
#            fold_incr = MP_prop/non_MP_prop
#        elseif non_MP_prop == 0
#            fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
#            fake_fold_incr = MP_prop/fake_non_MP_prop
#            fake_fold_incr_rd = round(digits=2, fake_fold_incr)
#            fold_incr = ">$(fake_fold_incr_rd)"
#        end
##        fold_incr_rd = round(digits=2, fold_incr)
#        observed = co_mut_count
#        expected = mut_prop_of_EPCI*adjusted_MP_seq_ct
#        if observed ≥ expected
#            fishexact_up_left = co_mut_count                                                                   ### seq with mut + meet search criteria (i.e. ≥ min_mut_ct muts from current mut pattern list)
#            fishexact_up_rt = max(adjusted_MP_seq_ct - co_mut_count, 0)                                        ### seq without mut + meet search criteria (i.e. ≥ min_mut_ct muts from current mut pattern list)
#            fishexact_down_left = max(EPCI_mut_ct - co_mut_count, 0)                                           ### seq with mut + does not meet search criteria (i.e. < min_mut_ct muts from current mut pattern list)
#            fishexact_down_rt = max(adjusted_EPCI_seq_ct - EPCI_mut_ct - adjusted_MP_seq_ct + co_mut_count, 0) ### seq w/o mut + does not meet search criteria (i.e. < min_mut_ct muts from current mut pattern list)
#            fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
#            pv_fish = pvalue(fish)
#            log_pv_fish = -log10(pv_fish)
#### Chi2 formula —— Straight from Claude.ai which says I was doing it all wrong before. Seems to make ~zero difference though ########
#            a = fishexact_up_left;  b = fishexact_up_rt;  c = fishexact_down_left;  d = fishexact_down_rt
#            n = a + b + c + d  # total sample size
#            chi_2_top = n*(a*d - b*c)^2
#            chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
#            chi_squared = 0.0
#            chi_squared_rd = 0.0
#            if chi_2_bottom > 0
#                chi_squared = chi_2_top/chi_2_bottom
#                expected_a = (a + b) * (a + c) / n
#                expected_b = (a + b) * (b + d) / n
#                expected_c = (c + d) * (a + c) / n
#                expected_d = (c + d) * (b + d) / n
#                if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
#                    corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
#                    chi_squared_yates = corrected_numerator/chi_2_bottom
#                    chi_squared = chi_squared_yates
#                    chi_squared_rd = round(digits=1, chi_squared)
#                end
##                        chi_squared_pvalue = 1 - cdf(Chisq(1), chi_squared)
#            end
#            props = (EPCI_prop, MP_prop, non_MP_prop, co_mut_count, chi_squared, pv_fish, log_pv_fish, fold_incr, EPCI_mut_ct, adjusted_MP_seq_ct)
#            prop_dict[mut] = props
#        end
#    end
############################################################################################################################################################################
#### Same as above section two above but excluding mutations in sel_muts
    for (mut, co_mut_count) in coAAmut_ct_nonlist
        if all_muts_0___disulfide_only_1___disulfide_chk_2 == 2
            break
        end
        unknown = get(coAAmut_ct_unknown, aa_gene_and_pos_comprehensive_dict[mut], 0)
        total_unknown = get(total_AApos_ct_unknown, aa_gene_and_pos_comprehensive_dict[mut], 0)
        inherited = get(MP_seqs_inherited, mut, 0)
        all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
        adjusted_MP_seq_ct = MP_seq_ct - unknown - inherited
        adjusted_EPCI_seq_ct = total_chronic_ct - total_unknown - all_chr_inherited
        EPCI_mut_ct = AA_muts_ct_combined[mut]
        EPCI_prop = co_mut_count/EPCI_mut_ct
        MP_prop = co_mut_count/adjusted_MP_seq_ct
        non_MP_prop = (EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)                               ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation 
        EPCI_pct = 100*EPCI_prop
        MP_pct = 100*MP_prop
        non_MP_pct = 100*non_MP_prop
        mut_prop_of_EPCI = EPCI_mut_ct/adjusted_EPCI_seq_ct 
        MP_prop_of_EPCI = adjusted_MP_seq_ct/adjusted_EPCI_seq_ct
        fold_incr::Union{Float64, String} = 0.0
        if non_MP_prop ≠ 0
            fold_incr = MP_prop/non_MP_prop
        elseif non_MP_prop == 0
            fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
            fake_fold_incr = MP_prop/fake_non_MP_prop
            fake_fold_incr_rd = round(digits=2, fake_fold_incr)
            fold_incr = ">$(fake_fold_incr_rd)"
        end
#        fold_incr_rd = round(digits=2, fold_incr)
        observed = co_mut_count
        expected = mut_prop_of_EPCI*adjusted_MP_seq_ct
        if observed ≥ expected
            diff = observed-expected
            chi_squared = diff^2/expected
            chi_squared_rd = round(digits=1, chi_squared)
            fishexact_up_left = Int(round(co_mut_count))                                                                      ### seq with mut + meet search criteria (usually 2+ muts in list)
            fishexact_up_rt = max(Int(round(adjusted_MP_seq_ct - co_mut_count)), 0)                                           ### seq without mut + meet search criteria (usually 2+ muts in list)
            fishexact_down_left = max(Int(round(EPCI_mut_ct - co_mut_count)), 0)                                              ### seq with mut + does not meet search criteria (usually 2+ muts in list)
            fishexact_down_rt = max(Int(round(adjusted_EPCI_seq_ct - EPCI_mut_ct - adjusted_MP_seq_ct + co_mut_count)), 0)    ### seq w/o mut + does not meet search criteria
            fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
            pv_fish = pvalue(fish)
            log_pv_fish = -log10(pv_fish)
            if log_pv_fish > max_log_pv_fish
                max_log_pv_fish = log_pv_fish
            end
            co_mut_count = Int(round(co_mut_count))
            props = (EPCI_prop, MP_prop, non_MP_prop, co_mut_count, chi_squared, pv_fish, log_pv_fish, fold_incr, EPCI_mut_ct, adjusted_MP_seq_ct)
            prop_dict_nonlist[mut] = props
            push!(df_mut_corr_unfiltered, (mut_str, mut, co_mut_count, adjusted_MP_seq_ct, EPCI_mut_ct, log_pv_fish, fold_incr, EPCI_pct, MP_pct, non_MP_pct, pv_fish))
        end
    end
#    for (mut, ct) in coAAmut_ct_unknown
#        mutpad = rpad(mut, 10)
#        ctpad = lpad(ct, 3)
#        println("$(mutpad) = $(ctpad)")
#    end
###########################################################################################################################################################################
#################################################### NTD-disulfide correlation calculations ###############################################################################
###########################################################################################################################################################################
    EPCI_NTD_disulfide_denominator = total_EPCI_seq_ct - NTD_disulfide_EPCI_dropout_seq_ct
    EPCI_NTD_disulfide_numerator = NTD_disulfide_EPCI_total_seq_ct
    EPCI_NTD_disulfide_prop = EPCI_NTD_disulfide_numerator/EPCI_NTD_disulfide_denominator
    MP_seqs_NTD_disulfide_denominator = length(MP_seqs) - NTD_disulfide_MP_seqs_dropout_total_seq_ct
    MP_seqs_NTD_disulfide_numerator = NTD_disulfide_MP_total_seq_ct
    MP_seqs_NTD_disulfide_prop = MP_seqs_NTD_disulfide_numerator/MP_seqs_NTD_disulfide_denominator
    non_MP_NTD_disulfide_denominator = (total_EPCI_seq_ct - NTD_disulfide_EPCI_dropout_seq_ct) - (length(MP_seqs) - NTD_disulfide_MP_seqs_dropout_total_seq_ct)
    non_MP_NTD_disulfide_numerator = (length(MP_seqs) - NTD_disulfide_MP_seqs_dropout_total_seq_ct) - NTD_disulfide_MP_total_seq_ct
    non_MP_NTD_disulfide_prop = non_MP_NTD_disulfide_numerator/non_MP_NTD_disulfide_denominator
    observed = MP_seqs_NTD_disulfide_numerator
    expected = EPCI_NTD_disulfide_prop*MP_seqs_NTD_disulfide_denominator
#    println("NTD_disulfide_expected = $(expected)")
#    println("NTD_disulfide_observed = $(observed)")
#    println("MP_seqs_NTD_disulfide_denominator = $(MP_seqs_NTD_disulfide_denominator)"); print("\n"^1)
    coAAmut_ct["NTD:disulfide"] = MP_seqs_NTD_disulfide_numerator
    coAAmut_ct_nonlist["NTD:disulfide"] = MP_seqs_NTD_disulfide_numerator
    if observed > expected
        co_mut_count = MP_seqs_NTD_disulfide_numerator
        EPCI_prop = MP_seqs_NTD_disulfide_numerator/NTD_disulfide_EPCI_total_seq_ct
        MP_prop = MP_seqs_NTD_disulfide_prop
        non_MP_prop = non_MP_NTD_disulfide_prop
        EPCI_pct = 100*EPCI_prop
        MP_pct = 100*MP_prop
        non_MP_pct = 100*non_MP_prop
        mut_prop_of_EPCI = EPCI_NTD_disulfide_prop
        MP_prop_of_EPCI = (length(MP_seqs) - NTD_disulfide_MP_seqs_dropout_total_seq_ct)/EPCI_NTD_disulfide_denominator                ###  adjusted_MP_seq_ct/adjusted_EPCI_ct 
        fold_incr = 0.0
        if non_MP_prop ≠ 0
            fold_incr = MP_prop/non_MP_prop
        elseif non_MP_prop == 0
            fake_non_MP_prop = 1/(EPCI_NTD_disulfide_denominator - MP_seqs_NTD_disulfide_denominator)
            fake_fold_incr = MP_prop/fake_non_MP_prop
            fake_fold_incr_rd = round(digits=2, fake_fold_incr)
            fold_incr = ">$(fake_fold_incr_rd)"
        end
#        fold_incr_rd = round(digits=2, fold_incr)
#######################
        EPCI_NTD_disulfide_or_dropout = union(NTD_disulfide_EPCI_seqs, NTD_disulfide_EPCI_dropout_seqs)
        EPCI_non_NTD_disulfide_or_dropout = setdiff(all_unique_chr_seqs_set, EPCI_NTD_disulfide_or_dropout)
        fishexact_up_left = NTD_disulfide_MP_total_seq_ct                                         ### seq with NTD_disulfide mut + in MP_seqs (i.e. ≥ min_ct muts from mut search list)
        fishexact_up_rt = MP_seqs_NTD_disulfide_denominator - MP_seqs_NTD_disulfide_numerator ### seq without NTD_disulfide mut + in MP_seqs 
        fishexact_down_left = length(setdiff(NTD_disulfide_EPCI_seqs, NTD_disulfide_MP_seqs))          ### seq with NTD_disulfide mut + not in MP_seqs (i.e. < min_ct muts from mut search list)
        fishexact_down_rt = length(setdiff(EPCI_non_NTD_disulfide_or_dropout, MP_seqs))                ### seq without NTD_disulfide mut + not in MP_seqs
        fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
        pv_fish = pvalue(fish)
        log_pv_fish = -log10(pv_fish)
#        println("NTD_disulfide fold_incr = $(fold_incr_rd)")
#        println("NTD_disulfide log_pv_fish = $(log_pv_fish)")
#        println("NTD_disulfide pv_fish = $(pv_fish)")
        a = fishexact_up_left;  b = fishexact_up_rt;  c = fishexact_down_left;  d = fishexact_down_rt
        n = a + b + c + d  # total sample size
        chi_2_top = n*(a*d - b*c)^2
        chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
        chi_squared = 0.0
        chi_squared_rd = 0.0
        if chi_2_bottom > 0
            chi_squared = chi_2_top/chi_2_bottom
            expected_a = (a + b) * (a + c) / n
            expected_b = (a + b) * (b + d) / n
            expected_c = (c + d) * (a + c) / n
            expected_d = (c + d) * (b + d) / n
            if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                chi_squared_yates = corrected_numerator/chi_2_bottom
                chi_squared = chi_squared_yates
                chi_squared_rd = round(digits=1, chi_squared)
            end
        end
        props = (EPCI_prop, MP_prop, non_MP_prop, co_mut_count, chi_squared, pv_fish, log_pv_fish, fold_incr, EPCI_NTD_disulfide_numerator, MP_seqs_NTD_disulfide_denominator)
        prop_dict["NTD:disulfide"] = props
        prop_dict_nonlist["NTD:disulfide"] = props
    end
###########################################################################################################################################################################
################################################# END: NTD-disulfide correlation calculations #############################################################################
###########################################################################################################################################################################
############################################################################################################################################################################
########################### All the sections directly below just sort the dictionaries according to their various components ###############################################
############################################################################################################################################################################
    coAAmut_ct_gene_sort = sort(collect(coAAmut_ct), by = x -> AA_gene_sortKey_2(x[1]) )
    coAAmut_ct_count_sort = sort(collect(coAAmut_ct), by = x -> AA_ct_sortKey2(x), rev=true)
    coAAmut_ct_unknown_gene_sort = sort(collect(coAAmut_ct_unknown), by = x -> AA_gene_pos_sortKey(x) )
    coAAmut_ct_unknown_count_sort = sort(collect(coAAmut_ct_unknown), by = x -> AA_ct_pos_sortKey2(x), rev=true)  
######################################################################################################################################
    coAAmut_ct_gene_sort_nonlist = sort(collect(coAAmut_ct_nonlist), by = x -> AA_gene_sortKey_2(x[1]) )
    coAAmut_ct_count_sort_nonlist = sort(collect(coAAmut_ct_nonlist), by = x -> AA_ct_sortKey2(x), rev=true)
######################################################################################################################################
    prop_dict_sort_Count = sort(collect(prop_dict), by = x -> (coAAmut_ct[x[1]], x[2][1]), rev=true)
    prop_dict_sort_ChiSq = sort(collect(prop_dict), by = x -> (x[2][4], x[2][3]), rev=true)
    prop_dict_sort_EPCIpct = sort(collect(prop_dict), by = x -> (x[2][1], x[2][3]), rev=true)
    prop_dict_sort_gene = sort(collect(prop_dict), by = x -> AA_gene_sortKey_2(x[1]) )
    prop_dict_sort_log_pv_fish = sort(collect(prop_dict), by = x -> (x[2][6], x[2][3]), rev=true)
#################################
    prop_dict_unknown_sort_Count = sort(collect(prop_dict_unknown), by = x -> coAAmut_ct_unknown[x[1]], rev=true)
    prop_dict_unknown_sort_gene = sort(collect(prop_dict_unknown), by = x -> AA_gene_pos_sortKey(x) )
#############################################################################
#################################
#    prop_dict_po2_unknown_sort_Count = sort(collect(prop_dict_po2_unknown), by = x -> coAAmut_ct_po2_unknown[x[1]], rev=true)
#    prop_dict_po2_unknown_sort_gene = sort(collect(prop_dict_po2_unknown), by = x -> AA_gene_pos_sortKey(x) )
#####################################################################################################################################
    prop_dict_sort_Count_nonlist = sort(collect(prop_dict_nonlist), by = x -> (coAAmut_ct_nonlist[x[1]], x[2][1]), rev=true)
    prop_dict_sort_ChiSq_nonlist = sort(collect(prop_dict_nonlist), by = x -> (x[2][4], x[2][3]), rev=true)
    prop_dict_sort_EPCIpct_nonlist = sort(collect(prop_dict_nonlist), by = x -> (x[2][1], x[2][3]), rev=true)
    prop_dict_sort_gene_nonlist = sort(collect(prop_dict_nonlist), by = x -> AA_gene_sortKey_2(x[1]) )
    prop_dict_sort_log_pv_fish_nonlist = sort(collect(prop_dict_nonlist), by = x -> (x[2][6], x[2][3]), rev=true)
#####################################################################################################################################
#  total_group_ct = length(rep_seq_grps_maxmut_seqs) | total_singlet_ct = length(non_rep_seqs) | total_chronic_ct = total_group_ct + total_singlet_ct 
    mut_perc = round(digits=1, length(MP_seqs)/total_chronic_ct)
    all_seqs_incl_repeats = Set{String}()
############################################################################################################################################################################
##### This is just to count the total number of sequences with a mutation, including repeat sequences from the same patient
    for (seq, mutset) in seq_AA_muts_combined
        seq_grp_ct = 0
        for (group, muts) in mut_groups
            group_ct = 0
            for mut in muts
                if mut in mutset
                    group_ct += 1
                end
            end
            if group_ct ≥ 1
                seq_grp_ct += 1
            end
        end
        if seq_grp_ct ≥ min_mut_ct
            push!(all_seqs_incl_repeats, seq)
        end
    end
############################################################################################################################################################################
    srt(n) = (length(n), n)
    all_seqs_incl_repeats_sort = sort(collect(all_seqs_incl_repeats), by = x -> srt(x) )
##############################################################################################################
#### Counts the clades/Pango lineages of each qualifying sequence
#### ADJ dictionaries sort by the proportion of a given clade/Pango lineage that qualify
    pango_dict_for_muts = Dict{String, Int}()
    clade_dict_for_muts = Dict{String, Int}()
    clade_dict_for_muts_ADJ = Dict{String, Float64}()
    clade_ADJ_factor_dict = Dict{String, Float64}()
    for clade in clade_set_complete
        seq_clade_ct[clade] = get(seq_clade_ct, clade, 0)
    end
    for clade in clade_set_complete
        if seq_clade_ct[clade] == 0
            clade_ADJ_factor_dict[clade] = 0
        end
        if seq_clade_ct[clade] ≠ 0
            clade_ADJ_factor_dict[clade] = 1000/seq_clade_ct[clade]
        end
        clade_dict_for_muts[clade] = 0
        clade_dict_for_muts_ADJ[clade] = 0
    end
#############################################################################################################
    for seq in MP_seqs
        pango_dict_for_muts[seq_pango[seq]] = get(pango_dict_for_muts, seq_pango[seq], 0) + 1
        clade_dict_for_muts[seq_clade[seq]] = get(clade_dict_for_muts, seq_clade[seq], 0) + 1
    end
#############################################################################################################
    for clade in clade_set_complete
        if haskey(clade_dict_for_muts_ADJ, clade)
            clade_dict_for_muts_ADJ[clade] = round(digits=1, clade_dict_for_muts[clade]*clade_ADJ_factor_dict[clade])
        else
            clade_dict_for_muts_ADJ[clade] = 0
        end
    end
#############################################################################################################
    pango_dict_for_muts_sort_by_ct = sort(collect(pango_dict_for_muts), by = x -> x[2], rev=true)
    pango_dict_for_muts_sort_by_pango = sort(collect(pango_dict_for_muts), by = x -> x[1])
    clade_dict_for_muts_sort_by_ct = sort(collect(clade_dict_for_muts), by = x -> x[2], rev=true)
    clade_dict_for_muts_sort_by_clade = sort(collect(clade_dict_for_muts), by = x -> x[1])
    clade_dict_for_muts_ADJ_sort_by_ct = sort(collect(clade_dict_for_muts_ADJ), by = x -> x[2], rev=true)
    clade_dict_for_muts_ADJ_sort_by_clade = sort(collect(clade_dict_for_muts_ADJ), by = x -> x[1])
#####################################################################################################################################
    preprint_time_rd = round(digits=1, time() - start_time)
#    println("Total preprint_time = $(preprint_time_rd)")
############################################################################################################################################################################
########################################### Everything Below Just Prints the Results in various Ways ########################################################################
############################################################################################################################################################################
    date = Dates.format(today(), "yyyy_mm_dd")
    mutPerc = round(digits=1, 100*MP_seq_ct/total_chronic_ct)
    padd = 15;    seq_title_pad = 24;    clade_title_pad = 5
    non_list_file = "$(folder)/$(mut_str_name)_NONLIST_minMut$(min_mut_ct)_minChiSq$(min_chi_sq)_minLogPVfish$(min_log_pv_fish)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date_now).txt"
    open(non_list_file, "a") do g
        if seq_factor_OFF_HALF_ON__0_1_2 == 2; println(g, "############  seq_factor Mode = ON ################"); println(g); end
        if seq_factor_OFF_HALF_ON__0_1_2 == 0; println(g, "############  seq_factor Mode = OFF ################"); println(g); end
        if seq_factor_OFF_HALF_ON__0_1_2 == 1; println(g, "############  seq_factor Mode = HALF ################"); println(g); end
        println(g, "************************* Clade/Pango Count for $(min_mut_ct)-of $(mut_str), sorted by Clade *************************")
            pct_seq_title = lpad("| Seqs % |", seq_title_pad)
            pct_clade_title = lpad(" Clade % |", clade_title_pad)
        println(g, "                                                |        |        |         |")
        println(g, "                                Clade/Pango     | Seq Ct |  % All | % Clade |")
        println(g, "                                                |        |        |         |")    
        for w in clade_dict_for_muts_sort_by_clade
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            clade_pct_of_mutSeqs = round(digits=1, 100*clade_ct/MP_seq_ct)
            mutSeqs_pct_of_clade = round(digits=1, 100*clade_ct/seq_clade_ct[clade])
            if clade_ct > 0
                cladePango = rpad("$(clade)|$(pango) ", 15)
                pctOfMutseqs = lpad("$(clade_pct_of_mutSeqs)%", 6)
                pctOfClade = lpad("$(mutSeqs_pct_of_clade)%", 6)
                fracTop = lpad(clade_ct, 3)
                fracBot = rpad(MP_seq_ct, 3)
                fract = "$(fracTop)/$(fracBot)"
                println(g, "                                $(cladePango) |$(fract) | $(pctOfMutseqs) | $(pctOfClade)  |")
            end
        end
        println(g)
        println(g, "************************* Clade/Pango Count for $(min_mut_ct)-of $(mut_str), sorted by Count *************************")
            pct_seq_title = lpad("| Seqs % |", seq_title_pad)
            pct_clade_title = lpad(" Clade % |", clade_title_pad)  
        println(g, "                                                |         |        |         |")
        println(g, "                                Clade/Pango     | Seq Ct  |  % All | % Clade |")
        println(g, "                                                |         |        |         |")    
        for w in clade_dict_for_muts_sort_by_ct
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            clade_pct_of_mutSeqs = round(digits=1, 100*clade_ct/MP_seq_ct)
            mutSeqs_pct_of_clade = round(digits=1, 100*clade_ct/seq_clade_ct[clade])
            if clade_ct > 0
                cladePango = rpad("$(clade)|$(pango) ", 15)
                pctOfMutseqs = lpad("$(clade_pct_of_mutSeqs)%", 6)
                pctOfClade = lpad("$(mutSeqs_pct_of_clade)%", 6)
                fracTop = lpad(clade_ct, 3)
                fracBot = lpad(MP_seq_ct, 3)
                fract = "$(fracTop)/$(fracBot)"
                println(g, "                                $(cladePango) |$(fract)  | $(pctOfMutseqs) | $(pctOfClade)  |")
            end
        end
        println(g)
        println(g, "************************* Normalized Clade/Pango Count for $(min_mut_ct)-of $(mut_str), sorted by Count *************************")
        pct_seq_title = lpad("| Seqs % |", seq_title_pad)
        pct_clade_title = lpad(" Clade % |", clade_title_pad)
        println(g, "                                                |           |        |         |")
        println(g, "                                Clade/Pango     | Score [Ct]|  % All | % Clade |")
        println(g, "                                                |           |        |         |")
        for w in clade_dict_for_muts_ADJ_sort_by_ct
            clade = w[1]
            pango = clade_to_pango[w[1]]
            cladePango = rpad("$(clade)|$(pango) ", 15)
            clade_ct = w[2]
            clade_ct_pad = lpad(clade_ct, 5)
            clade_ct_muts_pad = rpad("["*string(clade_dict_for_muts[clade])*"]", 4)
            if clade_ct > 0
                println(g, rpad("                                $(cladePango)", 13) * " | $(clade_ct_pad) $(clade_ct_muts_pad)|")
            end
        end
        println(g, "#######################################################################################################################")
        println(g, "* Mutation cts for seq with $(min_mut_ct)-of $(mut_str) NONLIST Sorted by Log10 Fisher's Exact p-value *")
        println(g, "#######################################################################################################################")
        println(g)
        println(g, rpad("                    Total number of Chronic Singlets = ", 54) * "$(total_singlet_ct)")
        println(g, rpad("                    Total number of Unrelated Chronic Groups = ", 54) * "$(total_group_ct)")
        println(g, rpad("                    Total number of Independent Chronic Singlets/Groups = ", 53) * "$(total_chronic_ct)")
        println(g, "                    Total Number of $(min_mut_ct)-of $(mut_str) seqs =         $(MP_seq_ct)")
        println(g, "                    Total Percentage of all Chronics with $(min_mut_ct)-of $(mut_str) =         $(mutPerc)%")
        println(g)
        println(g, "                             |                  |                   |                 |         |         |")
        println(g, "             Mutation        |   % Match Mut    |   % Search Mut    |  % All Chronics |logPVfish| pv_fish |")
        println(g, "                             |                  |                   |                 |         |         |")
        for m in prop_dict_sort_log_pv_fish_nonlist
            mut = m[1]
            if coAAmut_ct_nonlist[mut] ≥ print_min
                EPCI_prop          = m[2][1]
                MP_prop            = m[2][2]
                non_MP_prop        = m[2][3]
                count              = m[2][4]
                chi2               = m[2][5]
                pv_fish            = m[2][6]
                log_pv_fish        = m[2][7]
                fold_incr          = m[2][8]
                EPCI_mut_ct        = m[2][9]
                adjusted_MP_seq_ct = m[2][10]
###########################################################
                mut_pad = rpad(mut, padd)
                mut_perc_of_all = round(digits=1, 100*AA_muts_ct_combined[mut]/total_chronic_ct)
                fracTop = Int(round(coAAmut_ct_nonlist[mut]))
                fracTop_pad = lpad(fracTop, 4) 
                fracBot = lpad(AA_muts_ct_combined[mut], 3)
                fracBot2 = lpad(MP_seq_ct, 3)
                EPCI_pct = 100*EPCI_prop
                EPCI_pct_rd = round(digits=1, EPCI_pct)
                EPCI_pct_pad = lpad("$(EPCI_pct_rd)%", 6)
                MP_pct = 100*MP_prop
                MP_pct_rd = round(digits=1, MP_pct)
                MP_pct_pad = lpad("$(MP_pct_rd)%", 6)
                non_MP_pct = 100*non_MP_prop
                non_MP_pct_rd = round(digits=1, non_MP_pct)
                non_MP_pct_pad = lpad("$(non_MP_pct_rd)%", 6)
                mut_perc_of_all_pad = lpad(mut_perc_of_all, 5)
                chi = chi2;   fish = pv_fish;   log10_pvalue_fish = log_pv_fish
                fish_fin = lpad(number_to_string_to_two_decimal_places(pv_fish, 2), 7)
                log_pv_fish_fin = lpad(add_zeros_to_rounded(log_pv_fish, 2), 7)
#                chi2_fin = lpad(add_zeros_to_rounded(chi2, 2), 7)
#                chi_sq_rd = round(digits=1, chi2)
#                chi_sq_2dec = add_zeros_to_rounded(chi_sq_rd, 2)
#                chi_sq = lpad(chi_sq_2dec, 7)
                fish_rd = round(digits=2, pv_fish)
                fish_2dec = add_zeros_to_rounded(fish_rd, 2)
                fish = lpad(fish_2dec, 7)
                PV_fish_rd = round(digits=6, pv_fish)
                PV_fish_pad = rpad(PV_fish_rd, 9)
                if log_pv_fish ≥ min_log_pv_fish
#                    println("             $(mut_pad) | $(fracTop_pad)/$(fracBot) =$(EPCI_pct_pad)%| $(fracTop_pad)/$(fracBot2) =$(MP_pct_pad)  | PctAll = $(mut_perc_of_all_pad)  | $(log_pv_fish_fin) |$(PV_fish_pad)|")
                    println(g, "             $(mut_pad) | $(fracTop_pad)/$(fracBot) =$(EPCI_pct_pad)%| $(fracTop_pad)/$(fracBot2) =$(MP_pct_pad)  | PctAll = $(mut_perc_of_all_pad)  | $(log_pv_fish_fin) |$(PV_fish_pad)|")
                    push!(df_mut_corr, (mut_str, mut, count, adjusted_MP_seq_ct, EPCI_mut_ct, log_pv_fish, fold_incr, EPCI_pct, MP_pct, non_MP_pct, pv_fish))
                end
            end 
        end
        println(g, "##########################################################################################")
        println(g, "************* Mutation cts for seq with $(min_mut_ct)-of $(mut_str) NONLIST Sorted by Count *************")
        println(g, "##########################################################################################")
        println(g)
        println(g, rpad("                    Total number of Chronic Singlets = ", 54) * "$(total_singlet_ct)")
        println(g, rpad("                    Total number of Unrelated Chronic Groups = ", 54) * "$(total_group_ct)")
        println(g, rpad("                    Total number of Independent Chronic Singlets/Groups = ", 53) * "$(total_chronic_ct)")
        println(g, "                    Total Number of $(min_mut_ct)-of $(mut_str) seqs =         $(MP_seq_ct)")
        println(g, "                    Total Percentage of all Chronics with $(min_mut_ct)-of $(mut_str) =         $(mutPerc)%")
        println(g)
        println(g, "                             |                  |                   |                 |         |         |")
        println(g, "             Mutation        |   % Match Mut    |   % Search Mut    |  % All Chronics |logPVfish| pv_fish |")
        println(g, "                             |                  |                   |                 |         |         |")
        for m in prop_dict_sort_Count_nonlist
            mut = m[1]
            if coAAmut_ct_nonlist[mut] ≥ print_min
                EPCI_prop          = m[2][1]
                MP_prop            = m[2][2]
                non_MP_prop        = m[2][3]
                count              = m[2][4]
                chi2               = m[2][5]
                pv_fish            = m[2][6]
                log_pv_fish        = m[2][7]
                fold_incr          = m[2][8]
                EPCI_mut_ct        = m[2][9]
                adjusted_MP_seq_ct = m[2][10]
###########################################################
                mut_perc_of_all = round(digits=1, 100*AA_muts_ct_combined[mut]/length(all_unique_chr_seqs_set))
                mut_pad = rpad(mut, padd)
                fracTop = Int(round(coAAmut_ct_nonlist[mut]))
                fracTop_pad = lpad(fracTop, 4) 
                fracBot = lpad(AA_muts_ct_combined[mut], 3)
                fracBot2 = lpad(MP_seq_ct, 3)
                EPCI_pct = 100*EPCI_prop
                EPCI_pct_rd = round(digits=1, EPCI_pct)
                EPCI_pct_pad = lpad("$(EPCI_pct_rd)%", 6)
                MP_pct = 100*MP_prop
                MP_pct_rd = round(digits=1, MP_pct)
                MP_pct_pad = lpad("$(MP_pct_rd)%", 6)
                non_MP_pct = 100*non_MP_prop
                non_MP_pct_rd = round(digits=1, non_MP_pct)
                non_MP_pct_pad = lpad("$(non_MP_pct_rd)%", 6)
                mut_perc_of_all_pad = lpad(mut_perc_of_all, 5)
                pv_fish_fin = lpad(number_to_string_to_two_decimal_places(pv_fish, 2), 7)
                log_pv_fish_fin = lpad(add_zeros_to_rounded(log_pv_fish, 2), 7)
#                chi2_fin = lpad(add_zeros_to_rounded(chi2, 2), 7)
#                chi_sq_str = string(chi2) 
#                chi_sq_rd = round(digits=1, chi2)
#                chi_sq_2dec = add_zeros_to_rounded(chi_sq_rd, 2)
#                chi_sq = lpad(chi_sq_2dec, 7)
                fish_rd = round(digits=2, pv_fish)
                fish_2dec = add_zeros_to_rounded(fish_rd, 2)
                fish = lpad(fish_2dec, 7)
                PV_fish_rd = round(digits=6, pv_fish)
                PV_fish_pad = rpad(PV_fish_rd, 9)
                selmut_join = join(sel_muts)
            if log_pv_fish ≥ 5
                print("$(selmut_join)|$(mut), ")
            end
                if log_pv_fish ≥ min_log_pv_fish
                    println(g, "             $(mut_pad) | $(fracTop_pad)/$(fracBot) =$(EPCI_pct_pad)%| $(fracTop_pad)/$(fracBot2) =$(MP_pct_pad)  | PctAll = $(mut_perc_of_all_pad)  | $(log_pv_fish_fin) |$(PV_fish_pad)|")
#                    push!(df_mut_corr, (mut_str, mut, count, adjusted_MP_seq_ct, EPCI_mut_ct, log_pv_fish, fold_incr, EPCI_pct, MP_pct, non_MP_pct, pv_fish))
                end
            end
        end
######################################################################################################################################
        println(g, "##########################################################################################")
        println(g, "************ Mutation cts for seq with $(min_mut_ct)-of $(mut_str) NONLIST Sorted by Gene & Position *************")
        println(g, "##########################################################################################")
        println(g)
        println(g, rpad("                    Total number of Chronic Singlets = ", 54) * "$(total_singlet_ct)")
        println(g, rpad("                    Total number of Unrelated Chronic Groups = ", 54) * "$(total_group_ct)")
        println(g, rpad("                    Total number of Independent Chronic Singlets/Groups = ", 53) * "$(total_chronic_ct)")
        println(g, "                    Total Number of $(min_mut_ct)-of $(mut_str) seqs =         $(MP_seq_ct)")
        println(g, "                    Total Percentage of all Chronics with $(min_mut_ct)-of $(mut_str) =         $(mutPerc)%")
        println(g)
        println(g, "                             |                  |                   |                 |         |         |")
        println(g, "             Mutation        |   % Match Mut    |   % Search Mut    |  % All Chronics |logPVfish| pv_fish |")
        println(g, "                             |                  |                   |                 |         |         |")
        for m in prop_dict_sort_gene_nonlist
            mut = m[1]
            if coAAmut_ct_nonlist[mut] ≥ print_min
                EPCI_prop          = m[2][1]
                MP_prop            = m[2][2]
                non_MP_prop        = m[2][3]
                count              = m[2][4]
                chi2               = m[2][5]
                pv_fish            = m[2][6]
                log_pv_fish        = m[2][7]
                fold_incr          = m[2][8]
                EPCI_mut_ct        = m[2][9]
                adjusted_MP_seq_ct = m[2][10]
###########################################################
                mut_perc_of_all = round(digits=1, 100*AA_muts_ct_combined[mut]/total_EPCI_seq_ct)
                mut_pad = rpad(mut, padd)
                fracTop = Int(round(coAAmut_ct_nonlist[mut]))
                fracTop_pad = lpad(fracTop, 4) 
                fracBot = lpad(AA_muts_ct_combined[mut], 3)
                fracBot2 = lpad(MP_seq_ct, 3)
                EPCI_pct = 100*EPCI_prop
                EPCI_pct_rd = round(digits=1, EPCI_pct)
                EPCI_pct_pad = lpad("$(EPCI_pct_rd)%", 6)
                MP_pct = 100*MP_prop
                MP_pct_rd = round(digits=1, MP_pct)
                MP_pct_pad = lpad("$(MP_pct_rd)%", 6)
                non_MP_pct = 100*non_MP_prop
                non_MP_pct_rd = round(digits=1, non_MP_pct)
                non_MP_pct_pad = lpad("$(non_MP_pct_rd)%", 6)
                mut_perc_of_all_pad = lpad(mut_perc_of_all, 5)
#                chi2_fin = lpad(add_zeros_to_rounded(chi2, 2), 7)
                fishex_fin = lpad(number_to_string_to_two_decimal_places(pv_fish, 2), 7)
                log_pv_fish_fin = lpad(add_zeros_to_rounded(log_pv_fish, 2), 7)
#                fish_rd = round(digits=2, pv_fish)
#                fish_2dec = add_zeros_to_rounded(fish_rd, 2)
#                fish = lpad(fish_2dec, 7)
#                PV_fish = round(digits=6, pv_fish)
#                PV_fish_pad = rpad(PV_fish, 9)
                if log_pv_fish ≥ min_log_pv_fish
                    println(g, "             $(mut_pad) | $(fracTop_pad)/$(fracBot) =$(EPCI_pct_pad)%| $(fracTop_pad)/$(fracBot2) =$(MP_pct_pad)  | PctAll = $(mut_perc_of_all_pad)  | $(log_pv_fish_fin) |$(fishex_fin)|")
                    push!(df_mut_corr_gene_sort, (mut_str, mut, count, adjusted_MP_seq_ct, EPCI_mut_ct, log_pv_fish, fold_incr, EPCI_pct, MP_pct, non_MP_pct, pv_fish))
                end
            end
        end
    end
############################################################################################################################################################################
############################################################################################################################################################################
    gene_number(n) = mut_gene_Dict[aa_gene_comprehensive_dict[n]]
    finalKey(n) = (gene_number(n), aa_pos_comprehensive_dict[n])
    mut_seqs_file = "$(folder)/$(mut_str_name)_Seqs_minMut$(min_mut_ct)_minChiSq$(min_chi_sq)_minLogPVfish$(min_log_pv_fish)_$(HQCS_qc_string)__$(date_now).txt"
    open(mut_seqs_file, "a") do g
        println(g)
        println(g, "Number of all sequences without repeats = $(length(MP_seqs))" )
        println(g, "Number of all sequences including repeats = $(length(all_seqs_incl_repeats))" )
        println(g, "Average Number of AA Subs per Seq = $(avg_mutations_per_seq_no_dels)" )
        println(g, "Average Number of AA Subs per Chronic Sequence in Entire List = $(avg_AA_sub_ct_per_chronic_seq)")
        print(g, "\n"^3)
        println(g, "**** Individual Seq Private Mutations for Seqs with $(min_mut_ct) of $(mut_str) ****")
        println(g)
        vec_vec_header_mutsStr = Vector{Vector{String}}()
        for seq_EPI in MP_seqs_sort
            vec_header_mutsStr = Vector{String}()
            seq_muts_sort = sort(collect(seq_AA_muts[seq_EPI]), by = x -> finalKey(x))
            pango = seq_pango[seq_EPI]
            cntry = seq_country[seq_EPI]
            coll_date = seq_collection_date[seq_EPI]
            header = "$(pango), $(cntry), $(coll_date), $(seq_EPI)"
            seq_muts_sort_arr = Vector{String}()
            for seq_mut in seq_muts_sort
                push!(seq_muts_sort_arr, seq_mut)
            end
            seq_muts_sort_str = join(seq_muts_sort_arr, ", ")
            push!(vec_header_mutsStr, header)
            push!(vec_header_mutsStr, seq_muts_sort_str)
            push!(vec_vec_header_mutsStr, vec_header_mutsStr)
        end
        vec_vec_header_mutsStr_sort = sort(vec_vec_header_mutsStr, by = x -> x[1])
        for vec_header in vec_vec_header_mutsStr_sort
            println(g, vec_header[1])
            println(g, vec_header[2])
            println(g)
        end
    end
    if none_0___sub2pos_1___pos2sub_2 == 1
        namestart = "sub2pos"
    elseif none_0___sub2pos_1___pos2sub_2 == 2
        namestart = "pos2sub"
    elseif none_0___sub2pos_1___pos2sub_2 == 0
        if sub_0__posonly_1__mixed_2 == 0
            namestart = "sub"
        elseif sub_0__posonly_1__mixed_2 == 1
            namestart = "pos_only"
        end
    end
    if nonmulti_0___multi_1 == 0
        CSV.write("$(folder)/df_$(namestart)_$(mut_str_name)_df_main_$(HQCS_qc_string)__$(date_hour).csv", df_mut_corr)
        CSV.write("$(folder)/df_$(namestart)_$(mut_str_name)_df_gene_sort_$(HQCS_qc_string)__$(date_hour).csv", df_mut_corr_gene_sort)
        CSV.write("$(folder)/df_$(namestart)_$(mut_str_name)_df_unfiltered_$(HQCS_qc_string)__$(date_hour).csv", df_mut_corr_unfiltered)
    elseif nonmulti_0___multi_1 == 1
        if max_log_pv_fish ≥ min_log_pv_fish
            CSV.write("$(folder)/df_$(namestart)_$(mut_str_name)_df_main_$(HQCS_qc_string)__$(date_hour).csv", df_mut_corr)
            CSV.write("$(folder)/df_$(namestart)_$(mut_str_name)_df_gene_sort_$(HQCS_qc_string)__$(date_hour).csv", df_mut_corr_gene_sort)
            CSV.write("$(folder)/df_$(namestart)_$(mut_str_name)_df_unfiltered_$(HQCS_qc_string)__$(date_hour).csv", df_mut_corr_unfiltered)
        end
    end
##############################################################################################################
    return prop_dict_nonlist
end

2026_03_07__701PM
avg_AA_sub_ct_per_chronic_seq = 18.06
avg_AA_sub_ct_per_chronic_seq_v2 = 18.06


AA_2plus__mut2pos_pos2mut__2026_02_12 (generic function with 2 methods)

In [243]:
### Execute AA_2plus__mut2pos_pos2mut__2026_02_12          6_8_95 
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
min_mut_ct = 1; min_chi_sq = 0; min_log_pv_fish = 0.0; print_min = 1
exc_muts = [""]; exc_muts_po = [""]
sel_muts = ["S:P9L", "S:P9T", "S:S12P", "S:S13I", "S:C15-", "S:C15Y", "S:C15S", "S:C15F", "S:C15G", "S:C15L", "S:C15R", "S:R21I", "S:W64C", "S:S71C", "S:C136-", "S:C136F", "S:C136G", "S:C136R", "S:C136Y", "S:P139S", "S:P139T", "S:G142C", "S:D142C", "S:Y144C", "S:Y145C", "S:S151C", "S:W152C", "S:S247C", "S:Y248C", "S:W258C", "S:G261C"]
seq_factor_OFF_HALF_ON__0_1_2 = 0
nonmulti_0___multi_1 = 0
AA_2plus__mut2pos_pos2mut__2026_02_12(min_mut_ct, min_chi_sq, min_log_pv_fish, print_min, exc_muts, exc_muts_po, sel_muts, seq_factor_OFF_HALF_ON__0_1_2, nonmulti_0___multi_1)
#############################################################################################################################################################################
## AA_2plus__mut2pos_pos2mut__2026_02_12(filename, min_mut_ct, min_chi_sq, min_log_pv_fish::Int, print_min, exc_muts::Vector{String}, exc_muts_po::Vector{String}, sel_muts::Vector{String}, sel_muts_po::Vector{String}, seq_factor_OFF_HALF_ON__0_1_2::Int)
#############################################################################################################################################################################
#############################################################################################################################################################################

2026_02_18__652PM
S:V16-, S:Q14-, S:N17-, S:L18-, S:F140-, S:A243-, NTD:disulfide, S:D138-, S:T95I, S:I19-, S:T20-, S:R21-, S:T22-, S:Q23-, S:Y145-, S:D215G, S:S13-, S:R190S, S:R190K, S:S12-, 

Dict{String, Tuple{Float64, Float64, Int64, Float64, Float64, Float64, Float64, Int64, Int64}} with 2992 entries:
  "S:F939S"      => (0.5, 0.00251256, 1, 2.27474, 0.46595, 0.331661, 4.02638, 2…
  "ORF1b:I1227M" => (0.25, 0.00246305, 1, 0.517152, 0.820377, 0.0859865, 2.0227…
  "S:R214S"      => (0.25, 0.00518135, 2, 1.08628, 0.505527, 0.296255, 2.05699,…
  "ORF1a:A2388V" => (1.0, 0.00246305, 1, 6.2026, 0.247561, 0.606318, 8.07882, 1…
  "N:Q281R"      => (0.5, 0.00244499, 1, 2.27408, 0.466025, 0.331591, 4.02567, …
  "ORF1a:A1306S" => (0.25, 0.00251889, 1, 0.488224, 0.833263, 0.0792177, 1.9842…
  "ORF1a:P3515S" => (1.0, 0.00245098, 1, 6.15878, 0.248932, 0.603919, 8.03431, …
  "S:E154Q"      => (0.4, 0.00512821, 2, 3.14186, 0.232949, 0.632738, 3.26462, …
  "ORF1b:P1727S" => (0.181818, 0.004914, 2, 0.299299, 0.806388, 0.0934561, 1.46…
  "S:S98F"       => (0.206897, 0.0153061, 6, 1.6965, 0.268865, 0.570466, 1.6915…
  "N:A419T"      => (0.5, 0.00245098, 1, 2.26264, 0.467347, 0.330361, 4.0134

In [110]:
### mut auto-detect correlations, sub2pos/pos2sub version ——Used to detect DAM-associated mutations| Runtime: 
start = time()
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
NTD_disulfide_variations = Set(["NTD:disulfide", "NTD_disulfide", "S:NTD_disulfide", "NTDdisulfide", "NTD:Disulfide"])
Double_N_ORF9b_muts_AUTO_sub2pos = Set(["N:Q28-", "N:N29-", "N:G30-", "N:E31-", "N:R32-", "N:S33-", "N:L13-", "N:L13I", "N:L13H", "N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:G63D", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
Double_N_ORF9b_muts_po_AUTO_sub2pos = list_to_strings_set("N:4, N:5, N:6, N:7, N:8, N:9, N:10, N:11, N:12, N:13, N:14, N:15, N:16, N:17, N:18, N:19, N:20, N:21, N:22, N:23, N:24, N:25, N:26, N:27, N:28, N:29, N:30, N:31, N:32, N:33, N:34, N:35, N:36, N:37, N:38, N:39, N:40, N:41, N:42, N:43, N:44, N:45, N:46, N:47, N:48, N:49, N:50, N:51, N:52, N:53, N:54, N:55, N:56, N:57, N:58, N:59, N:60, N:61, N:62, N:63, N:64, N:65, N:66, N:67, N:68, N:69, N:70, N:71, N:72, N:73, N:74, N:75, N:76, N:77, N:78, N:79, N:80, N:81, N:82, N:83, N:84, N:85, N:86, N:87, N:88, N:89, N:90, N:91, N:92, N:93, N:94, N:95, N:96, N:97, N:98, N:99, N:100, N:101, N:102")
Double_N_ORF9b_combined_AUTO_sub2pos = union(Double_N_ORF9b_muts_AUTO_sub2pos, Double_N_ORF9b_muts_po_AUTO_sub2pos)
ORF9b_all_muts_set_AUTO_sub2pos = Set(["ORF9b:M1R","ORF9b:M1K","ORF9b:M1T","ORF9b:M1L","ORF9b:M1V","ORF9b:M1-","ORF9b:M1I","ORF9b:G2S","ORF9b:D2D","ORF9b:D2H","ORF9b:D2G","ORF9b:D2Y","ORF9b:D2V","ORF9b:D2E","ORF9b:D2N","ORF9b:D2-","ORF9b:D2A","ORF9b:P3S","ORF9b:P3F","ORF9b:H3L","ORF9b:P3-","ORF9b:P3H","ORF9b:H3N","ORF9b:H3D","ORF9b:H3Q","ORF9b:P3P","ORF9b:H3P","ORF9b:H3Y","ORF9b:P3R","ORF9b:P3T","ORF9b:P3L","ORF9b:P3A","ORF9b:H3R","ORF9b:K4R","ORF9b:K4K","ORF9b:K4I","ORF9b:K4-","ORF9b:K4T","ORF9b:K4E","ORF9b:K4Y","ORF9b:K4Q","ORF9b:K4N","ORF9b:I5L","ORF9b:T5N","ORF9b:I5F","ORF9b:I5V","ORF9b:T5P","ORF9b:T5V","ORF9b:I5-","ORF9b:T5I","ORF9b:I5N","ORF9b:T5A","ORF9b:I5T","ORF9b:I5I","ORF9b:I5S","ORF9b:I5M","ORF9b:T5T","ORF9b:I5C","ORF9b:S6T","ORF9b:S6N","ORF9b:S6G","ORF9b:S6I","ORF9b:S6-","ORF9b:S6C","ORF9b:S6P","ORF9b:S6R","ORF9b:C6-","ORF9b:C6F","ORF9b:C6R","ORF9b:E7*","ORF9b:E7K","ORF9b:E7S","ORF9b:E7E","ORF9b:E7V","ORF9b:E7Q","ORF9b:E7A","ORF9b:E7D","ORF9b:E7G","ORF9b:E7-","ORF9b:M8K","ORF9b:M8-","ORF9b:M8T","ORF9b:M8L","ORF9b:M8I","ORF9b:M8R","ORF9b:M8V","ORF9b:H9N","ORF9b:H9P","ORF9b:H9L","ORF9b:D9Y","ORF9b:H9H","ORF9b:D9E","ORF9b:H9-","ORF9b:H9Q","ORF9b:H9Y","ORF9b:H9D","ORF9b:H9R","ORF9b:F10C","ORF9b:P10-","ORF9b:P10T","ORF9b:P10R","ORF9b:F10P","ORF9b:F10Y","ORF9b:F10L","ORF9b:F10I","ORF9b:S10S","ORF9b:S10L","ORF9b:P10L","ORF9b:P10S","ORF9b:S10P","ORF9b:P10F","ORF9b:F10S","ORF9b:S10H","ORF9b:S10Y","ORF9b:S10A","ORF9b:S10C","ORF9b:S10-","ORF9b:P10H","ORF9b:F10-","ORF9b:S10T","ORF9b:P10A","ORF9b:S10F","ORF9b:P10D","ORF9b:A11S","ORF9b:A11P","ORF9b:V11I","ORF9b:A11T","ORF9b:A11G","ORF9b:A11I","ORF9b:A11V","ORF9b:A11E","ORF9b:A11-","ORF9b:L12I","ORF9b:L12-","ORF9b:L12L","ORF9b:L12V","ORF9b:L12S","ORF9b:L12Q","ORF9b:L12F","ORF9b:R13C","ORF9b:R13S","ORF9b:R13F","ORF9b:R13H","ORF9b:R13-","ORF9b:R13G","ORF9b:R13L","ORF9b:R13P","ORF9b:L14M","ORF9b:L14V","ORF9b:L14L","ORF9b:L14W","ORF9b:L14I","ORF9b:L14S","ORF9b:L14-","ORF9b:L14F","ORF9b:V15L","ORF9b:V15-","ORF9b:V15V","ORF9b:V15A","ORF9b:L15V","ORF9b:V15M","ORF9b:V15E","ORF9b:G16A","ORF9b:D16N","ORF9b:G16V","ORF9b:D16A","ORF9b:D16E","ORF9b:G16-","ORF9b:G16C","ORF9b:D16V","ORF9b:G16D","ORF9b:D16D","ORF9b:D16Y","ORF9b:G16S","ORF9b:D16-","ORF9b:D16H","ORF9b:D16G","ORF9b:P17S","ORF9b:P17T","ORF9b:P17A","ORF9b:P17L","ORF9b:P17-","ORF9b:P17R","ORF9b:P17P","ORF9b:P17H","ORF9b:Q18S","ORF9b:Q18H","ORF9b:Q18L","ORF9b:Q18R","ORF9b:Q18*","ORF9b:Q18-","ORF9b:Q18P","ORF9b:I19F","ORF9b:I19L","ORF9b:I19-","ORF9b:I19M","ORF9b:I19N","ORF9b:I19T","ORF9b:I19S","ORF9b:I19V","ORF9b:I19I","ORF9b:Q20L","ORF9b:Q20H","ORF9b:Q20*","ORF9b:Q20P","ORF9b:Q20R","ORF9b:Q20-","ORF9b:L21-","ORF9b:L21Q","ORF9b:L21I","ORF9b:L21L","ORF9b:L21R","ORF9b:L21M","ORF9b:L21V","ORF9b:L21P","ORF9b:A22-","ORF9b:A22S","ORF9b:A22A","ORF9b:A22E","ORF9b:A22T","ORF9b:A22P","ORF9b:A22L","ORF9b:A22V","ORF9b:V23A","ORF9b:V23-","ORF9b:V23L","ORF9b:V23I","ORF9b:V23E","ORF9b:T24N","ORF9b:T24I","ORF9b:T24A","ORF9b:T24S","ORF9b:T24P","ORF9b:T24D","ORF9b:T24-","ORF9b:R25C","ORF9b:R25G","ORF9b:R25T","ORF9b:R25K","ORF9b:R25-","ORF9b:R25S","ORF9b:R25I","ORF9b:R25M","ORF9b:M26T","ORF9b:M26I","ORF9b:M26V","ORF9b:M26L","ORF9b:M26K","ORF9b:M26-","ORF9b:M26R","ORF9b:E27K","ORF9b:E27D","ORF9b:E27Q","ORF9b:E27A","ORF9b:E27V","ORF9b:E27-","ORF9b:E27G","ORF9b:E27W","ORF9b:N28Y","ORF9b:N28T","ORF9b:N28D","ORF9b:N28S","ORF9b:N28I","ORF9b:N28K","ORF9b:N28H","ORF9b:N28-","ORF9b:A29G","ORF9b:A29L","ORF9b:A29P","ORF9b:A29I","ORF9b:I29A","ORF9b:I29N","ORF9b:I29S","ORF9b:I29-","ORF9b:A29V","ORF9b:I29T","ORF9b:A29T","ORF9b:A29E","ORF9b:A29S","ORF9b:I29V","ORF9b:A29-","ORF9b:L30V","ORF9b:L30F","ORF9b:V30-","ORF9b:L30-","ORF9b:L30S","ORF9b:V30L","ORF9b:V30E","ORF9b:V30G","ORF9b:V30F","ORF9b:L30I","ORF9b:A30-","ORF9b:V30M","ORF9b:V30A","ORF9b:G31G","ORF9b:E31-","ORF9b:G31-","ORF9b:R31G","ORF9b:G31V","ORF9b:G31A","ORF9b:G31W","ORF9b:G31E","ORF9b:G31T","ORF9b:G31R","ORF9b:R32I","ORF9b:R32P","ORF9b:R32-","ORF9b:R32C","ORF9b:R32L","ORF9b:R32S","ORF9b:R32N","ORF9b:R32F","ORF9b:R32G","ORF9b:R32H","ORF9b:D33G","ORF9b:D33Y","ORF9b:D33A","ORF9b:D33N","ORF9b:D33D","ORF9b:D33V","ORF9b:D33-","ORF9b:D33E","ORF9b:D33H","ORF9b:Q34H","ORF9b:Q34P","ORF9b:Q34L","ORF9b:Q34Q","ORF9b:Q34R","ORF9b:Q34-","ORF9b:Q34*","ORF9b:Q34E","ORF9b:N35-","ORF9b:N35H","ORF9b:N35Y","ORF9b:N35I","ORF9b:N35L","ORF9b:N35T","ORF9b:N35K","ORF9b:N35D","ORF9b:N35S","ORF9b:N36I","ORF9b:N36N","ORF9b:N36K","ORF9b:N36-","ORF9b:N36Y","ORF9b:N36D","ORF9b:N36T","ORF9b:N36H","ORF9b:N36S","ORF9b:V37I","ORF9b:V37L","ORF9b:V37F","ORF9b:V37-","ORF9b:V37G","ORF9b:V37A","ORF9b:V37D","ORF9b:G38A","ORF9b:G38R","ORF9b:G38-","ORF9b:G38F","ORF9b:G38D","ORF9b:G38S","ORF9b:G38C","ORF9b:D38N","ORF9b:G38V","ORF9b:P39R","ORF9b:P39A","ORF9b:P39L","ORF9b:P39H","ORF9b:P39S","ORF9b:P39-","ORF9b:K40T","ORF9b:R40M","ORF9b:K40Q","ORF9b:K40M","ORF9b:K40*","ORF9b:K40R","ORF9b:K40N","ORF9b:K40E","ORF9b:K40-","ORF9b:V41I","ORF9b:V41-","ORF9b:V41A","ORF9b:V41F","ORF9b:V41L","ORF9b:V41D","ORF9b:V41G","ORF9b:Y42H","ORF9b:Y42D","ORF9b:Y42C","ORF9b:Y42-","ORF9b:Y42N","ORF9b:P43R","ORF9b:P43L","ORF9b:P43-","ORF9b:P43S","ORF9b:P43A","ORF9b:P43T","ORF9b:P43P","ORF9b:P43Q","ORF9b:I44V","ORF9b:I44-","ORF9b:I44L","ORF9b:I44T","ORF9b:I44M","ORF9b:I45M","ORF9b:I45-","ORF9b:I45K","ORF9b:I45T","ORF9b:I45L","ORF9b:I45V","ORF9b:L46P","ORF9b:L46V","ORF9b:L46R","ORF9b:L46M","ORF9b:L46F","ORF9b:L46L","ORF9b:L46-","ORF9b:L46I","ORF9b:L46Q","ORF9b:R47G","ORF9b:R47L","ORF9b:R47P","ORF9b:R47C","ORF9b:R47H","ORF9b:R47F","ORF9b:R47-","ORF9b:L48R","ORF9b:L48H","ORF9b:L48P","ORF9b:L48F","ORF9b:L48-","ORF9b:G49-","ORF9b:G49C","ORF9b:G49D","ORF9b:G49V","ORF9b:S50L","ORF9b:S50P","ORF9b:S50T","ORF9b:S50-","ORF9b:L50I","ORF9b:P51S","ORF9b:P51R","ORF9b:P51A","ORF9b:P51Q","ORF9b:P51-","ORF9b:P51L","ORF9b:P51P","ORF9b:P51T","ORF9b:L52R","ORF9b:L52V","ORF9b:L52P","ORF9b:L52-","ORF9b:L52H","ORF9b:S53L","ORF9b:S53T","ORF9b:S53-","ORF9b:L54V","ORF9b:L54H","ORF9b:L54R","ORF9b:L54-","ORF9b:L54P","ORF9b:N55-","ORF9b:S55N","ORF9b:N55S","ORF9b:N55T","ORF9b:N55D","ORF9b:N55Y","ORF9b:M56-","ORF9b:M56T","ORF9b:M56I","ORF9b:M56V","ORF9b:A57-","ORF9b:A57P","ORF9b:A57V","ORF9b:A57S","ORF9b:A57E","ORF9b:A57G","ORF9b:A57T","ORF9b:R58K","ORF9b:R58G","ORF9b:R58-","ORF9b:R58E","ORF9b:R58T","ORF9b:R58W","ORF9b:R58S","ORF9b:R58M","ORF9b:K59R","ORF9b:K59E","ORF9b:K59Q","ORF9b:K59T","ORF9b:K59N","ORF9b:K59M","ORF9b:K59-","ORF9b:K59K","ORF9b:T60A","ORF9b:A60S","ORF9b:T60-","ORF9b:A60V","ORF9b:T60S","ORF9b:T60P","ORF9b:T60N","ORF9b:A60-","ORF9b:A60T","ORF9b:T60I","ORF9b:I60T","ORF9b:A60D","ORF9b:L61-","ORF9b:L61S","ORF9b:L61F","ORF9b:N62H","ORF9b:N62S","ORF9b:N62T","ORF9b:N62I","ORF9b:N62K","ORF9b:N62V","ORF9b:N62-","ORF9b:N62Y","ORF9b:N62D","ORF9b:S63A","ORF9b:S63T","ORF9b:S63F","ORF9b:S63Y","ORF9b:S63-","ORF9b:S63S","ORF9b:L64V","ORF9b:L64P","ORF9b:L64-","ORF9b:L64S","ORF9b:L64L","ORF9b:L64H","ORF9b:L64I","ORF9b:L64R","ORF9b:L64F","ORF9b:E65N","ORF9b:E65K","ORF9b:E65G","ORF9b:E65*","ORF9b:E65Q","ORF9b:E65A","ORF9b:E65S","ORF9b:E65D","ORF9b:E65-","ORF9b:E65V","ORF9b:D66Y","ORF9b:D66G","ORF9b:D66V","ORF9b:D66N","ORF9b:D66A","ORF9b:D66-","ORF9b:D66H","ORF9b:D66E","ORF9b:K67A","ORF9b:K67L","ORF9b:K67T","ORF9b:K67E","ORF9b:K67R","ORF9b:K67Q","ORF9b:K67-","ORF9b:K67M","ORF9b:A68D","ORF9b:A68-","ORF9b:A68E","ORF9b:A68V","ORF9b:A68S","ORF9b:F69C","ORF9b:F69S","ORF9b:F69Y","ORF9b:F69-","ORF9b:F69L","ORF9b:F69I","ORF9b:Q70L","ORF9b:Q70C","ORF9b:Q70H","ORF9b:Q70P","ORF9b:Q70-","ORF9b:Q70R","ORF9b:L71P","ORF9b:L71-","ORF9b:L71V","ORF9b:L71K","ORF9b:L71R","ORF9b:L71A","ORF9b:L71I","ORF9b:L71E","ORF9b:L71S","ORF9b:T72A","ORF9b:T72-","ORF9b:T72P","ORF9b:T72R","ORF9b:T72I","ORF9b:P73-","ORF9b:P73S","ORF9b:P73T","ORF9b:P73L","ORF9b:P73R","ORF9b:P73Q","ORF9b:I74V","ORF9b:I74L","ORF9b:I74-","ORF9b:I74T","ORF9b:I74M","ORF9b:A75-","ORF9b:A75T","ORF9b:A75K","ORF9b:A75S","ORF9b:A75V","ORF9b:A75P","ORF9b:A75A","ORF9b:V76L","ORF9b:V76D","ORF9b:V76C","ORF9b:V76N","ORF9b:V76S","ORF9b:V76H","ORF9b:V76F","ORF9b:V76A","ORF9b:V76-","ORF9b:V76G","ORF9b:V76V","ORF9b:V76I","ORF9b:Q77L","ORF9b:Q77-","ORF9b:E77*","ORF9b:Q77E","ORF9b:E77A","ORF9b:Q77H","ORF9b:Q77R","ORF9b:Q77P","ORF9b:E77G","ORF9b:E77D","ORF9b:Q77Y","ORF9b:Q77*","ORF9b:Q77K","ORF9b:Q77S","ORF9b:M78T","ORF9b:M78-","ORF9b:M78V","ORF9b:M78I","ORF9b:M78L","ORF9b:M78K","ORF9b:T79N","ORF9b:T79T","ORF9b:T79-","ORF9b:T79I","ORF9b:K80R","ORF9b:K80T","ORF9b:K80E","ORF9b:K80-","ORF9b:K80I","ORF9b:K80K","ORF9b:K80N","ORF9b:L81-","ORF9b:L81S","ORF9b:L81F","ORF9b:L81W","ORF9b:A82D","ORF9b:A82-","ORF9b:A82V","ORF9b:A82G","ORF9b:T83-","ORF9b:T83I","ORF9b:T84S","ORF9b:T84-","ORF9b:T84I","ORF9b:E85-","ORF9b:E85A","ORF9b:E85V","ORF9b:E85D","ORF9b:E85K","ORF9b:E85G","ORF9b:E86V","ORF9b:E86E","ORF9b:E86K","ORF9b:E86P","ORF9b:E86*","ORF9b:E86Q","ORF9b:E86Y","ORF9b:E86A","ORF9b:E86G","ORF9b:E86-","ORF9b:E86D","ORF9b:L87-","ORF9b:L87Q","ORF9b:L87I","ORF9b:L87P","ORF9b:L87L","ORF9b:L87V","ORF9b:P88L","ORF9b:P88-","ORF9b:P88Q","ORF9b:P88S","ORF9b:P88T","ORF9b:P88R","ORF9b:P88A","ORF9b:D89-","ORF9b:D89G","ORF9b:D89E","ORF9b:D89H","ORF9b:D89N","ORF9b:D89V","ORF9b:D89A","ORF9b:D89Y","ORF9b:E90A","ORF9b:E90K","ORF9b:E90D","ORF9b:E90-","ORF9b:E90V","ORF9b:E90Q","ORF9b:E90G","ORF9b:F91Y","ORF9b:F91C","ORF9b:F91S","ORF9b:F91P","ORF9b:F91L","ORF9b:F91-","ORF9b:F91V","ORF9b:V92A","ORF9b:V92L","ORF9b:V92V","ORF9b:V92I","ORF9b:V92M","ORF9b:V92E","ORF9b:V92-","ORF9b:V93-","ORF9b:V93F","ORF9b:V93L","ORF9b:V93E","ORF9b:V93V","ORF9b:V93M","ORF9b:V93A","ORF9b:V94V","ORF9b:V94L","ORF9b:V94-","ORF9b:V94M","ORF9b:V94A","ORF9b:T95P","ORF9b:T95V","ORF9b:T95E","ORF9b:T95M","ORF9b:T95-","ORF9b:M95T","ORF9b:T95R","ORF9b:T95A","ORF9b:T95K","ORF9b:V96G","ORF9b:V96T","ORF9b:V96-","ORF9b:V96I","ORF9b:V96A","ORF9b:V96L","ORF9b:V96E","ORF9b:K97Q","ORF9b:K97T","ORF9b:K97-","ORF9b:K97E","ORF9b:K97R","ORF9b:K97N","ORF9b:K97I","ORF9b:*98R","ORF9b:*98G","ORF9b:*98L","ORF9b:*98S","ORF9b:*98C","ORF9b:*98-","ORF9b:*98*"])
ORF9b_all_muts_set_po_AUTO_sub2pos = list_to_strings_set("ORF9b:1, ORF9b:2, ORF9b:3, ORF9b:4, ORF9b:5, ORF9b:6, ORF9b:7, ORF9b:8, ORF9b:9, ORF9b:10, ORF9b:11, ORF9b:12, ORF9b:13, ORF9b:14, ORF9b:15, ORF9b:16, ORF9b:17, ORF9b:18, ORF9b:19, ORF9b:20, ORF9b:21, ORF9b:22, ORF9b:23, ORF9b:24, ORF9b:25, ORF9b:26, ORF9b:27, ORF9b:28, ORF9b:29, ORF9b:30, ORF9b:31, ORF9b:32, ORF9b:33, ORF9b:34, ORF9b:35, ORF9b:36, ORF9b:37, ORF9b:38, ORF9b:39, ORF9b:40, ORF9b:41, ORF9b:42, ORF9b:43, ORF9b:44, ORF9b:45, ORF9b:46, ORF9b:47, ORF9b:48, ORF9b:49, ORF9b:50, ORF9b:51, ORF9b:52, ORF9b:53, ORF9b:54, ORF9b:55, ORF9b:56, ORF9b:57, ORF9b:58, ORF9b:59, ORF9b:60, ORF9b:61, ORF9b:62, ORF9b:63, ORF9b:64, ORF9b:65, ORF9b:66, ORF9b:67, ORF9b:68, ORF9b:69, ORF9b:70, ORF9b:71, ORF9b:72, ORF9b:73, ORF9b:74, ORF9b:75, ORF9b:76, ORF9b:77, ORF9b:78, ORF9b:79, ORF9b:80, ORF9b:81, ORF9b:82, ORF9b:83, ORF9b:84, ORF9b:85, ORF9b:86, ORF9b:87, ORF9b:88, ORF9b:89, ORF9b:90, ORF9b:91, ORF9b:92, ORF9b:93, ORF9b:94, ORF9b:95, ORF9b:96, ORF9b:97, ORF9b:98")
ORF9b_all_muts_combined_AUTO_sub2pos = union(ORF9b_all_muts_set_AUTO_sub2pos, ORF9b_all_muts_set_po_AUTO_sub2pos)
correlated_muts_AUTO_sub2pos = list_to_strings_set("ORF1a:I1367V, ORF1a:R1318G, ORF1a:N3443Y, ORF1a:T1567I, ORF1a:A2325V, ORF1a:N3443K, ORF1b:Y822H, ORF1a:P1220S, ORF1a:A1397V, ORF1a:H110Y, ORF1a:I114T, ORF1a:R232L, ORF1a:E233D, ORF1a:E233K, ORF1a:E233A, ORF1a:E233G, ORF1a:K958R, ORF1a:P959L, ORF1a:P959S, ORF1a:P1220H, ORF1a:P1220L, ORF1a:P1220S, ORF1a:S1221P, ORF1a:E1223K, ORF1a:Q1224R, ORF1a:S1272G, ORF1a:D1273N, ORF1a:D1273G, ORF1a:I1276T, ORF1a:T1322A, ORF1a:T1322I, ORF1a:D1323N, ORF1a:D1323G, ORF1a:I1326V, ORF1a:T1328N, ORF1a:S1361P, ORF1a:Q1365R, ORF1a:Q1365P, ORF1a:I1367L, ORF1a:L1368I, ORF1a:G1369R, ORF1a:S1372A, ORF1a:S1372C, ORF1a:E1394D, ORF1a:T1395N, ORF1a:T1395I, ORF1a:H1500Y, ORF1a:T1504I, ORF1a:I1505L, ORF1a:L1507F, ORF1a:T1542I, ORF1a:T1543I, ORF1a:R1566M, ORF1a:V1570A, ORF1a:V1570M, ORF1a:V1570L, ORF1a:T1572I, ORF1a:T1638A, ORF1a:T1638I, ORF1a:T1638N, ORF1a:D1639N, ORF1a:E1706D, ORF1a:N1709S, ORF1a:N1709T, ORF1a:I1714V, ORF1a:I1714T, ORF1a:I1714A, ORF1a:M1769V, ORF1a:M1771V, ORF1a:T1773I, ORF1a:F1779L, ORF1a:K1795Q, ORF1a:P2134S, ORF1a:P2134L, ORF1a:P2134H, ORF1a:D2136N, ORF1a:D2136G, ORF1a:I2138V, ORF1a:C2165F, ORF1a:T2166I, ORF1a:F2209L, ORF1a:S2214L, ORF1a:F2215L, ORF1a:T2274I, ORF1a:A2325T, ORF1a:A2325V, ORF1a:F2328V, ORF1a:F2328L, ORF1a:L2329F, ORF1a:S2466G, ORF1a:D2467E, ORF1a:E2468D, ORF1a:V2469I, ORF1a:A2470S, ORF1a:R2471G, ORF1a:R2471S, ORF1a:R2471K, ORF1a:D2472E, ORF1a:L2475I, ORF1a:A3648V, ORF1a:V3653A, ORF1a:A3657V, ORF1a:R3802S, ORF1a:R3802H, ORF1a:R3802C, ORF1a:Y3803H, ORF1a:L3808F, ORF1a:L3951F, ORF1a:P3952S, ORF1a:T4087A, ORF1a:T4087I, ORF1a:T4088I, ORF1a:T4090I, ORF1a:T4090A, ORF1a:T4164N, ORF1a:T4164I, ORF1a:D4165N, ORF1a:D4165V, ORF1a:D4165G, ORF1a:D4165A, ORF1a:D4166N, ORF1a:D4166Y, ORF1a:D4166G, ORF1a:N4167S, ORF1a:A4396V, ORF1a:A4396T, ORF1a:S4398L, ORF1b:Q72R, ORF1b:E75K, ORF1b:T76I, ORF1b:T730I, ORF1b:L820F, ORF1b:P821S, ORF1b:D824N, ORF1b:D824E, ORF1b:S826A, ORF1b:L1220F, N:V270L, N:T271I, N:P326H, N:P326L, ORF3a:Q213K, ORF7a:E22D, ORF7a:T39I, ORF9b:S10P, ORF9b:R13C, ORF9b:L64P, ORF9b:D66E, ORF9b:A68E, ORF9b:A68V, ORF9b:V93M, ORF9b:V93L, ORF9b:T95M, ORF9b:T95A, ORF9b:V96A")
correlated_muts_po_AUTO_sub2pos = list_to_strings_set("ORF1a:85, ORF1a:110, ORF1a:111, ORF1a:114, ORF1a:233, ORF1a:235, ORF1a:958, ORF1a:959, ORF1a:1218, ORF1a:1220, ORF1a:1221, ORF1a:1223, ORF1a:1224, ORF1a:1225, ORF1a:1272, ORF1a:1273, ORF1a:1274, ORF1a:1276, ORF1a:1322, ORF1a:1323, ORF1a:1326, ORF1a:1365, ORF1a:1367, ORF1a:1368, ORF1a:1369, ORF1a:1372, ORF1a:1394, ORF1a:1395, ORF1a:1397, ORF1a:1500, ORF1a:1504, ORF1a:1505, ORF1a:1507, ORF1a:1542, ORF1a:1543, ORF1a:1566, ORF1a:1570, ORF1a:1572, ORF1a:1638, ORF1a:1639, ORF1a:1709, ORF1a:1714, ORF1a:1716, ORF1a:1769, ORF1a:1773, ORF1a:2134, ORF1a:2136, ORF1a:2138, ORF1a:2166, ORF1a:2209, ORF1a:2214, ORF1a:2215, ORF1a:2274, ORF1a:2324, ORF1a:2325, ORF1a:2328, ORF1a:2345, ORF1a:2346, ORF1a:2349, ORF1a:2351, ORF1a:2353, ORF1a:2466, ORF1a:2467, ORF1a:2468, ORF1a:2469, ORF1a:2470, ORF1a:2471, ORF1a:2472, ORF1a:2475, ORF1a:3443, ORF1a:3648, ORF1a:3650, ORF1a:3652, ORF1a:3653, ORF1a:3657, ORF1a:3802, ORF1a:3803, ORF1a:3808, ORF1a:3949, ORF1a:3951, ORF1a:3952, ORF1a:4083, ORF1a:4087, ORF1a:4088, ORF1a:4090, ORF1a:4097, ORF1a:4102, ORF1a:4164, ORF1a:4165, ORF1a:4166, ORF1a:4167, ORF1a:4396, ORF1a:4397, ORF1a:4398, ORF1b:72, ORF1b:75, ORF1b:76, ORF1b:662, ORF1b:730, ORF1b:820, ORF1b:821, ORF1b:822, ORF1b:824, ORF1b:826, ORF1b:1220, N:270, N:271, N:326, ORF3a:213, ORF7a:22, ORF7a:39, ORF9b:93, ORF9b:95")
correlated_muts_combined_AUTO_sub2pos = union(correlated_muts_AUTO_sub2pos, correlated_muts_po_AUTO_sub2pos)
BAL_muts_AUTO_sub2pos = list_to_strings_set("ORF1a:K4176R, ORF1a:F3089L, ORF1a:T3032I, E:A32V, E:S68F, ORF1a:T2183I, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:F374L, S:F375-, S:A376V, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659P, S:A668V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:L1186F, E:V5A, E:V5F, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:F23S, E:T30I, E:A36V, E:Y42C, M:A2V, M:S4P, M:V10I, M:F28S, M:T77N, M:S94R, M:S94N, M:H125Y, M:H148R, M:A188T, N:P80L, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*")
BAL_muts_po_AUTO_sub2pos = Set(["ORF1a:2183","ORF1a:2972","ORF1a:3049","ORF1a:3058","ORF1a:3070","ORF1a:3072","ORF1a:3076","ORF1a:3123","ORF1a:3195","ORF1a:3359","ORF1a:3454","ORF1a:3456","ORF1a:4110","ORF1a:4175","ORF1a:4197","ORF1a:4205","ORF1a:4207","ORF1a:4387","ORF1b:314","ORF1b:1181","ORF1b:1247","ORF1b:1424","ORF1b:2339","ORF1b:2340","ORF1b:2537","S:50","S:330","S:354","S:367","S:374","S:375","S:376","S:405","S:508","S:551","S:573","S:647","S:653","S:657","S:659","S:668","S:859","S:944","S:958","S:978","S:1169","S:1179","S:1186","E:5","E:5","E:9","E:10","E:10","E:13","E:14","E:16","E:23","E:30","E:36","E:42","M:2","M:4","M:10","M:28","M:77","M:94","M:94","M:125","M:148","M:188","N:80","N:416","N:417","ORF7a:115","ORF9b:1","ORF9b:77"])
BAL_muts_combined_AUTO_sub2pos = union(BAL_muts_AUTO_sub2pos, BAL_muts_po_AUTO_sub2pos)
Others_AUTO_sub2pos = Set(["S:Y248F", "ORF1a:L2084I", "ORF1a:S2083-", "S:L24-", "S:P25-", "S:P26-", "S:N137-", "S:L141-", "S:G142-", "S:D142-", "S:V143-", "S:Y144-", "S:W152-", "S:M153-", "S:E154-", "S:S155-", "S:E156-", "S:L242-", "S:N137M", "ORF9b:V30-", "ORF7b:*44Y", "ORF7b:M1T", "S:V213P", "S:R214E", "ORF1a:S142T", "ORF1a:F143M", "S:Y144D", "S:V143D", "ORF7b:I2V", "ORF7b:M1T", "ORF3a:L15V", "ORF9b:P10S", "S:T19R", "S:A27S", "S:V213G", "ORF3a:N257D", "M:I8F", "S:D145Y", "S:I212L", "ORF9b:G16D", "S:L18F", "ORF8:V117L", "ORF8:L118V", "ORF8:F120V", "ORF8:F120L", "ORF8:I121L", "ORF7a:A82V", "S:I212L", "S:N211V", "S:R493Q", "ORF7b:M1I", "ORF7b:I2N", "ORF1b:R2613N", "ORF1b:P2612L", "ORF1a:L3606F", "S:H655Y", "N:I84V"])
Others_po_AUTO_sub2pos = Set(["S:248", "ORF1a:2084", "ORF1a:2083", "ORF1a:142", "ORF1a:143", "ORF1a:3606", "ORF1b:2612", "ORF1b:2613", "S:18", "S:24", "S:25", "S:26", "S:27", "S:137", "S:139", "S:141", "S:142", "S:142", "S:143", "S:143", "S:144", "S:144", "S:145", "S:152", "S:153", "S:154", "S:155", "S:156", "S:213", "S:213", "S:493", "S:655", "M:8", "N:84", "ORF3a:15", "ORF7a:82", "ORF7b:1", "ORF7b:1", "ORF7b:2", "ORF7b:2", "ORF7b:44", "ORF8:117", "ORF8:118", "ORF8:120", "ORF8:120", "ORF8:121", "ORF9b:10", "ORF9b:16", "ORF9b:30"])
Others_combined_AUTO_sub2pos = union(Others_AUTO_sub2pos, Others_po_AUTO_sub2pos)
Temp_Others_AUTO_sub2pos = Set(["N:T205I", "ORF1b:P1000L", "S:T19R"])
Temp_Others_po_AUTO_sub2pos = Set(["N:205", "ORF1b:1000", "S:19"])
Temp_Others_combined_AUTO_sub2pos = union(Temp_Others_AUTO_sub2pos, Temp_Others_po_AUTO_sub2pos)
Temp_Others2_AUTO_sub2pos = Set(["S:R493Q", "S:E484K"])  
Temp_Others2_po_AUTO_sub2pos = Set(["S:493", "S:484"])
Temp_Others2_combined_AUTO_sub2pos = union(Temp_Others2_AUTO_sub2pos, Temp_Others2_po_AUTO_sub2pos)

ORF3a_dels = list_to_strings_set("ORF3a:M1-, ORF3a:D2-, ORF3a:L3-, ORF3a:F4-, ORF3a:M5-, ORF3a:R6-, ORF3a:I7-, ORF3a:F8-, ORF3a:T9-, ORF3a:I10-, ORF3a:G11-, ORF3a:T12-, ORF3a:V13-, ORF3a:L15-, ORF3a:K16-, ORF3a:Q17-, ORF3a:G18-, ORF3a:E19-, ORF3a:I20-, ORF3a:K21-, ORF3a:D22-, ORF3a:A23-, ORF3a:T24-, ORF3a:P25-, ORF3a:S26-, ORF3a:D27-, ORF3a:F28-, ORF3a:V29-, ORF3a:R30-, ORF3a:A31-, ORF3a:T32-, ORF3a:A33-, ORF3a:T34-, ORF3a:I35-, ORF3a:P36-, ORF3a:I37-, ORF3a:Q38-, ORF3a:A39-, ORF3a:S40-, ORF3a:L41-, ORF3a:P42-, ORF3a:F43-, ORF3a:G44-, ORF3a:W45-, ORF3a:L46-, ORF3a:I47-, ORF3a:V48-, ORF3a:G49-, ORF3a:V50-, ORF3a:A51-, ORF3a:L52-, ORF3a:L53-, ORF3a:A54-, ORF3a:V55-, ORF3a:F56-, ORF3a:Q57-, ORF3a:S58-, ORF3a:A59-, ORF3a:S60-, ORF3a:K61-, ORF3a:I62-, ORF3a:I63-, ORF3a:T64-, ORF3a:L65-, ORF3a:K66-, ORF3a:K67-, ORF3a:R68-, ORF3a:W69-, ORF3a:P258-, ORF3a:V259-, ORF3a:M260-, ORF3a:E261-, ORF3a:P262-, ORF3a:I263-")
ORF6_dels = list_to_strings_set("ORF6:M1-, ORF6:F2-, ORF6:H3-, ORF6:L4-, ORF6:V5-, ORF6:D6-, ORF6:F7-, ORF6:Q8-, ORF6:V9-, ORF6:T10-, ORF6:I11-, ORF6:A12-, ORF6:E13-, ORF6:I14-, ORF6:L15-, ORF6:L16-, ORF6:I17-, ORF6:I18-, ORF6:M19-, ORF6:R20-, ORF6:T21-, ORF6:F22-, ORF6:K23-, ORF6:V24-, ORF6:S25-, ORF6:I26-, ORF6:W27-, ORF6:N28-, ORF6:L29-, ORF6:D30-, ORF6:Y31-, ORF6:I32-, ORF6:I33-, ORF6:N34-, ORF6:L35-, ORF6:I36-, ORF6:I37-, ORF6:K38-, ORF6:N39-, ORF6:L40-, ORF6:S41-, ORF6:K42-, ORF6:S43-, ORF6:L44-, ORF6:T45-, ORF6:E46-, ORF6:N47-, ORF6:K48-, ORF6:Y49-, ORF6:S50-, ORF6:Q51-, ORF6:L52-, ORF6:D53-, ORF6:E54-, ORF6:E55-, ORF6:Q56-, ORF6:P57-, ORF6:M58-, ORF6:E59-, ORF6:I60-, ORF6:D61-, ORF6:*62-")
ORF7_dels = list_to_strings_set("ORF7a:M1-, ORF7a:K2-, ORF7a:I3-, ORF7a:I4-, ORF7a:L5-, ORF7a:F6-, ORF7a:L7-, ORF7a:A8-, ORF7a:L9-, ORF7a:I10-, ORF7a:T11-, ORF7a:L12-, ORF7a:A13-, ORF7a:T14-, ORF7a:C15-, ORF7a:E16-, ORF7a:L17-, ORF7a:Y18-, ORF7a:H19-, ORF7a:Y20-, ORF7a:Q21-, ORF7a:E22-, ORF7a:C23-, ORF7a:V24-, ORF7a:R25-, ORF7a:G26-, ORF7a:T27-, ORF7a:T28-, ORF7a:V29-, ORF7a:L30-, ORF7a:L31-, ORF7a:K32-, ORF7a:E33-, ORF7a:P34-, ORF7a:C35-, ORF7a:S36-, ORF7a:S37-, ORF7a:G38-, ORF7a:T39-, ORF7a:Y40-, ORF7a:E41-, ORF7a:G42-, ORF7a:N43-, ORF7a:S44-, ORF7a:P45-, ORF7a:F46-, ORF7a:H47-, ORF7a:P48-, ORF7a:L49-, ORF7a:A50-, ORF7a:D51-, ORF7a:N52-, ORF7a:K53-, ORF7a:F54-, ORF7a:A55-, ORF7a:L56-, ORF7a:T57-, ORF7a:C58-, ORF7a:F59-, ORF7a:S60-, ORF7a:T61-, ORF7a:Q62-, ORF7a:F63-, ORF7a:A64-, ORF7a:F65-, ORF7a:A66-, ORF7a:C67-, ORF7a:P68-, ORF7a:D69-, ORF7a:G70-, ORF7a:V71-, ORF7a:K72-, ORF7a:H73-, ORF7a:V74-, ORF7a:Y75-, ORF7a:Q76-, ORF7a:L77-, ORF7a:R78-, ORF7a:A79-, ORF7a:R80-, ORF7a:S81-, ORF7a:V82-, ORF7a:S83-, ORF7a:P84-, ORF7a:K85-, ORF7a:L86-, ORF7a:F87-, ORF7a:I88-, ORF7a:R89-, ORF7a:Q90-, ORF7a:E91-, ORF7a:E92-, ORF7a:V93-, ORF7a:Q94-, ORF7a:E95-, ORF7a:L96-, ORF7a:Y97-, ORF7a:S98-, ORF7a:P99-, ORF7a:I100-, ORF7a:F101-, ORF7a:L102-, ORF7a:I103-, ORF7a:V104-, ORF7a:A105-, ORF7a:A106-, ORF7a:I107-, ORF7a:V108-, ORF7a:F109-, ORF7a:I110-, ORF7a:T111-, ORF7a:L112-, ORF7a:C113-, ORF7a:F114-, ORF7a:T115-, ORF7b:M1-, ORF7b:I2-, ORF7b:E3-, ORF7b:L4-, ORF7b:S5-, ORF7b:L6-, ORF7b:I7-, ORF7b:D8-, ORF7b:F9-, ORF7b:Y10-, ORF7b:L11-, ORF7b:C12-, ORF7b:F13-, ORF7b:L14-, ORF7b:A15-, ORF7b:F16-, ORF7b:L17-, ORF7b:L18-, ORF7b:F19-, ORF7b:L20-, ORF7b:V21-, ORF7b:L22-, ORF7b:I23-, ORF7b:M24-, ORF7b:L25-, ORF7b:I26-, ORF7b:I27-, ORF7b:F28-, ORF7b:W29-, ORF7b:F30-, ORF7b:S31-, ORF7b:L32-, ORF7b:E33-, ORF7b:L34-, ORF7b:Q35-, ORF7b:D36-, ORF7b:H37-, ORF7b:N38-, ORF7b:E39-, ORF7b:T40-, ORF7b:C41-, ORF7b:H42-, ORF7b:A43-, ORF7b:*44-")
ORF8_dels = list_to_strings_set("ORF8:M1-, ORF8:K2-, ORF8:F3-, ORF8:L4-, ORF8:V5-, ORF8:F6-, ORF8:L7-, ORF8:G8-, ORF8:I9-, ORF8:I10-, ORF8:T11-, ORF8:T12-, ORF8:V13-, ORF8:A14-, ORF8:A15-, ORF8:F16-, ORF8:H17-, ORF8:Q18-, ORF8:E19-, ORF8:C20-, ORF8:S21-, ORF8:L22-, ORF8:Q23-, ORF8:S24-, ORF8:C25-, ORF8:T26-, ORF8:Q27-, ORF8:H28-, ORF8:Q29-, ORF8:P30-, ORF8:Y31-, ORF8:V32-, ORF8:V33-, ORF8:D34-, ORF8:D35-, ORF8:P36-, ORF8:C37-, ORF8:P38-, ORF8:I39-, ORF8:H40-, ORF8:F41-, ORF8:Y42-, ORF8:S43-, ORF8:K44-, ORF8:W45-, ORF8:Y46-, ORF8:I47-, ORF8:R48-, ORF8:V49-, ORF8:G50-, ORF8:A51-, ORF8:R52-, ORF8:K53-, ORF8:S54-, ORF8:A55-, ORF8:P56-, ORF8:L57-, ORF8:I58-, ORF8:E59-, ORF8:L60-, ORF8:C61-, ORF8:V62-, ORF8:D63-, ORF8:E64-, ORF8:A65-, ORF8:G66-, ORF8:S67-, ORF8:K68-, ORF8:S69-, ORF8:P70-, ORF8:I71-, ORF8:Q72-, ORF8:Y73-, ORF8:I74-, ORF8:D75-, ORF8:I76-, ORF8:G77-, ORF8:N78-, ORF8:Y79-, ORF8:T80-, ORF8:V81-, ORF8:S82-, ORF8:C83-, ORF8:L84-, ORF8:P85-, ORF8:F86-, ORF8:T87-, ORF8:I88-, ORF8:N89-, ORF8:C90-, ORF8:Q91-, ORF8:E92-, ORF8:P93-, ORF8:K94-, ORF8:L95-, ORF8:G96-, ORF8:S97-, ORF8:L98-, ORF8:V99-, ORF8:V100-, ORF8:R101-, ORF8:C102-, ORF8:S103-, ORF8:F104-, ORF8:Y105-, ORF8:E106-, ORF8:D107-, ORF8:F108-, ORF8:L109-, ORF8:E110-, ORF8:Y111-, ORF8:H112-, ORF8:D113-, ORF8:V114-, ORF8:R115-, ORF8:V116-, ORF8:V117-, ORF8:L118-, ORF8:D119-, ORF8:F120-")
ORF9b_N_dels = list_to_strings_set("N:Q28-, N:N29-, N:G30-, N:E31-, N:R32-, N:S33-, N:G34-, ORF9b:T24-, ORF9b:R25-, ORF9b:M26-, ORF9b:E27-, ORF9b:N28-, ORF9b:I29-, ORF9b:V30-, ORF9b:L30-, ORF9b:G31-")
ORF1a_dels = list_to_strings_set("ORF1a:H81-, ORF1a:G82-, ORF1a:H83-, ORF1a:V84-, ORF1a:M85-, ORF1a:V86-, ORF1a:E87-, ORF1a:I1005-, ORF1a:V1006-, ORF1a:E1007-, ORF1a:L3674-")
S_dels = list_to_strings_set("S:L7-, S:L8-, S:P9-, S:L10-, S:V11-, S:S12-, S:S13-, S:Q14-, S:V16-, S:N17-, S:L18-, S:T19-, S:I19-, S:R19-, S:T20-, S:T21-, S:R21-, S:T22-, S:N22-, S:Q23-, S:S24-, S:L24-, S:P25-, S:P26-, S:A27-, S:S27-, S:Y28-, S:N61-, S:V62-, S:T63-, S:W64-, S:F65-, S:H66-, S:A67-, S:I68-, S:S71-, S:G72-, S:T73-, S:N74-, S:G75-, S:T76-, S:K77-, S:F135-, S:N137-, S:P139-, S:F140-, S:L141-, S:G142-, S:D142-, S:Y145-, S:D145-, S:H146-, S:Q146-, S:K147-, S:N148-, S:N149-, S:K150-, S:M153-, S:E154-, S:S155-, S:E156-, S:F157-, S:S157-, S:L157-, S:R158-, S:L176-, S:N188-, S:L179-, S:E180-, S:L241-, S:H245-, S:R246-, S:S247-, S:S256-, S:G257-, S:W258-, S:T259-, S:A260-, S:G261-, S:A262-, S:A263-, S:A264-")
M_dels = list_to_strings_set("M:N5-, M:G6-, M:T7-")
COMBINED_dels = union(ORF3a_dels, ORF6_dels, ORF7_dels, ORF8_dels, ORF9b_N_dels, ORF1a_dels, S_dels, M_dels)
 
Doubtful_Revs_AUTO_sub2pos = Set(["ORF9b:A60T, N:G63D, S:F18L", "ORF1a:G1307S", "ORF1a:R856K", "ORF6:L61D", "N:R413S", "S:G158R", "ORF7b:I40T", "N:R204G", "S:F981L", "S:D339G", "S:V67A", "S:I95T"])
Doubtful_Revs_po_AUTO_sub2pos = Set(["ORF9b:60, N:63, S:18", "ORF1a:1307", "ORF1a:856", "ORF6:61", "N:413", "S:158", "ORF7b:40", "N:204", "S:981", "S:339", "S:67"])
Doubtful_Revs_combined_AUTO_sub2pos = union(Doubtful_Revs_AUTO_sub2pos, Doubtful_Revs_po_AUTO_sub2pos)
Cryptics_AUTO_sub2pos = Set(["ORF3a:W45R", "ORF1b:F2314L", "S:K417T", "S:V1176F", "S:L828F", "ORF3a:Q213L", "ORF3a:H182D", "S:K1266N", "S:L828F", "S:T19K", "S:H655Y", "S:T859N", "ORF1a:V38A", "N:S37P", "ORF1b:F2314V", "S:D1153A", "ORF1a:L3606V", "ORF1b:E2311K"])
Cryptics_po_AUTO_sub2pos = Set(["ORF3a:45", "ORF1b:2314", "S:417", "S:1176", "S:828", "ORF3a:213", "ORF3a:182", "S:1266", "S:828", "S:19", "S:655", "S:859", "ORF1a:38", "N:37", "ORF1b:2314", "S:1153", "ORF1a:3606", "ORF1b:2311"])
Cryptics_combined_AUTO_sub2pos = union(Cryptics_AUTO_sub2pos, Cryptics_po_AUTO_sub2pos)
#################### Note: commenting out lines below retains them from `excluded_muts`; NOT commenting them out excludes them from `excluded_muts` ########################
            correlated_muts_combined_AUTO_sub2pos = Set{String}()
            BAL_muts_combined_AUTO_sub2pos = Set{String}()
            Cryptics_combined_AUTO_sub2pos = Set{String}()
#            Others_combined_AUTO_sub2pos = Set{String}()
#            Temp_Others2_combined_AUTO_sub2pos = Set{String}()
#            Double_N_ORF9b_combined_AUTO_sub2pos = Set{String}()
            ORF9b_all_muts_combined_AUTO_sub2pos = Set{String}()
#             = Set{String}()
###########################################################################################################################################################################
###########################################################################################################################################################################
excluded_muts_AUTO_sub2pos = union(COMBINED_dels, Double_N_ORF9b_combined_AUTO_sub2pos, ORF9b_all_muts_combined_AUTO_sub2pos, Cryptics_combined_AUTO_sub2pos, BAL_muts_combined_AUTO_sub2pos, Others_combined_AUTO_sub2pos, Temp_Others_combined_AUTO_sub2pos, Doubtful_Revs_combined_AUTO_sub2pos, artifactual_private_muts, RBM_muts)  #, RBD_not_RBM_muts) #,BAL_muts)
###########################################################################################################################################################################
###########################################################################################################################################################################
### This section is to determine the number of sequences with mutations that are clearly associated with C15-C136 NTD disulfide deletion or rearrangement.
### Later, this number os used to calculate the degree to which seqs w/the searched-for mut correlate with seqs w/NTD-disulfide-changing mutations
NTD_disulfide_filter_positions = list_to_strings_set("S:9, S:13, S:15, S:136, S:139")
NTD_C15_disulfide_muts = list_to_strings_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R")
NTD_C15_disulfide_positions = list_to_strings_set("S:9, S:13, S:15")
OG_NTD_C136_disulfide_muts = list_to_strings_set("S:C136-, S:D138-, S:C136R, S:C136H, S:C136F, S:C136Y, S:W64C, S:W152C")
NTD_C136_disulfide_muts = list_to_strings_set("S:W64C, S:S71C, S:C136-, S:D138-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:Y248C, S:W258C, S:G261C")
##### Added to core OG_NTD_C136_disulfide_muts due to log10pvfish ≥ 4 for NTD_disulfide or major disulfide mut (like P9L): S:W64C, S:P139S, S:P139T, S:S151C, (S:D142C, S:G142C, S:Y144C, S:Y145C——tested & added collectively), 
#   comprehensive_NTD_disulfide_muts = list_to_strings_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:W64C, S:S71C, S:C136-, S:D138-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:G142C, S:D142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:Y248C, S:W258C, S:G261C")
NTD_C136_disulfide_positions = list_to_strings_set("S:136, S:138, S:139")
#############################################################################################################################################################################
df_auto_mut_sub2pos_correlations = DataFrame(
    Mutation = String[],
    MatchMut = String[],
    Mut_plus_MatchMut_seq_ct = Int[],
    EPCI_Mut_ct = Int[],
    logpvFish = Float64[],
    Fold_Incr = Union{Float64, String}[],
    EPCI_MatchMut_ct = Int[],
    EPCI_pct = Float64[],
    MP_pct = Float64[],
    non_MP_pct = Float64[],
    pvFish = Float64[])
##############################################
println("none_0___sub2pos_1___pos2sub_2 = $(none_0___sub2pos_1___pos2sub_2)")
println("sub_0__posonly_1__mixed_2 = $(sub_0__posonly_1__mixed_2)")
# function AA_2plus__mut2pos_pos2mut__2026_02_12(min_mut_ct::Int, min_chi_sq::Int, min_log_pv_fish::Int, print_min::Int, exc_muts::Vector{String}, exc_muts_po::Vector{String}, sel_muts::Vector{String}, seq_factor_OFF_HALF_ON__0_1_2::Int)
min_mut_ct = 3
min_chi_sq = 0
min_log_pv_fish = 2.0
min_log_pv_fish_str = string(min_log_pv_fish)
min_fish_str = ""
if string(min_log_pv_fish_str[end]) == "0"
    min_fish_str = string(min_log_pv_fish_str[1])
else
    min_fish_str = min_log_pv_fish_str
end
print_min = 1
exc_muts = String[]
exc_muts_po = String[]
seq_factor_OFF_HALF_ON__0_1_2 = 0
nonmulti_0___multi_1 = 1
fold_type = ""
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp")
global folder_date = "$(date_now)"
        if none_0___sub2pos_1___pos2sub_2 == 1
            fold_type = "sub2pos"
        elseif none_0___sub2pos_1___pos2sub_2 == 2
            fold_type = "pos2sub"
        elseif none_0___sub2pos_1___pos2sub_2 == 0
            if sub_0__posonly_1__mixed_2 == 0
                fold_type = "sub_corr"
            elseif sub_0__posonly_1__mixed_2 == 1
                fold_type = "pos_only_corr"
            end
        end
#############################################################################################################################################################################
#############################################################################################################################################################################
NTD_disulfide_muts_combined_no_D138del = list_to_strings_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:W64C, S:S71C, S:C136-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:Y248C, S:W258C, S:G261C")
NTD_disulfide_muts_combined = list_to_strings_set("S:P9L, S:P9T, S:S12P, S:S13I, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:W64C, S:S71C, S:C136-, S:C136R, S:C136H, S:C136F, S:C136Y, S:D138-, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:Y248C, S:W258C, S:G261C")
############################################################################################
NTD_disulfide_positions_combined = list_to_strings_set("S:9, S:13, S:15, S:136")
NTD_disulfide_all_combined = union(NTD_disulfide_muts_combined, NTD_disulfide_positions_combined)
############################################################################################
disulfide_comprehensive_for_corr_muts_with_D138del = list_to_strings_set("NTD:Disulfide, NTD_disulfide, NTD:disulfide, S:P9S, S:P9L, S:P9T, S:P9Q, S:S12P, S:S13I, S:S13C, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64G, S:W64C, S:S71C, S:C136-, S:D138-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:R158I, S:Y248C, S:W258C, S:G261C")
disulfide_comprehensive_for_corr_muts_without_D138del = list_to_strings_set("NTD:Disulfide, NTD_disulfide, NTD:disulfide, S:P9S, S:P9L, S:P9T, S:P9Q, S:S12P, S:S13I, S:S13C, S:C15-, S:C15Y, S:C15S, S:C15F, S:C15G, S:C15L, S:C15R, S:R21I, S:W64G, S:W64C, S:S71C, S:C136-, S:C136R, S:C136H, S:C136F, S:C136Y, S:P139S, S:P139T, S:D142C, S:G142C, S:Y144C, S:Y145C, S:S151C, S:W152C, S:R158I, S:Y248C, S:W258C, S:G261C")
#############################################################################################################################################################################
#############################################################################################################################################################################
disulfide_comprehensive_for_corr_muts = Set{String}()
with_D138del_0____without_D138del_1 = 0
D138del_attach = ""
if with_D138del_0____without_D138del_1 == 0
    disulfide_comprehensive_for_corr_muts = disulfide_comprehensive_for_corr_muts_with_D138del
    D138del_attach = "WITH_D138del"
elseif with_D138del_0____without_D138del_1 == 1
    disulfide_comprehensive_for_corr_muts = disulfide_comprehensive_for_corr_muts_without_D138del
    D138del_attach = "WITHOUT_D138del"
end
##############################################################
all_muts_0___disulfide_only_1___disulfide_chk_2 = 2
filter_mut_set = Set{String}()
all_or_disulfide_tag = ""
if all_muts_0___disulfide_only_1___disulfide_chk_2 == 0
    filter_mut_set = Set(collect(keys(AA_muts_ct)))
    all_or_disulfide_tag = "ALLmuts"
elseif all_muts_0___disulfide_only_1___disulfide_chk_2 == 1
    filter_mut_set = disulfide_comprehensive_for_corr_muts
    all_or_disulfide_tag = "DISULFIDEonly"
elseif all_muts_0___disulfide_only_1___disulfide_chk_2 == 2
    filter_mut_set = Set(collect(keys(AA_muts_ct)))
    all_or_disulfide_tag = "DISULFIDE_CHECK"
end
##############################################################   
println("with_D138del_0____without_D138del_1 = $(with_D138del_0____without_D138del_1)")
println("all_muts_0___disulfide_only_1___disulfide_chk_2 = $(all_muts_0___disulfide_only_1___disulfide_chk_2)")
##############################################################
mkdir("$(fold_type)/$(folder_date)")
double_count = 0
qualified_pair_ct = 0
already_used_pairs = Set{Set{String}}()
for (moot, count) in AA_muts_ct
    if moot in filter_mut_set
        if count ≥ 2 && !(moot in excluded_muts_AUTO_sub2pos)
            moot1 = string(split(moot, ":")[1])
            moot2 = string(split(moot, ":")[2])
            moot12 = "$(moot1)_$(moot2)"
            prop_dict_nonlist = AA_2plus__mut2pos_pos2mut__2026_02_12(min_mut_ct, min_chi_sq, min_log_pv_fish, print_min, exc_muts, exc_muts_po, [moot], seq_factor_OFF_HALF_ON__0_1_2, nonmulti_0___multi_1, with_D138del_0____without_D138del_1, all_muts_0___disulfide_only_1___disulfide_chk_2)
            for (matchmut, tuple) in prop_dict_nonlist
                if !(matchmut in excluded_muts_AUTO_sub2pos)
                    if !(Set([moot, matchmut]) in already_used_pairs)
                        EPCI_prop          = tuple[1]
                        MP_prop            = tuple[2]
                        non_MP_prop        = tuple[3]
                        match_ct           = tuple[4]
                        chi_squared        = tuple[5]
                        pvFish             = tuple[6]
                        log_pv_fish        = tuple[7]
                        fold_incr          = tuple[8]
                        match_EPCI_ct      = tuple[9]
                        adjusted_MP_seq_ct = tuple[10]
                        EPCI_pct = 100*EPCI_prop
                        MP_pct = 100*MP_prop
                        non_MP_pct = 100*non_MP_prop
                        if log_pv_fish ≥ min_log_pv_fish && match_ct ≥ 2
                            if !( (aa_gene_comprehensive_dict[moot] == aa_gene_comprehensive_dict[matchmut]) && (abs(aa_pos_comprehensive_dict[moot] - aa_pos_comprehensive_dict[matchmut]) < 3) )
                                if !( (aa_gene_comprehensive_dict[moot] == aa_gene_comprehensive_dict[matchmut]) && (abs(aa_pos_comprehensive_dict[moot] - aa_pos_comprehensive_dict[matchmut]) < 15) && (qryAA_comprehensive_dict[moot] == "-" && qryAA_comprehensive_dict[matchmut] == "-") )
                                    qualified_pair_ct += 1
                                    if qualified_pair_ct%100 == 0
                                        print("\n"^1)
                                        println("qualified_pair_ct = $(qualified_pair_ct)")
                                        print("\n"^1)
                                    end
                                    push!(df_auto_mut_sub2pos_correlations, (moot, matchmut, match_ct, adjusted_MP_seq_ct, log_pv_fish, fold_incr, match_EPCI_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish))
                                    push!(already_used_pairs, Set([moot, matchmut]))
                                end
                            end
                        end
                    end
                end
            end
        end
    end
end
# prop_dict_nonlist[mutmatch] = (EPCI_prop, MP_prop, co_mut_count, chi_squared, pv_fish, log_pv_fish, fold_incr, EPCI_NTD_disulfide_numerator, MP_seqs_NTD_disulfide_denominator)
# AA_2plus__Pt1__2025_08_10_TEST_ONLY(filename, minFish, min_match_ct, min_mut_ct, min_chi_sq, min_log_pv_fish, print_min, exc_muts, exc_muts_po, sel_muts, sel_muts_po, seq_factor_OFF_HALF_ON__0_1_2)
##############################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
CSV.write("df_auto_mut_correlations__$(fold_type)__minFish$(min_fish_str)_minCt$(min_mut_ct)__EPCI_$(EPCI_qc_str)__$(all_or_disulfide_tag)_$(D138del_attach)__$(date_now).csv", df_auto_mut_sub2pos_correlations)
########################################################################################################
########################################################################################################

2026_03_07__702PM
none_0___sub2pos_1___pos2sub_2 = 0
sub_0__posonly_1__mixed_2 = 0
with_D138del_0____without_D138del_1 = 0
all_muts_0___disulfide_only_1___disulfide_chk_2 = 2
S:R190K|NTD:disulfide, S:S151C|NTD:disulfide, S:R403K|NTD:disulfide, S:V642G|NTD:disulfide, S:D215G|NTD:disulfide, S:C136F|NTD:disulfide, S:W152C|NTD:disulfide, S:A243-|NTD:disulfide, S:C136-|NTD:disulfide, S:R21I|NTD:disulfide, S:A688V|NTD:disulfide, S:T95I|NTD:disulfide, S:N211-|NTD:disulfide, S:C15-|NTD:disulfide, S:D138-|NTD:disulfide, S:R190S|NTD:disulfide, S:P9L|NTD:disulfide, Runtime v0 = 1032.9933488368988 seconds
Runtime v2 = 0 hr, 17 min, 12.99 sec
2026_03_07__719PM


"df_auto_mut_correlations__sub_corr__minFish2_minCt3__EPCI_8_10_95__DISULFIDE_CHECK_WITH_D138del__2026_03_07__719PM.csv"